In [1]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd

air_korea_final = pd.read_pickle("/Users/drewbaldwin/PM2_5 Research/air_korea_final_imputed_with_blh.pkl")
air_korea_final.head()


,Datetime,SO2,CO,O3,NO2,PM10,PM25,Station_ID,Year,lon,...,urban_landuse_area_m2_3km,green_space_area_3km,building_footprint_area_3km,railway_length_3km,dist_to_coast_km,dist_to_major_road_km,industrial_area_m2_3km,traffic_points_count_3km,boundary_layer_height,cloud_cover
0,2016-01-01 00:00:00,7.0,1000.0,2.0,76.0,77.0,53.0,111121,2016,126.9747,...,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0,30.0,6
1,2016-01-01 01:00:00,7.0,1100.0,2.0,77.0,70.0,48.0,111121,2016,126.9747,...,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0,25.0,3
2,2016-01-01 02:00:00,7.0,1200.0,2.0,78.0,75.0,53.0,111121,2016,126.9747,...,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0,15.0,2
3,2016-01-01 03:00:00,6.0,1400.0,2.0,78.0,77.0,53.0,111121,2016,126.9747,...,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0,15.0,1
4,2016-01-01 04:00:00,6.0,1500.0,2.0,77.0,83.0,52.0,111121,2016,126.9747,...,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0,15.0,1


In [1]:
# ================================================================
# LABEL FIX: the region-extreme-event target now asks "does a 2h-
# sustained exceedance (>=75ug/m3, matching Korea's real advisory
# definition) start ANYWHERE in the next 36 hours" instead of "does
# it start at EXACTLY hour 71" (the old point-target, which produced
# a narrow, high-variance label that only fired for windows whose
# horizon-end happened to land precisely on an episode's onset hour --
# real episodes lasting 10h would only positively label a handful of
# windows near their exact start, mislabeling everything else nearby
# as negative even though a real event was happening in-horizon).
#
# The underlying 2h-sustained/75ug/m3 criterion is unchanged (still
# matches the actual Korean statute) -- only WHERE in the 36h horizon
# we require it to occur has changed, via a forward rolling MAX over
# HORIZON hours of the original point-in-time flag.
#
# Also keeps the hybrid station-graph edges from the last validated
# test: station pairs <=20km apart use pure distance-decay (wind data
# is ~duplicated at that range), pairs beyond it keep the wind-
# alignment formula.
#
# IMPORTANT: fully restart the kernel before running this (Kernel ->
# Restart, not just interrupt) -- this script is structurally almost
# identical to the hybrid-edges run that completed successfully
# before, so the crash is most likely leftover memory from earlier
# partial runs in this kernel session, not a new bug here.
#
# Single split (seed 0), 2 model seeds -- first read on whether the
# label fix matters, before scaling to multi-split CV.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)
print(f"station graph: {E} directed edges, {close_edge_mask.sum()} ({100*close_edge_mask.mean():.1f}%) are 'close' (distance-only)")

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

representative_station_idx = np.zeros(n_regions, dtype=int)
for r in range(n_regions):
    idxs = np.where(station_region_idx == r)[0]
    representative_station_idx[r] = idxs[np.argmin(dist_to_region[idxs, r])]

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)
print(f"region graph: {len(r_src_idx)} directed edges (fully connected, {n_regions} regions)")

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months_arr = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)  # used for "far" edges only
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = wind_speed_arr[:, representative_station_idx].astype(np.float32)
region_wdir_sin = wdir_sin_station[:, representative_station_idx].astype(np.float32)
region_wdir_cos = wdir_cos_station[:, representative_station_idx].astype(np.float32)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(pm25_raw_arr)
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wdir_sin_station, wdir_cos_station, season_sin, season_cos
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

rev2 = region_episode_label[::-1]
roll_max_rev = pd.DataFrame(rev2).rolling(window=HORIZON, min_periods=HORIZON).max().to_numpy()
horizon_episode_label = np.nan_to_num(roll_max_rev[::-1], nan=0.0).astype(np.float32)
print(f"point-target positive rate: {np.nanmean(region_episode_label):.4f}  |  horizon-target positive rate: {np.nanmean(horizon_episode_label):.4f}")

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, hidden=32, gru_hidden=32, dropout=0.5):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, hidden=32, gru_hidden=32, dropout=0.5):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            h_region = torch.relu(self.region_fc1(h_region_pooled))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_horizon_label"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 5, 4
WEIGHT_DECAY_P1, WEIGHT_DECAY_P2 = 1e-4, 5e-4
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset, apply_recent_mask=True):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    if apply_recent_mask and GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

SPLIT_SEEDS = [4,5,6,7]
MODEL_SEEDS = [0, 1]
all_split_results = []

for split_seed in SPLIT_SEEDS:
    print(f"\n{'='*20} SPLIT {split_seed} {'='*20}")
    year_offset = years - years.min()
    block_id = year_offset * 12 + (months_arr - 1)
    n_blocks = int(block_id.max()) + 1
    rng = np.random.RandomState(split_seed)
    block_order = rng.permutation(n_blocks)
    n_train_blocks = int(round(0.6 * n_blocks))
    n_val_blocks = int(round(0.2 * n_blocks))
    block_to_split = np.empty(n_blocks, dtype=int)
    block_to_split[block_order[:n_train_blocks]] = 0
    block_to_split[block_order[n_train_blocks:n_train_blocks + n_val_blocks]] = 1
    block_to_split[block_order[n_train_blocks + n_val_blocks:]] = 2
    split_id_per_hour = block_to_split[block_id]
    TRAIN_MASK = split_id_per_hour == 0

    ref_speed = np.nanmean(wind_speed_arr[TRAIN_MASK])
    print(f"hybrid close-edge reference speed (train-period mean): {ref_speed:.3f}")
    component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
    edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
    del component
    gc.collect()

    train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
    weight_scale = train_nonzero.std()
    edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
    region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
    region_weight_scale = region_train_nonzero.std()
    region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
    del train_nonzero, region_train_nonzero, edge_weight_by_hour_raw
    gc.collect()

    LOOKBACK = GRAPH_RECENT_HOURS
    edge_roll_mean = pd.DataFrame(region_edge_weight_by_hour).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
    pm25_roll_mean = pd.DataFrame(region_pm25).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
    region_pm25_median_train = np.nanmedian(region_pm25[TRAIN_MASK], axis=0)
    src_elevated = pm25_roll_mean[:, r_src_idx] > region_pm25_median_train[r_src_idx][None, :]
    transport_signal_per_edge = np.where(src_elevated, edge_roll_mean, 0.0)
    transport_score = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        edge_mask = (r_dst_idx == r)
        transport_score[:, r] = np.nan_to_num(transport_signal_per_edge[:, edge_mask]).max(axis=1)
    edge_threshold_train = np.percentile(region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0], 75)
    transport_flag_per_region = transport_score > edge_threshold_train
    del edge_roll_mean, pm25_roll_mean, src_elevated, transport_signal_per_edge, transport_score
    gc.collect()

    t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
    t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
    time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
    s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
    static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

    p1_buckets = {0: ([], [], []), 1: ([], [], [])}
    p2_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
    for t in range(0, n_time - WINDOW - HORIZON + 1):
        target_t = t + WINDOW + HORIZON - 1
        input_end_t = t + WINDOW - 1
        s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
        if s_start != s_target:
            continue
        x_win = time_arr_std[t:t + WINDOW]
        if s_start in p1_buckets:
            Xl, yreg_l, sl = p1_buckets[s_start]
            Xl.append(x_win); yreg_l.append(time_arr_std[target_t, :, pm25_col_idx]); sl.append(t)
        Xl2, ycls_l2, mask_l2, sl2 = p2_buckets[s_start]
        Xl2.append(x_win); ycls_l2.append(horizon_episode_label[t + WINDOW])
        mask_l2.append(transport_flag_per_region[input_end_t]); sl2.append(t)

    X_train, yreg_train, starts_train = (np.stack(v) for v in p1_buckets[0])
    X_val, yreg_val, starts_val = (np.stack(v) for v in p1_buckets[1])
    X2_train, ycls2_train, mask2_train, starts2_train = (np.stack(v) for v in p2_buckets[0])
    X2_val, ycls2_val, mask2_val, starts2_val = (np.stack(v) for v in p2_buckets[1])
    X2_test, ycls2_test, mask2_test, starts2_test = (np.stack(v) for v in p2_buckets[2])
    del p1_buckets, p2_buckets, time_arr_std
    gc.collect()
    print(f"windows: phase1 train={len(X_train)} val={len(X_val)}  phase2 train={len(X2_train)} val={len(X2_val)} test={len(X2_test)}")
    print(f"masked-in fraction: train={mask2_train.mean():.4f} val={mask2_val.mean():.4f} test={mask2_test.mean():.4f}")
    print(f"positive rate (horizon label): train={ycls2_train.mean():.4f} val={ycls2_val.mean():.4f} test={ycls2_test.mean():.4f}")

    Xtr_t, ytr_reg_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(yreg_train, dtype=torch.float32)
    Xva_t, yva_reg_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(yreg_val, dtype=torch.float32)
    starts_train_t, starts_val_t = torch.tensor(starts_train, dtype=torch.long), torch.tensor(starts_val, dtype=torch.long)
    del X_train, X_val, yreg_train, yreg_val, starts_train, starts_val
    gc.collect()

    X2tr_t, y2tr_cls_t, m2tr_t = torch.tensor(X2_train, dtype=torch.float32), torch.tensor(ycls2_train, dtype=torch.float32), torch.tensor(mask2_train, dtype=torch.float32)
    X2va_t, y2va_cls_t, m2va_t = torch.tensor(X2_val, dtype=torch.float32), torch.tensor(ycls2_val, dtype=torch.float32), torch.tensor(mask2_val, dtype=torch.float32)
    X2te_t, y2te_cls_t, m2te_t = torch.tensor(X2_test, dtype=torch.float32), torch.tensor(ycls2_test, dtype=torch.float32), torch.tensor(mask2_test, dtype=torch.float32)
    starts2_train_t, starts2_val_t, starts2_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts2_train, starts2_val, starts2_test))
    del X2_train, X2_val, X2_test, ycls2_train, ycls2_val, ycls2_test, mask2_train, mask2_val, mask2_test, starts2_train, starts2_val, starts2_test
    gc.collect()

    masked_train_labels = y2tr_cls_t[m2tr_t.bool()]
    POS_WEIGHT = min(float((masked_train_labels.numel() - masked_train_labels.sum()) / masked_train_labels.sum().clamp(min=1)), 50.0)
    print(f"pos_weight: {POS_WEIGHT:.2f}")

    edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
    region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)
    del edge_weight_by_hour, region_edge_weight_by_hour
    gc.collect()

    region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT), reduction="none")

    def run_p1_epoch(model, use_graph, X, y, starts, optimizer, train):
        n = X.shape[0]
        idx = torch.randperm(n) if train else torch.arange(n)
        model.train(train)
        total_loss, total_n = 0.0, 0
        eff_batch = MICRO_BATCH * ACCUM_STEPS
        for start in range(0, n, eff_batch):
            if train: optimizer.zero_grad()
            batch_idx = idx[start:start + eff_batch]
            for ms in range(0, len(batch_idx), MICRO_BATCH):
                mb_idx = batch_idx[ms:ms + MICRO_BATCH]
                if len(mb_idx) == 0: continue
                xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
                yb = y[mb_idx].to(DEVICE)
                with torch.set_grad_enabled(train):
                    if use_graph:
                        ew_seq = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
                        pred = model(xb, edge_index, ew_seq)
                    else:
                        pred = model(xb)
                    loss = pinball_loss(pred, yb, QUANTILES)
                if train: (loss * len(mb_idx) / len(batch_idx)).backward()
                total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
            if train: optimizer.step()
        return total_loss / total_n

    def train_p1(name, model, use_graph):
        model = model.to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY_P1)
        best_val, best_epoch = float("inf"), -1
        ckpt_path = f"{CKPT_DIR}/{name}.pt"
        for epoch in range(1, MAX_EPOCHS_P1 + 1):
            t0 = time.time()
            tl = run_p1_epoch(model, use_graph, Xtr_t, ytr_reg_t, starts_train_t, opt, True)
            vl = run_p1_epoch(model, use_graph, Xva_t, yva_reg_t, starts_val_t, opt, False)
            print(f"[P1 {name}] ep{epoch} train={tl:.4f} val={vl:.4f} ({time.time()-t0:.0f}s)", flush=True)
            if vl < best_val:
                best_val, best_epoch = vl, epoch
                torch.save(model.state_dict(), ckpt_path)
        model.load_state_dict(torch.load(ckpt_path))
        print(f"[P1 {name}] BEST ep{best_epoch} val={best_val:.4f}")
        return model

    def run_p2_epoch(model, use_graph, X, y, m, starts, optimizer, train, return_probs=False):
        n = X.shape[0]
        idx = torch.randperm(n) if train else torch.arange(n)
        model.train(train)
        total_loss, total_maskn = 0.0, 0.0
        probs_list = [] if return_probs else None
        eff_batch = MICRO_BATCH * ACCUM_STEPS
        for start in range(0, n, eff_batch):
            if train: optimizer.zero_grad()
            batch_idx = idx[start:start + eff_batch]
            for ms in range(0, len(batch_idx), MICRO_BATCH):
                mb_idx = batch_idx[ms:ms + MICRO_BATCH]
                if len(mb_idx) == 0: continue
                xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
                yb = y[mb_idx].to(DEVICE)
                mb = m[mb_idx].to(DEVICE)
                with torch.set_grad_enabled(train):
                    if use_graph:
                        ew_s = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
                        ew_r = gather_seq(region_edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
                        logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                    else:
                        logits = model(xb, n_regions)
                    loss_pe = region_criterion(logits, yb)
                    masked_loss = (loss_pe * mb).sum() / mb.sum().clamp(min=1)
                if train: masked_loss.backward()
                if return_probs: probs_list.append(torch.sigmoid(logits).detach().cpu())
                total_loss += masked_loss.item() * mb.sum().item(); total_maskn += mb.sum().item()
            if train: optimizer.step()
        avg_loss = total_loss / max(total_maskn, 1)
        if return_probs: return avg_loss, torch.cat(probs_list, dim=0).numpy()
        return avg_loss

    def train_p2(name, model, use_graph):
        model = model.to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY_P2)
        best_val_auc_pr, best_epoch = -1.0, -1
        ckpt_path = f"{CKPT_DIR}/{name}.pt"
        y_val_masked = y2va_cls_t.numpy()[m2va_t.numpy().astype(bool)]
        for epoch in range(1, MAX_EPOCHS_P2 + 1):
            t0 = time.time()
            tl = run_p2_epoch(model, use_graph, X2tr_t, y2tr_cls_t, m2tr_t, starts2_train_t, opt, True)
            vl, vp = run_p2_epoch(model, use_graph, X2va_t, y2va_cls_t, m2va_t, starts2_val_t, opt, False, True)
            vp_masked = vp[m2va_t.numpy().astype(bool)]
            va_auc_pr = average_precision_score(y_val_masked, vp_masked)
            print(f"[P2 {name}] ep{epoch} train={tl:.4f} val_loss={vl:.4f} val_AUCPR={va_auc_pr:.4f} ({time.time()-t0:.0f}s)", flush=True)
            if va_auc_pr > best_val_auc_pr:
                best_val_auc_pr, best_epoch = va_auc_pr, epoch
                torch.save(model.state_dict(), ckpt_path)
        model.load_state_dict(torch.load(ckpt_path))
        print(f"[P2 {name}] BEST ep{best_epoch} val_AUCPR={best_val_auc_pr:.4f}")
        return model

    @torch.no_grad()
    def predict_test(model, use_graph):
        model.eval()
        n = X2te_t.shape[0]
        preds = []
        for start in range(0, n, MICRO_BATCH):
            xb = add_static_fn(X2te_t[start:start + MICRO_BATCH].to(DEVICE), static_tensor)
            if use_graph:
                ew_s = gather_seq(edge_weight_by_hour_t, starts2_test_t[start:start + MICRO_BATCH]).to(DEVICE)
                ew_r = gather_seq(region_edge_weight_by_hour_t, starts2_test_t[start:start + MICRO_BATCH]).to(DEVICE)
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            else:
                logits = model(xb, n_regions)
            preds.append(torch.sigmoid(logits).cpu())
        return torch.cat(preds, dim=0).numpy()

    wind_probs_by_seed, nograph_probs_by_seed = [], []
    for model_seed in MODEL_SEEDS:
        torch.manual_seed(model_seed); np.random.seed(model_seed)
        p1w = train_p1(f"s{split_seed}_m{model_seed}_p1_gcn", StationQuantileGCN(n_feats), True)
        torch.manual_seed(model_seed); np.random.seed(model_seed)
        p1n = train_p1(f"s{split_seed}_m{model_seed}_p1_mlp", StationQuantileMLP(n_feats), False)
        sc_copy = copy.deepcopy(p1w.station_conv)
        fc1_copy, fc2_copy = copy.deepcopy(p1n.fc1), copy.deepcopy(p1n.fc2)
        del p1w, p1n; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

        p2w = RegionFromQuantileGCN(sc_copy, region_membership_t).to(DEVICE)
        p2w = train_p2(f"s{split_seed}_m{model_seed}_p2_gcn", p2w, True)
        wind_probs_by_seed.append(predict_test(p2w, True))

        p2n = RegionFromQuantileMLP(fc1_copy, fc2_copy, region_membership_t).to(DEVICE)
        p2n = train_p2(f"s{split_seed}_m{model_seed}_p2_mlp", p2n, False)
        nograph_probs_by_seed.append(predict_test(p2n, False))
        del p2w, p2n; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

    wind_ensemble = np.mean(wind_probs_by_seed, axis=0)
    nograph_ensemble = np.mean(nograph_probs_by_seed, axis=0)
    mask_test = m2te_t.numpy().astype(bool)
    y_true = y2te_cls_t.numpy()[mask_test]

    for name, proba in [("wind_graph_horizonlabel", wind_ensemble[mask_test]), ("no_graph", nograph_ensemble[mask_test])]:
        auc_roc = roc_auc_score(y_true, proba)
        auc_pr = average_precision_score(y_true, proba)
        pred_pos = proba >= 0.5
        actual_pos = y_true >= 0.5
        tp = int(np.sum(pred_pos & actual_pos)); fp = int(np.sum(pred_pos & ~actual_pos))
        fn = int(np.sum(~pred_pos & actual_pos))
        precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
        recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        print(f"[SPLIT {split_seed} ENSEMBLE {name}] AUC-ROC={auc_roc:.4f} AUC-PR={auc_pr:.4f} P={precision:.4f} R={recall:.4f}")
        all_split_results.append({"split": split_seed, "model": name, "auc_roc": auc_roc, "auc_pr": auc_pr, "precision": precision, "recall": recall})

    del Xtr_t, Xva_t, ytr_reg_t, yva_reg_t, starts_train_t, starts_val_t
    del X2tr_t, y2tr_cls_t, m2tr_t, X2va_t, y2va_cls_t, m2va_t, X2te_t, y2te_cls_t, m2te_t
    del starts2_train_t, starts2_val_t, starts2_test_t, edge_weight_by_hour_t, region_edge_weight_by_hour_t, static_tensor
    gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

del wind_component
gc.collect()

results_df = pd.DataFrame(all_split_results)
print("\n" + "="*20 + " HORIZON-LABEL RESULT (single split) " + "="*20)
print(results_df.to_string(index=False))
results_df.to_csv(f"{BASE}/results_horizon_label_hybrid_edges_splits4to7.csv", index=False)



station graph: 21414 directed edges, 1994 (9.3%) are 'close' (distance-only)
region graph: 272 directed edges (fully connected, 17 regions)
point-target positive rate: 0.0083  |  horizon-target positive rate: 0.0349
device=mps

==================== SPLIT 4 ====================
hybrid close-edge reference speed (train-period mean): 10.027
windows: phase1 train=30160 val=9444  phase2 train=30160 val=9444 test=10093
masked-in fraction: train=0.2500 val=0.2312 test=0.2194
positive rate (horizon label): train=0.0311 val=0.0365 test=0.0457
pos_weight: 23.00
[P1 s4_m0_p1_gcn] ep1 train=0.1965 val=0.1912 (227s)
[P1 s4_m0_p1_gcn] ep2 train=0.1871 val=0.1863 (211s)
[P1 s4_m0_p1_gcn] ep3 train=0.1848 val=0.1873 (218s)
[P1 s4_m0_p1_gcn] ep4 train=0.1838 val=0.1842 (209s)
[P1 s4_m0_p1_gcn] ep5 train=0.1827 val=0.1851 (210s)
[P1 s4_m0_p1_gcn] BEST ep4 val=0.1842
[P1 s4_m0_p1_mlp] ep1 train=0.1971 val=0.1909 (41s)
[P1 s4_m0_p1_mlp] ep2 train=0.1908 val=0.1940 (39s)
[P1 s4_m0_p1_mlp] ep3 train=0.1890 

: 

In [2]:
import pandas as pd

BASE = "/Users/drewbaldwin/PM2_5 Research"

recovered_results = [
    {"split": 4, "model": "wind_graph_horizonlabel", "auc_roc": 0.9082, "auc_pr": 0.6083, "precision": 0.3213, "recall": 0.7675},
    {"split": 4, "model": "no_graph",                "auc_roc": 0.8949, "auc_pr": 0.5926, "precision": 0.2487, "recall": 0.8057},
    {"split": 5, "model": "wind_graph_horizonlabel", "auc_roc": 0.8482, "auc_pr": 0.3590, "precision": 0.2131, "recall": 0.6001},
    {"split": 5, "model": "no_graph",                "auc_roc": 0.7936, "auc_pr": 0.3674, "precision": 0.1585, "recall": 0.5800},
]

results_df = pd.DataFrame(recovered_results)
results_df.to_csv(f"{BASE}/results_horizon_label_hybrid_edges_splits4to5.csv", index=False)
print(results_df.to_string(index=False))
print(f"\nsaved to {BASE}/results_horizon_label_hybrid_edges_splits4to5.csv")


 split                   model  auc_roc  auc_pr  precision  recall
     4 wind_graph_horizonlabel   0.9082  0.6083     0.3213  0.7675
     4                no_graph   0.8949  0.5926     0.2487  0.8057
     5 wind_graph_horizonlabel   0.8482  0.3590     0.2131  0.6001
     5                no_graph   0.7936  0.3674     0.1585  0.5800

saved to /Users/drewbaldwin/PM2_5 Research/results_horizon_label_hybrid_edges_splits4to5.csv


In [3]:
#total combined 
import pandas as pd
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df_0to3 = pd.read_csv(f"{BASE}/results_horizon_label_hybrid_edges.csv")
df_4to7 = pd.read_csv(f"{BASE}/results_horizon_label_hybrid_edges_splits4to5.csv")

combined = pd.concat([df_0to3, df_4to7], ignore_index=True)
combined.to_csv(f"{BASE}/results_horizon_label_hybrid_edges_combined8.csv", index=False)

print(f"combined: {combined['split'].nunique()} splits, {len(combined)} rows")
print(combined.to_string(index=False))

print("\n=== aggregated across all 8 splits ===")
print(combined.groupby("model")[["auc_roc", "auc_pr", "precision", "recall"]].agg(["mean", "std"]))

print("\n=== paired comparison across all 8 splits ===")
n_splits = combined["split"].nunique()
for metric in ["auc_roc", "auc_pr", "recall", "precision"]:
    pivot = combined.pivot(index="split", columns="model", values=metric)
    t, p = stats.ttest_rel(pivot["wind_graph_horizonlabel"], pivot["no_graph"])
    wins = int((pivot["wind_graph_horizonlabel"] > pivot["no_graph"]).sum())
    print(f"[{metric}] wind={pivot['wind_graph_horizonlabel'].mean():.4f} no_graph={pivot['no_graph'].mean():.4f} t={t:.3f} p={p:.4f} wind_wins={wins}/{n_splits}")


combined: 6 splits, 12 rows
 split                   model  auc_roc   auc_pr  precision   recall
     0 wind_graph_horizonlabel 0.917733 0.625851   0.334250 0.840873
     0                no_graph 0.893530 0.579604   0.258346 0.836083
     1 wind_graph_horizonlabel 0.867678 0.429893   0.191700 0.784014
     1                no_graph 0.805293 0.405278   0.174561 0.720748
     2 wind_graph_horizonlabel 0.921478 0.571721   0.188736 0.855355
     2                no_graph 0.868115 0.497136   0.147856 0.786787
     3 wind_graph_horizonlabel 0.946633 0.664165   0.333434 0.870866
     3                no_graph 0.938300 0.640798   0.276401 0.877953
     4 wind_graph_horizonlabel 0.908200 0.608300   0.321300 0.767500
     4                no_graph 0.894900 0.592600   0.248700 0.805700
     5 wind_graph_horizonlabel 0.848200 0.359000   0.213100 0.600100
     5                no_graph 0.793600 0.367400   0.158500 0.580000

=== aggregated across all 8 splits ===
                          auc_roc  

In [4]:
# ================================================================
# TRUE HELD-OUT-YEAR TEST: same modeling approach (hybrid station
# edges, horizon-aggregated label, transport-driven masking,
# two-phase quantile-pretrain -> region classification) as every
# multi-split result so far, but with a strict CHRONOLOGICAL split
# instead of randomized month-block reshuffling:
#   TRAIN = 2016-2019, VAL = 2020, TEST = 2021 (never touched by
#   anything -- not training, not validation, not model selection).
# This is the generalization check flagged earlier: does the
# wind-graph advantage hold on data from a genuinely unseen year,
# not just a reshuffled partition of the same six years.
#
# Includes the two real memory fixes found during the crash
# debugging: ref_speed cast to float32 (was silently upcasting a
# ~4.5GB array to ~9GB via float64 promotion), and windows built
# once per split (not duplicated across phase1/phase2).
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)
print(f"station graph: {E} directed edges, {close_edge_mask.sum()} ({100*close_edge_mask.mean():.1f}%) are 'close' (distance-only)")

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

representative_station_idx = np.zeros(n_regions, dtype=int)
for r in range(n_regions):
    idxs = np.where(station_region_idx == r)[0]
    representative_station_idx[r] = idxs[np.argmin(dist_to_region[idxs, r])]

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)
print(f"region graph: {len(r_src_idx)} directed edges (fully connected, {n_regions} regions)")

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)
print(f"year range: {years.min()}-{years.max()}  |  hours per year: {dict(zip(*np.unique(years, return_counts=True)))}")

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = wind_speed_arr[:, representative_station_idx].astype(np.float32)
region_wdir_sin = wdir_sin_station[:, representative_station_idx].astype(np.float32)
region_wdir_cos = wdir_cos_station[:, representative_station_idx].astype(np.float32)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(pm25_raw_arr)
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wdir_sin_station, wdir_cos_station, season_sin, season_cos, wind_dir_arr, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

rev2 = region_episode_label[::-1]
roll_max_rev = pd.DataFrame(rev2).rolling(window=HORIZON, min_periods=HORIZON).max().to_numpy()
horizon_episode_label = np.nan_to_num(roll_max_rev[::-1], nan=0.0).astype(np.float32)
print(f"point-target positive rate: {np.nanmean(region_episode_label):.4f}  |  horizon-target positive rate: {np.nanmean(horizon_episode_label):.4f}")

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, hidden=32, gru_hidden=32, dropout=0.5):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, hidden=32, gru_hidden=32, dropout=0.5):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            h_region = torch.relu(self.region_fc1(h_region_pooled))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_heldout_year"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 5, 4
WEIGHT_DECAY_P1, WEIGHT_DECAY_P2 = 1e-4, 5e-4
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset, apply_recent_mask=True):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    if apply_recent_mask and GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

# ---- CHRONOLOGICAL split: train<=2019, val=2020, test=2021 ----
split_id_per_hour = np.where(years <= 2019, 0, np.where(years == 2020, 1, 2))
TRAIN_MASK = split_id_per_hour == 0
print(f"train hours (<=2019): {TRAIN_MASK.sum()}  val hours (2020): {(split_id_per_hour==1).sum()}  test hours (2021): {(split_id_per_hour==2).sum()}")

ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
print(f"hybrid close-edge reference speed (train-period mean): {ref_speed:.3f}")
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
region_weight_scale = region_train_nonzero.std()
region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
del train_nonzero, region_train_nonzero, edge_weight_by_hour_raw
gc.collect()

LOOKBACK = GRAPH_RECENT_HOURS
edge_roll_mean = pd.DataFrame(region_edge_weight_by_hour).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
pm25_roll_mean = pd.DataFrame(region_pm25).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
region_pm25_median_train = np.nanmedian(region_pm25[TRAIN_MASK], axis=0)
src_elevated = pm25_roll_mean[:, r_src_idx] > region_pm25_median_train[r_src_idx][None, :]
transport_signal_per_edge = np.where(src_elevated, edge_roll_mean, 0.0)
transport_score = np.zeros((n_time, n_regions), dtype=np.float32)
for r in range(n_regions):
    edge_mask = (r_dst_idx == r)
    transport_score[:, r] = np.nan_to_num(transport_signal_per_edge[:, edge_mask]).max(axis=1)
edge_threshold_train = np.percentile(region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0], 75)
transport_flag_per_region = transport_score > edge_threshold_train
del edge_roll_mean, pm25_roll_mean, src_elevated, transport_signal_per_edge, transport_score
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], [], []), 1: ([], [], [], [], []), 2: ([], [], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    input_end_t = t + WINDOW - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    Xl, yregl, yclsl, maskl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    yclsl.append(horizon_episode_label[t + WINDOW])
    maskl.append(transport_flag_per_region[input_end_t])
    sl.append(t)

X0, yreg0, ycls0, mask0, starts0 = (np.stack(v) for v in p_buckets[0])  # train
X1, yreg1, ycls1, mask1, starts1 = (np.stack(v) for v in p_buckets[1])  # val
X2, yreg2, ycls2, mask2, starts2 = (np.stack(v) for v in p_buckets[2])  # TRUE HOLDOUT test (2021)
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test(2021 holdout)={len(X2)}")
print(f"masked-in fraction: train={mask0.mean():.4f} val={mask1.mean():.4f} test={mask2.mean():.4f}")
print(f"positive rate (horizon label): train={ycls0.mean():.4f} val={ycls1.mean():.4f} test={ycls2.mean():.4f}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ycls0_t, ycls1_t, ycls2_t = torch.tensor(ycls0, dtype=torch.float32), torch.tensor(ycls1, dtype=torch.float32), torch.tensor(ycls2, dtype=torch.float32)
mask0_t, mask1_t, mask2_t = torch.tensor(mask0, dtype=torch.float32), torch.tensor(mask1, dtype=torch.float32), torch.tensor(mask2, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1, ycls0, ycls1, ycls2, mask0, mask1, mask2, starts0, starts1, starts2
gc.collect()

masked_train_labels = ycls0_t[mask0_t.bool()]
POS_WEIGHT = min(float((masked_train_labels.numel() - masked_train_labels.sum()) / masked_train_labels.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")

edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)
del edge_weight_by_hour, region_edge_weight_by_hour
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT), reduction="none")

def run_p1_epoch(model, use_graph, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                if use_graph:
                    ew_seq = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
                    pred = model(xb, edge_index, ew_seq)
                else:
                    pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def train_p1(name, model, use_graph):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY_P1)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        t0 = time.time()
        tl = run_p1_epoch(model, use_graph, X0_t, yreg0_t, starts0_t, opt, True)
        vl = run_p1_epoch(model, use_graph, X1_t, yreg1_t, starts1_t, opt, False)
        print(f"[P1 {name}] ep{epoch} train={tl:.4f} val={vl:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if vl < best_val:
            best_val, best_epoch = vl, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    print(f"[P1 {name}] BEST ep{best_epoch} val={best_val:.4f}")
    return model

def run_p2_epoch(model, use_graph, X, y, m, starts, optimizer, train, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_maskn = 0.0, 0.0
    probs_list = [] if return_probs else None
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            mb = m[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                if use_graph:
                    ew_s = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
                    ew_r = gather_seq(region_edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
                    logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                else:
                    logits = model(xb, n_regions)
                loss_pe = region_criterion(logits, yb)
                masked_loss = (loss_pe * mb).sum() / mb.sum().clamp(min=1)
            if train: masked_loss.backward()
            if return_probs: probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += masked_loss.item() * mb.sum().item(); total_maskn += mb.sum().item()
        if train: optimizer.step()
    avg_loss = total_loss / max(total_maskn, 1)
    if return_probs: return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

def train_p2(name, model, use_graph):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY_P2)
    best_val_auc_pr, best_epoch = -1.0, -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    y_val_masked = ycls1_t.numpy()[mask1_t.numpy().astype(bool)]
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        t0 = time.time()
        tl = run_p2_epoch(model, use_graph, X0_t, ycls0_t, mask0_t, starts0_t, opt, True)
        vl, vp = run_p2_epoch(model, use_graph, X1_t, ycls1_t, mask1_t, starts1_t, opt, False, True)
        vp_masked = vp[mask1_t.numpy().astype(bool)]
        va_auc_pr = average_precision_score(y_val_masked, vp_masked)
        print(f"[P2 {name}] ep{epoch} train={tl:.4f} val_loss={vl:.4f} val_AUCPR={va_auc_pr:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if va_auc_pr > best_val_auc_pr:
            best_val_auc_pr, best_epoch = va_auc_pr, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    print(f"[P2 {name}] BEST ep{best_epoch} val_AUCPR={best_val_auc_pr:.4f}")
    return model

@torch.no_grad()
def predict_test(model, use_graph):
    model.eval()
    n = X2_t.shape[0]
    preds = []
    for start in range(0, n, MICRO_BATCH):
        xb = add_static_fn(X2_t[start:start + MICRO_BATCH].to(DEVICE), static_tensor)
        if use_graph:
            ew_s = gather_seq(edge_weight_by_hour_t, starts2_t[start:start + MICRO_BATCH]).to(DEVICE)
            ew_r = gather_seq(region_edge_weight_by_hour_t, starts2_t[start:start + MICRO_BATCH]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
        else:
            logits = model(xb, n_regions)
        preds.append(torch.sigmoid(logits).cpu())
    return torch.cat(preds, dim=0).numpy()

wind_probs_by_seed, nograph_probs_by_seed = [], []
MODEL_SEEDS = [0, 1]
for model_seed in MODEL_SEEDS:
    torch.manual_seed(model_seed); np.random.seed(model_seed)
    p1w = train_p1(f"heldout_m{model_seed}_p1_gcn", StationQuantileGCN(n_feats), True)
    torch.manual_seed(model_seed); np.random.seed(model_seed)
    p1n = train_p1(f"heldout_m{model_seed}_p1_mlp", StationQuantileMLP(n_feats), False)
    sc_copy = copy.deepcopy(p1w.station_conv)
    fc1_copy, fc2_copy = copy.deepcopy(p1n.fc1), copy.deepcopy(p1n.fc2)
    del p1w, p1n; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    p2w = RegionFromQuantileGCN(sc_copy, region_membership_t).to(DEVICE)
    p2w = train_p2(f"heldout_m{model_seed}_p2_gcn", p2w, True)
    wind_probs_by_seed.append(predict_test(p2w, True))

    p2n = RegionFromQuantileMLP(fc1_copy, fc2_copy, region_membership_t).to(DEVICE)
    p2n = train_p2(f"heldout_m{model_seed}_p2_mlp", p2n, False)
    nograph_probs_by_seed.append(predict_test(p2n, False))
    del p2w, p2n; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

wind_ensemble = np.mean(wind_probs_by_seed, axis=0)
nograph_ensemble = np.mean(nograph_probs_by_seed, axis=0)
mask_test = mask2_t.numpy().astype(bool)
y_true = ycls2_t.numpy()[mask_test]

heldout_results = []
for name, proba in [("wind_graph_horizonlabel", wind_ensemble[mask_test]), ("no_graph", nograph_ensemble[mask_test])]:
    auc_roc = roc_auc_score(y_true, proba)
    auc_pr = average_precision_score(y_true, proba)
    pred_pos = proba >= 0.5
    actual_pos = y_true >= 0.5
    tp = int(np.sum(pred_pos & actual_pos)); fp = int(np.sum(pred_pos & ~actual_pos))
    fn = int(np.sum(~pred_pos & actual_pos))
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    print(f"[HELDOUT-2021 {name}] AUC-ROC={auc_roc:.4f} AUC-PR={auc_pr:.4f} P={precision:.4f} R={recall:.4f}")
    heldout_results.append({"model": name, "auc_roc": auc_roc, "auc_pr": auc_pr, "precision": precision, "recall": recall})

pd.DataFrame(heldout_results).to_csv(f"{BASE}/results_heldout_2021.csv", index=False)
print(f"\nsaved to {BASE}/results_heldout_2021.csv")


station graph: 21414 directed edges, 1994 (9.3%) are 'close' (distance-only)
region graph: 272 directed edges (fully connected, 17 regions)
year range: 2016-2021  |  hours per year: {np.int32(2016): np.int64(8784), np.int32(2017): np.int64(8760), np.int32(2018): np.int64(8760), np.int32(2019): np.int64(8760), np.int32(2020): np.int64(8784), np.int32(2021): np.int64(8760)}
point-target positive rate: 0.0083  |  horizon-target positive rate: 0.0349
device=mps
train hours (<=2019): 35064  val hours (2020): 8784  test hours (2021): 8760
hybrid close-edge reference speed (train-period mean): 9.921
windows: train=34993 val=8713 test(2021 holdout)=8689
masked-in fraction: train=0.2480 val=0.1797 test=0.1478
positive rate (horizon label): train=0.0417 val=0.0134 test=0.0290
pos_weight: 15.62
[P1 heldout_m0_p1_gcn] ep1 train=0.1930 val=0.1617 (253s)
[P1 heldout_m0_p1_gcn] ep2 train=0.1852 val=0.1546 (227s)
[P1 heldout_m0_p1_gcn] ep3 train=0.1832 val=0.1618 (231s)
[P1 heldout_m0_p1_gcn] ep4 trai

In [3]:
import pandas as pd
from scipy import stats

results_df = pd.read_csv(f"{BASE}/results_horizon_label_hybrid_edges.csv")
print(results_df.groupby("model")[["auc_roc", "auc_pr", "precision", "recall"]].agg(["mean", "std"]))

n_splits = results_df["split"].nunique()
for metric in ["auc_roc", "auc_pr", "recall", "precision"]:
    pivot = results_df.pivot(index="split", columns="model", values=metric)
    t, p = stats.ttest_rel(pivot["wind_graph_horizonlabel"], pivot["no_graph"])
    wins = int((pivot["wind_graph_horizonlabel"] > pivot["no_graph"]).sum())
    print(f"[{metric}] wind={pivot['wind_graph_horizonlabel'].mean():.4f} no_graph={pivot['no_graph'].mean():.4f} t={t:.3f} p={p:.4f} wind_wins={wins}/{n_splits}")


                          auc_roc              auc_pr           precision  \
                             mean       std      mean       std      mean   
model                                                                       
no_graph                 0.876309  0.055527  0.530704  0.102259  0.214291   
wind_graph_horizonlabel  0.913380  0.033061  0.572907  0.102609  0.262030   

                                     recall            
                              std      mean       std  
model                                                  
no_graph                 0.062691  0.805393  0.067621  
wind_graph_horizonlabel  0.082931  0.837777  0.037877  
[auc_roc] wind=0.9134 no_graph=0.8763 t=2.948 p=0.0601 wind_wins=4/4
[auc_pr] wind=0.5729 no_graph=0.5307 t=3.516 p=0.0390 wind_wins=4/4
[recall] wind=0.8378 no_graph=0.8054 t=1.657 p=0.1961 wind_wins=3/4
[precision] wind=0.2620 no_graph=0.2143 t=3.831 p=0.0313 wind_wins=4/4


In [1]:
#doing graph ablation (zero out regions) to check if graph is actually helping. 

# ================================================================
# Leave-one-region-out EDGE ablation on the 2021 true holdout test
# set, using the ALREADY-TRAINED checkpoints from the held-out-year
# run (ckpt_heldout_year/heldout_m{0,1}_p2_gcn.pt) -- inference only,
# no retraining needed since those checkpoints are saved to disk.
#
# For each of the 17 regions in turn: zero out every edge whose
# SOURCE belongs to that region, at BOTH the station-graph level
# (any station in that region) and the region-graph level (that
# region's outgoing edges), re-run inference on the 2021 holdout
# test set, and record the change in AUC-PR for every downwind
# region vs the unablated baseline -> 17x17 (source x downwind)
# delta matrix.
#
# Silencing is done on the small per-batch gathered edge-weight
# tensor at inference time (not by duplicating the full n_time-sized
# array 17 times), to avoid the memory blowups that crashed this
# notebook earlier.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, gc
from sklearn.metrics import average_precision_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
CKPT_DIR = f"{BASE}/ckpt_heldout_year"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

representative_station_idx = np.zeros(n_regions, dtype=int)
for r in range(n_regions):
    idxs = np.where(station_region_idx == r)[0]
    representative_station_idx[r] = idxs[np.argmin(dist_to_region[idxs, r])]

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = wind_speed_arr[:, representative_station_idx].astype(np.float32)
region_wdir_sin = wdir_sin_station[:, representative_station_idx].astype(np.float32)
region_wdir_cos = wdir_cos_station[:, representative_station_idx].astype(np.float32)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(pm25_raw_arr)
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wdir_sin_station, wdir_cos_station, season_sin, season_cos, wind_dir_arr, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

rev2 = region_episode_label[::-1]
roll_max_rev = pd.DataFrame(rev2).rolling(window=HORIZON, min_periods=HORIZON).max().to_numpy()
horizon_episode_label = np.nan_to_num(roll_max_rev[::-1], nan=0.0).astype(np.float32)

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, hidden=32, gru_hidden=32, dropout=0.5):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH = 16
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset, apply_recent_mask=True):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    if apply_recent_mask and GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

# ---- chronological split, matching the held-out-year run ----
split_id_per_hour = np.where(years <= 2019, 0, np.where(years == 2020, 1, 2))
TRAIN_MASK = split_id_per_hour == 0

ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
region_weight_scale = region_train_nonzero.std()
region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
del train_nonzero, region_train_nonzero, edge_weight_by_hour_raw
gc.collect()

LOOKBACK = GRAPH_RECENT_HOURS
edge_roll_mean = pd.DataFrame(region_edge_weight_by_hour).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
pm25_roll_mean = pd.DataFrame(region_pm25).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
region_pm25_median_train = np.nanmedian(region_pm25[TRAIN_MASK], axis=0)
src_elevated = pm25_roll_mean[:, r_src_idx] > region_pm25_median_train[r_src_idx][None, :]
transport_signal_per_edge = np.where(src_elevated, edge_roll_mean, 0.0)
transport_score = np.zeros((n_time, n_regions), dtype=np.float32)
for r in range(n_regions):
    edge_mask = (r_dst_idx == r)
    transport_score[:, r] = np.nan_to_num(transport_signal_per_edge[:, edge_mask]).max(axis=1)
edge_threshold_train = np.percentile(region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0], 75)
transport_flag_per_region = transport_score > edge_threshold_train
del edge_roll_mean, pm25_roll_mean, src_elevated, transport_signal_per_edge, transport_score
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

# only need the TEST (2021) bucket -- no train/val windows needed for inference
test_bucket = ([], [], [], [])
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    input_end_t = t + WINDOW - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start != 2:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    Xl, yclsl, maskl, sl = test_bucket
    Xl.append(x_win); yclsl.append(horizon_episode_label[t + WINDOW])
    maskl.append(transport_flag_per_region[input_end_t]); sl.append(t)

X2, ycls2, mask2, starts2 = (np.stack(v) for v in test_bucket)
del test_bucket, time_arr_std
gc.collect()
print(f"2021 holdout test windows: {len(X2)}")

X2_t = torch.tensor(X2, dtype=torch.float32); del X2; gc.collect()
starts2_t = torch.tensor(starts2, dtype=torch.long)
mask_test = mask2.astype(bool)   # (n_windows, n_regions)
y_true_all = ycls2                # (n_windows, n_regions)

edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour); del edge_weight_by_hour; gc.collect()
region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour); del region_edge_weight_by_hour; gc.collect()

# precompute, per source region, which station-edges and region-edges to silence
station_src_region = station_region_idx[src_idx]          # (E_station,)
region_src_region = r_src_idx                              # (272,)

def load_gcn(model_seed):
    sc_placeholder = WindConvLayer(n_feats, 32)
    model = RegionFromQuantileGCN(sc_placeholder, region_membership_t).to(DEVICE)
    model.load_state_dict(torch.load(f"{CKPT_DIR}/heldout_m{model_seed}_p2_gcn.pt", map_location=DEVICE))
    model.eval()
    return model

models = [load_gcn(0), load_gcn(1)]

@torch.no_grad()
def run_inference(silence_region):
    station_mask = None if silence_region is None else torch.tensor(station_src_region == silence_region)
    region_mask = None if silence_region is None else torch.tensor(region_src_region == silence_region)
    all_seed_preds = []
    for model in models:
        n = X2_t.shape[0]
        preds = []
        for start in range(0, n, MICRO_BATCH):
            xb = add_static_fn(X2_t[start:start + MICRO_BATCH].to(DEVICE), static_tensor)
            ew_s = gather_seq(edge_weight_by_hour_t, starts2_t[start:start + MICRO_BATCH]).to(DEVICE)
            ew_r = gather_seq(region_edge_weight_by_hour_t, starts2_t[start:start + MICRO_BATCH]).to(DEVICE)
            if silence_region is not None:
                ew_s = ew_s.clone(); ew_s[:, :, station_mask] = 0.0
                ew_r = ew_r.clone(); ew_r[:, :, region_mask] = 0.0
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            preds.append(torch.sigmoid(logits).cpu())
        all_seed_preds.append(torch.cat(preds, dim=0).numpy())
    return np.mean(all_seed_preds, axis=0)  # (n_windows, n_regions)

def per_region_aucpr(proba_all):
    out = {}
    for r in range(n_regions):
        m = mask_test[:, r]
        if m.sum() < 5 or y_true_all[m, r].sum() < 2:
            out[region_names[r]] = np.nan
            continue
        out[region_names[r]] = average_precision_score(y_true_all[m, r], proba_all[m, r])
    return out

print("running baseline (no silencing)...")
baseline_proba = run_inference(None)
baseline_aucpr = per_region_aucpr(baseline_proba)
print("baseline per-region AUC-PR:", {k: round(v, 4) for k, v in baseline_aucpr.items() if not np.isnan(v)})

delta_rows = []
for s in range(n_regions):
    print(f"silencing source region: {region_names[s]}")
    ablated_proba = run_inference(s)
    ablated_aucpr = per_region_aucpr(ablated_proba)
    for t in range(n_regions):
        if s == t:
            continue
        base = baseline_aucpr[region_names[t]]
        abl = ablated_aucpr[region_names[t]]
        delta = np.nan if (np.isnan(base) or np.isnan(abl)) else (abl - base)
        delta_rows.append({"source_region": region_names[s], "downwind_region": region_names[t],
                            "baseline_aucpr": base, "ablated_aucpr": abl, "delta_aucpr": delta})

delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv(f"{BASE}/leave_one_region_out_2021holdout.csv", index=False)
print(f"\nsaved {len(delta_df)} rows to {BASE}/leave_one_region_out_2021holdout.csv")

pivot = delta_df.pivot(index="source_region", columns="downwind_region", values="delta_aucpr")
print("\n=== delta AUC-PR matrix (source x downwind); negative = silencing that source HURT that downwind region ===")
print(pivot.round(4).to_string())

print("\n=== top 10 strongest source->downwind dependencies (most negative delta) ===")
print(delta_df.dropna(subset=["delta_aucpr"]).sort_values("delta_aucpr").head(10).to_string(index=False))


device=mps
2021 holdout test windows: 8689
running baseline (no silencing)...
baseline per-region AUC-PR: {'Seoul': 0.4866, 'Busan': 0.224, 'Daegu': 0.1715, 'Incheon': 0.488, 'Gwangju': 0.6706, 'Daejeon': 0.0673, 'Ulsan': 0.0701, 'Sejong': 0.4116, 'Gyeonggi': 0.3482, 'Gangwon': 0.0649, 'Chungbuk': 0.2858, 'Chungnam': 0.5015, 'Jeonbuk': 0.4774, 'Jeonnam': 0.2099, 'Gyeongbuk': 0.2393, 'Gyeongnam': 0.1432}
silencing source region: Seoul
silencing source region: Busan
silencing source region: Daegu
silencing source region: Incheon
silencing source region: Gwangju
silencing source region: Daejeon
silencing source region: Ulsan
silencing source region: Sejong
silencing source region: Gyeonggi
silencing source region: Gangwon
silencing source region: Chungbuk
silencing source region: Chungnam
silencing source region: Jeonbuk
silencing source region: Jeonnam
silencing source region: Gyeongbuk
silencing source region: Gyeongnam
silencing source region: Jeju

saved 272 rows to /Users/drewbaldwin

In [1]:
# ================================================================
# REVISED, FASTER: drops wind_graph_gat entirely given its ~5x
# slower-than-expected per-config cost (2739s vs ~550s for the
# others) -- not worth the uncertain, possibly ~6-hour addition
# given the time constraint. wind_graph reuses its already-found
# best config (lr=3e-3, dropout=0.5, wd=5e-4) with one confirmatory
# run instead of repeating its full 8-config search. Full search
# kept for augmented_matched and no_graph.
#
# TRAIN = 2016-2017, VAL = 2018, TEST = 2019 (held out). 2020/2021
# untouched, as always.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)
print(f"station graph: {E} directed edges, {close_edge_mask.sum()} ({100*close_edge_mask.mean():.1f}%) are 'close' (distance-only)")

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

representative_station_idx = np.zeros(n_regions, dtype=int)
for r in range(n_regions):
    idxs = np.where(station_region_idx == r)[0]
    representative_station_idx[r] = idxs[np.argmin(dist_to_region[idxs, r])]

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)
print(f"region graph: {len(r_src_idx)} directed edges (fully connected, {n_regions} regions)")

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = wind_speed_arr[:, representative_station_idx].astype(np.float32)
region_wdir_sin = wdir_sin_station[:, representative_station_idx].astype(np.float32)
region_wdir_cos = wdir_cos_station[:, representative_station_idx].astype(np.float32)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(pm25_raw_arr)
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wdir_sin_station, wdir_cos_station, season_sin, season_cos, wind_dir_arr, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

rev2 = region_episode_label[::-1]
roll_max_rev = pd.DataFrame(rev2).rolling(window=HORIZON, min_periods=HORIZON).max().to_numpy()
horizon_episode_label = np.nan_to_num(roll_max_rev[::-1], nan=0.0).astype(np.float32)

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

def raw_neighbor_agg_and_connectivity(x, edge_index, edge_weight, num_nodes):
    src, dst = edge_index[0], edge_index[1]
    messages = x[src] * edge_weight.unsqueeze(-1)
    agg_sum = x.new_zeros(num_nodes, x.size(-1))
    agg_sum.index_add_(0, dst, messages)
    weight_sum = x.new_zeros(num_nodes)
    weight_sum.index_add_(0, dst, edge_weight)
    agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
    connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
    return agg_mean, connectivity

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP_Augmented(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim * 2 + 1, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            agg_mean, connectivity = raw_neighbor_agg_and_connectivity(xt, ei_b, ew_b, num_nodes)
            combined = torch.cat([xt, agg_mean, connectivity], dim=-1)
            h = torch.relu(self.fc1(combined))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            h_region = torch.relu(self.region_fc1(h_region_pooled))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP_Augmented(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden * 2 + 1, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            s_agg_mean, s_connectivity = raw_neighbor_agg_and_connectivity(xt, station_ei_b, ew_station_b, num_station_nodes)
            combined = torch.cat([xt, s_agg_mean, s_connectivity], dim=-1)
            h = torch.relu(self.fc1(combined))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            r_agg_mean, r_connectivity = raw_neighbor_agg_and_connectivity(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes)
            region_combined = torch.cat([h_region_pooled, r_agg_mean, r_connectivity], dim=-1)
            h_region = torch.relu(self.region_fc1(region_combined))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_hpsearch"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset, apply_recent_mask=True):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    if apply_recent_mask and GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0
print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
region_weight_scale = region_train_nonzero.std()
region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
del train_nonzero, region_train_nonzero, edge_weight_by_hour_raw
gc.collect()

LOOKBACK = GRAPH_RECENT_HOURS
edge_roll_mean = pd.DataFrame(region_edge_weight_by_hour).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
pm25_roll_mean = pd.DataFrame(region_pm25).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
region_pm25_median_train = np.nanmedian(region_pm25[TRAIN_MASK], axis=0)
src_elevated = pm25_roll_mean[:, r_src_idx] > region_pm25_median_train[r_src_idx][None, :]
transport_signal_per_edge = np.where(src_elevated, edge_roll_mean, 0.0)
transport_score = np.zeros((n_time, n_regions), dtype=np.float32)
for r in range(n_regions):
    edge_mask = (r_dst_idx == r)
    transport_score[:, r] = np.nan_to_num(transport_signal_per_edge[:, edge_mask]).max(axis=1)
edge_threshold_train = np.percentile(region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0], 75)
transport_flag_per_region = transport_score > edge_threshold_train
del edge_roll_mean, pm25_roll_mean, src_elevated, transport_signal_per_edge, transport_score
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], [], []), 1: ([], [], [], [], []), 2: ([], [], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    input_end_t = t + WINDOW - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    Xl, yregl, yclsl, maskl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    yclsl.append(horizon_episode_label[t + WINDOW])
    maskl.append(transport_flag_per_region[input_end_t])
    sl.append(t)

X0, yreg0, ycls0, mask0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ycls1, mask1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ycls2, mask2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")
print(f"masked-in fraction: train={mask0.mean():.4f} val={mask1.mean():.4f} test={mask2.mean():.4f}")
print(f"positive rate (horizon label): train={ycls0.mean():.4f} val={ycls1.mean():.4f} test={ycls2.mean():.4f}")

def pool_to_region_features(X):
    out = np.zeros((X.shape[0], n_regions, n_time_feats), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r, :] = X[:, :, cols, :].mean(axis=(1, 2))
    return out

def masked_flat(region_feat, ycls, mask):
    m = mask.astype(bool)
    return region_feat[m], ycls[m]

X0_lr, y0_lr = masked_flat(pool_to_region_features(X0), ycls0, mask0)
X1_lr, y1_lr = masked_flat(pool_to_region_features(X1), ycls1, mask1)
X2_lr, y2_lr = masked_flat(pool_to_region_features(X2), ycls2, mask2)

print(f"\n{'='*15} Logistic Regression baseline {'='*15}")
C_GRID = [0.01, 0.1, 1.0, 10.0]
best_C, best_C_score = None, -1.0
for C in C_GRID:
    clf = LogisticRegression(C=C, max_iter=1000, class_weight="balanced")
    clf.fit(X0_lr, y0_lr)
    val_proba = clf.predict_proba(X1_lr)[:, 1]
    val_aucpr = average_precision_score(y1_lr, val_proba)
    print(f"[logistic_regression] C={C} -> val_AUCPR={val_aucpr:.4f}")
    if val_aucpr > best_C_score:
        best_C_score, best_C = val_aucpr, C

clf_final = LogisticRegression(C=best_C, max_iter=1000, class_weight="balanced")
clf_final.fit(X0_lr, y0_lr)
test_proba = clf_final.predict_proba(X2_lr)[:, 1]
lr_test_auc_roc = roc_auc_score(y2_lr, test_proba)
lr_test_auc_pr = average_precision_score(y2_lr, test_proba)
print(f"[logistic_regression] BEST C={best_C} val_AUCPR={best_C_score:.4f} TEST AUC-ROC={lr_test_auc_roc:.4f} AUC-PR={lr_test_auc_pr:.4f}")

LR_BASELINE_RESULT = {"C": best_C, "val_aucpr": best_C_score, "test_auc_roc": lr_test_auc_roc, "test_auc_pr": lr_test_auc_pr}
del X0_lr, y0_lr, X1_lr, y1_lr, X2_lr, y2_lr, clf, clf_final
gc.collect()

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ycls0_t, ycls1_t, ycls2_t = torch.tensor(ycls0, dtype=torch.float32), torch.tensor(ycls1, dtype=torch.float32), torch.tensor(ycls2, dtype=torch.float32)
mask0_t, mask1_t, mask2_t = torch.tensor(mask0, dtype=torch.float32), torch.tensor(mask1, dtype=torch.float32), torch.tensor(mask2, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1, ycls0, ycls1, ycls2, mask0, mask1, mask2, starts0, starts1, starts2
gc.collect()

masked_train_labels = ycls0_t[mask0_t.bool()]
POS_WEIGHT = min(float((masked_train_labels.numel() - masked_train_labels.sum()) / masked_train_labels.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")

edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)
del edge_weight_by_hour, region_edge_weight_by_hour
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT), reduction="none")

def run_p1_epoch(model, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch(model, X, y, m, starts, optimizer, train, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_maskn = 0.0, 0.0
    probs_list = [] if return_probs else None
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            mb = m[mb_idx].to(DEVICE)
            ew_s = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(region_edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss_pe = region_criterion(logits, yb)
                masked_loss = (loss_pe * mb).sum() / mb.sum().clamp(min=1)
            if train: masked_loss.backward()
            if return_probs: probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += masked_loss.item() * mb.sum().item(); total_maskn += mb.sum().item()
        if train: optimizer.step()
    avg_loss = total_loss / max(total_maskn, 1)
    if return_probs: return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

def run_config(arch_name, StationCls, RegionCls, lr, dropout, weight_decay, seed=0, eval_test=False):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationCls(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch(p1, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val = vl
            best_p1_state = copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)

    if arch_name == "wind_graph":
        encoder_copy = copy.deepcopy(p1.station_conv)
        p2 = RegionCls(encoder_copy, region_membership_t, dropout).to(DEVICE)
    else:
        fc1_copy, fc2_copy = copy.deepcopy(p1.fc1), copy.deepcopy(p1.fc2)
        p2 = RegionCls(fc1_copy, fc2_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_masked = ycls1_t.numpy()[mask1_t.numpy().astype(bool)]
    best_val_aucpr = -1.0
    best_state = None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch(p2, X0_t, ycls0_t, mask0_t, starts0_t, opt2, True)
        _, vp = run_p2_epoch(p2, X1_t, ycls1_t, mask1_t, starts1_t, opt2, False, True)
        vp_masked = vp[mask1_t.numpy().astype(bool)]
        va = average_precision_score(y_val_masked, vp_masked)
        if va > best_val_aucpr:
            best_val_aucpr = va
            if eval_test: best_state = copy.deepcopy(p2.state_dict())

    test_metrics = None
    if eval_test:
        p2.load_state_dict(best_state)
        p2.eval()
        with torch.no_grad():
            _, tp = run_p2_epoch(p2, X2_t, ycls2_t, mask2_t, starts2_t, opt2, False, True)
        mask_test = mask2_t.numpy().astype(bool)
        y_test = ycls2_t.numpy()[mask_test]
        proba_test = tp[mask_test]
        test_metrics = {"test_auc_roc": roc_auc_score(y_test, proba_test), "test_auc_pr": average_precision_score(y_test, proba_test)}

    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    return best_val_aucpr, test_metrics

search_results = []
best_config = {"logistic_regression": LR_BASELINE_RESULT}

# --- wind_graph: ONE confirmatory run at its already-known best config ---
print(f"\n{'='*15} wind_graph: confirmatory run at known best config {'='*15}")
t0 = time.time()
wg_val, wg_test = run_config("wind_graph", StationQuantileGCN, RegionFromQuantileGCN, 3e-3, 0.5, 5e-4, eval_test=True)
print(f"[wind_graph] val_AUCPR={wg_val:.4f} TEST AUC-ROC={wg_test['test_auc_roc']:.4f} AUC-PR={wg_test['test_auc_pr']:.4f} ({time.time()-t0:.0f}s)")
best_config["wind_graph"] = {"lr": 3e-3, "dropout": 0.5, "weight_decay": 5e-4, "val_aucpr": wg_val,
                             "test_auc_roc": wg_test["test_auc_roc"], "test_auc_pr": wg_test["test_auc_pr"]}

# --- full search for augmented_matched and no_graph ---
ARCH_SPECS = [("augmented_matched", StationQuantileMLP_Augmented, RegionFromQuantileMLP_Augmented),
              ("no_graph", StationQuantileMLP, RegionFromQuantileMLP)]

DEFAULT_DROPOUT, DEFAULT_WD = 0.5, 5e-4
LR_GRID = [3e-4, 1e-3, 3e-3, 1e-2]
REG_GRID = [(0.3, 1e-4), (0.5, 5e-4), (0.7, 2e-3)]

for arch_name, StationCls, RegionCls in ARCH_SPECS:
    print(f"\n{'='*15} STAGE 1: LR sweep, {arch_name} {'='*15}")
    best_lr, best_lr_score = None, -1.0
    for lr in LR_GRID:
        t0 = time.time()
        score, _ = run_config(arch_name, StationCls, RegionCls, lr, DEFAULT_DROPOUT, DEFAULT_WD)
        print(f"[{arch_name}] lr={lr:.0e} dropout={DEFAULT_DROPOUT} wd={DEFAULT_WD:.0e} -> val_AUCPR={score:.4f} ({time.time()-t0:.0f}s)")
        search_results.append({"arch": arch_name, "stage": 1, "lr": lr, "dropout": DEFAULT_DROPOUT, "weight_decay": DEFAULT_WD, "val_aucpr": score})
        if score > best_lr_score:
            best_lr_score, best_lr = score, lr

    print(f"\n{'='*15} STAGE 2: dropout/weight_decay sweep @ lr={best_lr:.0e}, {arch_name} {'='*15}")
    best_reg, best_reg_score = (DEFAULT_DROPOUT, DEFAULT_WD), best_lr_score
    for dropout, wd in REG_GRID:
        t0 = time.time()
        score, _ = run_config(arch_name, StationCls, RegionCls, best_lr, dropout, wd)
        print(f"[{arch_name}] lr={best_lr:.0e} dropout={dropout} wd={wd:.0e} -> val_AUCPR={score:.4f} ({time.time()-t0:.0f}s)")
        search_results.append({"arch": arch_name, "stage": 2, "lr": best_lr, "dropout": dropout, "weight_decay": wd, "val_aucpr": score})
        if score > best_reg_score:
            best_reg_score, best_reg = score, (dropout, wd)

    print(f"\n{'='*15} FINAL: re-run best config on 2019 TEST (held out from search), {arch_name} {'='*15}")
    final_val_score, test_metrics = run_config(arch_name, StationCls, RegionCls, best_lr, best_reg[0], best_reg[1], eval_test=True)
    print(f"[{arch_name}] TEST (2019): AUC-ROC={test_metrics['test_auc_roc']:.4f} AUC-PR={test_metrics['test_auc_pr']:.4f}")

    best_config[arch_name] = {"lr": best_lr, "dropout": best_reg[0], "weight_decay": best_reg[1],
                               "val_aucpr": best_reg_score, "test_auc_roc": test_metrics["test_auc_roc"], "test_auc_pr": test_metrics["test_auc_pr"]}
    print(f"\n>>> BEST for {arch_name}: lr={best_lr:.0e} dropout={best_reg[0]} weight_decay={best_reg[1]:.0e}  val_AUCPR={best_reg_score:.4f}  test_AUCPR(2019)={test_metrics['test_auc_pr']:.4f}")

results_df = pd.DataFrame(search_results)
results_df.to_csv(f"{BASE}/hpsearch_results.csv", index=False)
print("\n=== best config per model (incl. 2019 test performance) ===")
for arch, cfg in best_config.items():
    print(f"{arch}: {cfg}")

import json
with open(f"{BASE}/hpsearch_best_config.json", "w") as f:
    json.dump(best_config, f, indent=2)
print(f"\nsaved best configs to {BASE}/hpsearch_best_config.json")


station graph: 21414 directed edges, 1994 (9.3%) are 'close' (distance-only)
region graph: 272 directed edges (fully connected, 17 regions)
device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689
masked-in fraction: train=0.2493 val=0.1780 test=0.1973
positive rate (horizon label): train=0.0370 val=0.0390 test=0.0537

=============== Logistic Regression baseline ===============
[logistic_regression] C=0.01 -> val_AUCPR=0.3092
[logistic_regression] C=0.1 -> val_AUCPR=0.3087
[logistic_regression] C=1.0 -> val_AUCPR=0.3086
[logistic_regression] C=10.0 -> val_AUCPR=0.3087
[logistic_regression] BEST C=0.01 val_AUCPR=0.3092 TEST AUC-ROC=0.8395 AUC-PR=0.4289
pos_weight: 19.01

=============== wind_graph: confirmatory run at known best config ===============
[wind_graph] val_AUCPR=0.5754 TEST AUC-ROC=0.9252 AUC-PR=0.7097 (586s)

=============== STAGE 1: LR sweep, augmented_matched ===============
[augmented_matched] lr=

In [1]:
#full dataset search.

# FULL DATA (no transport-spike masking): hyperparameter search + 3-seed final retrain
# ================================================================
# Removes the transport_flag_per_region mask entirely -- training
# loss and evaluation now cover EVERY region-hour in the window,
# not just hours flagged as "transport-driven" by the wind-alignment
# formula. This avoids the circularity where that mask was built
# from the same wind-decay/speed/bearing computation that wind_graph
# uses as its edge weights (see discussion: selecting eval hours
# using the graph's own inductive-bias signal biased the head-to-head
# comparison in the graph's favor).
#
# TRAIN = 2016-2017, VAL = 2018, TEST = 2019 (unchanged, pre-COVID).
# For each of wind_graph, augmented_matched, no_graph: full LR sweep
# (stage 1) + dropout/weight_decay sweep (stage 2), THEN a 3-seed
# final retrain at the winning config, reporting mean +/- std of
# test AUC-ROC / AUC-PR on the FULL (unmasked) 2019 test set.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)
print(f"station graph: {len(src_idx)} directed edges, {close_edge_mask.sum()} ({100*close_edge_mask.mean():.1f}%) are 'close' (distance-only)")

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

representative_station_idx = np.zeros(n_regions, dtype=int)
for r in range(n_regions):
    idxs = np.where(station_region_idx == r)[0]
    representative_station_idx[r] = idxs[np.argmin(dist_to_region[idxs, r])]

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)
print(f"region graph: {len(r_src_idx)} directed edges (fully connected, {n_regions} regions)")

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = wind_speed_arr[:, representative_station_idx].astype(np.float32)
region_wdir_sin = wdir_sin_station[:, representative_station_idx].astype(np.float32)
region_wdir_cos = wdir_cos_station[:, representative_station_idx].astype(np.float32)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(pm25_raw_arr)
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wdir_sin_station, wdir_cos_station, season_sin, season_cos, wind_dir_arr, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

rev2 = region_episode_label[::-1]
roll_max_rev = pd.DataFrame(rev2).rolling(window=HORIZON, min_periods=HORIZON).max().to_numpy()
horizon_episode_label = np.nan_to_num(roll_max_rev[::-1], nan=0.0).astype(np.float32)

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

def raw_neighbor_agg_and_connectivity(x, edge_index, edge_weight, num_nodes):
    src, dst = edge_index[0], edge_index[1]
    messages = x[src] * edge_weight.unsqueeze(-1)
    agg_sum = x.new_zeros(num_nodes, x.size(-1))
    agg_sum.index_add_(0, dst, messages)
    weight_sum = x.new_zeros(num_nodes)
    weight_sum.index_add_(0, dst, edge_weight)
    agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
    connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
    return agg_mean, connectivity

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP_Augmented(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim * 2 + 1, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            agg_mean, connectivity = raw_neighbor_agg_and_connectivity(xt, ei_b, ew_b, num_nodes)
            combined = torch.cat([xt, agg_mean, connectivity], dim=-1)
            h = torch.relu(self.fc1(combined))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            h_region = torch.relu(self.region_fc1(h_region_pooled))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP_Augmented(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden * 2 + 1, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            s_agg_mean, s_connectivity = raw_neighbor_agg_and_connectivity(xt, station_ei_b, ew_station_b, num_station_nodes)
            combined = torch.cat([xt, s_agg_mean, s_connectivity], dim=-1)
            h = torch.relu(self.fc1(combined))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            r_agg_mean, r_connectivity = raw_neighbor_agg_and_connectivity(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes)
            region_combined = torch.cat([h_region_pooled, r_agg_mean, r_connectivity], dim=-1)
            h_region = torch.relu(self.region_fc1(region_combined))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset, apply_recent_mask=True):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    if apply_recent_mask and GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0
print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
region_weight_scale = region_train_nonzero.std()
region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
del train_nonzero, region_train_nonzero, edge_weight_by_hour_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

# NOTE: no transport mask -- every (window, region) pair is kept.
p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    Xl, yregl, yclsl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    yclsl.append(horizon_episode_label[t + WINDOW])
    sl.append(t)

X0, yreg0, ycls0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ycls1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ycls2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")
print(f"positive rate (horizon label, ALL region-hours): train={ycls0.mean():.4f} val={ycls1.mean():.4f} test={ycls2.mean():.4f}")

def pool_to_region_features(X):
    out = np.zeros((X.shape[0], n_regions, n_time_feats), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r, :] = X[:, :, cols, :].mean(axis=(1, 2))
    return out

X0_lr = pool_to_region_features(X0).reshape(-1, n_time_feats)
y0_lr = ycls0.reshape(-1)
X1_lr = pool_to_region_features(X1).reshape(-1, n_time_feats)
y1_lr = ycls1.reshape(-1)
X2_lr = pool_to_region_features(X2).reshape(-1, n_time_feats)
y2_lr = ycls2.reshape(-1)

print(f"\n{'='*15} Logistic Regression baseline (full data) {'='*15}")
C_GRID = [0.01, 0.1, 1.0, 10.0]
best_C, best_C_score = None, -1.0
for C in C_GRID:
    clf = LogisticRegression(C=C, max_iter=1000, class_weight="balanced")
    clf.fit(X0_lr, y0_lr)
    val_proba = clf.predict_proba(X1_lr)[:, 1]
    val_aucpr = average_precision_score(y1_lr, val_proba)
    print(f"[logistic_regression] C={C} -> val_AUCPR={val_aucpr:.4f}")
    if val_aucpr > best_C_score:
        best_C_score, best_C = val_aucpr, C

clf_final = LogisticRegression(C=best_C, max_iter=1000, class_weight="balanced")
clf_final.fit(X0_lr, y0_lr)
test_proba = clf_final.predict_proba(X2_lr)[:, 1]
LR_BASELINE_RESULT = {
    "C": best_C, "val_aucpr": best_C_score,
    "test_auc_roc": roc_auc_score(y2_lr, test_proba),
    "test_auc_pr": average_precision_score(y2_lr, test_proba),
}
print(f"[logistic_regression] BEST C={best_C} val_AUCPR={best_C_score:.4f} "
      f"TEST AUC-ROC={LR_BASELINE_RESULT['test_auc_roc']:.4f} AUC-PR={LR_BASELINE_RESULT['test_auc_pr']:.4f}")

del X0_lr, y0_lr, X1_lr, y1_lr, X2_lr, y2_lr, clf, clf_final
gc.collect()

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ycls0_t, ycls1_t, ycls2_t = torch.tensor(ycls0, dtype=torch.float32), torch.tensor(ycls1, dtype=torch.float32), torch.tensor(ycls2, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1, ycls0, ycls1, ycls2, starts0, starts1, starts2
gc.collect()

POS_WEIGHT = min(float((ycls0_t.numel() - ycls0_t.sum()) / ycls0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (full data): {POS_WEIGHT:.2f}")

edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)
del edge_weight_by_hour, region_edge_weight_by_hour
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch(model, X, y, starts, optimizer, train, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    probs_list = [] if return_probs else None
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_s = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(region_edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            if return_probs: probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    avg_loss = total_loss / max(total_n, 1)
    if return_probs: return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

def run_config(arch_name, StationCls, RegionCls, lr, dropout, weight_decay, seed=0, eval_test=False):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationCls(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch(p1, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val = vl
            best_p1_state = copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)

    if arch_name == "wind_graph":
        encoder_copy = copy.deepcopy(p1.station_conv)
        p2 = RegionCls(encoder_copy, region_membership_t, dropout).to(DEVICE)
    else:
        fc1_copy, fc2_copy = copy.deepcopy(p1.fc1), copy.deepcopy(p1.fc2)
        p2 = RegionCls(fc1_copy, fc2_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val = ycls1_t.numpy().reshape(-1)
    best_val_aucpr = -1.0
    best_state = None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch(p2, X0_t, ycls0_t, starts0_t, opt2, True)
        _, vp = run_p2_epoch(p2, X1_t, ycls1_t, starts1_t, opt2, False, True)
        va = average_precision_score(y_val, vp.reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr = va
            if eval_test: best_state = copy.deepcopy(p2.state_dict())

    test_metrics = None
    if eval_test:
        p2.load_state_dict(best_state)
        p2.eval()
        with torch.no_grad():
            _, tp = run_p2_epoch(p2, X2_t, ycls2_t, starts2_t, opt2, False, True)
        y_test = ycls2_t.numpy().reshape(-1)
        proba_test = tp.reshape(-1)
        test_metrics = {"test_auc_roc": roc_auc_score(y_test, proba_test), "test_auc_pr": average_precision_score(y_test, proba_test)}

    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    return best_val_aucpr, test_metrics

def multi_seed_final_eval(arch_name, StationCls, RegionCls, lr, dropout, weight_decay, seeds):
    per_seed = []
    for seed in seeds:
        t0 = time.time()
        val_aucpr, tm = run_config(arch_name, StationCls, RegionCls, lr, dropout, weight_decay, seed=seed, eval_test=True)
        elapsed = time.time() - t0
        print(f"  [{arch_name}] seed={seed} -> val_AUCPR={val_aucpr:.4f} TEST AUC-ROC={tm['test_auc_roc']:.4f} AUC-PR={tm['test_auc_pr']:.4f} ({elapsed:.0f}s)")
        per_seed.append({"seed": seed, "val_aucpr": val_aucpr, "test_auc_roc": tm["test_auc_roc"], "test_auc_pr": tm["test_auc_pr"]})
    roc_vals = [r["test_auc_roc"] for r in per_seed]
    pr_vals = [r["test_auc_pr"] for r in per_seed]
    val_vals = [r["val_aucpr"] for r in per_seed]
    return {
        "lr": lr, "dropout": dropout, "weight_decay": weight_decay, "n_seeds": len(seeds),
        "val_aucpr_mean": float(np.mean(val_vals)), "val_aucpr_std": float(np.std(val_vals)),
        "test_auc_roc_mean": float(np.mean(roc_vals)), "test_auc_roc_std": float(np.std(roc_vals)),
        "test_auc_pr_mean": float(np.mean(pr_vals)), "test_auc_pr_std": float(np.std(pr_vals)),
        "per_seed": per_seed,
    }

search_results = []
best_config = {"logistic_regression": LR_BASELINE_RESULT}

ARCH_SPECS = [("wind_graph", StationQuantileGCN, RegionFromQuantileGCN),
              ("augmented_matched", StationQuantileMLP_Augmented, RegionFromQuantileMLP_Augmented),
              ("no_graph", StationQuantileMLP, RegionFromQuantileMLP)]

DEFAULT_DROPOUT, DEFAULT_WD = 0.5, 5e-4
LR_GRID = [3e-4, 1e-3, 3e-3, 1e-2]
REG_GRID = [(0.3, 1e-4), (0.5, 5e-4), (0.7, 2e-3)]
N_SEEDS = 3
SEEDS = list(range(N_SEEDS))

for arch_name, StationCls, RegionCls in ARCH_SPECS:
    print(f"\n{'='*15} STAGE 1: LR sweep (full data), {arch_name} {'='*15}")
    best_lr, best_lr_score = None, -1.0
    for lr in LR_GRID:
        t0 = time.time()
        score, _ = run_config(arch_name, StationCls, RegionCls, lr, DEFAULT_DROPOUT, DEFAULT_WD)
        print(f"[{arch_name}] lr={lr:.0e} dropout={DEFAULT_DROPOUT} wd={DEFAULT_WD:.0e} -> val_AUCPR={score:.4f} ({time.time()-t0:.0f}s)")
        search_results.append({"arch": arch_name, "stage": 1, "lr": lr, "dropout": DEFAULT_DROPOUT, "weight_decay": DEFAULT_WD, "val_aucpr": score})
        if score > best_lr_score:
            best_lr_score, best_lr = score, lr

    print(f"\n{'='*15} STAGE 2: dropout/weight_decay sweep @ lr={best_lr:.0e} (full data), {arch_name} {'='*15}")
    best_reg, best_reg_score = (DEFAULT_DROPOUT, DEFAULT_WD), best_lr_score
    for dropout, wd in REG_GRID:
        t0 = time.time()
        score, _ = run_config(arch_name, StationCls, RegionCls, best_lr, dropout, wd)
        print(f"[{arch_name}] lr={best_lr:.0e} dropout={dropout} wd={wd:.0e} -> val_AUCPR={score:.4f} ({time.time()-t0:.0f}s)")
        search_results.append({"arch": arch_name, "stage": 2, "lr": best_lr, "dropout": dropout, "weight_decay": wd, "val_aucpr": score})
        if score > best_reg_score:
            best_reg_score, best_reg = score, (dropout, wd)

    print(f"\n{'='*15} FINAL: {N_SEEDS}-seed retrain of best config on 2019 TEST (full data), {arch_name} {'='*15}")
    best_config[arch_name] = multi_seed_final_eval(arch_name, StationCls, RegionCls, best_lr, best_reg[0], best_reg[1], SEEDS)
    s = best_config[arch_name]
    print(f">>> BEST for {arch_name}: lr={best_lr:.0e} dropout={best_reg[0]} weight_decay={best_reg[1]:.0e}  "
          f"TEST AUC-ROC={s['test_auc_roc_mean']:.4f}+/-{s['test_auc_roc_std']:.4f}  AUC-PR={s['test_auc_pr_mean']:.4f}+/-{s['test_auc_pr_std']:.4f}")

results_df = pd.DataFrame(search_results)
results_df.to_csv(f"{BASE}/hpsearch_results_fulldata.csv", index=False)
print(f"\n{'='*15} SUMMARY: full-data (no transport masking) test performance, {N_SEEDS} seeds {'='*15}")
for arch, cfg in best_config.items():
    if arch == "logistic_regression":
        print(f"{arch}: TEST AUC-ROC={cfg['test_auc_roc']:.4f}  AUC-PR={cfg['test_auc_pr']:.4f}  (deterministic)")
    else:
        print(f"{arch}: lr={cfg['lr']:.0e} dropout={cfg['dropout']} wd={cfg['weight_decay']:.0e}  "
              f"TEST AUC-ROC={cfg['test_auc_roc_mean']:.4f}+/-{cfg['test_auc_roc_std']:.4f}  "
              f"AUC-PR={cfg['test_auc_pr_mean']:.4f}+/-{cfg['test_auc_pr_std']:.4f}")

with open(f"{BASE}/hpsearch_best_config_fulldata.json", "w") as f:
    json.dump(best_config, f, indent=2)
print(f"\nsaved full-data best configs to {BASE}/hpsearch_best_config_fulldata.json")


station graph: 21414 directed edges, 1994 (9.3%) are 'close' (distance-only)
region graph: 272 directed edges (fully connected, 17 regions)
device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689
positive rate (horizon label, ALL region-hours): train=0.0370 val=0.0390 test=0.0537

=============== Logistic Regression baseline (full data) ===============
[logistic_regression] C=0.01 -> val_AUCPR=0.3036
[logistic_regression] C=0.1 -> val_AUCPR=0.3030
[logistic_regression] C=1.0 -> val_AUCPR=0.3029
[logistic_regression] C=10.0 -> val_AUCPR=0.3029
[logistic_regression] BEST C=0.01 val_AUCPR=0.3036 TEST AUC-ROC=0.9294 AUC-PR=0.5341
pos_weight (full data): 26.01

=============== STAGE 1: LR sweep (full data), wind_graph ===============
[wind_graph] lr=3e-04 dropout=0.5 wd=5e-04 -> val_AUCPR=0.4608 (544s)
[wind_graph] lr=1e-03 dropout=0.5 wd=5e-04 -> val_AUCPR=0.4626 (537s)
[wind_graph] lr=3e-03 dropout=0.5 wd=5e-04 -> 

In [1]:
# 10-SEED final eval, full data (no transport masking), using ALREADY-FOUND best configs
# ================================================================
# Extends the 3-seed full-data run to 10 seeds per architecture, for
# tighter confidence intervals on the wind_graph vs augmented_matched
# vs no_graph comparison. Reuses seeds 0,1,2 already saved in
# hpsearch_best_config_fulldata.json and only trains seeds 3-9 new,
# then merges to report mean/std across all 10.
#
# TRAIN = 2016-2017, VAL = 2018, TEST = 2019 (full, unmasked data).
# No hyperparameter search here -- configs are loaded from the prior
# run's json.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

representative_station_idx = np.zeros(n_regions, dtype=int)
for r in range(n_regions):
    idxs = np.where(station_region_idx == r)[0]
    representative_station_idx[r] = idxs[np.argmin(dist_to_region[idxs, r])]

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = wind_speed_arr[:, representative_station_idx].astype(np.float32)
region_wdir_sin = wdir_sin_station[:, representative_station_idx].astype(np.float32)
region_wdir_cos = wdir_cos_station[:, representative_station_idx].astype(np.float32)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(pm25_raw_arr)
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wdir_sin_station, wdir_cos_station, season_sin, season_cos, wind_dir_arr, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

rev2 = region_episode_label[::-1]
roll_max_rev = pd.DataFrame(rev2).rolling(window=HORIZON, min_periods=HORIZON).max().to_numpy()
horizon_episode_label = np.nan_to_num(roll_max_rev[::-1], nan=0.0).astype(np.float32)

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

def raw_neighbor_agg_and_connectivity(x, edge_index, edge_weight, num_nodes):
    src, dst = edge_index[0], edge_index[1]
    messages = x[src] * edge_weight.unsqueeze(-1)
    agg_sum = x.new_zeros(num_nodes, x.size(-1))
    agg_sum.index_add_(0, dst, messages)
    weight_sum = x.new_zeros(num_nodes)
    weight_sum.index_add_(0, dst, edge_weight)
    agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
    connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
    return agg_mean, connectivity

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP_Augmented(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim * 2 + 1, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            agg_mean, connectivity = raw_neighbor_agg_and_connectivity(xt, ei_b, ew_b, num_nodes)
            combined = torch.cat([xt, agg_mean, connectivity], dim=-1)
            h = torch.relu(self.fc1(combined))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            h_region = torch.relu(self.region_fc1(h_region_pooled))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP_Augmented(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden * 2 + 1, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            s_agg_mean, s_connectivity = raw_neighbor_agg_and_connectivity(xt, station_ei_b, ew_station_b, num_station_nodes)
            combined = torch.cat([xt, s_agg_mean, s_connectivity], dim=-1)
            h = torch.relu(self.fc1(combined))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            r_agg_mean, r_connectivity = raw_neighbor_agg_and_connectivity(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes)
            region_combined = torch.cat([h_region_pooled, r_agg_mean, r_connectivity], dim=-1)
            h_region = torch.relu(self.region_fc1(region_combined))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset, apply_recent_mask=True):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    if apply_recent_mask and GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0
print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
region_weight_scale = region_train_nonzero.std()
region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
del train_nonzero, region_train_nonzero, edge_weight_by_hour_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    Xl, yregl, yclsl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    yclsl.append(horizon_episode_label[t + WINDOW])
    sl.append(t)

X0, yreg0, ycls0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ycls1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ycls2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ycls0_t, ycls1_t, ycls2_t = torch.tensor(ycls0, dtype=torch.float32), torch.tensor(ycls1, dtype=torch.float32), torch.tensor(ycls2, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1, ycls0, ycls1, ycls2, starts0, starts1, starts2
gc.collect()

POS_WEIGHT = min(float((ycls0_t.numel() - ycls0_t.sum()) / ycls0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (full data): {POS_WEIGHT:.2f}")

edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)
del edge_weight_by_hour, region_edge_weight_by_hour
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch(model, X, y, starts, optimizer, train, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    probs_list = [] if return_probs else None
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_s = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(region_edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            if return_probs: probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    avg_loss = total_loss / max(total_n, 1)
    if return_probs: return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

def run_config(arch_name, StationCls, RegionCls, lr, dropout, weight_decay, seed=0, eval_test=False):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationCls(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch(p1, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val = vl
            best_p1_state = copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)

    if arch_name == "wind_graph":
        encoder_copy = copy.deepcopy(p1.station_conv)
        p2 = RegionCls(encoder_copy, region_membership_t, dropout).to(DEVICE)
    else:
        fc1_copy, fc2_copy = copy.deepcopy(p1.fc1), copy.deepcopy(p1.fc2)
        p2 = RegionCls(fc1_copy, fc2_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val = ycls1_t.numpy().reshape(-1)
    best_val_aucpr = -1.0
    best_state = None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch(p2, X0_t, ycls0_t, starts0_t, opt2, True)
        _, vp = run_p2_epoch(p2, X1_t, ycls1_t, starts1_t, opt2, False, True)
        va = average_precision_score(y_val, vp.reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr = va
            if eval_test: best_state = copy.deepcopy(p2.state_dict())

    test_metrics = None
    if eval_test:
        p2.load_state_dict(best_state)
        p2.eval()
        with torch.no_grad():
            _, tp = run_p2_epoch(p2, X2_t, ycls2_t, starts2_t, opt2, False, True)
        y_test = ycls2_t.numpy().reshape(-1)
        proba_test = tp.reshape(-1)
        test_metrics = {"test_auc_roc": roc_auc_score(y_test, proba_test), "test_auc_pr": average_precision_score(y_test, proba_test)}

    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    return best_val_aucpr, test_metrics

# --- load prior 3-seed results and best configs; only train the NEW seeds ---
with open(f"{BASE}/hpsearch_best_config_fulldata.json") as f:
    prior = json.load(f)
print("loaded prior full-data results (3 seeds):", {k: v.get("n_seeds") for k, v in prior.items() if k != "logistic_regression"})

ARCH_MODEL_MAP = {
    "wind_graph": (StationQuantileGCN, RegionFromQuantileGCN),
    "augmented_matched": (StationQuantileMLP_Augmented, RegionFromQuantileMLP_Augmented),
    "no_graph": (StationQuantileMLP, RegionFromQuantileMLP),
}

TOTAL_SEEDS = 10
best_config = {"logistic_regression": prior["logistic_regression"]}

for arch_name, (StationCls, RegionCls) in ARCH_MODEL_MAP.items():
    cfg = prior[arch_name]
    existing_per_seed = cfg["per_seed"]  # seeds 0,1,2 already run
    existing_seeds = {r["seed"] for r in existing_per_seed}
    new_seeds = [s for s in range(TOTAL_SEEDS) if s not in existing_seeds]
    print(f"\n{'='*15} {arch_name}: training {len(new_seeds)} NEW seeds {new_seeds} "
          f"(config lr={cfg['lr']:.0e}, dropout={cfg['dropout']}, wd={cfg['weight_decay']:.0e}) {'='*15}")

    all_per_seed = list(existing_per_seed)
    for seed in new_seeds:
        t0 = time.time()
        val_aucpr, tm = run_config(arch_name, StationCls, RegionCls, cfg["lr"], cfg["dropout"], cfg["weight_decay"], seed=seed, eval_test=True)
        elapsed = time.time() - t0
        print(f"  [{arch_name}] seed={seed} -> val_AUCPR={val_aucpr:.4f} TEST AUC-ROC={tm['test_auc_roc']:.4f} AUC-PR={tm['test_auc_pr']:.4f} ({elapsed:.0f}s)")
        all_per_seed.append({"seed": seed, "val_aucpr": val_aucpr, "test_auc_roc": tm["test_auc_roc"], "test_auc_pr": tm["test_auc_pr"]})

    all_per_seed.sort(key=lambda r: r["seed"])
    roc_vals = [r["test_auc_roc"] for r in all_per_seed]
    pr_vals = [r["test_auc_pr"] for r in all_per_seed]
    val_vals = [r["val_aucpr"] for r in all_per_seed]
    best_config[arch_name] = {
        "lr": cfg["lr"], "dropout": cfg["dropout"], "weight_decay": cfg["weight_decay"],
        "n_seeds": len(all_per_seed),
        "val_aucpr_mean": float(np.mean(val_vals)), "val_aucpr_std": float(np.std(val_vals)),
        "test_auc_roc_mean": float(np.mean(roc_vals)), "test_auc_roc_std": float(np.std(roc_vals)),
        "test_auc_pr_mean": float(np.mean(pr_vals)), "test_auc_pr_std": float(np.std(pr_vals)),
        "per_seed": all_per_seed,
    }
    s = best_config[arch_name]
    print(f">>> {arch_name} ({s['n_seeds']} seeds): TEST AUC-ROC={s['test_auc_roc_mean']:.4f}+/-{s['test_auc_roc_std']:.4f}  "
          f"AUC-PR={s['test_auc_pr_mean']:.4f}+/-{s['test_auc_pr_std']:.4f}")

print(f"\n{'='*15} SUMMARY: full-data (no transport masking) test performance, {TOTAL_SEEDS} seeds {'='*15}")
for arch, cfg in best_config.items():
    if arch == "logistic_regression":
        print(f"{arch}: TEST AUC-ROC={cfg['test_auc_roc']:.4f}  AUC-PR={cfg['test_auc_pr']:.4f}  (deterministic)")
    else:
        print(f"{arch}: lr={cfg['lr']:.0e} dropout={cfg['dropout']} wd={cfg['weight_decay']:.0e}  "
              f"TEST AUC-ROC={cfg['test_auc_roc_mean']:.4f}+/-{cfg['test_auc_roc_std']:.4f}  "
              f"AUC-PR={cfg['test_auc_pr_mean']:.4f}+/-{cfg['test_auc_pr_std']:.4f}")

with open(f"{BASE}/hpsearch_best_config_fulldata_10seed.json", "w") as f:
    json.dump(best_config, f, indent=2)
print(f"\nsaved 10-seed full-data results to {BASE}/hpsearch_best_config_fulldata_10seed.json")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689
pos_weight (full data): 26.01
loaded prior full-data results (3 seeds): {'wind_graph': 3, 'augmented_matched': 3, 'no_graph': 3}

=============== wind_graph: training 7 NEW seeds [3, 4, 5, 6, 7, 8, 9] (config lr=3e-03, dropout=0.5, wd=5e-04) ===============
  [wind_graph] seed=3 -> val_AUCPR=0.4835 TEST AUC-ROC=0.9546 AUC-PR=0.6917 (586s)
  [wind_graph] seed=4 -> val_AUCPR=0.4607 TEST AUC-ROC=0.9499 AUC-PR=0.6783 (582s)
  [wind_graph] seed=5 -> val_AUCPR=0.4843 TEST AUC-ROC=0.9522 AUC-PR=0.6909 (594s)
  [wind_graph] seed=6 -> val_AUCPR=0.5033 TEST AUC-ROC=0.9445 AUC-PR=0.6830 (594s)
  [wind_graph] seed=7 -> val_AUCPR=0.5041 TEST AUC-ROC=0.9561 AUC-PR=0.7089 (593s)
  [wind_graph] seed=8 -> val_AUCPR=0.4763 TEST AUC-ROC=0.9524 AUC-PR=0.6869 (593s)
  [wind_graph] seed=9 -> val_AUCPR=0.4810 TEST AUC-ROC=0.9481 AUC-PR=0.6811 (593s)
>>> wind_graph (10 seeds): TES

In [2]:
# Paired significance tests across the 10 matched seeds
# ================================================================
# Since wind_graph, augmented_matched, and no_graph were each
# trained/evaluated with the SAME seed values (0-9) on the SAME
# train/val/test split, we can pair results by seed and remove the
# shared across-seed variance -- a much more powerful test than the
# unpaired comparison, since seed-level noise (e.g. an unlucky init
# for wind_graph on seed 6 also shows up if it happens to no_graph)
# cancels out rather than inflating the variance estimate.
#
# Runs paired t-test AND Wilcoxon signed-rank (nonparametric, no
# normality assumption, more appropriate with n=10) for every
# architecture pair, on both AUC-ROC and AUC-PR.
# ================================================================
import json
import numpy as np
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
with open(f"{BASE}/hpsearch_best_config_fulldata_10seed.json") as f:
    results = json.load(f)

ARCHS = ["no_graph", "wind_graph", "augmented_matched"]
METRICS = ["test_auc_roc", "test_auc_pr"]

# build seed -> metric value dicts per arch, then align by seed
per_arch_by_seed = {}
for arch in ARCHS:
    per_arch_by_seed[arch] = {r["seed"]: r for r in results[arch]["per_seed"]}

common_seeds = sorted(set.intersection(*[set(per_arch_by_seed[a].keys()) for a in ARCHS]))
print(f"comparing over {len(common_seeds)} matched seeds: {common_seeds}\n")

pairs = [("no_graph", "wind_graph"), ("no_graph", "augmented_matched"), ("wind_graph", "augmented_matched")]

summary_rows = []
for metric in METRICS:
    print(f"{'='*20} {metric} {'='*20}")
    for a, b in pairs:
        vals_a = np.array([per_arch_by_seed[a][s][metric] for s in common_seeds])
        vals_b = np.array([per_arch_by_seed[b][s][metric] for s in common_seeds])
        diff = vals_b - vals_a  # positive = b beats a

        t_stat, t_p = stats.ttest_rel(vals_b, vals_a)
        try:
            w_stat, w_p = stats.wilcoxon(vals_b, vals_a)
        except ValueError:
            w_stat, w_p = np.nan, np.nan  # e.g. if all diffs are zero (won't happen here)

        n_wins_b = int((diff > 0).sum())
        print(f"[{b} vs {a}]  mean_diff={diff.mean():+.4f}  std_diff={diff.std(ddof=1):.4f}  "
              f"b_wins={n_wins_b}/{len(common_seeds)}  "
              f"paired_t: t={t_stat:.3f} p={t_p:.4f}   wilcoxon: W={w_stat:.1f} p={w_p:.4f}")

        summary_rows.append({
            "metric": metric, "a": a, "b": b, "mean_diff_b_minus_a": float(diff.mean()),
            "std_diff": float(diff.std(ddof=1)), "b_wins": n_wins_b, "n": len(common_seeds),
            "paired_t_stat": float(t_stat), "paired_t_p": float(t_p),
            "wilcoxon_stat": float(w_stat), "wilcoxon_p": float(w_p),
        })
    print()

with open(f"{BASE}/paired_seed_significance_tests.json", "w") as f:
    json.dump(summary_rows, f, indent=2)
print(f"saved paired test results to {BASE}/paired_seed_significance_tests.json")


comparing over 10 matched seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

==================== test_auc_roc ====================
[wind_graph vs no_graph]  mean_diff=+0.0050  std_diff=0.0036  b_wins=9/10  paired_t: t=4.405 p=0.0017   wilcoxon: W=2.0 p=0.0059
[augmented_matched vs no_graph]  mean_diff=+0.0086  std_diff=0.0049  b_wins=9/10  paired_t: t=5.501 p=0.0004   wilcoxon: W=1.0 p=0.0039
[augmented_matched vs wind_graph]  mean_diff=+0.0036  std_diff=0.0058  b_wins=8/10  paired_t: t=1.969 p=0.0805   wilcoxon: W=11.0 p=0.1055

==================== test_auc_pr ====================
[wind_graph vs no_graph]  mean_diff=+0.0139  std_diff=0.0123  b_wins=8/10  paired_t: t=3.569 p=0.0060   wilcoxon: W=3.0 p=0.0098
[augmented_matched vs no_graph]  mean_diff=+0.0238  std_diff=0.0092  b_wins=10/10  paired_t: t=8.204 p=0.0000   wilcoxon: W=0.0 p=0.0020
[augmented_matched vs wind_graph]  mean_diff=+0.0098  std_diff=0.0121  b_wins=9/10  paired_t: t=2.576 p=0.0299   wilcoxon: W=9.0 p=0.0645

saved paired tes

In [1]:
# QUICK CHECK: region-to-region wind uses ALL sensors in a region (circular mean),
# instead of a single representative station
# ================================================================
# The current region_windspeed/region_wdir_sin/region_wdir_cos are
# sampled from ONE representative_station_idx per region (nearest
# station to the region centroid). This is the wind signal that
# drives the region-to-region graph edges in wind_graph (and
# augmented_matched, which also consumes region edges).
#
# CHANGE: replace that single-station sample with the circular mean
# of wind direction (average of sin/cos components) and mean wind
# speed across EVERY station in the region, using the same
# regional_flat_mean() helper already used for PM2.5. This is a
# proper vector average -- if all stations in a region report
# identical wind (e.g. because they share an underlying reanalysis
# grid cell), averaging trivially returns that same value; if they
# disagree, it produces a genuine predominant-direction estimate.
# Everything else (architecture, station graph, split, features) is
# UNCHANGED from the full-data (no transport masking) setup.
#
# This is a quick check: 5 seeds, wind_graph only, at its already-
# found best config (lr=3e-3, dropout=0.5, wd=5e-4), compared
# against the existing 10-seed single-station baseline saved in
# hpsearch_best_config_fulldata_10seed.json.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))

# ---- CHANGED: predominant regional wind = circular mean over ALL stations in the
# region, instead of a single representative_station_idx sample ----
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
# ---- end change ----

region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(pm25_raw_arr)
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wdir_sin_station, wdir_cos_station, season_sin, season_cos, wind_dir_arr, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

rev2 = region_episode_label[::-1]
roll_max_rev = pd.DataFrame(rev2).rolling(window=HORIZON, min_periods=HORIZON).max().to_numpy()
horizon_episode_label = np.nan_to_num(roll_max_rev[::-1], nan=0.0).astype(np.float32)

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset, apply_recent_mask=True):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    if apply_recent_mask and GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0
print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
region_weight_scale = region_train_nonzero.std()
region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
del train_nonzero, region_train_nonzero, edge_weight_by_hour_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    Xl, yregl, yclsl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    yclsl.append(horizon_episode_label[t + WINDOW])
    sl.append(t)

X0, yreg0, ycls0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ycls1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ycls2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ycls0_t, ycls1_t, ycls2_t = torch.tensor(ycls0, dtype=torch.float32), torch.tensor(ycls1, dtype=torch.float32), torch.tensor(ycls2, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1, ycls0, ycls1, ycls2, starts0, starts1, starts2
gc.collect()

POS_WEIGHT = min(float((ycls0_t.numel() - ycls0_t.sum()) / ycls0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (full data): {POS_WEIGHT:.2f}")

edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)
del edge_weight_by_hour, region_edge_weight_by_hour
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch(model, X, y, starts, optimizer, train, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    probs_list = [] if return_probs else None
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_s = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(region_edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            if return_probs: probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    avg_loss = total_loss / max(total_n, 1)
    if return_probs: return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

def run_config(StationCls, RegionCls, lr, dropout, weight_decay, seed=0, eval_test=False):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationCls(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch(p1, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val = vl
            best_p1_state = copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)

    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionCls(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val = ycls1_t.numpy().reshape(-1)
    best_val_aucpr = -1.0
    best_state = None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch(p2, X0_t, ycls0_t, starts0_t, opt2, True)
        _, vp = run_p2_epoch(p2, X1_t, ycls1_t, starts1_t, opt2, False, True)
        va = average_precision_score(y_val, vp.reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr = va
            if eval_test: best_state = copy.deepcopy(p2.state_dict())

    test_metrics = None
    if eval_test:
        p2.load_state_dict(best_state)
        p2.eval()
        with torch.no_grad():
            _, tp = run_p2_epoch(p2, X2_t, ycls2_t, starts2_t, opt2, False, True)
        y_test = ycls2_t.numpy().reshape(-1)
        proba_test = tp.reshape(-1)
        test_metrics = {"test_auc_roc": roc_auc_score(y_test, proba_test), "test_auc_pr": average_precision_score(y_test, proba_test)}

    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    return best_val_aucpr, test_metrics

with open(f"{BASE}/hpsearch_best_config_fulldata_10seed.json") as f:
    prior = json.load(f)
wg_cfg = prior["wind_graph"]
print(f"using wind_graph's known best config: lr={wg_cfg['lr']:.0e} dropout={wg_cfg['dropout']} wd={wg_cfg['weight_decay']:.0e}")

N_SEEDS = 5
SEEDS = list(range(N_SEEDS))
per_seed = []
for seed in SEEDS:
    t0 = time.time()
    val_aucpr, tm = run_config(StationQuantileGCN, RegionFromQuantileGCN, wg_cfg["lr"], wg_cfg["dropout"], wg_cfg["weight_decay"], seed=seed, eval_test=True)
    elapsed = time.time() - t0
    print(f"  [wind_graph_regionmean] seed={seed} -> val_AUCPR={val_aucpr:.4f} TEST AUC-ROC={tm['test_auc_roc']:.4f} AUC-PR={tm['test_auc_pr']:.4f} ({elapsed:.0f}s)")
    per_seed.append({"seed": seed, "val_aucpr": val_aucpr, "test_auc_roc": tm["test_auc_roc"], "test_auc_pr": tm["test_auc_pr"]})

roc_vals = [r["test_auc_roc"] for r in per_seed]
pr_vals = [r["test_auc_pr"] for r in per_seed]
new_result = {
    "lr": wg_cfg["lr"], "dropout": wg_cfg["dropout"], "weight_decay": wg_cfg["weight_decay"], "n_seeds": N_SEEDS,
    "test_auc_roc_mean": float(np.mean(roc_vals)), "test_auc_roc_std": float(np.std(roc_vals)),
    "test_auc_pr_mean": float(np.mean(pr_vals)), "test_auc_pr_std": float(np.std(pr_vals)),
    "per_seed": per_seed,
}

old_result = prior["wind_graph"]
print(f"\n{'='*15} COMPARISON: single-station vs region-mean wind, wind_graph {'='*15}")
print(f"OLD (single representative station, {old_result['n_seeds']} seeds):  "
      f"TEST AUC-ROC={old_result['test_auc_roc_mean']:.4f}+/-{old_result['test_auc_roc_std']:.4f}  "
      f"AUC-PR={old_result['test_auc_pr_mean']:.4f}+/-{old_result['test_auc_pr_std']:.4f}")
print(f"NEW (region-wide circular mean, {N_SEEDS} seeds):  "
      f"TEST AUC-ROC={new_result['test_auc_roc_mean']:.4f}+/-{new_result['test_auc_roc_std']:.4f}  "
      f"AUC-PR={new_result['test_auc_pr_mean']:.4f}+/-{new_result['test_auc_pr_std']:.4f}")

with open(f"{BASE}/wind_graph_regionmean_wind_quickcheck.json", "w") as f:
    json.dump(new_result, f, indent=2)
print(f"\nsaved to {BASE}/wind_graph_regionmean_wind_quickcheck.json")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689
pos_weight (full data): 26.01
using wind_graph's known best config: lr=3e-03 dropout=0.5 wd=5e-04
  [wind_graph_regionmean] seed=0 -> val_AUCPR=0.4972 TEST AUC-ROC=0.9579 AUC-PR=0.7074 (584s)
  [wind_graph_regionmean] seed=1 -> val_AUCPR=0.4997 TEST AUC-ROC=0.9525 AUC-PR=0.6919 (580s)
  [wind_graph_regionmean] seed=2 -> val_AUCPR=0.4693 TEST AUC-ROC=0.9517 AUC-PR=0.6853 (578s)
  [wind_graph_regionmean] seed=3 -> val_AUCPR=0.4912 TEST AUC-ROC=0.9546 AUC-PR=0.6955 (584s)
  [wind_graph_regionmean] seed=4 -> val_AUCPR=0.4530 TEST AUC-ROC=0.9501 AUC-PR=0.6811 (579s)

=============== COMPARISON: single-station vs region-mean wind, wind_graph ===============
OLD (single representative station, 10 seeds):  TEST AUC-ROC=0.9511+/-0.0041  AUC-PR=0.6871+/-0.0112
NEW (region-wide circular mean, 5 seeds):  TEST AUC-ROC=0.9534+/-0.0027  AUC-PR=0.6922+/-0.0091

saved to /U

In [1]:
# Extend region-mean wind_graph to 10 seeds (matching the single-station baseline)
# ================================================================
# Trains seeds 5-9 (region-mean wind formulation, same as the quick
# check) and merges with the already-saved seeds 0-4 from
# wind_graph_regionmean_wind_quickcheck.json, to get a full 10-seed
# result directly comparable -- seed-for-seed -- against the
# single-station baseline in hpsearch_best_config_fulldata_10seed.json.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))

# ---- region-mean wind (all stations in region, circular mean) ----
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
# ---- end ----

region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(pm25_raw_arr)
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wdir_sin_station, wdir_cos_station, season_sin, season_cos, wind_dir_arr, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

rev2 = region_episode_label[::-1]
roll_max_rev = pd.DataFrame(rev2).rolling(window=HORIZON, min_periods=HORIZON).max().to_numpy()
horizon_episode_label = np.nan_to_num(roll_max_rev[::-1], nan=0.0).astype(np.float32)

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset, apply_recent_mask=True):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    if apply_recent_mask and GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0
print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
region_weight_scale = region_train_nonzero.std()
region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
del train_nonzero, region_train_nonzero, edge_weight_by_hour_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    Xl, yregl, yclsl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    yclsl.append(horizon_episode_label[t + WINDOW])
    sl.append(t)

X0, yreg0, ycls0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ycls1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ycls2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ycls0_t, ycls1_t, ycls2_t = torch.tensor(ycls0, dtype=torch.float32), torch.tensor(ycls1, dtype=torch.float32), torch.tensor(ycls2, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1, ycls0, ycls1, ycls2, starts0, starts1, starts2
gc.collect()

POS_WEIGHT = min(float((ycls0_t.numel() - ycls0_t.sum()) / ycls0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (full data): {POS_WEIGHT:.2f}")

edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)
del edge_weight_by_hour, region_edge_weight_by_hour
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch(model, X, y, starts, optimizer, train, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    probs_list = [] if return_probs else None
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_s = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(region_edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            if return_probs: probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    avg_loss = total_loss / max(total_n, 1)
    if return_probs: return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

def run_config(StationCls, RegionCls, lr, dropout, weight_decay, seed=0, eval_test=False):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationCls(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch(p1, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val = vl
            best_p1_state = copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)

    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionCls(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val = ycls1_t.numpy().reshape(-1)
    best_val_aucpr = -1.0
    best_state = None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch(p2, X0_t, ycls0_t, starts0_t, opt2, True)
        _, vp = run_p2_epoch(p2, X1_t, ycls1_t, starts1_t, opt2, False, True)
        va = average_precision_score(y_val, vp.reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr = va
            if eval_test: best_state = copy.deepcopy(p2.state_dict())

    test_metrics = None
    if eval_test:
        p2.load_state_dict(best_state)
        p2.eval()
        with torch.no_grad():
            _, tp = run_p2_epoch(p2, X2_t, ycls2_t, starts2_t, opt2, False, True)
        y_test = ycls2_t.numpy().reshape(-1)
        proba_test = tp.reshape(-1)
        test_metrics = {"test_auc_roc": roc_auc_score(y_test, proba_test), "test_auc_pr": average_precision_score(y_test, proba_test)}

    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    return best_val_aucpr, test_metrics

with open(f"{BASE}/wind_graph_regionmean_wind_quickcheck.json") as f:
    prior_regionmean = json.load(f)
with open(f"{BASE}/hpsearch_best_config_fulldata_10seed.json") as f:
    old_baseline = json.load(f)["wind_graph"]

cfg = {"lr": prior_regionmean["lr"], "dropout": prior_regionmean["dropout"], "weight_decay": prior_regionmean["weight_decay"]}
print(f"config: lr={cfg['lr']:.0e} dropout={cfg['dropout']} wd={cfg['weight_decay']:.0e}")

existing_per_seed = prior_regionmean["per_seed"]  # seeds 0-4
existing_seeds = {r["seed"] for r in existing_per_seed}
TOTAL_SEEDS = 10
new_seeds = [s for s in range(TOTAL_SEEDS) if s not in existing_seeds]
print(f"training {len(new_seeds)} NEW seeds: {new_seeds}")

all_per_seed = list(existing_per_seed)
for seed in new_seeds:
    t0 = time.time()
    val_aucpr, tm = run_config(StationQuantileGCN, RegionFromQuantileGCN, cfg["lr"], cfg["dropout"], cfg["weight_decay"], seed=seed, eval_test=True)
    elapsed = time.time() - t0
    print(f"  [wind_graph_regionmean] seed={seed} -> val_AUCPR={val_aucpr:.4f} TEST AUC-ROC={tm['test_auc_roc']:.4f} AUC-PR={tm['test_auc_pr']:.4f} ({elapsed:.0f}s)")
    all_per_seed.append({"seed": seed, "val_aucpr": val_aucpr, "test_auc_roc": tm["test_auc_roc"], "test_auc_pr": tm["test_auc_pr"]})

all_per_seed.sort(key=lambda r: r["seed"])
roc_vals = [r["test_auc_roc"] for r in all_per_seed]
pr_vals = [r["test_auc_pr"] for r in all_per_seed]
new_result = {
    "lr": cfg["lr"], "dropout": cfg["dropout"], "weight_decay": cfg["weight_decay"], "n_seeds": len(all_per_seed),
    "test_auc_roc_mean": float(np.mean(roc_vals)), "test_auc_roc_std": float(np.std(roc_vals)),
    "test_auc_pr_mean": float(np.mean(pr_vals)), "test_auc_pr_std": float(np.std(pr_vals)),
    "per_seed": all_per_seed,
}

print(f"\n{'='*15} FULL 10-SEED COMPARISON: single-station vs region-mean wind, wind_graph {'='*15}")
print(f"OLD (single representative station, {old_baseline['n_seeds']} seeds):  "
      f"TEST AUC-ROC={old_baseline['test_auc_roc_mean']:.4f}+/-{old_baseline['test_auc_roc_std']:.4f}  "
      f"AUC-PR={old_baseline['test_auc_pr_mean']:.4f}+/-{old_baseline['test_auc_pr_std']:.4f}")
print(f"NEW (region-wide circular mean, {new_result['n_seeds']} seeds):  "
      f"TEST AUC-ROC={new_result['test_auc_roc_mean']:.4f}+/-{new_result['test_auc_roc_std']:.4f}  "
      f"AUC-PR={new_result['test_auc_pr_mean']:.4f}+/-{new_result['test_auc_pr_std']:.4f}")

# paired comparison, seed-by-seed (both used the same seeds 0-9)
old_by_seed = {r["seed"]: r for r in old_baseline["per_seed"]}
diffs_roc = np.array([new_result["per_seed"][i]["test_auc_roc"] - old_by_seed[i]["test_auc_roc"] for i in range(TOTAL_SEEDS)])
diffs_pr = np.array([new_result["per_seed"][i]["test_auc_pr"] - old_by_seed[i]["test_auc_pr"] for i in range(TOTAL_SEEDS)])
from scipy import stats
t_roc, p_roc = stats.ttest_1samp(diffs_roc, 0)
t_pr, p_pr = stats.ttest_1samp(diffs_pr, 0)
w_roc, wp_roc = stats.wilcoxon(diffs_roc)
w_pr, wp_pr = stats.wilcoxon(diffs_pr)
print(f"\npaired diffs (region-mean minus single-station), n={TOTAL_SEEDS} matched seeds:")
print(f"  AUC-ROC: mean_diff={diffs_roc.mean():+.4f}  wins={int((diffs_roc>0).sum())}/{TOTAL_SEEDS}  paired_t p={p_roc:.4f}  wilcoxon p={wp_roc:.4f}")
print(f"  AUC-PR:  mean_diff={diffs_pr.mean():+.4f}  wins={int((diffs_pr>0).sum())}/{TOTAL_SEEDS}  paired_t p={p_pr:.4f}  wilcoxon p={wp_pr:.4f}")

with open(f"{BASE}/wind_graph_regionmean_wind_10seed.json", "w") as f:
    json.dump(new_result, f, indent=2)
print(f"\nsaved to {BASE}/wind_graph_regionmean_wind_10seed.json")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689
pos_weight (full data): 26.01
config: lr=3e-03 dropout=0.5 wd=5e-04
training 5 NEW seeds: [5, 6, 7, 8, 9]
  [wind_graph_regionmean] seed=5 -> val_AUCPR=0.4950 TEST AUC-ROC=0.9526 AUC-PR=0.6986 (581s)
  [wind_graph_regionmean] seed=6 -> val_AUCPR=0.5001 TEST AUC-ROC=0.9473 AUC-PR=0.6923 (578s)
  [wind_graph_regionmean] seed=7 -> val_AUCPR=0.5000 TEST AUC-ROC=0.9576 AUC-PR=0.7073 (579s)
  [wind_graph_regionmean] seed=8 -> val_AUCPR=0.4790 TEST AUC-ROC=0.9517 AUC-PR=0.6897 (579s)
  [wind_graph_regionmean] seed=9 -> val_AUCPR=0.4779 TEST AUC-ROC=0.9478 AUC-PR=0.6882 (584s)

=============== FULL 10-SEED COMPARISON: single-station vs region-mean wind, wind_graph ===============
OLD (single representative station, 10 seeds):  TEST AUC-ROC=0.9511+/-0.0041  AUC-PR=0.6871+/-0.0112
NEW (region-wide circular mean, 10 seeds):  TEST AUC-ROC=0.9524+/-0.0034  AUC-PR=0.6937

In [1]:
# ================================================================
# FINAL HEAD-TO-HEAD: wind_graph, augmented_matched, no_graph (each
# using its own tuned config from hpsearch_best_config.json) plus
# logistic_regression (retrained with its tuned C) -- full rigor
# (5/4 epoch budget, 2-seed ensembling) on TRAIN=2016-2019,
# VAL=2020, TEST=2021.
#
# FIX: pool_to_region_features rewritten as a single vectorized
# matrix multiply instead of a 17-region loop of boolean-indexed
# copies -- the old version was copying up to ~2GB per region out
# of a ~10GB array, 17 times over, which is what caused the 2-hour
# stall. Also adds validation-tuned decision thresholds (same
# max-F1-on-val method used earlier in this project) alongside the
# naive 0.5 cutoff for precision/recall, since threshold choice
# doesn't affect AUC-ROC/AUC-PR (already threshold-free) but does
# meaningfully affect reported precision/recall.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.linear_model import LogisticRegression

BASE = "/Users/drewbaldwin/PM2_5 Research"

with open(f"{BASE}/hpsearch_best_config.json") as f:
    BEST_CONFIG = json.load(f)
print("loaded tuned configs:")
for arch, cfg in BEST_CONFIG.items():
    print(f"  {arch}: {cfg}")

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)
print(f"station graph: {E} directed edges, {close_edge_mask.sum()} ({100*close_edge_mask.mean():.1f}%) are 'close' (distance-only)")

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

representative_station_idx = np.zeros(n_regions, dtype=int)
for r in range(n_regions):
    idxs = np.where(station_region_idx == r)[0]
    representative_station_idx[r] = idxs[np.argmin(dist_to_region[idxs, r])]

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)
print(f"region graph: {len(r_src_idx)} directed edges (fully connected, {n_regions} regions)")

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = wind_speed_arr[:, representative_station_idx].astype(np.float32)
region_wdir_sin = wdir_sin_station[:, representative_station_idx].astype(np.float32)
region_wdir_cos = wdir_cos_station[:, representative_station_idx].astype(np.float32)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(pm25_raw_arr)
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wdir_sin_station, wdir_cos_station, season_sin, season_cos, wind_dir_arr, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

rev2 = region_episode_label[::-1]
roll_max_rev = pd.DataFrame(rev2).rolling(window=HORIZON, min_periods=HORIZON).max().to_numpy()
horizon_episode_label = np.nan_to_num(roll_max_rev[::-1], nan=0.0).astype(np.float32)

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

def raw_neighbor_agg_and_connectivity(x, edge_index, edge_weight, num_nodes):
    src, dst = edge_index[0], edge_index[1]
    messages = x[src] * edge_weight.unsqueeze(-1)
    agg_sum = x.new_zeros(num_nodes, x.size(-1))
    agg_sum.index_add_(0, dst, messages)
    weight_sum = x.new_zeros(num_nodes)
    weight_sum.index_add_(0, dst, edge_weight)
    agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
    connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
    return agg_mean, connectivity

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP_Augmented(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim * 2 + 1, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            agg_mean, connectivity = raw_neighbor_agg_and_connectivity(xt, ei_b, ew_b, num_nodes)
            combined = torch.cat([xt, agg_mean, connectivity], dim=-1)
            h = torch.relu(self.fc1(combined))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            h_region = torch.relu(self.region_fc1(h_region_pooled))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP_Augmented(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden * 2 + 1, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            s_agg_mean, s_connectivity = raw_neighbor_agg_and_connectivity(xt, station_ei_b, ew_station_b, num_station_nodes)
            combined = torch.cat([xt, s_agg_mean, s_connectivity], dim=-1)
            h = torch.relu(self.fc1(combined))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            r_agg_mean, r_connectivity = raw_neighbor_agg_and_connectivity(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes)
            region_combined = torch.cat([h_region_pooled, r_agg_mean, r_connectivity], dim=-1)
            h_region = torch.relu(self.region_fc1(region_combined))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_final_headtohead"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 5, 4
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset, apply_recent_mask=True):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    if apply_recent_mask and GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

split_id_per_hour = np.where(years <= 2019, 0, np.where(years == 2020, 1, 2))
TRAIN_MASK = split_id_per_hour == 0
print(f"train hours (<=2019): {TRAIN_MASK.sum()}  val hours (2020): {(split_id_per_hour==1).sum()}  test hours (2021): {(split_id_per_hour==2).sum()}")

ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
region_weight_scale = region_train_nonzero.std()
region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
del train_nonzero, region_train_nonzero, edge_weight_by_hour_raw
gc.collect()

LOOKBACK = GRAPH_RECENT_HOURS
edge_roll_mean = pd.DataFrame(region_edge_weight_by_hour).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
pm25_roll_mean = pd.DataFrame(region_pm25).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
region_pm25_median_train = np.nanmedian(region_pm25[TRAIN_MASK], axis=0)
src_elevated = pm25_roll_mean[:, r_src_idx] > region_pm25_median_train[r_src_idx][None, :]
transport_signal_per_edge = np.where(src_elevated, edge_roll_mean, 0.0)
transport_score = np.zeros((n_time, n_regions), dtype=np.float32)
for r in range(n_regions):
    edge_mask = (r_dst_idx == r)
    transport_score[:, r] = np.nan_to_num(transport_signal_per_edge[:, edge_mask]).max(axis=1)
edge_threshold_train = np.percentile(region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0], 75)
transport_flag_per_region = transport_score > edge_threshold_train
del edge_roll_mean, pm25_roll_mean, src_elevated, transport_signal_per_edge, transport_score
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], [], []), 1: ([], [], [], [], []), 2: ([], [], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    input_end_t = t + WINDOW - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    Xl, yregl, yclsl, maskl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    yclsl.append(horizon_episode_label[t + WINDOW])
    maskl.append(transport_flag_per_region[input_end_t])
    sl.append(t)

X0, yreg0, ycls0, mask0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ycls1, mask1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ycls2, mask2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test(2021)={len(X2)}")
print(f"masked-in fraction: train={mask0.mean():.4f} val={mask1.mean():.4f} test={mask2.mean():.4f}")

# ---- FIXED: vectorized region pooling (matmul, not per-region boolean copies) ----
region_avg_matrix = region_membership / (region_membership.sum(axis=0, keepdims=True) + 1e-8)

def pool_to_region_features(X):
    pooled_by_station = np.einsum('bwsf,sr->bwrf', X, region_avg_matrix)
    return pooled_by_station.mean(axis=1)

def masked_flat(region_feat, ycls, mask):
    m = mask.astype(bool)
    return region_feat[m], ycls[m]

print("pooling region features for logistic regression...")
t0 = time.time()
X0_lr, y0_lr = masked_flat(pool_to_region_features(X0), ycls0, mask0)
X2_lr, y2_lr = masked_flat(pool_to_region_features(X2), ycls2, mask2)
print(f"done ({time.time()-t0:.0f}s)")

lr_cfg = BEST_CONFIG["logistic_regression"]
clf = LogisticRegression(C=lr_cfg["C"], max_iter=1000, class_weight="balanced")
clf.fit(X0_lr, y0_lr)
lr_proba = clf.predict_proba(X2_lr)[:, 1]
lr_result = {"model": "logistic_regression", "auc_roc": roc_auc_score(y2_lr, lr_proba),
             "auc_pr": average_precision_score(y2_lr, lr_proba)}
pred_pos = lr_proba >= 0.5
tp = int(np.sum(pred_pos & (y2_lr >= 0.5))); fp = int(np.sum(pred_pos & (y2_lr < 0.5))); fn = int(np.sum(~pred_pos & (y2_lr >= 0.5)))
lr_result["precision_05"] = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
lr_result["recall_05"] = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
lr_result["threshold_tuned"] = None
lr_result["precision_tuned"] = None
lr_result["recall_tuned"] = None
print(f"[FINAL logistic_regression] AUC-ROC={lr_result['auc_roc']:.4f} AUC-PR={lr_result['auc_pr']:.4f} P={lr_result['precision_05']:.4f} R={lr_result['recall_05']:.4f}")
del X0_lr, y0_lr, X2_lr, y2_lr, clf
gc.collect()

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ycls0_t, ycls1_t, ycls2_t = torch.tensor(ycls0, dtype=torch.float32), torch.tensor(ycls1, dtype=torch.float32), torch.tensor(ycls2, dtype=torch.float32)
mask0_t, mask1_t, mask2_t = torch.tensor(mask0, dtype=torch.float32), torch.tensor(mask1, dtype=torch.float32), torch.tensor(mask2, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1, ycls0, ycls1, ycls2, mask0, mask1, mask2, starts0, starts1, starts2
gc.collect()

masked_train_labels = ycls0_t[mask0_t.bool()]
POS_WEIGHT = min(float((masked_train_labels.numel() - masked_train_labels.sum()) / masked_train_labels.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")

edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)
del edge_weight_by_hour, region_edge_weight_by_hour
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT), reduction="none")

def run_p1_epoch(model, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def train_p1(name, model, lr, weight_decay):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        t0 = time.time()
        tl = run_p1_epoch(model, X0_t, yreg0_t, starts0_t, opt, True)
        vl = run_p1_epoch(model, X1_t, yreg1_t, starts1_t, opt, False)
        print(f"[P1 {name}] ep{epoch} train={tl:.4f} val={vl:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if vl < best_val:
            best_val, best_epoch = vl, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    print(f"[P1 {name}] BEST ep{best_epoch} val={best_val:.4f}")
    return model

def run_p2_epoch(model, X, y, m, starts, optimizer, train, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_maskn = 0.0, 0.0
    probs_list = [] if return_probs else None
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            mb = m[mb_idx].to(DEVICE)
            ew_s = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(region_edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss_pe = region_criterion(logits, yb)
                masked_loss = (loss_pe * mb).sum() / mb.sum().clamp(min=1)
            if train: masked_loss.backward()
            if return_probs: probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += masked_loss.item() * mb.sum().item(); total_maskn += mb.sum().item()
        if train: optimizer.step()
    avg_loss = total_loss / max(total_maskn, 1)
    if return_probs: return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

def train_p2(name, model, lr, weight_decay):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_auc_pr, best_epoch = -1.0, -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    y_val_masked = ycls1_t.numpy()[mask1_t.numpy().astype(bool)]
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        t0 = time.time()
        tl = run_p2_epoch(model, X0_t, ycls0_t, mask0_t, starts0_t, opt, True)
        vl, vp = run_p2_epoch(model, X1_t, ycls1_t, mask1_t, starts1_t, opt, False, True)
        vp_masked = vp[mask1_t.numpy().astype(bool)]
        va_auc_pr = average_precision_score(y_val_masked, vp_masked)
        print(f"[P2 {name}] ep{epoch} train={tl:.4f} val_loss={vl:.4f} val_AUCPR={va_auc_pr:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if va_auc_pr > best_val_auc_pr:
            best_val_auc_pr, best_epoch = va_auc_pr, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    print(f"[P2 {name}] BEST ep{best_epoch} val_AUCPR={best_val_auc_pr:.4f}")
    return model

@torch.no_grad()
def predict_test(model):
    model.eval()
    n = X2_t.shape[0]
    preds = []
    for start in range(0, n, MICRO_BATCH):
        xb = add_static_fn(X2_t[start:start + MICRO_BATCH].to(DEVICE), static_tensor)
        ew_s = gather_seq(edge_weight_by_hour_t, starts2_t[start:start + MICRO_BATCH]).to(DEVICE)
        ew_r = gather_seq(region_edge_weight_by_hour_t, starts2_t[start:start + MICRO_BATCH]).to(DEVICE)
        logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
        preds.append(torch.sigmoid(logits).cpu())
    return torch.cat(preds, dim=0).numpy()

MODEL_SEEDS = [0, 1]
ARCH_SPECS = [("wind_graph", StationQuantileGCN, RegionFromQuantileGCN),
              ("augmented_matched", StationQuantileMLP_Augmented, RegionFromQuantileMLP_Augmented),
              ("no_graph", StationQuantileMLP, RegionFromQuantileMLP)]

final_results = [lr_result]
for arch_name, StationCls, RegionCls in ARCH_SPECS:
    cfg = BEST_CONFIG[arch_name]
    lr, dropout, weight_decay = cfg["lr"], cfg["dropout"], cfg["weight_decay"]
    print(f"\n{'='*20} {arch_name}  (lr={lr:.0e} dropout={dropout} wd={weight_decay:.0e}) {'='*20}")
    probs_by_seed, val_probs_by_seed = [], []
    for model_seed in MODEL_SEEDS:
        torch.manual_seed(model_seed); np.random.seed(model_seed)
        p1 = train_p1(f"final_{arch_name}_m{model_seed}_p1", StationCls(n_feats, dropout), lr, weight_decay)
        if arch_name == "wind_graph":
            encoder_copy = copy.deepcopy(p1.station_conv)
            p2 = RegionCls(encoder_copy, region_membership_t, dropout).to(DEVICE)
        else:
            fc1_copy, fc2_copy = copy.deepcopy(p1.fc1), copy.deepcopy(p1.fc2)
            p2 = RegionCls(fc1_copy, fc2_copy, region_membership_t, dropout).to(DEVICE)
        del p1; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

        p2 = train_p2(f"final_{arch_name}_m{model_seed}_p2", p2, lr, weight_decay)
        probs_by_seed.append(predict_test(p2))
        p2.eval()
        with torch.no_grad():
            _, vp = run_p2_epoch(p2, X1_t, ycls1_t, mask1_t, starts1_t, None, False, True)
        val_probs_by_seed.append(vp)
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

    ensemble = np.mean(probs_by_seed, axis=0)
    val_ensemble = np.mean(val_probs_by_seed, axis=0)
    mask_test = mask2_t.numpy().astype(bool)
    mask_val = mask1_t.numpy().astype(bool)
    y_true = ycls2_t.numpy()[mask_test]
    y_val = ycls1_t.numpy()[mask_val]
    proba = ensemble[mask_test]
    proba_val = val_ensemble[mask_val]

    auc_roc = roc_auc_score(y_true, proba)
    auc_pr = average_precision_score(y_true, proba)

    pred_pos_05 = proba >= 0.5
    actual_pos = y_true >= 0.5
    tp05 = int(np.sum(pred_pos_05 & actual_pos)); fp05 = int(np.sum(pred_pos_05 & ~actual_pos)); fn05 = int(np.sum(~pred_pos_05 & actual_pos))
    precision_05 = tp05 / (tp05 + fp05) if (tp05 + fp05) > 0 else float("nan")
    recall_05 = tp05 / (tp05 + fn05) if (tp05 + fn05) > 0 else float("nan")

    prec_curve, rec_curve, thresh_curve = precision_recall_curve(y_val, proba_val)
    f1_curve = 2 * prec_curve[:-1] * rec_curve[:-1] / (prec_curve[:-1] + rec_curve[:-1] + 1e-12)
    best_thresh = thresh_curve[int(np.argmax(f1_curve))]
    pred_pos_tuned = proba >= best_thresh
    tp_t = int(np.sum(pred_pos_tuned & actual_pos)); fp_t = int(np.sum(pred_pos_tuned & ~actual_pos)); fn_t = int(np.sum(~pred_pos_tuned & actual_pos))
    precision_tuned = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else float("nan")
    recall_tuned = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else float("nan")

    print(f"\n[FINAL {arch_name}] AUC-ROC={auc_roc:.4f} AUC-PR={auc_pr:.4f}")
    print(f"  @0.5:    P={precision_05:.4f} R={recall_05:.4f}")
    print(f"  @tuned={best_thresh:.3f}: P={precision_tuned:.4f} R={recall_tuned:.4f}")
    final_results.append({"model": arch_name, "lr": lr, "dropout": dropout, "weight_decay": weight_decay,
                           "auc_roc": auc_roc, "auc_pr": auc_pr,
                           "precision_05": precision_05, "recall_05": recall_05,
                           "threshold_tuned": best_thresh, "precision_tuned": precision_tuned, "recall_tuned": recall_tuned})

results_df = pd.DataFrame(final_results)
results_df.to_csv(f"{BASE}/results_final_headtohead_2021.csv", index=False)
print("\n=== FINAL HEAD-TO-HEAD, 4 models, 2021 holdout ===")
print(results_df.to_string(index=False))


loaded tuned configs:
  logistic_regression: {'C': 0.01, 'val_aucpr': 0.30921876312403185, 'test_auc_roc': 0.839453602862475, 'test_auc_pr': 0.42886132968703805}
  wind_graph: {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.0005, 'val_aucpr': 0.5753801386622154, 'test_auc_roc': 0.9252028125349959, 'test_auc_pr': 0.7097272456251205}
  augmented_matched: {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.0005, 'val_aucpr': 0.5879353578861042, 'test_auc_roc': 0.9237375336712002, 'test_auc_pr': 0.7037623859115992}
  no_graph: {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005, 'val_aucpr': 0.5231158426080988, 'test_auc_roc': 0.9092271502247076, 'test_auc_pr': 0.6682175687661341}
station graph: 21414 directed edges, 1994 (9.3%) are 'close' (distance-only)
region graph: 272 directed edges (fully connected, 17 regions)
device=mps
train hours (<=2019): 35064  val hours (2020): 8784  test hours (2021): 8760
windows: train=34993 val=8713 test(2021)=8689
masked-in fraction: train=0.2480 val=0.1797

In [ ]:
# ================================================================
# FINAL HEAD-TO-HEAD: all 5 models, each using its own tuned config
# from hpsearch_best_config.json, at full rigor (5/4 epoch budget,
# 2-seed ensembling for the 4 neural architectures) on
# TRAIN=2016-2019, VAL=2020, TEST=2021 -- the same protocol as every
# other "real" comparison in this project, so these numbers are
# directly comparable to results already on record.
#
# Models: wind_graph, wind_graph_gat, augmented_matched, no_graph
# (all neural, tuned hyperparameters loaded per-architecture) plus
# logistic_regression (retrained on the full 2016-2019 train period
# with its winning C, region-pooled features, same masking).
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression

BASE = "/Users/drewbaldwin/PM2_5 Research"

with open(f"{BASE}/hpsearch_best_config.json") as f:
    BEST_CONFIG = json.load(f)
print("loaded tuned configs:")
for arch, cfg in BEST_CONFIG.items():
    print(f"  {arch}: {cfg}")

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)
print(f"station graph: {E} directed edges, {close_edge_mask.sum()} ({100*close_edge_mask.mean():.1f}%) are 'close' (distance-only)")

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

representative_station_idx = np.zeros(n_regions, dtype=int)
for r in range(n_regions):
    idxs = np.where(station_region_idx == r)[0]
    representative_station_idx[r] = idxs[np.argmin(dist_to_region[idxs, r])]

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)
print(f"region graph: {len(r_src_idx)} directed edges (fully connected, {n_regions} regions)")

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = wind_speed_arr[:, representative_station_idx].astype(np.float32)
region_wdir_sin = wdir_sin_station[:, representative_station_idx].astype(np.float32)
region_wdir_cos = wdir_cos_station[:, representative_station_idx].astype(np.float32)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(pm25_raw_arr)
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wdir_sin_station, wdir_cos_station, season_sin, season_cos, wind_dir_arr, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

rev2 = region_episode_label[::-1]
roll_max_rev = pd.DataFrame(rev2).rolling(window=HORIZON, min_periods=HORIZON).max().to_numpy()
horizon_episode_label = np.nan_to_num(roll_max_rev[::-1], nan=0.0).astype(np.float32)

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class WindGATLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin = nn.Linear(in_dim, out_dim)
        self.attn_src = nn.Linear(out_dim, 1)
        self.attn_dst = nn.Linear(out_dim, 1)
        self.lin_connectivity = nn.Linear(1, out_dim)
        self.prior_scale = nn.Parameter(torch.tensor(1.0))

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        h = self.lin(x)
        e = self.attn_src(h[src]).squeeze(-1) + self.attn_dst(h[dst]).squeeze(-1)
        e = F.leaky_relu(e, 0.2) + self.prior_scale * torch.log1p(edge_weight.clamp(min=0))
        e = e - e.max()
        exp_e = torch.exp(e)
        denom = x.new_zeros(num_nodes)
        denom.index_add_(0, dst, exp_e)
        alpha = exp_e / (denom[dst] + 1e-8)

        messages = h[src] * alpha.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, h.size(-1))
        agg_sum.index_add_(0, dst, messages)

        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)

        return self.lin_self(x) + agg_sum + self.lin_connectivity(connectivity)

def raw_neighbor_agg_and_connectivity(x, edge_index, edge_weight, num_nodes):
    src, dst = edge_index[0], edge_index[1]
    messages = x[src] * edge_weight.unsqueeze(-1)
    agg_sum = x.new_zeros(num_nodes, x.size(-1))
    agg_sum.index_add_(0, dst, messages)
    weight_sum = x.new_zeros(num_nodes)
    weight_sum.index_add_(0, dst, edge_weight)
    agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
    connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
    return agg_mean, connectivity

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileGAT(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindGATLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP_Augmented(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim * 2 + 1, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            agg_mean, connectivity = raw_neighbor_agg_and_connectivity(xt, ei_b, ew_b, num_nodes)
            combined = torch.cat([xt, agg_mean, connectivity], dim=-1)
            h = torch.relu(self.fc1(combined))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileGAT(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindGATLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            h_region = torch.relu(self.region_fc1(h_region_pooled))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP_Augmented(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden * 2 + 1, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            s_agg_mean, s_connectivity = raw_neighbor_agg_and_connectivity(xt, station_ei_b, ew_station_b, num_station_nodes)
            combined = torch.cat([xt, s_agg_mean, s_connectivity], dim=-1)
            h = torch.relu(self.fc1(combined))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            r_agg_mean, r_connectivity = raw_neighbor_agg_and_connectivity(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes)
            region_combined = torch.cat([h_region_pooled, r_agg_mean, r_connectivity], dim=-1)
            h_region = torch.relu(self.region_fc1(region_combined))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_final_headtohead"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 5, 4
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset, apply_recent_mask=True):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    if apply_recent_mask and GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

split_id_per_hour = np.where(years <= 2019, 0, np.where(years == 2020, 1, 2))
TRAIN_MASK = split_id_per_hour == 0
print(f"train hours (<=2019): {TRAIN_MASK.sum()}  val hours (2020): {(split_id_per_hour==1).sum()}  test hours (2021): {(split_id_per_hour==2).sum()}")

ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
region_weight_scale = region_train_nonzero.std()
region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
del train_nonzero, region_train_nonzero, edge_weight_by_hour_raw
gc.collect()

LOOKBACK = GRAPH_RECENT_HOURS
edge_roll_mean = pd.DataFrame(region_edge_weight_by_hour).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
pm25_roll_mean = pd.DataFrame(region_pm25).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
region_pm25_median_train = np.nanmedian(region_pm25[TRAIN_MASK], axis=0)
src_elevated = pm25_roll_mean[:, r_src_idx] > region_pm25_median_train[r_src_idx][None, :]
transport_signal_per_edge = np.where(src_elevated, edge_roll_mean, 0.0)
transport_score = np.zeros((n_time, n_regions), dtype=np.float32)
for r in range(n_regions):
    edge_mask = (r_dst_idx == r)
    transport_score[:, r] = np.nan_to_num(transport_signal_per_edge[:, edge_mask]).max(axis=1)
edge_threshold_train = np.percentile(region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0], 75)
transport_flag_per_region = transport_score > edge_threshold_train
del edge_roll_mean, pm25_roll_mean, src_elevated, transport_signal_per_edge, transport_score
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], [], []), 1: ([], [], [], [], []), 2: ([], [], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    input_end_t = t + WINDOW - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    Xl, yregl, yclsl, maskl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    yclsl.append(horizon_episode_label[t + WINDOW])
    maskl.append(transport_flag_per_region[input_end_t])
    sl.append(t)

X0, yreg0, ycls0, mask0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ycls1, mask1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ycls2, mask2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test(2021)={len(X2)}")
print(f"masked-in fraction: train={mask0.mean():.4f} val={mask1.mean():.4f} test={mask2.mean():.4f}")

# ---- logistic regression: retrain on full train (2016-2019) with tuned C, eval on 2021 ----
def pool_to_region_features(X):
    out = np.zeros((X.shape[0], n_regions, n_time_feats), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r, :] = X[:, :, cols, :].mean(axis=(1, 2))
    return out

def masked_flat(region_feat, ycls, mask):
    m = mask.astype(bool)
    return region_feat[m], ycls[m]

X0_lr, y0_lr = masked_flat(pool_to_region_features(X0), ycls0, mask0)
X2_lr, y2_lr = masked_flat(pool_to_region_features(X2), ycls2, mask2)
lr_cfg = BEST_CONFIG["logistic_regression"]
clf = LogisticRegression(C=lr_cfg["C"], max_iter=1000, class_weight="balanced")
clf.fit(X0_lr, y0_lr)
lr_proba = clf.predict_proba(X2_lr)[:, 1]
lr_result = {"model": "logistic_regression", "auc_roc": roc_auc_score(y2_lr, lr_proba),
             "auc_pr": average_precision_score(y2_lr, lr_proba)}
pred_pos = lr_proba >= 0.5
tp = int(np.sum(pred_pos & (y2_lr >= 0.5))); fp = int(np.sum(pred_pos & (y2_lr < 0.5))); fn = int(np.sum(~pred_pos & (y2_lr >= 0.5)))
lr_result["precision"] = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
lr_result["recall"] = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
print(f"[FINAL logistic_regression] AUC-ROC={lr_result['auc_roc']:.4f} AUC-PR={lr_result['auc_pr']:.4f} P={lr_result['precision']:.4f} R={lr_result['recall']:.4f}")
del X0_lr, y0_lr, X2_lr, y2_lr, clf
gc.collect()

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ycls0_t, ycls1_t, ycls2_t = torch.tensor(ycls0, dtype=torch.float32), torch.tensor(ycls1, dtype=torch.float32), torch.tensor(ycls2, dtype=torch.float32)
mask0_t, mask1_t, mask2_t = torch.tensor(mask0, dtype=torch.float32), torch.tensor(mask1, dtype=torch.float32), torch.tensor(mask2, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1, ycls0, ycls1, ycls2, mask0, mask1, mask2, starts0, starts1, starts2
gc.collect()

masked_train_labels = ycls0_t[mask0_t.bool()]
POS_WEIGHT = min(float((masked_train_labels.numel() - masked_train_labels.sum()) / masked_train_labels.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")

edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)
del edge_weight_by_hour, region_edge_weight_by_hour
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT), reduction="none")

def run_p1_epoch(model, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def train_p1(name, model, lr, weight_decay):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        t0 = time.time()
        tl = run_p1_epoch(model, X0_t, yreg0_t, starts0_t, opt, True)
        vl = run_p1_epoch(model, X1_t, yreg1_t, starts1_t, opt, False)
        print(f"[P1 {name}] ep{epoch} train={tl:.4f} val={vl:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if vl < best_val:
            best_val, best_epoch = vl, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    print(f"[P1 {name}] BEST ep{best_epoch} val={best_val:.4f}")
    return model

def run_p2_epoch(model, X, y, m, starts, optimizer, train, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_maskn = 0.0, 0.0
    probs_list = [] if return_probs else None
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            mb = m[mb_idx].to(DEVICE)
            ew_s = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(region_edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss_pe = region_criterion(logits, yb)
                masked_loss = (loss_pe * mb).sum() / mb.sum().clamp(min=1)
            if train: masked_loss.backward()
            if return_probs: probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += masked_loss.item() * mb.sum().item(); total_maskn += mb.sum().item()
        if train: optimizer.step()
    avg_loss = total_loss / max(total_maskn, 1)
    if return_probs: return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

def train_p2(name, model, lr, weight_decay):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_auc_pr, best_epoch = -1.0, -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    y_val_masked = ycls1_t.numpy()[mask1_t.numpy().astype(bool)]
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        t0 = time.time()
        tl = run_p2_epoch(model, X0_t, ycls0_t, mask0_t, starts0_t, opt, True)
        vl, vp = run_p2_epoch(model, X1_t, ycls1_t, mask1_t, starts1_t, opt, False, True)
        vp_masked = vp[mask1_t.numpy().astype(bool)]
        va_auc_pr = average_precision_score(y_val_masked, vp_masked)
        print(f"[P2 {name}] ep{epoch} train={tl:.4f} val_loss={vl:.4f} val_AUCPR={va_auc_pr:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if va_auc_pr > best_val_auc_pr:
            best_val_auc_pr, best_epoch = va_auc_pr, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    print(f"[P2 {name}] BEST ep{best_epoch} val_AUCPR={best_val_auc_pr:.4f}")
    return model

@torch.no_grad()
def predict_test(model):
    model.eval()
    n = X2_t.shape[0]
    preds = []
    for start in range(0, n, MICRO_BATCH):
        xb = add_static_fn(X2_t[start:start + MICRO_BATCH].to(DEVICE), static_tensor)
        ew_s = gather_seq(edge_weight_by_hour_t, starts2_t[start:start + MICRO_BATCH]).to(DEVICE)
        ew_r = gather_seq(region_edge_weight_by_hour_t, starts2_t[start:start + MICRO_BATCH]).to(DEVICE)
        logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
        preds.append(torch.sigmoid(logits).cpu())
    return torch.cat(preds, dim=0).numpy()

GRAPH_STATION_CONV_ARCHS = ("wind_graph", "wind_graph_gat")
MODEL_SEEDS = [0, 1]
ARCH_SPECS = [("wind_graph", StationQuantileGCN, RegionFromQuantileGCN),
              ("wind_graph_gat", StationQuantileGAT, RegionFromQuantileGAT),
              ("augmented_matched", StationQuantileMLP_Augmented, RegionFromQuantileMLP_Augmented),
              ("no_graph", StationQuantileMLP, RegionFromQuantileMLP)]

final_results = [lr_result]
for arch_name, StationCls, RegionCls in ARCH_SPECS:
    cfg = BEST_CONFIG[arch_name]
    lr, dropout, weight_decay = cfg["lr"], cfg["dropout"], cfg["weight_decay"]
    print(f"\n{'='*20} {arch_name}  (lr={lr:.0e} dropout={dropout} wd={weight_decay:.0e}) {'='*20}")
    probs_by_seed = []
    for model_seed in MODEL_SEEDS:
        torch.manual_seed(model_seed); np.random.seed(model_seed)
        p1 = train_p1(f"final_{arch_name}_m{model_seed}_p1", StationCls(n_feats, dropout), lr, weight_decay)
        if arch_name in GRAPH_STATION_CONV_ARCHS:
            encoder_copy = copy.deepcopy(p1.station_conv)
            p2 = RegionCls(encoder_copy, region_membership_t, dropout).to(DEVICE)
        else:
            fc1_copy, fc2_copy = copy.deepcopy(p1.fc1), copy.deepcopy(p1.fc2)
            p2 = RegionCls(fc1_copy, fc2_copy, region_membership_t, dropout).to(DEVICE)
        del p1; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

        p2 = train_p2(f"final_{arch_name}_m{model_seed}_p2", p2, lr, weight_decay)
        probs_by_seed.append(predict_test(p2))
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

    ensemble = np.mean(probs_by_seed, axis=0)
    mask_test = mask2_t.numpy().astype(bool)
    y_true = ycls2_t.numpy()[mask_test]
    proba = ensemble[mask_test]
    auc_roc = roc_auc_score(y_true, proba)
    auc_pr = average_precision_score(y_true, proba)
    pred_pos = proba >= 0.5
    actual_pos = y_true >= 0.5
    tp = int(np.sum(pred_pos & actual_pos)); fp = int(np.sum(pred_pos & ~actual_pos))
    fn = int(np.sum(~pred_pos & actual_pos))
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    print(f"\n[FINAL {arch_name}] AUC-ROC={auc_roc:.4f} AUC-PR={auc_pr:.4f} P={precision:.4f} R={recall:.4f}")
    final_results.append({"model": arch_name, "lr": lr, "dropout": dropout, "weight_decay": weight_decay,
                           "auc_roc": auc_roc, "auc_pr": auc_pr, "precision": precision, "recall": recall})

results_df = pd.DataFrame(final_results)
results_df.to_csv(f"{BASE}/results_final_headtohead_2021.csv", index=False)
print("\n=== FINAL HEAD-TO-HEAD, all 5 models, 2021 holdout ===")
print(results_df.to_string(index=False))


In [ ]:
# ================================================================
# FINAL TUNED HEAD-TO-HEAD: uses each architecture's OWN best
# hyperparameters found by the search (hpsearch_best_config.json),
# at full rigor -- 5/4 epoch budget, 2-seed ensembling -- on
# TRAIN=2016-2019, VAL=2020, TEST=2021. This is the comparison that
# actually counts; the search itself was just to find these configs.
#
# 2021 was never touched by the search (search only used
# 2016-2017/2018/2019), so this is a clean, independent evaluation --
# directly comparable to the untuned baselines already on record:
#   wind_graph (untuned):        AUC-ROC=0.776 AUC-PR=0.332 P=0.222 R=0.599
#   augmented_no_graph (untuned): AUC-ROC=0.768 AUC-PR=0.356 P=0.219 R=0.622
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/Users/drewbaldwin/PM2_5 Research"

with open(f"{BASE}/hpsearch_best_config.json") as f:
    BEST_CONFIG = json.load(f)
print("loaded tuned configs:")
for arch, cfg in BEST_CONFIG.items():
    print(f"  {arch}: lr={cfg['lr']:.0e} dropout={cfg['dropout']} weight_decay={cfg['weight_decay']:.0e}")

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)
print(f"station graph: {E} directed edges, {close_edge_mask.sum()} ({100*close_edge_mask.mean():.1f}%) are 'close' (distance-only)")

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

representative_station_idx = np.zeros(n_regions, dtype=int)
for r in range(n_regions):
    idxs = np.where(station_region_idx == r)[0]
    representative_station_idx[r] = idxs[np.argmin(dist_to_region[idxs, r])]

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)
print(f"region graph: {len(r_src_idx)} directed edges (fully connected, {n_regions} regions)")

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = wind_speed_arr[:, representative_station_idx].astype(np.float32)
region_wdir_sin = wdir_sin_station[:, representative_station_idx].astype(np.float32)
region_wdir_cos = wdir_cos_station[:, representative_station_idx].astype(np.float32)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(pm25_raw_arr)
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wdir_sin_station, wdir_cos_station, season_sin, season_cos, wind_dir_arr, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

rev2 = region_episode_label[::-1]
roll_max_rev = pd.DataFrame(rev2).rolling(window=HORIZON, min_periods=HORIZON).max().to_numpy()
horizon_episode_label = np.nan_to_num(roll_max_rev[::-1], nan=0.0).astype(np.float32)

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

def raw_neighbor_agg_and_connectivity(x, edge_index, edge_weight, num_nodes):
    src, dst = edge_index[0], edge_index[1]
    messages = x[src] * edge_weight.unsqueeze(-1)
    agg_sum = x.new_zeros(num_nodes, x.size(-1))
    agg_sum.index_add_(0, dst, messages)
    weight_sum = x.new_zeros(num_nodes)
    weight_sum.index_add_(0, dst, edge_weight)
    agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
    connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
    return agg_mean, connectivity

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP_Augmented(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim * 2 + 1, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            agg_mean, connectivity = raw_neighbor_agg_and_connectivity(xt, ei_b, ew_b, num_nodes)
            combined = torch.cat([xt, agg_mean, connectivity], dim=-1)
            h = torch.relu(self.fc1(combined))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP_Augmented(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden * 2 + 1, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            s_agg_mean, s_connectivity = raw_neighbor_agg_and_connectivity(xt, station_ei_b, ew_station_b, num_station_nodes)
            combined = torch.cat([xt, s_agg_mean, s_connectivity], dim=-1)
            h = torch.relu(self.fc1(combined))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            r_agg_mean, r_connectivity = raw_neighbor_agg_and_connectivity(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes)
            region_combined = torch.cat([h_region_pooled, r_agg_mean, r_connectivity], dim=-1)
            h_region = torch.relu(self.region_fc1(region_combined))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_final_tuned"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 5, 4
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset, apply_recent_mask=True):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    if apply_recent_mask and GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

split_id_per_hour = np.where(years <= 2019, 0, np.where(years == 2020, 1, 2))
TRAIN_MASK = split_id_per_hour == 0
print(f"train hours (<=2019): {TRAIN_MASK.sum()}  val hours (2020): {(split_id_per_hour==1).sum()}  test hours (2021): {(split_id_per_hour==2).sum()}")

ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
region_weight_scale = region_train_nonzero.std()
region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
del train_nonzero, region_train_nonzero, edge_weight_by_hour_raw
gc.collect()

LOOKBACK = GRAPH_RECENT_HOURS
edge_roll_mean = pd.DataFrame(region_edge_weight_by_hour).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
pm25_roll_mean = pd.DataFrame(region_pm25).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
region_pm25_median_train = np.nanmedian(region_pm25[TRAIN_MASK], axis=0)
src_elevated = pm25_roll_mean[:, r_src_idx] > region_pm25_median_train[r_src_idx][None, :]
transport_signal_per_edge = np.where(src_elevated, edge_roll_mean, 0.0)
transport_score = np.zeros((n_time, n_regions), dtype=np.float32)
for r in range(n_regions):
    edge_mask = (r_dst_idx == r)
    transport_score[:, r] = np.nan_to_num(transport_signal_per_edge[:, edge_mask]).max(axis=1)
edge_threshold_train = np.percentile(region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0], 75)
transport_flag_per_region = transport_score > edge_threshold_train
del edge_roll_mean, pm25_roll_mean, src_elevated, transport_signal_per_edge, transport_score
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], [], []), 1: ([], [], [], [], []), 2: ([], [], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    input_end_t = t + WINDOW - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    Xl, yregl, yclsl, maskl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    yclsl.append(horizon_episode_label[t + WINDOW])
    maskl.append(transport_flag_per_region[input_end_t])
    sl.append(t)

X0, yreg0, ycls0, mask0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ycls1, mask1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ycls2, mask2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test(2021)={len(X2)}")
print(f"masked-in fraction: train={mask0.mean():.4f} val={mask1.mean():.4f} test={mask2.mean():.4f}")
print(f"positive rate (horizon label): train={ycls0.mean():.4f} val={ycls1.mean():.4f} test={ycls2.mean():.4f}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ycls0_t, ycls1_t, ycls2_t = torch.tensor(ycls0, dtype=torch.float32), torch.tensor(ycls1, dtype=torch.float32), torch.tensor(ycls2, dtype=torch.float32)
mask0_t, mask1_t, mask2_t = torch.tensor(mask0, dtype=torch.float32), torch.tensor(mask1, dtype=torch.float32), torch.tensor(mask2, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1, ycls0, ycls1, ycls2, mask0, mask1, mask2, starts0, starts1, starts2
gc.collect()

masked_train_labels = ycls0_t[mask0_t.bool()]
POS_WEIGHT = min(float((masked_train_labels.numel() - masked_train_labels.sum()) / masked_train_labels.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")

edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)
del edge_weight_by_hour, region_edge_weight_by_hour
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT), reduction="none")

def run_p1_epoch(model, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def train_p1(name, model, lr, weight_decay):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        t0 = time.time()
        tl = run_p1_epoch(model, X0_t, yreg0_t, starts0_t, opt, True)
        vl = run_p1_epoch(model, X1_t, yreg1_t, starts1_t, opt, False)
        print(f"[P1 {name}] ep{epoch} train={tl:.4f} val={vl:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if vl < best_val:
            best_val, best_epoch = vl, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    print(f"[P1 {name}] BEST ep{best_epoch} val={best_val:.4f}")
    return model

def run_p2_epoch(model, X, y, m, starts, optimizer, train, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_maskn = 0.0, 0.0
    probs_list = [] if return_probs else None
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            mb = m[mb_idx].to(DEVICE)
            ew_s = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(region_edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss_pe = region_criterion(logits, yb)
                masked_loss = (loss_pe * mb).sum() / mb.sum().clamp(min=1)
            if train: masked_loss.backward()
            if return_probs: probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += masked_loss.item() * mb.sum().item(); total_maskn += mb.sum().item()
        if train: optimizer.step()
    avg_loss = total_loss / max(total_maskn, 1)
    if return_probs: return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

def train_p2(name, model, lr, weight_decay):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_auc_pr, best_epoch = -1.0, -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    y_val_masked = ycls1_t.numpy()[mask1_t.numpy().astype(bool)]
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        t0 = time.time()
        tl = run_p2_epoch(model, X0_t, ycls0_t, mask0_t, starts0_t, opt, True)
        vl, vp = run_p2_epoch(model, X1_t, ycls1_t, mask1_t, starts1_t, opt, False, True)
        vp_masked = vp[mask1_t.numpy().astype(bool)]
        va_auc_pr = average_precision_score(y_val_masked, vp_masked)
        print(f"[P2 {name}] ep{epoch} train={tl:.4f} val_loss={vl:.4f} val_AUCPR={va_auc_pr:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if va_auc_pr > best_val_auc_pr:
            best_val_auc_pr, best_epoch = va_auc_pr, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    print(f"[P2 {name}] BEST ep{best_epoch} val_AUCPR={best_val_auc_pr:.4f}")
    return model

@torch.no_grad()
def predict_test(model):
    model.eval()
    n = X2_t.shape[0]
    preds = []
    for start in range(0, n, MICRO_BATCH):
        xb = add_static_fn(X2_t[start:start + MICRO_BATCH].to(DEVICE), static_tensor)
        ew_s = gather_seq(edge_weight_by_hour_t, starts2_t[start:start + MICRO_BATCH]).to(DEVICE)
        ew_r = gather_seq(region_edge_weight_by_hour_t, starts2_t[start:start + MICRO_BATCH]).to(DEVICE)
        logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
        preds.append(torch.sigmoid(logits).cpu())
    return torch.cat(preds, dim=0).numpy()

MODEL_SEEDS = [0, 1]
ARCH_SPECS = [("wind_graph", StationQuantileGCN, RegionFromQuantileGCN),
              ("augmented_matched", StationQuantileMLP_Augmented, RegionFromQuantileMLP_Augmented)]

final_results = []
for arch_name, StationCls, RegionCls in ARCH_SPECS:
    cfg = BEST_CONFIG[arch_name]
    lr, dropout, weight_decay = cfg["lr"], cfg["dropout"], cfg["weight_decay"]
    print(f"\n{'='*20} {arch_name}  (lr={lr:.0e} dropout={dropout} wd={weight_decay:.0e}) {'='*20}")
    probs_by_seed = []
    for model_seed in MODEL_SEEDS:
        torch.manual_seed(model_seed); np.random.seed(model_seed)
        p1 = train_p1(f"final_{arch_name}_m{model_seed}_p1", StationCls(n_feats, dropout), lr, weight_decay)
        if arch_name == "wind_graph":
            encoder_copy = copy.deepcopy(p1.station_conv)
            p2 = RegionCls(encoder_copy, region_membership_t, dropout).to(DEVICE)
        else:
            fc1_copy, fc2_copy = copy.deepcopy(p1.fc1), copy.deepcopy(p1.fc2)
            p2 = RegionCls(fc1_copy, fc2_copy, region_membership_t, dropout).to(DEVICE)
        del p1; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

        p2 = train_p2(f"final_{arch_name}_m{model_seed}_p2", p2, lr, weight_decay)
        probs_by_seed.append(predict_test(p2))
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

    ensemble = np.mean(probs_by_seed, axis=0)
    mask_test = mask2_t.numpy().astype(bool)
    y_true = ycls2_t.numpy()[mask_test]
    proba = ensemble[mask_test]
    auc_roc = roc_auc_score(y_true, proba)
    auc_pr = average_precision_score(y_true, proba)
    pred_pos = proba >= 0.5
    actual_pos = y_true >= 0.5
    tp = int(np.sum(pred_pos & actual_pos)); fp = int(np.sum(pred_pos & ~actual_pos))
    fn = int(np.sum(~pred_pos & actual_pos))
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    print(f"\n[FINAL TUNED {arch_name}] AUC-ROC={auc_roc:.4f} AUC-PR={auc_pr:.4f} P={precision:.4f} R={recall:.4f}")
    final_results.append({"model": arch_name, "lr": lr, "dropout": dropout, "weight_decay": weight_decay,
                           "auc_roc": auc_roc, "auc_pr": auc_pr, "precision": precision, "recall": recall})

results_df = pd.DataFrame(final_results)
results_df.to_csv(f"{BASE}/results_final_tuned_2021.csv", index=False)
print("\n=== FINAL TUNED comparison, 2021 holdout ===")
print(results_df.to_string(index=False))
print("\n=== vs untuned baselines (from earlier run) ===")
print("wind_graph (untuned):         AUC-ROC=0.7762 AUC-PR=0.3321 P=0.2217 R=0.5991")
print("augmented_no_graph (untuned): AUC-ROC=0.7680 AUC-PR=0.3559 P=0.2190 R=0.6223")


In [3]:
# ================================================================
# Adds 4 more splits (seeds 1-4) to the hybrid-edge run, appending to
# the existing all_split_results (which already has split 0), for a
# 5-split paired comparison against no_graph -- higher-powered than
# the original 3-split CV. Reuses everything already built in memory
# (graph structures, wind_component, time_arr_raw, static_arr, model
# classes, etc.) -- only run this in the SAME kernel session as the
# hybrid-edge script that already produced split 0's results.
# ================================================================
from scipy import stats

SPLIT_SEEDS_NEW = [1, 2, 3, 4]

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

for split_seed in SPLIT_SEEDS_NEW:
    print(f"\n{'='*20} SPLIT {split_seed} {'='*20}")
    year_offset = years - years.min()
    block_id = year_offset * 12 + (months_arr - 1)
    n_blocks = int(block_id.max()) + 1
    rng = np.random.RandomState(split_seed)
    block_order = rng.permutation(n_blocks)
    n_train_blocks = int(round(0.6 * n_blocks))
    n_val_blocks = int(round(0.2 * n_blocks))
    block_to_split = np.empty(n_blocks, dtype=int)
    block_to_split[block_order[:n_train_blocks]] = 0
    block_to_split[block_order[n_train_blocks:n_train_blocks + n_val_blocks]] = 1
    block_to_split[block_order[n_train_blocks + n_val_blocks:]] = 2
    split_id_per_hour = block_to_split[block_id]
    TRAIN_MASK = split_id_per_hour == 0

    ref_speed = np.nanmean(wind_speed_arr[TRAIN_MASK])
    print(f"hybrid close-edge reference speed (train-period mean): {ref_speed:.3f}")
    component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
    edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
    del component
    gc.collect()

    train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
    weight_scale = train_nonzero.std()
    edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
    region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
    region_weight_scale = region_train_nonzero.std()
    region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
    del train_nonzero, region_train_nonzero, edge_weight_by_hour_raw
    gc.collect()

    LOOKBACK = GRAPH_RECENT_HOURS
    edge_roll_mean = pd.DataFrame(region_edge_weight_by_hour).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
    pm25_roll_mean = pd.DataFrame(region_pm25).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
    region_pm25_median_train = np.nanmedian(region_pm25[TRAIN_MASK], axis=0)
    src_elevated = pm25_roll_mean[:, r_src_idx] > region_pm25_median_train[r_src_idx][None, :]
    transport_signal_per_edge = np.where(src_elevated, edge_roll_mean, 0.0)
    transport_score = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        edge_mask = (r_dst_idx == r)
        transport_score[:, r] = np.nan_to_num(transport_signal_per_edge[:, edge_mask]).max(axis=1)
    edge_threshold_train = np.percentile(region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0], 75)
    transport_flag_per_region = transport_score > edge_threshold_train
    del edge_roll_mean, pm25_roll_mean, src_elevated, transport_signal_per_edge, transport_score
    gc.collect()

    t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
    t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
    time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
    s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
    static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

    p1_buckets = {0: ([], [], []), 1: ([], [], [])}
    p2_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
    for t in range(0, n_time - WINDOW - HORIZON + 1):
        target_t = t + WINDOW + HORIZON - 1
        input_end_t = t + WINDOW - 1
        s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
        if s_start != s_target:
            continue
        x_win = time_arr_std[t:t + WINDOW]
        if s_start in p1_buckets:
            Xl, yreg_l, sl = p1_buckets[s_start]
            Xl.append(x_win); yreg_l.append(time_arr_std[target_t, :, pm25_col_idx]); sl.append(t)
        Xl2, ycls_l2, mask_l2, sl2 = p2_buckets[s_start]
        Xl2.append(x_win); ycls_l2.append(region_episode_label[target_t])
        mask_l2.append(transport_flag_per_region[input_end_t]); sl2.append(t)

    X_train, yreg_train, starts_train = (np.stack(v) for v in p1_buckets[0])
    X_val, yreg_val, starts_val = (np.stack(v) for v in p1_buckets[1])
    X2_train, ycls2_train, mask2_train, starts2_train = (np.stack(v) for v in p2_buckets[0])
    X2_val, ycls2_val, mask2_val, starts2_val = (np.stack(v) for v in p2_buckets[1])
    X2_test, ycls2_test, mask2_test, starts2_test = (np.stack(v) for v in p2_buckets[2])
    del p1_buckets, p2_buckets, time_arr_std
    gc.collect()
    print(f"windows: phase1 train={len(X_train)} val={len(X_val)}  phase2 train={len(X2_train)} val={len(X2_val)} test={len(X2_test)}")
    print(f"masked-in fraction: train={mask2_train.mean():.4f} val={mask2_val.mean():.4f} test={mask2_test.mean():.4f}")

    Xtr_t, ytr_reg_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(yreg_train, dtype=torch.float32)
    Xva_t, yva_reg_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(yreg_val, dtype=torch.float32)
    starts_train_t, starts_val_t = torch.tensor(starts_train, dtype=torch.long), torch.tensor(starts_val, dtype=torch.long)
    del X_train, X_val, yreg_train, yreg_val, starts_train, starts_val
    gc.collect()

    X2tr_t, y2tr_cls_t, m2tr_t = torch.tensor(X2_train, dtype=torch.float32), torch.tensor(ycls2_train, dtype=torch.float32), torch.tensor(mask2_train, dtype=torch.float32)
    X2va_t, y2va_cls_t, m2va_t = torch.tensor(X2_val, dtype=torch.float32), torch.tensor(ycls2_val, dtype=torch.float32), torch.tensor(mask2_val, dtype=torch.float32)
    X2te_t, y2te_cls_t, m2te_t = torch.tensor(X2_test, dtype=torch.float32), torch.tensor(ycls2_test, dtype=torch.float32), torch.tensor(mask2_test, dtype=torch.float32)
    starts2_train_t, starts2_val_t, starts2_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts2_train, starts2_val, starts2_test))
    del X2_train, X2_val, X2_test, ycls2_train, ycls2_val, ycls2_test, mask2_train, mask2_val, mask2_test, starts2_train, starts2_val, starts2_test
    gc.collect()

    masked_train_labels = y2tr_cls_t[m2tr_t.bool()]
    POS_WEIGHT = min(float((masked_train_labels.numel() - masked_train_labels.sum()) / masked_train_labels.sum().clamp(min=1)), 50.0)
    print(f"pos_weight: {POS_WEIGHT:.2f}")

    edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
    region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)
    del edge_weight_by_hour, region_edge_weight_by_hour
    gc.collect()

    region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT), reduction="none")

    def run_p1_epoch(model, use_graph, X, y, starts, optimizer, train):
        n = X.shape[0]
        idx = torch.randperm(n) if train else torch.arange(n)
        model.train(train)
        total_loss, total_n = 0.0, 0
        eff_batch = MICRO_BATCH * ACCUM_STEPS
        for start in range(0, n, eff_batch):
            if train: optimizer.zero_grad()
            batch_idx = idx[start:start + eff_batch]
            for ms in range(0, len(batch_idx), MICRO_BATCH):
                mb_idx = batch_idx[ms:ms + MICRO_BATCH]
                if len(mb_idx) == 0: continue
                xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
                yb = y[mb_idx].to(DEVICE)
                with torch.set_grad_enabled(train):
                    if use_graph:
                        ew_seq = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
                        pred = model(xb, edge_index, ew_seq)
                    else:
                        pred = model(xb)
                    loss = pinball_loss(pred, yb, QUANTILES)
                if train: (loss * len(mb_idx) / len(batch_idx)).backward()
                total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
            if train: optimizer.step()
        return total_loss / total_n

    def train_p1(name, model, use_graph):
        model = model.to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY_P1)
        best_val, best_epoch = float("inf"), -1
        ckpt_path = f"{CKPT_DIR}/{name}.pt"
        for epoch in range(1, MAX_EPOCHS_P1 + 1):
            t0 = time.time()
            tl = run_p1_epoch(model, use_graph, Xtr_t, ytr_reg_t, starts_train_t, opt, True)
            vl = run_p1_epoch(model, use_graph, Xva_t, yva_reg_t, starts_val_t, opt, False)
            print(f"[P1 {name}] ep{epoch} train={tl:.4f} val={vl:.4f} ({time.time()-t0:.0f}s)", flush=True)
            if vl < best_val:
                best_val, best_epoch = vl, epoch
                torch.save(model.state_dict(), ckpt_path)
        model.load_state_dict(torch.load(ckpt_path))
        print(f"[P1 {name}] BEST ep{best_epoch} val={best_val:.4f}")
        return model

    def run_p2_epoch(model, use_graph, X, y, m, starts, optimizer, train, return_probs=False):
        n = X.shape[0]
        idx = torch.randperm(n) if train else torch.arange(n)
        model.train(train)
        total_loss, total_maskn = 0.0, 0.0
        probs_list = [] if return_probs else None
        eff_batch = MICRO_BATCH * ACCUM_STEPS
        for start in range(0, n, eff_batch):
            if train: optimizer.zero_grad()
            batch_idx = idx[start:start + eff_batch]
            for ms in range(0, len(batch_idx), MICRO_BATCH):
                mb_idx = batch_idx[ms:ms + MICRO_BATCH]
                if len(mb_idx) == 0: continue
                xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
                yb = y[mb_idx].to(DEVICE)
                mb = m[mb_idx].to(DEVICE)
                with torch.set_grad_enabled(train):
                    if use_graph:
                        ew_s = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
                        ew_r = gather_seq(region_edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
                        logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                    else:
                        logits = model(xb, n_regions)
                    loss_pe = region_criterion(logits, yb)
                    masked_loss = (loss_pe * mb).sum() / mb.sum().clamp(min=1)
                if train: masked_loss.backward()
                if return_probs: probs_list.append(torch.sigmoid(logits).detach().cpu())
                total_loss += masked_loss.item() * mb.sum().item(); total_maskn += mb.sum().item()
            if train: optimizer.step()
        avg_loss = total_loss / max(total_maskn, 1)
        if return_probs: return avg_loss, torch.cat(probs_list, dim=0).numpy()
        return avg_loss

    def train_p2(name, model, use_graph):
        model = model.to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY_P2)
        best_val_auc_pr, best_epoch = -1.0, -1
        ckpt_path = f"{CKPT_DIR}/{name}.pt"
        y_val_masked = y2va_cls_t.numpy()[m2va_t.numpy().astype(bool)]
        for epoch in range(1, MAX_EPOCHS_P2 + 1):
            t0 = time.time()
            tl = run_p2_epoch(model, use_graph, X2tr_t, y2tr_cls_t, m2tr_t, starts2_train_t, opt, True)
            vl, vp = run_p2_epoch(model, use_graph, X2va_t, y2va_cls_t, m2va_t, starts2_val_t, opt, False, True)
            vp_masked = vp[m2va_t.numpy().astype(bool)]
            va_auc_pr = average_precision_score(y_val_masked, vp_masked)
            print(f"[P2 {name}] ep{epoch} train={tl:.4f} val_loss={vl:.4f} val_AUCPR={va_auc_pr:.4f} ({time.time()-t0:.0f}s)", flush=True)
            if va_auc_pr > best_val_auc_pr:
                best_val_auc_pr, best_epoch = va_auc_pr, epoch
                torch.save(model.state_dict(), ckpt_path)
        model.load_state_dict(torch.load(ckpt_path))
        print(f"[P2 {name}] BEST ep{best_epoch} val_AUCPR={best_val_auc_pr:.4f}")
        return model

    @torch.no_grad()
    def predict_test(model, use_graph):
        model.eval()
        n = X2te_t.shape[0]
        preds = []
        for start in range(0, n, MICRO_BATCH):
            xb = add_static_fn(X2te_t[start:start + MICRO_BATCH].to(DEVICE), static_tensor)
            if use_graph:
                ew_s = gather_seq(edge_weight_by_hour_t, starts2_test_t[start:start + MICRO_BATCH]).to(DEVICE)
                ew_r = gather_seq(region_edge_weight_by_hour_t, starts2_test_t[start:start + MICRO_BATCH]).to(DEVICE)
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            else:
                logits = model(xb, n_regions)
            preds.append(torch.sigmoid(logits).cpu())
        return torch.cat(preds, dim=0).numpy()

    wind_probs_by_seed, nograph_probs_by_seed = [], []
    for model_seed in MODEL_SEEDS:
        torch.manual_seed(model_seed); np.random.seed(model_seed)
        p1w = train_p1(f"s{split_seed}_m{model_seed}_p1_gcn", StationQuantileGCN(n_feats), True)
        torch.manual_seed(model_seed); np.random.seed(model_seed)
        p1n = train_p1(f"s{split_seed}_m{model_seed}_p1_mlp", StationQuantileMLP(n_feats), False)
        sc_copy = copy.deepcopy(p1w.station_conv)
        fc1_copy, fc2_copy = copy.deepcopy(p1n.fc1), copy.deepcopy(p1n.fc2)
        del p1w, p1n; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

        p2w = RegionFromQuantileGCN(sc_copy, region_membership_t).to(DEVICE)
        p2w = train_p2(f"s{split_seed}_m{model_seed}_p2_gcn", p2w, True)
        wind_probs_by_seed.append(predict_test(p2w, True))

        p2n = RegionFromQuantileMLP(fc1_copy, fc2_copy, region_membership_t).to(DEVICE)
        p2n = train_p2(f"s{split_seed}_m{model_seed}_p2_mlp", p2n, False)
        nograph_probs_by_seed.append(predict_test(p2n, False))
        del p2w, p2n; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

    wind_ensemble = np.mean(wind_probs_by_seed, axis=0)
    nograph_ensemble = np.mean(nograph_probs_by_seed, axis=0)
    mask_test = m2te_t.numpy().astype(bool)
    y_true = y2te_cls_t.numpy()[mask_test]

    for name, proba in [("wind_graph_hybrid", wind_ensemble[mask_test]), ("no_graph", nograph_ensemble[mask_test])]:
        auc_roc = roc_auc_score(y_true, proba)
        auc_pr = average_precision_score(y_true, proba)
        pred_pos = proba >= 0.5
        actual_pos = y_true >= 0.5
        tp = int(np.sum(pred_pos & actual_pos)); fp = int(np.sum(pred_pos & ~actual_pos))
        fn = int(np.sum(~pred_pos & actual_pos))
        precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
        recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        print(f"[SPLIT {split_seed} ENSEMBLE {name}] AUC-ROC={auc_roc:.4f} AUC-PR={auc_pr:.4f} P={precision:.4f} R={recall:.4f}")
        all_split_results.append({"split": split_seed, "model": name, "auc_roc": auc_roc, "auc_pr": auc_pr, "precision": precision, "recall": recall})

    del Xtr_t, Xva_t, ytr_reg_t, yva_reg_t, starts_train_t, starts_val_t
    del X2tr_t, y2tr_cls_t, m2tr_t, X2va_t, y2va_cls_t, m2va_t, X2te_t, y2te_cls_t, m2te_t
    del starts2_train_t, starts2_val_t, starts2_test_t, edge_weight_by_hour_t, region_edge_weight_by_hour_t, static_tensor
    gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# Final aggregation across all 5 splits (0-4) + paired t-test vs
# no_graph, same style as the original validated CV comparison.
# ================================================================
results_df = pd.DataFrame(all_split_results)
print("\n" + "="*20 + " HYBRID-EDGE FINAL RESULTS (5 splits) " + "="*20)
print(results_df.to_string(index=False))
print("\n=== aggregated across splits ===")
print(results_df.groupby("model")[["auc_roc", "auc_pr", "precision", "recall"]].agg(["mean", "std"]))
print("\n=== paired comparison across splits ===")
n_splits = results_df["split"].nunique()
for metric in ["auc_roc", "auc_pr", "recall", "precision"]:
    pivot = results_df.pivot(index="split", columns="model", values=metric)
    t, p = stats.ttest_rel(pivot["wind_graph_hybrid"], pivot["no_graph"])
    wins = int((pivot["wind_graph_hybrid"] > pivot["no_graph"]).sum())
    print(f"[{metric}] wind_hybrid={pivot['wind_graph_hybrid'].mean():.4f} no_graph={pivot['no_graph'].mean():.4f} t={t:.3f} p={p:.4f} wind_wins={wins}/{n_splits}")



==================== SPLIT 1 ====================
hybrid close-edge reference speed (train-period mean): 9.930
windows: phase1 train=30163 val=9609  phase2 train=30163 val=9609 test=9925
masked-in fraction: train=0.2442 val=0.3019 test=0.3284
pos_weight: 50.00


KeyboardInterrupt: 

In [1]:
# ================================================================
# Cross-validation across 3 independent splits x 2 model seeds each,
# with seed-ensembling within each split.
#
# FIX from last crash: the numpy edge-weight arrays (station-level
# ~4.5GB) were being implicitly overwritten each split iteration
# rather than explicitly freed, so for a brief window both the old
# and new copies existed simultaneously -- across 3 splits that was
# enough to exhaust memory. Now explicitly deleted right after their
# torch tensor copies are made, same for the transient train_nonzero/
# region_train_nonzero subset arrays.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"station graph: {E} directed edges")

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

representative_station_idx = np.zeros(n_regions, dtype=int)
for r in range(n_regions):
    idxs = np.where(station_region_idx == r)[0]
    representative_station_idx[r] = idxs[np.argmin(dist_to_region[idxs, r])]

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)
print(f"region graph: {len(r_src_idx)} directed edges (fully connected, {n_regions} regions)")

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months_arr = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = wind_speed_arr[:, representative_station_idx].astype(np.float32)
region_wdir_sin = wdir_sin_station[:, representative_station_idx].astype(np.float32)
region_wdir_cos = wdir_cos_station[:, representative_station_idx].astype(np.float32)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(pm25_raw_arr)
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wind_speed_arr, wind_dir_arr, blh_arr, wdir_sin_station, wdir_cos_station, season_sin, season_cos
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, hidden=32, gru_hidden=32, dropout=0.5):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, hidden=32, gru_hidden=32, dropout=0.5):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            h_region = torch.relu(self.region_fc1(h_region_pooled))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_transport_test_cv"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 5, 4
WEIGHT_DECAY_P1, WEIGHT_DECAY_P2 = 1e-4, 5e-4
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset, apply_recent_mask=True):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    if apply_recent_mask and GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

SPLIT_SEEDS = [0, 1, 2]
MODEL_SEEDS = [0, 1]
all_split_results = []

for split_seed in SPLIT_SEEDS:
    print(f"\n{'='*20} SPLIT {split_seed} {'='*20}")
    year_offset = years - years.min()
    block_id = year_offset * 12 + (months_arr - 1)
    n_blocks = int(block_id.max()) + 1
    rng = np.random.RandomState(split_seed)
    block_order = rng.permutation(n_blocks)
    n_train_blocks = int(round(0.6 * n_blocks))
    n_val_blocks = int(round(0.2 * n_blocks))
    block_to_split = np.empty(n_blocks, dtype=int)
    block_to_split[block_order[:n_train_blocks]] = 0
    block_to_split[block_order[n_train_blocks:n_train_blocks + n_val_blocks]] = 1
    block_to_split[block_order[n_train_blocks + n_val_blocks:]] = 2
    split_id_per_hour = block_to_split[block_id]
    TRAIN_MASK = split_id_per_hour == 0

    train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
    weight_scale = train_nonzero.std()
    edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
    region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
    region_weight_scale = region_train_nonzero.std()
    region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
    del train_nonzero, region_train_nonzero
    gc.collect()

    LOOKBACK = GRAPH_RECENT_HOURS
    edge_roll_mean = pd.DataFrame(region_edge_weight_by_hour).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
    pm25_roll_mean = pd.DataFrame(region_pm25).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
    region_pm25_median_train = np.nanmedian(region_pm25[TRAIN_MASK], axis=0)
    src_elevated = pm25_roll_mean[:, r_src_idx] > region_pm25_median_train[r_src_idx][None, :]
    transport_signal_per_edge = np.where(src_elevated, edge_roll_mean, 0.0)
    transport_score = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        edge_mask = (r_dst_idx == r)
        transport_score[:, r] = np.nan_to_num(transport_signal_per_edge[:, edge_mask]).max(axis=1)
    edge_threshold_train = np.percentile(region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0], 75)
    transport_flag_per_region = transport_score > edge_threshold_train
    del edge_roll_mean, pm25_roll_mean, src_elevated, transport_signal_per_edge, transport_score
    gc.collect()

    t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
    t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
    time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
    s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
    static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

    p1_buckets = {0: ([], [], []), 1: ([], [], [])}
    p2_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
    for t in range(0, n_time - WINDOW - HORIZON + 1):
        target_t = t + WINDOW + HORIZON - 1
        input_end_t = t + WINDOW - 1
        s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
        if s_start != s_target:
            continue
        x_win = time_arr_std[t:t + WINDOW]
        if s_start in p1_buckets:
            Xl, yreg_l, sl = p1_buckets[s_start]
            Xl.append(x_win); yreg_l.append(time_arr_std[target_t, :, pm25_col_idx]); sl.append(t)
        Xl2, ycls_l2, mask_l2, sl2 = p2_buckets[s_start]
        Xl2.append(x_win); ycls_l2.append(region_episode_label[target_t])
        mask_l2.append(transport_flag_per_region[input_end_t]); sl2.append(t)

    X_train, yreg_train, starts_train = (np.stack(v) for v in p1_buckets[0])
    X_val, yreg_val, starts_val = (np.stack(v) for v in p1_buckets[1])
    X2_train, ycls2_train, mask2_train, starts2_train = (np.stack(v) for v in p2_buckets[0])
    X2_val, ycls2_val, mask2_val, starts2_val = (np.stack(v) for v in p2_buckets[1])
    X2_test, ycls2_test, mask2_test, starts2_test = (np.stack(v) for v in p2_buckets[2])
    del p1_buckets, p2_buckets, time_arr_std
    gc.collect()
    print(f"windows: phase1 train={len(X_train)} val={len(X_val)}  phase2 train={len(X2_train)} val={len(X2_val)} test={len(X2_test)}")
    print(f"masked-in fraction: train={mask2_train.mean():.4f} val={mask2_val.mean():.4f} test={mask2_test.mean():.4f}")

    Xtr_t, ytr_reg_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(yreg_train, dtype=torch.float32)
    Xva_t, yva_reg_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(yreg_val, dtype=torch.float32)
    starts_train_t, starts_val_t = torch.tensor(starts_train, dtype=torch.long), torch.tensor(starts_val, dtype=torch.long)
    del X_train, X_val, yreg_train, yreg_val, starts_train, starts_val
    gc.collect()

    X2tr_t, y2tr_cls_t, m2tr_t = torch.tensor(X2_train, dtype=torch.float32), torch.tensor(ycls2_train, dtype=torch.float32), torch.tensor(mask2_train, dtype=torch.float32)
    X2va_t, y2va_cls_t, m2va_t = torch.tensor(X2_val, dtype=torch.float32), torch.tensor(ycls2_val, dtype=torch.float32), torch.tensor(mask2_val, dtype=torch.float32)
    X2te_t, y2te_cls_t, m2te_t = torch.tensor(X2_test, dtype=torch.float32), torch.tensor(ycls2_test, dtype=torch.float32), torch.tensor(mask2_test, dtype=torch.float32)
    starts2_train_t, starts2_val_t, starts2_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts2_train, starts2_val, starts2_test))
    del X2_train, X2_val, X2_test, ycls2_train, ycls2_val, ycls2_test, mask2_train, mask2_val, mask2_test, starts2_train, starts2_val, starts2_test
    gc.collect()

    masked_train_labels = y2tr_cls_t[m2tr_t.bool()]
    POS_WEIGHT = min(float((masked_train_labels.numel() - masked_train_labels.sum()) / masked_train_labels.sum().clamp(min=1)), 50.0)
    print(f"pos_weight: {POS_WEIGHT:.2f}")

    edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
    region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)
    del edge_weight_by_hour, region_edge_weight_by_hour
    gc.collect()

    region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT), reduction="none")

    def run_p1_epoch(model, use_graph, X, y, starts, optimizer, train):
        n = X.shape[0]
        idx = torch.randperm(n) if train else torch.arange(n)
        model.train(train)
        total_loss, total_n = 0.0, 0
        eff_batch = MICRO_BATCH * ACCUM_STEPS
        for start in range(0, n, eff_batch):
            if train: optimizer.zero_grad()
            batch_idx = idx[start:start + eff_batch]
            for ms in range(0, len(batch_idx), MICRO_BATCH):
                mb_idx = batch_idx[ms:ms + MICRO_BATCH]
                if len(mb_idx) == 0: continue
                xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
                yb = y[mb_idx].to(DEVICE)
                with torch.set_grad_enabled(train):
                    if use_graph:
                        ew_seq = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
                        pred = model(xb, edge_index, ew_seq)
                    else:
                        pred = model(xb)
                    loss = pinball_loss(pred, yb, QUANTILES)
                if train: (loss * len(mb_idx) / len(batch_idx)).backward()
                total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
            if train: optimizer.step()
        return total_loss / total_n

    def train_p1(name, model, use_graph):
        model = model.to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY_P1)
        best_val, best_epoch = float("inf"), -1
        ckpt_path = f"{CKPT_DIR}/{name}.pt"
        for epoch in range(1, MAX_EPOCHS_P1 + 1):
            t0 = time.time()
            tl = run_p1_epoch(model, use_graph, Xtr_t, ytr_reg_t, starts_train_t, opt, True)
            vl = run_p1_epoch(model, use_graph, Xva_t, yva_reg_t, starts_val_t, opt, False)
            print(f"[P1 {name}] ep{epoch} train={tl:.4f} val={vl:.4f} ({time.time()-t0:.0f}s)", flush=True)
            if vl < best_val:
                best_val, best_epoch = vl, epoch
                torch.save(model.state_dict(), ckpt_path)
        model.load_state_dict(torch.load(ckpt_path))
        print(f"[P1 {name}] BEST ep{best_epoch} val={best_val:.4f}")
        return model

    def run_p2_epoch(model, use_graph, X, y, m, starts, optimizer, train, return_probs=False):
        n = X.shape[0]
        idx = torch.randperm(n) if train else torch.arange(n)
        model.train(train)
        total_loss, total_maskn = 0.0, 0.0
        probs_list = [] if return_probs else None
        eff_batch = MICRO_BATCH * ACCUM_STEPS
        for start in range(0, n, eff_batch):
            if train: optimizer.zero_grad()
            batch_idx = idx[start:start + eff_batch]
            for ms in range(0, len(batch_idx), MICRO_BATCH):
                mb_idx = batch_idx[ms:ms + MICRO_BATCH]
                if len(mb_idx) == 0: continue
                xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
                yb = y[mb_idx].to(DEVICE)
                mb = m[mb_idx].to(DEVICE)
                with torch.set_grad_enabled(train):
                    if use_graph:
                        ew_s = gather_seq(edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
                        ew_r = gather_seq(region_edge_weight_by_hour_t, starts[mb_idx]).to(DEVICE)
                        logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                    else:
                        logits = model(xb, n_regions)
                    loss_pe = region_criterion(logits, yb)
                    masked_loss = (loss_pe * mb).sum() / mb.sum().clamp(min=1)
                if train: masked_loss.backward()
                if return_probs: probs_list.append(torch.sigmoid(logits).detach().cpu())
                total_loss += masked_loss.item() * mb.sum().item(); total_maskn += mb.sum().item()
            if train: optimizer.step()
        avg_loss = total_loss / max(total_maskn, 1)
        if return_probs: return avg_loss, torch.cat(probs_list, dim=0).numpy()
        return avg_loss

    def train_p2(name, model, use_graph):
        model = model.to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY_P2)
        best_val_auc_pr, best_epoch = -1.0, -1
        ckpt_path = f"{CKPT_DIR}/{name}.pt"
        y_val_masked = y2va_cls_t.numpy()[m2va_t.numpy().astype(bool)]
        for epoch in range(1, MAX_EPOCHS_P2 + 1):
            t0 = time.time()
            tl = run_p2_epoch(model, use_graph, X2tr_t, y2tr_cls_t, m2tr_t, starts2_train_t, opt, True)
            vl, vp = run_p2_epoch(model, use_graph, X2va_t, y2va_cls_t, m2va_t, starts2_val_t, opt, False, True)
            vp_masked = vp[m2va_t.numpy().astype(bool)]
            va_auc_pr = average_precision_score(y_val_masked, vp_masked)
            print(f"[P2 {name}] ep{epoch} train={tl:.4f} val_loss={vl:.4f} val_AUCPR={va_auc_pr:.4f} ({time.time()-t0:.0f}s)", flush=True)
            if va_auc_pr > best_val_auc_pr:
                best_val_auc_pr, best_epoch = va_auc_pr, epoch
                torch.save(model.state_dict(), ckpt_path)
        model.load_state_dict(torch.load(ckpt_path))
        print(f"[P2 {name}] BEST ep{best_epoch} val_AUCPR={best_val_auc_pr:.4f}")
        return model

    @torch.no_grad()
    def predict_test(model, use_graph):
        model.eval()
        n = X2te_t.shape[0]
        preds = []
        for start in range(0, n, MICRO_BATCH):
            xb = add_static_fn(X2te_t[start:start + MICRO_BATCH].to(DEVICE), static_tensor)
            if use_graph:
                ew_s = gather_seq(edge_weight_by_hour_t, starts2_test_t[start:start + MICRO_BATCH]).to(DEVICE)
                ew_r = gather_seq(region_edge_weight_by_hour_t, starts2_test_t[start:start + MICRO_BATCH]).to(DEVICE)
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            else:
                logits = model(xb, n_regions)
            preds.append(torch.sigmoid(logits).cpu())
        return torch.cat(preds, dim=0).numpy()

    wind_probs_by_seed, nograph_probs_by_seed = [], []
    for model_seed in MODEL_SEEDS:
        torch.manual_seed(model_seed); np.random.seed(model_seed)
        p1w = train_p1(f"s{split_seed}_m{model_seed}_p1_gcn", StationQuantileGCN(n_feats), True)
        torch.manual_seed(model_seed); np.random.seed(model_seed)
        p1n = train_p1(f"s{split_seed}_m{model_seed}_p1_mlp", StationQuantileMLP(n_feats), False)
        sc_copy = copy.deepcopy(p1w.station_conv)
        fc1_copy, fc2_copy = copy.deepcopy(p1n.fc1), copy.deepcopy(p1n.fc2)
        del p1w, p1n; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

        p2w = RegionFromQuantileGCN(sc_copy, region_membership_t).to(DEVICE)
        p2w = train_p2(f"s{split_seed}_m{model_seed}_p2_gcn", p2w, True)
        wind_probs_by_seed.append(predict_test(p2w, True))

        p2n = RegionFromQuantileMLP(fc1_copy, fc2_copy, region_membership_t).to(DEVICE)
        p2n = train_p2(f"s{split_seed}_m{model_seed}_p2_mlp", p2n, False)
        nograph_probs_by_seed.append(predict_test(p2n, False))
        del p2w, p2n; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

    wind_ensemble = np.mean(wind_probs_by_seed, axis=0)
    nograph_ensemble = np.mean(nograph_probs_by_seed, axis=0)
    mask_test = m2te_t.numpy().astype(bool)
    y_true = y2te_cls_t.numpy()[mask_test]

    for name, proba in [("wind_graph", wind_ensemble[mask_test]), ("no_graph", nograph_ensemble[mask_test])]:
        auc_roc = roc_auc_score(y_true, proba)
        auc_pr = average_precision_score(y_true, proba)
        pred_pos = proba >= 0.5
        actual_pos = y_true >= 0.5
        tp = int(np.sum(pred_pos & actual_pos)); fp = int(np.sum(pred_pos & ~actual_pos))
        fn = int(np.sum(~pred_pos & actual_pos))
        precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
        recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        print(f"[SPLIT {split_seed} ENSEMBLE {name}] AUC-ROC={auc_roc:.4f} AUC-PR={auc_pr:.4f} P={precision:.4f} R={recall:.4f}")
        all_split_results.append({"split": split_seed, "model": name, "auc_roc": auc_roc, "auc_pr": auc_pr, "precision": precision, "recall": recall})

    del Xtr_t, Xva_t, ytr_reg_t, yva_reg_t, starts_train_t, starts_val_t
    del X2tr_t, y2tr_cls_t, m2tr_t, X2va_t, y2va_cls_t, m2va_t, X2te_t, y2te_cls_t, m2te_t
    del starts2_train_t, starts2_val_t, starts2_test_t, edge_weight_by_hour_t, region_edge_weight_by_hour_t, static_tensor
    gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

results_df = pd.DataFrame(all_split_results)
print("\n" + "="*20 + " FINAL CROSS-VALIDATED RESULTS " + "="*20)
print(results_df.to_string(index=False))
print("\n=== aggregated across 3 splits (each an ensemble of 2 seeds) ===")
print(results_df.groupby("model")[["auc_roc", "auc_pr", "precision", "recall"]].agg(["mean", "std"]))
print("\n=== paired comparison across splits ===")
for metric in ["auc_roc", "auc_pr", "recall", "precision"]:
    pivot = results_df.pivot(index="split", columns="model", values=metric)
    t, p = stats.ttest_rel(pivot["wind_graph"], pivot["no_graph"])
    wins = int((pivot["wind_graph"] > pivot["no_graph"]).sum())
    print(f"[{metric}] wind={pivot['wind_graph'].mean():.4f} no_graph={pivot['no_graph'].mean():.4f} t={t:.3f} p={p:.4f} wind_wins={wins}/3")


station graph: 21414 directed edges
region graph: 272 directed edges (fully connected, 17 regions)
device=mps

==================== SPLIT 0 ====================
windows: phase1 train=30543 val=9251  phase2 train=30543 val=9251 test=10045
masked-in fraction: train=0.2441 val=0.2884 test=0.2426
pos_weight: 50.00
[P1 s0_m0_p1_gcn] ep1 train=0.1945 val=0.2135 (230s)
[P1 s0_m0_p1_gcn] ep2 train=0.1854 val=0.2127 (207s)
[P1 s0_m0_p1_gcn] ep3 train=0.1829 val=0.2141 (203s)
[P1 s0_m0_p1_gcn] ep4 train=0.1813 val=0.2151 (209s)
[P1 s0_m0_p1_gcn] ep5 train=0.1801 val=0.2178 (203s)
[P1 s0_m0_p1_gcn] BEST ep2 val=0.2127
[P1 s0_m0_p1_mlp] ep1 train=0.1954 val=0.2185 (41s)
[P1 s0_m0_p1_mlp] ep2 train=0.1891 val=0.2188 (39s)
[P1 s0_m0_p1_mlp] ep3 train=0.1871 val=0.2165 (39s)
[P1 s0_m0_p1_mlp] ep4 train=0.1855 val=0.2184 (39s)
[P1 s0_m0_p1_mlp] ep5 train=0.1844 val=0.2223 (39s)
[P1 s0_m0_p1_mlp] BEST ep3 val=0.2165
[P2 s0_m0_p2_gcn] ep1 train=0.6583 val_loss=0.6801 val_AUCPR=0.2200 (244s)
[P2 s0_m0_p2

In [3]:
# ================================================================
# Threshold-tuned precision/recall, reusing the ALREADY-TRAINED
# checkpoints from the cross-validation run (no retraining needed).
# For each split, picks the decision threshold that maximizes F1 on
# the VALIDATION set (per model, per split), then applies that
# threshold to the held-out TEST set. AUC-ROC/AUC-PR are recomputed
# too as a sanity check -- they're threshold-free so should match
# the original run's numbers exactly.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, gc
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

BASE = "/Users/drewbaldwin/PM2_5 Research"
CKPT_DIR = f"{BASE}/ckpt_transport_test_cv"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

representative_station_idx = np.zeros(n_regions, dtype=int)
for r in range(n_regions):
    idxs = np.where(station_region_idx == r)[0]
    representative_station_idx[r] = idxs[np.argmin(dist_to_region[idxs, r])]

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months_arr = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour_raw = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = wind_speed_arr[:, representative_station_idx].astype(np.float32)
region_wdir_sin = wdir_sin_station[:, representative_station_idx].astype(np.float32)
region_wdir_cos = wdir_cos_station[:, representative_station_idx].astype(np.float32)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(pm25_raw_arr)
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wind_speed_arr, wind_dir_arr, blh_arr, wdir_sin_station, wdir_cos_station, season_sin, season_cos
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, hidden=32, gru_hidden=32, dropout=0.5):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP(nn.Module):
    def __init__(self, fc1, fc2, region_membership_t_local, hidden=32, gru_hidden=32, dropout=0.5):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            h_region = torch.relu(self.region_fc1(h_region_pooled))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH = 16
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset, apply_recent_mask=True):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    if apply_recent_mask and GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

SPLIT_SEEDS = [0, 1, 2]
MODEL_SEEDS = [0, 1]
CKPT_DIR = f"{BASE}/ckpt_transport_test_cv"
all_tuned_results = []

@torch.no_grad()
def run_inference(model, use_graph, X, starts, ew_station_t, ew_region_t):
    model.eval()
    n = X.shape[0]
    preds = []
    for start in range(0, n, MICRO_BATCH):
        xb = add_static_fn(X[start:start + MICRO_BATCH].to(DEVICE), static_tensor)
        if use_graph:
            ew_s = gather_seq(ew_station_t, starts[start:start + MICRO_BATCH]).to(DEVICE)
            ew_r = gather_seq(ew_region_t, starts[start:start + MICRO_BATCH]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
        else:
            logits = model(xb, n_regions)
        preds.append(torch.sigmoid(logits).cpu())
    return torch.cat(preds, dim=0).numpy()

def best_f1_threshold(y_true, proba):
    precision, recall, thresh = precision_recall_curve(y_true, proba)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = int(np.argmax(f1))
    return thresh[best_idx], f1[best_idx]

for split_seed in SPLIT_SEEDS:
    print(f"\n{'='*20} SPLIT {split_seed} {'='*20}")
    year_offset = years - years.min()
    block_id = year_offset * 12 + (months_arr - 1)
    n_blocks = int(block_id.max()) + 1
    rng = np.random.RandomState(split_seed)
    block_order = rng.permutation(n_blocks)
    n_train_blocks = int(round(0.6 * n_blocks))
    n_val_blocks = int(round(0.2 * n_blocks))
    block_to_split = np.empty(n_blocks, dtype=int)
    block_to_split[block_order[:n_train_blocks]] = 0
    block_to_split[block_order[n_train_blocks:n_train_blocks + n_val_blocks]] = 1
    block_to_split[block_order[n_train_blocks + n_val_blocks:]] = 2
    split_id_per_hour = block_to_split[block_id]
    TRAIN_MASK = split_id_per_hour == 0

    train_nonzero = edge_weight_by_hour_raw[TRAIN_MASK][edge_weight_by_hour_raw[TRAIN_MASK] > 0]
    weight_scale = train_nonzero.std()
    edge_weight_by_hour = (edge_weight_by_hour_raw / weight_scale).astype(np.float32)
    region_train_nonzero = region_edge_weight_by_hour_raw[TRAIN_MASK][region_edge_weight_by_hour_raw[TRAIN_MASK] > 0]
    region_weight_scale = region_train_nonzero.std()
    region_edge_weight_by_hour = (region_edge_weight_by_hour_raw / region_weight_scale).astype(np.float32)
    del train_nonzero, region_train_nonzero
    gc.collect()

    LOOKBACK = GRAPH_RECENT_HOURS
    edge_roll_mean = pd.DataFrame(region_edge_weight_by_hour).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
    pm25_roll_mean = pd.DataFrame(region_pm25).rolling(window=LOOKBACK, min_periods=LOOKBACK).mean().to_numpy()
    region_pm25_median_train = np.nanmedian(region_pm25[TRAIN_MASK], axis=0)
    src_elevated = pm25_roll_mean[:, r_src_idx] > region_pm25_median_train[r_src_idx][None, :]
    transport_signal_per_edge = np.where(src_elevated, edge_roll_mean, 0.0)
    transport_score = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        edge_mask = (r_dst_idx == r)
        transport_score[:, r] = np.nan_to_num(transport_signal_per_edge[:, edge_mask]).max(axis=1)
    edge_threshold_train = np.percentile(region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0], 75)
    transport_flag_per_region = transport_score > edge_threshold_train
    del edge_roll_mean, pm25_roll_mean, src_elevated, transport_signal_per_edge, transport_score
    gc.collect()

    t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
    t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
    time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)

    p2_buckets = {1: ([], [], [], []), 2: ([], [], [], [])}
    for t in range(0, n_time - WINDOW - HORIZON + 1):
        target_t = t + WINDOW + HORIZON - 1
        input_end_t = t + WINDOW - 1
        s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
        if s_start != s_target or s_start not in p2_buckets:
            continue
        x_win = time_arr_std[t:t + WINDOW]
        Xl2, ycls_l2, mask_l2, sl2 = p2_buckets[s_start]
        Xl2.append(x_win); ycls_l2.append(region_episode_label[target_t])
        mask_l2.append(transport_flag_per_region[input_end_t]); sl2.append(t)

    X2_val, ycls2_val, mask2_val, starts2_val = (np.stack(v) for v in p2_buckets[1])
    X2_test, ycls2_test, mask2_test, starts2_test = (np.stack(v) for v in p2_buckets[2])
    del p2_buckets, time_arr_std
    gc.collect()
    print(f"windows: val={len(X2_val)} test={len(X2_test)}  masked-in: val={mask2_val.mean():.4f} test={mask2_test.mean():.4f}")

    X2va_t = torch.tensor(X2_val, dtype=torch.float32)
    X2te_t = torch.tensor(X2_test, dtype=torch.float32)
    starts2_val_t = torch.tensor(starts2_val, dtype=torch.long)
    starts2_test_t = torch.tensor(starts2_test, dtype=torch.long)
    mask_val = mask2_val.astype(bool)
    mask_test = mask2_test.astype(bool)
    y_val_masked = ycls2_val[mask_val]
    y_test_masked = ycls2_test[mask_test]
    del X2_val, X2_test
    gc.collect()

    edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
    region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)
    del edge_weight_by_hour, region_edge_weight_by_hour
    gc.collect()

    wind_val_by_seed, wind_test_by_seed = [], []
    nograph_val_by_seed, nograph_test_by_seed = [], []
    for model_seed in MODEL_SEEDS:
        sc_placeholder = WindConvLayer(n_feats, 32)
        p2w = RegionFromQuantileGCN(sc_placeholder, region_membership_t).to(DEVICE)
        p2w.load_state_dict(torch.load(f"{CKPT_DIR}/s{split_seed}_m{model_seed}_p2_gcn.pt", map_location=DEVICE))
        wind_val_by_seed.append(run_inference(p2w, True, X2va_t, starts2_val_t, edge_weight_by_hour_t, region_edge_weight_by_hour_t))
        wind_test_by_seed.append(run_inference(p2w, True, X2te_t, starts2_test_t, edge_weight_by_hour_t, region_edge_weight_by_hour_t))
        del p2w, sc_placeholder; gc.collect()

        fc1_placeholder, fc2_placeholder = nn.Linear(n_feats, 32), nn.Linear(32, 32)
        p2n = RegionFromQuantileMLP(fc1_placeholder, fc2_placeholder, region_membership_t).to(DEVICE)
        p2n.load_state_dict(torch.load(f"{CKPT_DIR}/s{split_seed}_m{model_seed}_p2_mlp.pt", map_location=DEVICE))
        nograph_val_by_seed.append(run_inference(p2n, False, X2va_t, starts2_val_t, edge_weight_by_hour_t, region_edge_weight_by_hour_t))
        nograph_test_by_seed.append(run_inference(p2n, False, X2te_t, starts2_test_t, edge_weight_by_hour_t, region_edge_weight_by_hour_t))
        del p2n, fc1_placeholder, fc2_placeholder; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

    wind_val_ens = np.mean(wind_val_by_seed, axis=0)
    wind_test_ens = np.mean(wind_test_by_seed, axis=0)
    nograph_val_ens = np.mean(nograph_val_by_seed, axis=0)
    nograph_test_ens = np.mean(nograph_test_by_seed, axis=0)

    for name, val_ens, test_ens in [("wind_graph", wind_val_ens, wind_test_ens), ("no_graph", nograph_val_ens, nograph_test_ens)]:
        val_proba_masked = val_ens[mask_val]
        test_proba_masked = test_ens[mask_test]
        thresh, val_f1 = best_f1_threshold(y_val_masked, val_proba_masked)

        auc_roc = roc_auc_score(y_test_masked, test_proba_masked)
        auc_pr = average_precision_score(y_test_masked, test_proba_masked)

        pred_pos_fixed = test_proba_masked >= 0.5
        pred_pos_tuned = test_proba_masked >= thresh
        actual_pos = y_test_masked >= 0.5

        def prf(pred_pos):
            tp = int(np.sum(pred_pos & actual_pos)); fp = int(np.sum(pred_pos & ~actual_pos)); fn = int(np.sum(~pred_pos & actual_pos))
            p = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
            r = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
            f1 = 2 * p * r / (p + r) if (p + r) > 0 else float("nan")
            return p, r, f1

        p_fixed, r_fixed, f1_fixed = prf(pred_pos_fixed)
        p_tuned, r_tuned, f1_tuned = prf(pred_pos_tuned)

        print(f"[SPLIT {split_seed} {name}] thresh={thresh:.3f} (val_F1={val_f1:.4f}) | "
              f"AUC-ROC={auc_roc:.4f} AUC-PR={auc_pr:.4f} | "
              f"@0.5: P={p_fixed:.4f} R={r_fixed:.4f} F1={f1_fixed:.4f} | "
              f"@tuned: P={p_tuned:.4f} R={r_tuned:.4f} F1={f1_tuned:.4f}")
        all_tuned_results.append({"split": split_seed, "model": name, "threshold": thresh, "auc_roc": auc_roc, "auc_pr": auc_pr,
                                   "precision_fixed": p_fixed, "recall_fixed": r_fixed, "f1_fixed": f1_fixed,
                                   "precision_tuned": p_tuned, "recall_tuned": r_tuned, "f1_tuned": f1_tuned})

    del X2va_t, X2te_t, starts2_val_t, starts2_test_t, edge_weight_by_hour_t, region_edge_weight_by_hour_t
    gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

tuned_df = pd.DataFrame(all_tuned_results)
print("\n" + "="*20 + " THRESHOLD-TUNED RESULTS " + "="*20)
print(tuned_df.to_string(index=False))
print("\n=== aggregated across 3 splits ===")
print(tuned_df.groupby("model")[["auc_roc", "auc_pr", "precision_tuned", "recall_tuned", "f1_tuned"]].agg(["mean", "std"]))
print("\n=== paired comparison across splits (tuned threshold) ===")
for metric in ["auc_roc", "auc_pr", "precision_tuned", "recall_tuned", "f1_tuned"]:
    pivot = tuned_df.pivot(index="split", columns="model", values=metric)
    t, p = stats.ttest_rel(pivot["wind_graph"], pivot["no_graph"])
    wins = int((pivot["wind_graph"] > pivot["no_graph"]).sum())
    print(f"[{metric}] wind={pivot['wind_graph'].mean():.4f} no_graph={pivot['no_graph'].mean():.4f} t={t:.3f} p={p:.4f} wind_wins={wins}/3")


device=mps

==================== SPLIT 0 ====================
windows: val=9251 test=10045  masked-in: val=0.2884 test=0.2426
[SPLIT 0 wind_graph] thresh=0.790 (val_F1=0.2813) | AUC-ROC=0.8867 AUC-PR=0.1144 | @0.5: P=0.0612 R=0.8300 F1=0.1140 | @tuned: P=0.1515 R=0.4597 F1=0.2279
[SPLIT 0 no_graph] thresh=0.816 (val_F1=0.3516) | AUC-ROC=0.8619 AUC-PR=0.1013 | @0.5: P=0.0616 R=0.7666 F1=0.1140 | @tuned: P=0.1338 R=0.1311 F1=0.1325

==================== SPLIT 1 ====================
windows: val=9609 test=9925  masked-in: val=0.3019 test=0.3284
[SPLIT 1 wind_graph] thresh=0.830 (val_F1=0.1425) | AUC-ROC=0.8186 AUC-PR=0.0296 | @0.5: P=0.0267 R=0.6260 F1=0.0513 | @tuned: P=0.0334 R=0.2547 F1=0.0590
[SPLIT 1 no_graph] thresh=0.899 (val_F1=0.1377) | AUC-ROC=0.7996 AUC-PR=0.0260 | @0.5: P=0.0204 R=0.6883 F1=0.0397 | @tuned: P=0.0411 R=0.1789 F1=0.0668

==================== SPLIT 2 ====================
windows: val=9704 test=9902  masked-in: val=0.2093 test=0.2491
[SPLIT 2 wind_graph] thresh=0.

In [1]:
# ================================================================
# Rebuilt with the two fixes identified:
#  1) Transport flag is now evaluated at the END OF THE INPUT WINDOW
#     (hour t+35, using an 18h lookback = GRAPH_RECENT_HOURS, i.e.
#     hours [t+18, t+35]) -- NOT the old version, which checked
#     conditions right before the TARGET (t+71), a period the model
#     never observes. This aligns the filter with what the model and
#     the graph mechanism can actually see.
#  2) Filtering is now PER-REGION, not a global "any of 17 regions"
#     flag. Every window is kept (all 17 regions still needed as
#     graph nodes so message-passing works), but the loss and
#     evaluation metrics are MASKED to only count a given region's
#     prediction when THAT region specifically shows a transport
#     signature at that window's input-end hour.
#
# Because we're no longer dropping whole windows, Phase 2's dataset
# is back to full size (not the ~19-20K filtered subset from before)
# -- expect meaningfully longer runtime than the last comparison.
# 7 epochs, 3 seeds, as requested.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"station graph: {E} directed edges")

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

representative_station_idx = np.zeros(n_regions, dtype=int)
for r in range(n_regions):
    idxs = np.where(station_region_idx == r)[0]
    local_dists = dist_to_region[idxs, r]
    representative_station_idx[r] = idxs[np.argmin(local_dists)]

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
R_E = len(r_src_idx)
region_edge_index = torch.tensor(np.stack([r_src_idx, r_dst_idx]), dtype=torch.long)
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)
print(f"region graph: {R_E} directed edges (fully connected, {n_regions} regions)")

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months_arr = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = wind_speed_arr[:, representative_station_idx].astype(np.float32)
region_wdir_sin = wdir_sin_station[:, representative_station_idx].astype(np.float32)
region_wdir_cos = wdir_cos_station[:, representative_station_idx].astype(np.float32)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(df.pivot(index="Datetime", columns="Station_ID", values="PM25")[station_order].to_numpy())

year_offset = years - years.min()
block_id = year_offset * 12 + (months_arr - 1)
n_blocks = int(block_id.max()) + 1
rng = np.random.RandomState(0)
block_order = rng.permutation(n_blocks)
n_train_blocks = int(round(0.6 * n_blocks))
n_val_blocks = int(round(0.2 * n_blocks))
block_to_split = np.empty(n_blocks, dtype=int)
block_to_split[block_order[:n_train_blocks]] = 0
block_to_split[block_order[n_train_blocks:n_train_blocks + n_val_blocks]] = 1
block_to_split[block_order[n_train_blocks + n_val_blocks:]] = 2
split_id_per_hour = block_to_split[block_id]
TRAIN_MASK = split_id_per_hour == 0

train_nonzero = edge_weight_by_hour[TRAIN_MASK][edge_weight_by_hour[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
del train_nonzero
region_train_nonzero = region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0]
region_weight_scale = region_train_nonzero.std()
region_edge_weight_by_hour_scaled = (region_edge_weight_by_hour / region_weight_scale).astype(np.float32)
del region_train_nonzero
gc.collect()

# ---- FIX 1 + 2: per-region transport flag, evaluated with an 18h lookback
# ENDING AT EACH HOUR (not 24h ending at a future target) ----
LOOKBACK_TRANSPORT = GRAPH_RECENT_HOURS  # 18h, matching what the graph actually attends to
edge_roll_mean = pd.DataFrame(region_edge_weight_by_hour).rolling(window=LOOKBACK_TRANSPORT, min_periods=LOOKBACK_TRANSPORT).mean().to_numpy()
pm25_roll_mean = pd.DataFrame(region_pm25).rolling(window=LOOKBACK_TRANSPORT, min_periods=LOOKBACK_TRANSPORT).mean().to_numpy()
region_pm25_median_train = np.nanmedian(region_pm25[TRAIN_MASK], axis=0)
src_elevated = pm25_roll_mean[:, r_src_idx] > region_pm25_median_train[r_src_idx][None, :]
transport_signal_per_edge = np.where(src_elevated, edge_roll_mean, 0.0)
transport_score = np.zeros((n_time, n_regions), dtype=np.float32)
for r in range(n_regions):
    edge_mask = (r_dst_idx == r)
    transport_score[:, r] = np.nan_to_num(transport_signal_per_edge[:, edge_mask]).max(axis=1)
edge_threshold_train = np.percentile(region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0], 75)
transport_flag_per_region = transport_score > edge_threshold_train   # (T, n_regions) -- per-region, per-hour
print(f"per-region transport flag positive rate: {transport_flag_per_region.mean():.4f}")
print("per-region rates:", dict(zip(region_names, transport_flag_per_region.mean(axis=0).round(3))))
del edge_roll_mean, pm25_roll_mean, src_elevated, transport_signal_per_edge, transport_score, region_edge_weight_by_hour
gc.collect()

wdir_sin, wdir_cos = wdir_sin_station, wdir_cos_station
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats

time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                     [wind_speed_arr, wdir_sin, wdir_cos, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wind_speed_arr, wind_dir_arr, blh_arr, wdir_sin_station, wdir_cos_station, wdir_sin, wdir_cos, season_sin, season_cos
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[TRAIN_MASK]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
del time_arr, time_train
gc.collect()

s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

# single pass: build Phase 1 (all data) and Phase 2 (ALL windows kept, plus a per-region
# transport mask looked up at the INPUT WINDOW's last hour, t+WINDOW-1)
p1_buckets = {0: ([], [], []), 1: ([], [], []), 2: ([], [], [])}
p2_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
n_dropped = 0
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    input_end_t = t + WINDOW - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target:
        n_dropped += 1
        continue
    x_win = time_arr_std[t:t + WINDOW]
    Xl, yreg_l, sl = p1_buckets[s_start]
    Xl.append(x_win); yreg_l.append(time_arr_std[target_t, :, pm25_col_idx]); sl.append(t)
    Xl2, ycls_l2, mask_l2, sl2 = p2_buckets[s_start]
    Xl2.append(x_win); ycls_l2.append(region_episode_label[target_t])
    mask_l2.append(transport_flag_per_region[input_end_t])   # per-region mask, from INPUT window's end
    sl2.append(t)
print(f"windows dropped (split boundary): {n_dropped}")

X_train, yreg_train, starts_train = (np.stack(v) for v in p1_buckets[0])
X_val, yreg_val, starts_val = (np.stack(v) for v in p1_buckets[1])
X2_train, ycls2_train, mask2_train, starts2_train = (np.stack(v) for v in p2_buckets[0])
X2_val, ycls2_val, mask2_val, starts2_val = (np.stack(v) for v in p2_buckets[1])
X2_test, ycls2_test, mask2_test, starts2_test = (np.stack(v) for v in p2_buckets[2])
del p1_buckets, p2_buckets, time_arr_std
gc.collect()
print(f"PHASE1 windows: train={len(X_train)}, val={len(X_val)}")
print(f"PHASE2 windows (all kept, per-region masked): train={len(X2_train)}, val={len(X2_val)}, test={len(X2_test)}, n_feats={n_feats}")
print(f"masked-in fraction: train={mask2_train.mean():.4f}, val={mask2_val.mean():.4f}, test={mask2_test.mean():.4f}")

Xtr_t, ytr_reg_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(yreg_train, dtype=torch.float32)
Xva_t, yva_reg_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(yreg_val, dtype=torch.float32)
starts_train_t, starts_val_t = torch.tensor(starts_train, dtype=torch.long), torch.tensor(starts_val, dtype=torch.long)
del X_train, X_val, yreg_train, yreg_val, starts_train, starts_val
gc.collect()

X2tr_t, y2tr_cls_t, m2tr_t = torch.tensor(X2_train, dtype=torch.float32), torch.tensor(ycls2_train, dtype=torch.float32), torch.tensor(mask2_train, dtype=torch.float32)
X2va_t, y2va_cls_t, m2va_t = torch.tensor(X2_val, dtype=torch.float32), torch.tensor(ycls2_val, dtype=torch.float32), torch.tensor(mask2_val, dtype=torch.float32)
X2te_t, y2te_cls_t, m2te_t = torch.tensor(X2_test, dtype=torch.float32), torch.tensor(ycls2_test, dtype=torch.float32), torch.tensor(mask2_test, dtype=torch.float32)
starts2_train_t, starts2_val_t, starts2_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts2_train, starts2_val, starts2_test))
del X2_train, X2_val, X2_test, ycls2_train, ycls2_val, ycls2_test, mask2_train, mask2_val, mask2_test, starts2_train, starts2_val, starts2_test
gc.collect()

# pos_weight computed from MASKED-IN train instances only
masked_train_labels = y2tr_cls_t[m2tr_t.bool()]
POS_WEIGHT = min(float((masked_train_labels.numel() - masked_train_labels.sum()) / masked_train_labels.sum().clamp(min=1)), 50.0)
print(f"pos_weight (masked-in train instances only, capped at 50): {POS_WEIGHT:.2f}")

edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
del edge_weight_by_hour
region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour_scaled)
del region_edge_weight_by_hour_scaled
gc.collect()

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def make_gather_fn(ew_by_hour_t):
    def gather_edge_weight_seq(starts_subset):
        idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
        ew = ew_by_hour_t[idx]
        if GRAPH_RECENT_HOURS < WINDOW:
            ew = ew.clone()
            ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
        return ew
    return gather_edge_weight_seq

gather_station_edges = make_gather_fn(edge_weight_by_hour_t)
gather_region_edges = make_gather_fn(region_edge_weight_by_hour_t)

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, station_edge_weight_seq, region_edge_weight_seq):
        B, W, N, Fin = x_window.shape
        station_ei_b = batch_edge_index(edge_index, N, B)
        region_ei_b = batch_edge_index(region_edge_index, n_regions, B)
        num_station_nodes = B * N
        num_region_nodes = B * n_regions
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station).reshape(B * n_regions, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_regions, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_regions, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_regions, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP(nn.Module):
    def __init__(self, fc1, fc2, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station).reshape(B * n_regions, -1)
            h_region = torch.relu(self.region_fc1(h_region_pooled))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_regions, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_regions, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_regions, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_transport_test_v2"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 7, 1e-4
print(f"device={DEVICE}")
region_membership_t = region_membership_t.to(DEVICE)
edge_index = edge_index.to(DEVICE)
region_edge_index = region_edge_index.to(DEVICE)

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT), reduction="none")

def run_phase1_epoch(model, use_graph, X, y, starts, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                if use_graph:
                    ew_seq = gather_station_edges(starts[mb_idx]).to(DEVICE)
                    pred = model(xb, edge_index, ew_seq)
                else:
                    pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_phase1(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_phase1_epoch(model, use_graph, Xtr_t, ytr_reg_t, starts_train_t, opt, train=True)
        val_loss = run_phase1_epoch(model, use_graph, Xva_t, yva_reg_t, starts_val_t, opt, train=False)
        print(f"[PHASE1 {name}] epoch {epoch}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    print(f"[PHASE1 {name}] BEST epoch={best_epoch}  val_pinball={best_val:.4f}\n", flush=True)
    return model

def run_phase2_epoch(model, use_graph, X, y, m, starts, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_maskn = 0.0, 0.0
    probs_list = [] if return_probs else None
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            mb = m[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                if use_graph:
                    ew_station = gather_station_edges(starts[mb_idx]).to(DEVICE)
                    ew_region = gather_region_edges(starts[mb_idx]).to(DEVICE)
                    logits = model(xb, ew_station, ew_region)
                else:
                    logits = model(xb)
                loss_per_elem = region_criterion(logits, yb)          # (B, n_regions)
                masked_loss = (loss_per_elem * mb).sum() / mb.sum().clamp(min=1)
            if train:
                (masked_loss * mb.sum() / mb.sum().clamp(min=1)).backward()
            if return_probs:
                probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += masked_loss.item() * mb.sum().item()
            total_maskn += mb.sum().item()
        if train:
            optimizer.step()
    avg_loss = total_loss / max(total_maskn, 1)
    if return_probs:
        return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

@torch.no_grad()
def predict_proba_phase2(model, use_graph, X, starts, micro_batch=16):
    model.eval()
    n = X.shape[0]
    preds = []
    for start in range(0, n, micro_batch):
        xb = add_static(X[start:start + micro_batch]).to(DEVICE)
        if use_graph:
            ew_station = gather_station_edges(starts[start:start + micro_batch]).to(DEVICE)
            ew_region = gather_region_edges(starts[start:start + micro_batch]).to(DEVICE)
            logits = model(xb, ew_station, ew_region)
        else:
            logits = model(xb)
        preds.append(torch.sigmoid(logits).cpu())
    return torch.cat(preds, dim=0).numpy()

def train_phase2(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val_auc_pr, best_epoch = -1.0, -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    y_val_masked = y2va_cls_t.numpy()[m2va_t.numpy().astype(bool)]
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_phase2_epoch(model, use_graph, X2tr_t, y2tr_cls_t, m2tr_t, starts2_train_t, opt, train=True)
        val_loss, val_probs = run_phase2_epoch(model, use_graph, X2va_t, y2va_cls_t, m2va_t, starts2_val_t, opt, train=False, return_probs=True)
        val_probs_masked = val_probs[m2va_t.numpy().astype(bool)]
        val_auc_pr = average_precision_score(y_val_masked, val_probs_masked)
        print(f"[PHASE2 {name}] epoch {epoch}  train={train_loss:.4f}  val_loss={val_loss:.4f}  val_AUC-PR={val_auc_pr:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_auc_pr > best_val_auc_pr:
            best_val_auc_pr, best_epoch = val_auc_pr, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    proba = predict_proba_phase2(model, use_graph, X2te_t, starts2_test_t)
    mask_test = m2te_t.numpy().astype(bool)
    y_true = y2te_cls_t.numpy()[mask_test]
    y_pred_proba = proba[mask_test]
    auc_roc = roc_auc_score(y_true, y_pred_proba)
    auc_pr = average_precision_score(y_true, y_pred_proba)
    pred_pos = y_pred_proba >= 0.5
    actual_pos = y_true >= 0.5
    tp = int(np.sum(pred_pos & actual_pos)); fp = int(np.sum(pred_pos & ~actual_pos))
    fn = int(np.sum(~pred_pos & actual_pos)); tn = int(np.sum(~pred_pos & ~actual_pos))
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    print(f"[PHASE2 {name}] BEST epoch={best_epoch} (val_AUC-PR={best_val_auc_pr:.4f})  "
          f"AUC-ROC={auc_roc:.4f}  AUC-PR={auc_pr:.4f}  P={precision:.4f}  R={recall:.4f}  (tp={tp} fp={fp} fn={fn} tn={tn}, masked test n={mask_test.sum()})\n", flush=True)
    return {"name": name, "auc_roc": auc_roc, "auc_pr": auc_pr, "precision": precision, "recall": recall}

SEEDS = [0, 1, 2]
all_results = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    phase1_wind = train_phase1(f"phase1_quantile_gcn_seed{seed}", StationQuantileGCN(n_feats), use_graph=True)
    torch.manual_seed(seed); np.random.seed(seed)
    phase1_nograph = train_phase1(f"phase1_quantile_mlp_seed{seed}", StationQuantileMLP(n_feats), use_graph=False)

    station_conv_copy = copy.deepcopy(phase1_wind.station_conv)
    fc1_copy, fc2_copy = copy.deepcopy(phase1_nograph.fc1), copy.deepcopy(phase1_nograph.fc2)
    del phase1_wind, phase1_nograph
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    phase2_wind_model = RegionFromQuantileGCN(station_conv_copy).to(DEVICE)
    res_wind = train_phase2(f"phase2_region_gcn_v2_seed{seed}", phase2_wind_model, use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_graph"
    all_results.append(res_wind)

    phase2_nograph_model = RegionFromQuantileMLP(fc1_copy, fc2_copy).to(DEVICE)
    res_nograph = train_phase2(f"phase2_region_mlp_v2_seed{seed}", phase2_nograph_model, use_graph=False)
    res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    all_results.append(res_nograph)

results_df = pd.DataFrame(all_results)
print(results_df[["model", "seed", "auc_roc", "auc_pr", "precision", "recall"]].to_string(index=False))
print("\n=== per-region-masked transport comparison, 7 epochs, 3 seeds ===")
print(results_df.groupby("model")[["auc_roc", "auc_pr", "precision", "recall"]].agg(["mean", "std"]))

print("\n=== paired comparison ===")
for metric in ["auc_roc", "auc_pr", "recall", "precision"]:
    pivot = results_df.pivot(index="seed", columns="model", values=metric)
    t, p = stats.ttest_rel(pivot["wind_graph"], pivot["no_graph"])
    wins = int((pivot["wind_graph"] > pivot["no_graph"]).sum())
    print(f"[{metric}] wind={pivot['wind_graph'].mean():.4f}  no_graph={pivot['no_graph'].mean():.4f}  "
          f"paired t={t:.3f} p={p:.4f}  wind_wins={wins}/3")


station graph: 21414 directed edges
region graph: 272 directed edges (fully connected, 17 regions)
per-region transport flag positive rate: 0.2535
per-region rates: {'Seoul': np.float64(0.313), 'Busan': np.float64(0.293), 'Daegu': np.float64(0.243), 'Incheon': np.float64(0.128), 'Gwangju': np.float64(0.17), 'Daejeon': np.float64(0.334), 'Ulsan': np.float64(0.316), 'Sejong': np.float64(0.335), 'Gyeonggi': np.float64(0.344), 'Gangwon': np.float64(0.236), 'Chungbuk': np.float64(0.296), 'Chungnam': np.float64(0.215), 'Jeonbuk': np.float64(0.235), 'Jeonnam': np.float64(0.241), 'Gyeongbuk': np.float64(0.237), 'Gyeongnam': np.float64(0.263), 'Jeju': np.float64(0.113)}
windows dropped (split boundary): 2698
PHASE1 windows: train=30543, val=9251
PHASE2 windows (all kept, per-region masked): train=30543, val=9251, test=10045, n_feats=23
masked-in fraction: train=0.2441, val=0.2884, test=0.2426
pos_weight (masked-in train instances only, capped at 50): 50.00
device=mps
[PHASE1 phase1_quantile_gcn

KeyboardInterrupt: 

In [1]:
# ================================================================
# Full pipeline, updated per two specific requirements:
#  1) Station -> region hidden-state mapping now uses a LEARNED
#     neural network layer (AttentionPool: a small trainable scoring
#     layer + softmax within each region), replacing max-pooling.
#  2) Region-level wind (direction + speed) is now ONE real, physical
#     reading per region per hour -- NOT an average of any kind.
#     Each region is represented by its single station closest to
#     the region's geographic centroid, and that station's actual
#     wind reading is used directly to build the region-to-region
#     graph edges. This keeps wind a literal physical quantity (real
#     degrees/speed), separate from the learned hidden-state pooling.
#
# Everything else (station quantile pretraining via pinball loss,
# transport-driven window filtering, memory-hardened data loading,
# 3 epochs / 1 seed) is unchanged from the last version.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"station graph: {E} directed edges")

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

# ONE representative station per region (nearest to that region's centroid) --
# used to give each region a single REAL wind reading, not a blended average.
representative_station_idx = np.zeros(n_regions, dtype=int)
for r in range(n_regions):
    idxs = np.where(station_region_idx == r)[0]
    local_dists = dist_to_region[idxs, r]
    representative_station_idx[r] = idxs[np.argmin(local_dists)]
print("representative station per region:", dict(zip(region_names, station_order[i] if False else representative_station_idx)))

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
R_E = len(r_src_idx)
region_edge_index = torch.tensor(np.stack([r_src_idx, r_dst_idx]), dtype=torch.long)
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)
print(f"region graph: {R_E} directed edges (fully connected, {n_regions} regions)")

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months_arr = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))

# region wind = the representative (nearest-to-centroid) station's OWN reading --
# one real number per region per hour, not any kind of average.
region_windspeed = wind_speed_arr[:, representative_station_idx].astype(np.float32)
region_wdir_sin = wdir_sin_station[:, representative_station_idx].astype(np.float32)
region_wdir_cos = wdir_cos_station[:, representative_station_idx].astype(np.float32)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

region_pm25 = regional_flat_mean(df.pivot(index="Datetime", columns="Station_ID", values="PM25")[station_order].to_numpy())

year_offset = years - years.min()
block_id = year_offset * 12 + (months_arr - 1)
n_blocks = int(block_id.max()) + 1
rng = np.random.RandomState(0)
block_order = rng.permutation(n_blocks)
n_train_blocks = int(round(0.6 * n_blocks))
n_val_blocks = int(round(0.2 * n_blocks))
block_to_split = np.empty(n_blocks, dtype=int)
block_to_split[block_order[:n_train_blocks]] = 0
block_to_split[block_order[n_train_blocks:n_train_blocks + n_val_blocks]] = 1
block_to_split[block_order[n_train_blocks + n_val_blocks:]] = 2
split_id_per_hour = block_to_split[block_id]
TRAIN_MASK = split_id_per_hour == 0

train_nonzero = edge_weight_by_hour[TRAIN_MASK][edge_weight_by_hour[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
del train_nonzero
region_train_nonzero = region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0]
region_weight_scale = region_train_nonzero.std()
region_edge_weight_by_hour_scaled = (region_edge_weight_by_hour / region_weight_scale).astype(np.float32)
del region_train_nonzero
gc.collect()

edge_roll_mean = pd.DataFrame(region_edge_weight_by_hour).rolling(window=24, min_periods=24).mean().to_numpy()
pm25_roll_mean = pd.DataFrame(region_pm25).rolling(window=24, min_periods=24).mean().to_numpy()
region_pm25_median_train = np.nanmedian(region_pm25[TRAIN_MASK], axis=0)
src_elevated = pm25_roll_mean[:, r_src_idx] > region_pm25_median_train[r_src_idx][None, :]
transport_signal_per_edge = np.where(src_elevated, edge_roll_mean, 0.0)
transport_score = np.zeros((n_time, n_regions), dtype=np.float32)
for r in range(n_regions):
    edge_mask = (r_dst_idx == r)
    transport_score[:, r] = np.nan_to_num(transport_signal_per_edge[:, edge_mask]).max(axis=1)
edge_threshold_train = np.percentile(region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0], 75)
transport_flag_per_region = transport_score > edge_threshold_train
transport_flag_any_region = transport_flag_per_region.any(axis=1)
print(f"hours with any-region transport signature: {transport_flag_any_region.mean():.4f} "
      f"({transport_flag_any_region[TRAIN_MASK].sum()} train / {transport_flag_any_region[split_id_per_hour==2].sum()} test)")
del edge_roll_mean, pm25_roll_mean, src_elevated, transport_signal_per_edge, transport_score, region_edge_weight_by_hour
gc.collect()

wdir_sin, wdir_cos = wdir_sin_station, wdir_cos_station
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats

time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                     [wind_speed_arr, wdir_sin, wdir_cos, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, wind_speed_arr, wind_dir_arr, blh_arr, wdir_sin_station, wdir_cos_station, wdir_sin, wdir_cos, season_sin, season_cos
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[TRAIN_MASK]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
del time_arr, time_train
gc.collect()

s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

p1_buckets = {0: ([], [], []), 1: ([], [], []), 2: ([], [], [])}
p2_buckets = {0: ([], [], []), 1: ([], [], []), 2: ([], [], [])}
n_dropped, n_p2_filtered = 0, 0
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target:
        n_dropped += 1
        continue
    x_win = time_arr_std[t:t + WINDOW]
    Xl, yreg_l, sl = p1_buckets[s_start]
    Xl.append(x_win); yreg_l.append(time_arr_std[target_t, :, pm25_col_idx]); sl.append(t)
    if transport_flag_any_region[target_t]:
        Xl2, ycls_l2, sl2 = p2_buckets[s_start]
        Xl2.append(x_win); ycls_l2.append(region_episode_label[target_t]); sl2.append(t)
    else:
        n_p2_filtered += 1
print(f"windows dropped (split boundary): {n_dropped}, phase-2 filtered out (not transport-driven): {n_p2_filtered}")

X_train, yreg_train, starts_train = (np.stack(v) for v in p1_buckets[0])
X_val, yreg_val, starts_val = (np.stack(v) for v in p1_buckets[1])
X2_train, ycls2_train, starts2_train = (np.stack(v) for v in p2_buckets[0])
X2_val, ycls2_val, starts2_val = (np.stack(v) for v in p2_buckets[1])
X2_test, ycls2_test, starts2_test = (np.stack(v) for v in p2_buckets[2])
del p1_buckets, p2_buckets, time_arr_std
gc.collect()
print(f"PHASE1 windows: train={len(X_train)}, val={len(X_val)}")
print(f"PHASE2 windows: train={len(X2_train)}, val={len(X2_val)}, test={len(X2_test)}, n_feats={n_feats}")

Xtr_t, ytr_reg_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(yreg_train, dtype=torch.float32)
Xva_t, yva_reg_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(yreg_val, dtype=torch.float32)
starts_train_t, starts_val_t = torch.tensor(starts_train, dtype=torch.long), torch.tensor(starts_val, dtype=torch.long)
del X_train, X_val, yreg_train, yreg_val, starts_train, starts_val
gc.collect()

X2tr_t, y2tr_cls_t = torch.tensor(X2_train, dtype=torch.float32), torch.tensor(ycls2_train, dtype=torch.float32)
X2va_t, y2va_cls_t = torch.tensor(X2_val, dtype=torch.float32), torch.tensor(ycls2_val, dtype=torch.float32)
X2te_t, y2te_cls_t = torch.tensor(X2_test, dtype=torch.float32), torch.tensor(ycls2_test, dtype=torch.float32)
starts2_train_t, starts2_val_t, starts2_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts2_train, starts2_val, starts2_test))
del X2_train, X2_val, X2_test, ycls2_train, ycls2_val, ycls2_test, starts2_train, starts2_val, starts2_test
gc.collect()

POS_WEIGHT = min(float((y2tr_cls_t.numel() - y2tr_cls_t.sum()) / y2tr_cls_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight for region BCE on transport-filtered subset (capped at 50): {POS_WEIGHT:.2f}")

edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
del edge_weight_by_hour
region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour_scaled)
del region_edge_weight_by_hour_scaled
gc.collect()

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def make_gather_fn(ew_by_hour_t):
    def gather_edge_weight_seq(starts_subset):
        idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
        ew = ew_by_hour_t[idx]
        if GRAPH_RECENT_HOURS < WINDOW:
            ew = ew.clone()
            ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
        return ew
    return gather_edge_weight_seq

gather_station_edges = make_gather_fn(edge_weight_by_hour_t)
gather_region_edges = make_gather_fn(region_edge_weight_by_hour_t)

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station):   # (B, N, H) -> (B, n_regions, H), learned weighted pool
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationQuantileMLP(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, station_edge_weight_seq, region_edge_weight_seq):
        B, W, N, Fin = x_window.shape
        station_ei_b = batch_edge_index(edge_index, N, B)
        region_ei_b = batch_edge_index(region_edge_index, n_regions, B)
        num_station_nodes = B * N
        num_region_nodes = B * n_regions
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station).reshape(B * n_regions, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_regions, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_regions, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_regions, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP(nn.Module):
    def __init__(self, fc1, fc2, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
        self.attn_pool = AttentionPool(hidden)
        self.region_fc1 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station).reshape(B * n_regions, -1)
            h_region = torch.relu(self.region_fc1(h_region_pooled))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_regions, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_regions, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_regions, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_transport_test"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 3, 1e-4
print(f"device={DEVICE}")
region_membership_t = region_membership_t.to(DEVICE)
edge_index = edge_index.to(DEVICE)
region_edge_index = region_edge_index.to(DEVICE)

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_phase1_epoch(model, use_graph, X, y, starts, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                if use_graph:
                    ew_seq = gather_station_edges(starts[mb_idx]).to(DEVICE)
                    pred = model(xb, edge_index, ew_seq)
                else:
                    pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_phase1(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_phase1_epoch(model, use_graph, Xtr_t, ytr_reg_t, starts_train_t, opt, train=True)
        val_loss = run_phase1_epoch(model, use_graph, Xva_t, yva_reg_t, starts_val_t, opt, train=False)
        print(f"[PHASE1 {name}] epoch {epoch}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    print(f"[PHASE1 {name}] BEST epoch={best_epoch}  val_pinball={best_val:.4f}\n", flush=True)
    return model

def run_phase2_epoch(model, use_graph, X, y, starts, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    probs_list = [] if return_probs else None
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                if use_graph:
                    ew_station = gather_station_edges(starts[mb_idx]).to(DEVICE)
                    ew_region = gather_region_edges(starts[mb_idx]).to(DEVICE)
                    logits = model(xb, ew_station, ew_region)
                else:
                    logits = model(xb)
                loss = region_criterion(logits, yb)
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            if return_probs:
                probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    avg_loss = total_loss / total_n
    if return_probs:
        return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

@torch.no_grad()
def predict_proba_phase2(model, use_graph, X, starts, micro_batch=16):
    model.eval()
    n = X.shape[0]
    preds = []
    for start in range(0, n, micro_batch):
        xb = add_static(X[start:start + micro_batch]).to(DEVICE)
        if use_graph:
            ew_station = gather_station_edges(starts[start:start + micro_batch]).to(DEVICE)
            ew_region = gather_region_edges(starts[start:start + micro_batch]).to(DEVICE)
            logits = model(xb, ew_station, ew_region)
        else:
            logits = model(xb)
        preds.append(torch.sigmoid(logits).cpu())
    return torch.cat(preds, dim=0).numpy()

def train_phase2(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val_auc_pr, best_epoch = -1.0, -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    y_val_flat = y2va_cls_t.numpy().ravel()
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_phase2_epoch(model, use_graph, X2tr_t, y2tr_cls_t, starts2_train_t, opt, train=True)
        val_loss, val_probs = run_phase2_epoch(model, use_graph, X2va_t, y2va_cls_t, starts2_val_t, opt, train=False, return_probs=True)
        val_auc_pr = average_precision_score(y_val_flat, val_probs.ravel())
        print(f"[PHASE2 {name}] epoch {epoch}  train={train_loss:.4f}  val_loss={val_loss:.4f}  val_AUC-PR={val_auc_pr:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_auc_pr > best_val_auc_pr:
            best_val_auc_pr, best_epoch = val_auc_pr, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    proba = predict_proba_phase2(model, use_graph, X2te_t, starts2_test_t)
    y_true = y2te_cls_t.numpy().ravel()
    y_pred_proba = proba.ravel()
    auc_roc = roc_auc_score(y_true, y_pred_proba)
    auc_pr = average_precision_score(y_true, y_pred_proba)
    pred_pos = y_pred_proba >= 0.5
    actual_pos = y_true >= 0.5
    tp = int(np.sum(pred_pos & actual_pos)); fp = int(np.sum(pred_pos & ~actual_pos))
    fn = int(np.sum(~pred_pos & actual_pos)); tn = int(np.sum(~pred_pos & ~actual_pos))
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    print(f"[PHASE2 {name}] BEST epoch={best_epoch} (val_AUC-PR={best_val_auc_pr:.4f})  "
          f"AUC-ROC={auc_roc:.4f}  AUC-PR={auc_pr:.4f}  P={precision:.4f}  R={recall:.4f}  (tp={tp} fp={fp} fn={fn} tn={tn})\n", flush=True)
    return {"name": name, "auc_roc": auc_roc, "auc_pr": auc_pr, "precision": precision, "recall": recall}

seed = 0
torch.manual_seed(seed); np.random.seed(seed)
phase1_wind = train_phase1("phase1_quantile_gcn", StationQuantileGCN(n_feats), use_graph=True)
torch.manual_seed(seed); np.random.seed(seed)
phase1_nograph = train_phase1("phase1_quantile_mlp", StationQuantileMLP(n_feats), use_graph=False)

station_conv_copy = copy.deepcopy(phase1_wind.station_conv)
fc1_copy, fc2_copy = copy.deepcopy(phase1_nograph.fc1), copy.deepcopy(phase1_nograph.fc2)

del Xtr_t, Xva_t, ytr_reg_t, yva_reg_t, starts_train_t, starts_val_t, phase1_wind, phase1_nograph
gc.collect()
if DEVICE.type == "mps":
    torch.mps.empty_cache()

phase2_wind_model = RegionFromQuantileGCN(station_conv_copy).to(DEVICE)
res_wind = train_phase2("phase2_region_gcn_transport", phase2_wind_model, use_graph=True)

phase2_nograph_model = RegionFromQuantileMLP(fc1_copy, fc2_copy).to(DEVICE)
res_nograph = train_phase2("phase2_region_mlp_transport", phase2_nograph_model, use_graph=False)

print("\n=== transport-driven-subset comparison (3 epochs, 1 seed -- directional only) ===")
print(pd.DataFrame([res_wind, res_nograph]).to_string(index=False))


station graph: 21414 directed edges
representative station per region: {'Seoul': np.int64(0), 'Busan': np.int64(67), 'Daegu': np.int64(111), 'Incheon': np.int64(162), 'Gwangju': np.int64(91), 'Daejeon': np.int64(127), 'Ulsan': np.int64(76), 'Sejong': np.int64(140), 'Gyeonggi': np.int64(49), 'Gangwon': np.int64(54), 'Chungbuk': np.int64(134), 'Chungnam': np.int64(138), 'Jeonbuk': np.int64(96), 'Jeonnam': np.int64(101), 'Gyeongbuk': np.int64(150), 'Gyeongnam': np.int64(81), 'Jeju': np.int64(108)}
region graph: 272 directed edges (fully connected, 17 regions)
hours with any-region transport signature: 0.6331 (20063 train / 6142 test)
windows dropped (split boundary): 2698, phase-2 filtered out (not transport-driven): 18322
PHASE1 windows: train=30543, val=9251
PHASE2 windows: train=19322, val=6533, test=5662, n_feats=23
pos_weight for region BCE on transport-filtered subset (capped at 50): 50.00
device=mps
[PHASE1 phase1_quantile_gcn] epoch 1  train=0.1945  val=0.2135  (284s)
[PHASE1 phas

In [2]:
# ================================================================
# Re-derive ONLY what was freed for memory (Phase 1's standardized
# features + windows) -- reuses the already-built station/region
# graphs, edge weights, Phase 2 windows, and model classes from the
# prior run without recomputing any of that. Then runs the full
# comparison at 7 epochs, 2 seeds.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score

# ---- re-derive Phase 1 standardized features + windows only ----
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

_time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
_dt_index = _time_panels["PM25"].index
_wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(_dt_index).to_numpy().astype(float)
_wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(_dt_index).to_numpy().astype(float)
_blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(_dt_index).to_numpy().astype(float)
_wdir_sin = np.sin(np.radians(_wind_dir_arr))
_wdir_cos = np.cos(np.radians(_wind_dir_arr))
_doy = _dt_index.dayofyear.to_numpy().astype(float)
_season_sin = np.tile(np.sin(2 * np.pi * _doy / 365.25)[:, None], (1, n_stations))
_season_cos = np.tile(np.cos(2 * np.pi * _doy / 365.25)[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
_time_arr = np.stack([_time_panels[c].to_numpy() for c in TIME_FEATS] +
                      [_wind_speed_arr, _wdir_sin, _wdir_cos, _blh_arr, _season_sin, _season_cos], axis=-1)
del _time_panels, _wind_speed_arr, _wind_dir_arr, _blh_arr, _wdir_sin, _wdir_cos, _season_sin, _season_cos
gc.collect()

_time_train = _time_arr[TRAIN_MASK]
_t_mean = np.nanmean(_time_train, axis=(0, 1), keepdims=True)
_t_std = np.nanstd(_time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((_time_arr - _t_mean) / _t_std, nan=0.0)
del _time_arr, _time_train
gc.collect()
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

p1_buckets = {0: ([], [], []), 1: ([], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start not in p1_buckets:
        continue
    Xl, yreg_l, sl = p1_buckets[s_start]
    Xl.append(time_arr_std[t:t + WINDOW]); yreg_l.append(time_arr_std[target_t, :, pm25_col_idx]); sl.append(t)
X_train, yreg_train, starts_train = (np.stack(v) for v in p1_buckets[0])
X_val, yreg_val, starts_val = (np.stack(v) for v in p1_buckets[1])
del p1_buckets, time_arr_std
gc.collect()
print(f"PHASE1 windows rebuilt: train={len(X_train)}, val={len(X_val)}")

Xtr_t, ytr_reg_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(yreg_train, dtype=torch.float32)
Xva_t, yva_reg_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(yreg_val, dtype=torch.float32)
starts_train_t, starts_val_t = torch.tensor(starts_train, dtype=torch.long), torch.tensor(starts_val, dtype=torch.long)
del X_train, X_val, yreg_train, yreg_val, starts_train, starts_val
gc.collect()

# ---- everything below reuses what's already in memory from the last run:
# edge_index, region_edge_index, edge_weight_by_hour_t, region_edge_weight_by_hour_t,
# static_tensor, region_membership_t, X2tr_t/y2tr_cls_t/X2va_t/y2va_cls_t/X2te_t/y2te_cls_t,
# starts2_*_t, POS_WEIGHT, n_feats, n_regions, DEVICE, model class definitions
# (WindConvLayer, AttentionPool, StationQuantileGCN, StationQuantileMLP,
# RegionFromQuantileGCN, RegionFromQuantileMLP), and helper functions
# (add_static, batch_edge_index, gather_station_edges, gather_region_edges, region_criterion)

CKPT_DIR = f"{BASE}/ckpt_transport_test"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 7, 1e-4   # 7 epochs now

def run_phase1_epoch(model, use_graph, X, y, starts, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                if use_graph:
                    ew_seq = gather_station_edges(starts[mb_idx]).to(DEVICE)
                    pred = model(xb, edge_index, ew_seq)
                else:
                    pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_phase1(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_phase1_epoch(model, use_graph, Xtr_t, ytr_reg_t, starts_train_t, opt, train=True)
        val_loss = run_phase1_epoch(model, use_graph, Xva_t, yva_reg_t, starts_val_t, opt, train=False)
        print(f"[PHASE1 {name}] epoch {epoch}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    print(f"[PHASE1 {name}] BEST epoch={best_epoch}  val_pinball={best_val:.4f}\n", flush=True)
    return model

def run_phase2_epoch(model, use_graph, X, y, starts, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    probs_list = [] if return_probs else None
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                if use_graph:
                    ew_station = gather_station_edges(starts[mb_idx]).to(DEVICE)
                    ew_region = gather_region_edges(starts[mb_idx]).to(DEVICE)
                    logits = model(xb, ew_station, ew_region)
                else:
                    logits = model(xb)
                loss = region_criterion(logits, yb)
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            if return_probs:
                probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    avg_loss = total_loss / total_n
    if return_probs:
        return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

@torch.no_grad()
def predict_proba_phase2(model, use_graph, X, starts, micro_batch=16):
    model.eval()
    n = X.shape[0]
    preds = []
    for start in range(0, n, micro_batch):
        xb = add_static(X[start:start + micro_batch]).to(DEVICE)
        if use_graph:
            ew_station = gather_station_edges(starts[start:start + micro_batch]).to(DEVICE)
            ew_region = gather_region_edges(starts[start:start + micro_batch]).to(DEVICE)
            logits = model(xb, ew_station, ew_region)
        else:
            logits = model(xb)
        preds.append(torch.sigmoid(logits).cpu())
    return torch.cat(preds, dim=0).numpy()

def train_phase2(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val_auc_pr, best_epoch = -1.0, -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    y_val_flat = y2va_cls_t.numpy().ravel()
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_phase2_epoch(model, use_graph, X2tr_t, y2tr_cls_t, starts2_train_t, opt, train=True)
        val_loss, val_probs = run_phase2_epoch(model, use_graph, X2va_t, y2va_cls_t, starts2_val_t, opt, train=False, return_probs=True)
        val_auc_pr = average_precision_score(y_val_flat, val_probs.ravel())
        print(f"[PHASE2 {name}] epoch {epoch}  train={train_loss:.4f}  val_loss={val_loss:.4f}  val_AUC-PR={val_auc_pr:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_auc_pr > best_val_auc_pr:
            best_val_auc_pr, best_epoch = val_auc_pr, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    proba = predict_proba_phase2(model, use_graph, X2te_t, starts2_test_t)
    y_true = y2te_cls_t.numpy().ravel()
    y_pred_proba = proba.ravel()
    auc_roc = roc_auc_score(y_true, y_pred_proba)
    auc_pr = average_precision_score(y_true, y_pred_proba)
    pred_pos = y_pred_proba >= 0.5
    actual_pos = y_true >= 0.5
    tp = int(np.sum(pred_pos & actual_pos)); fp = int(np.sum(pred_pos & ~actual_pos))
    fn = int(np.sum(~pred_pos & actual_pos)); tn = int(np.sum(~pred_pos & ~actual_pos))
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    print(f"[PHASE2 {name}] BEST epoch={best_epoch} (val_AUC-PR={best_val_auc_pr:.4f})  "
          f"AUC-ROC={auc_roc:.4f}  AUC-PR={auc_pr:.4f}  P={precision:.4f}  R={recall:.4f}  (tp={tp} fp={fp} fn={fn} tn={tn})\n", flush=True)
    return {"name": name, "auc_roc": auc_roc, "auc_pr": auc_pr, "precision": precision, "recall": recall}

SEEDS = [0, 1]
all_results = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    phase1_wind = train_phase1(f"phase1_quantile_gcn_seed{seed}", StationQuantileGCN(n_feats), use_graph=True)
    torch.manual_seed(seed); np.random.seed(seed)
    phase1_nograph = train_phase1(f"phase1_quantile_mlp_seed{seed}", StationQuantileMLP(n_feats), use_graph=False)

    station_conv_copy = copy.deepcopy(phase1_wind.station_conv)
    fc1_copy, fc2_copy = copy.deepcopy(phase1_nograph.fc1), copy.deepcopy(phase1_nograph.fc2)
    del phase1_wind, phase1_nograph
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    phase2_wind_model = RegionFromQuantileGCN(station_conv_copy).to(DEVICE)
    res_wind = train_phase2(f"phase2_region_gcn_transport_seed{seed}", phase2_wind_model, use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_graph"
    all_results.append(res_wind)

    phase2_nograph_model = RegionFromQuantileMLP(fc1_copy, fc2_copy).to(DEVICE)
    res_nograph = train_phase2(f"phase2_region_mlp_transport_seed{seed}", phase2_nograph_model, use_graph=False)
    res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    all_results.append(res_nograph)

results_df = pd.DataFrame(all_results)
print(results_df[["model", "seed", "auc_roc", "auc_pr", "precision", "recall"]].to_string(index=False))
print("\n=== transport-driven-subset comparison, 7 epochs, 2 seeds ===")
print(results_df.groupby("model")[["auc_roc", "auc_pr", "precision", "recall"]].agg(["mean", "std"]))

print("\n=== paired comparison ===")
for metric in ["auc_roc", "auc_pr", "recall", "precision"]:
    pivot = results_df.pivot(index="seed", columns="model", values=metric)
    t, p = stats.ttest_rel(pivot["wind_graph"], pivot["no_graph"])
    wins = int((pivot["wind_graph"] > pivot["no_graph"]).sum())
    print(f"[{metric}] wind={pivot['wind_graph'].mean():.4f}  no_graph={pivot['no_graph'].mean():.4f}  "
          f"paired t={t:.3f} p={p:.4f}  wind_wins={wins}/2")


PHASE1 windows rebuilt: train=30543, val=9251
[PHASE1 phase1_quantile_gcn_seed0] epoch 1  train=0.1945  val=0.2135  (228s)
[PHASE1 phase1_quantile_gcn_seed0] epoch 2  train=0.1854  val=0.2127  (214s)
[PHASE1 phase1_quantile_gcn_seed0] epoch 3  train=0.1829  val=0.2143  (205s)
[PHASE1 phase1_quantile_gcn_seed0] epoch 4  train=0.1813  val=0.2155  (205s)
[PHASE1 phase1_quantile_gcn_seed0] epoch 5  train=0.1801  val=0.2181  (204s)
[PHASE1 phase1_quantile_gcn_seed0] epoch 6  train=0.1792  val=0.2166  (205s)
[PHASE1 phase1_quantile_gcn_seed0] epoch 7  train=0.1782  val=0.2182  (205s)
[PHASE1 phase1_quantile_gcn_seed0] BEST epoch=2  val_pinball=0.2127

[PHASE1 phase1_quantile_mlp_seed0] epoch 1  train=0.1954  val=0.2185  (41s)
[PHASE1 phase1_quantile_mlp_seed0] epoch 2  train=0.1891  val=0.2188  (42s)
[PHASE1 phase1_quantile_mlp_seed0] epoch 3  train=0.1871  val=0.2165  (41s)
[PHASE1 phase1_quantile_mlp_seed0] epoch 4  train=0.1855  val=0.2184  (41s)
[PHASE1 phase1_quantile_mlp_seed0] epoch 5

In [7]:
# ================================================================
# 2) Every other variable in the dataset vs PM2.5: pooled correlation
# (raw and deseasonalized) for time-varying predictors, and
# correlation against per-station mean PM2.5 for static covariates.
# Flags expected near-circular relationships (PM10) rather than
# treating them as a genuine finding.
# ================================================================
TIME_VARYING_COLS = ["SO2", "CO", "O3", "NO2", "PM10",
                      "windspeed_10m", "surface_pressure", "temperature_2m",
                      "relative_humidity_2m", "precipitation",
                      "boundary_layer_height", "cloud_cover"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

def pooled_corr(a_wide, b_wide):
    a, b = a_wide.to_numpy().ravel(), b_wide.to_numpy().ravel()
    valid = ~(np.isnan(a) | np.isnan(b))
    return np.corrcoef(a[valid], b[valid])[0, 1] if valid.sum() > 1 else np.nan

print("=== Time-varying predictors vs PM2.5 (pooled across all station-hours) ===")
results = []
for col in TIME_VARYING_COLS:
    wide_var = df.pivot(index="Datetime", columns="Station_ID", values=col)[station_order]
    raw_corr = pooled_corr(pm25_wide, wide_var)
    resid_var = wide_var.sub(wide_var.mean(axis=1), axis=0)
    deseason_corr = pooled_corr(resid, resid_var)
    flag = "  <-- PM10 physically contains PM2.5, expect near-circular correlation" if col == "PM10" else ""
    results.append({"variable": col, "raw_corr": raw_corr, "deseasonalized_corr": deseason_corr, "note": flag})

# wind direction is circular -- plain correlation isn't meaningful, use sin/cos components
wdir_wide = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order]
wdir_rad = np.radians(wdir_wide)
for label, comp in [("winddirection_10m (sin)", np.sin(wdir_rad)), ("winddirection_10m (cos)", np.cos(wdir_rad))]:
    raw_corr = pooled_corr(pm25_wide, comp)
    comp_resid = comp.sub(comp.mean(axis=1), axis=0)
    deseason_corr = pooled_corr(resid, comp_resid)
    results.append({"variable": label, "raw_corr": raw_corr, "deseasonalized_corr": deseason_corr, "note": ""})

results_df = pd.DataFrame(results).sort_values("deseasonalized_corr", key=abs, ascending=False)
print(results_df.to_string(index=False))

print("\n=== Static covariates vs per-station mean PM2.5 ===")
station_mean_pm25 = pm25_wide.mean(axis=0)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]

static_results = []
for col in STATIC_COLS:
    c = np.corrcoef(static_df[col].to_numpy(), station_mean_pm25.to_numpy())[0, 1]
    static_results.append({"variable": col, "corr_with_station_mean_PM25": c})

static_results_df = pd.DataFrame(static_results).sort_values("corr_with_station_mean_PM25", key=abs, ascending=False)
print(static_results_df.to_string(index=False))


=== Time-varying predictors vs PM2.5 (pooled across all station-hours) ===
               variable  raw_corr  deseasonalized_corr                                                                   note
                   PM10  0.757078             0.698141   <-- PM10 physically contains PM2.5, expect near-circular correlation
                     CO  0.519613             0.307338                                                                       
                    NO2  0.444787             0.267554                                                                       
                    SO2  0.267276             0.149887                                                                       
         temperature_2m -0.201745            -0.097103                                                                       
  boundary_layer_height -0.204373            -0.092809                                                                       
          windspeed_10m -0.214496          

In [ ]:
# ================================================================
# Two-phase pipeline, as requested:
#   PHASE 1: train the SENSOR-LEVEL wind graph (WindConvLayer, same
#     250km cutoff, GRAPH_RECENT_HOURS=18 fix, connectivity branch)
#     using PINBALL LOSS / quantile regression -- this is the exact
#     setup that beat no_graph statistically significantly earlier
#     (t=5.68, p=0.0047). Matched no-graph GRU trained the same way.
#   PHASE 2: take the trained station-level encoder from Phase 1,
#     MAX-POOL its per-timestep hidden states into region-level
#     features (17 regions), connect regions HOURLY via a wind-
#     conditioned region graph (same formula, region-averaged wind),
#     run a region GRU, and train a region-level classification head
#     (2h-sustained advisory_75, matching the real system) -- the
#     station encoder continues to fine-tune during this phase.
#   Matched no-graph pipeline gets the identical two-phase structure
#   at every stage, differing only in whether graph edges are used.
#
# 2 seeds, to keep runtime manageable. Expect roughly 60-90 minutes
# total (Phase 1 + Phase 2, both models, both seeds) -- this is a
# bigger combined pipeline than previous single-stage runs.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"station graph: {E} directed edges")

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
R_E = len(r_src_idx)
region_edge_index = torch.tensor(np.stack([r_src_idx, r_dst_idx]), dtype=torch.long)
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)
print(f"region graph: {R_E} directed edges (fully connected, {n_regions} regions)")

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months_arr = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

def regional_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_mean(wind_speed_arr)
region_wdir_sin = regional_mean(wdir_sin_station)
region_wdir_cos = regional_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360
r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_edge_weight_by_hour = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)

year_offset = years - years.min()
block_id = year_offset * 12 + (months_arr - 1)
n_blocks = int(block_id.max()) + 1
rng = np.random.RandomState(0)
block_order = rng.permutation(n_blocks)
n_train_blocks = int(round(0.6 * n_blocks))
n_val_blocks = int(round(0.2 * n_blocks))
block_to_split = np.empty(n_blocks, dtype=int)
block_to_split[block_order[:n_train_blocks]] = 0
block_to_split[block_order[n_train_blocks:n_train_blocks + n_val_blocks]] = 1
block_to_split[block_order[n_train_blocks + n_val_blocks:]] = 2
split_id_per_hour = block_to_split[block_id]
TRAIN_MASK = split_id_per_hour == 0

train_nonzero = edge_weight_by_hour[TRAIN_MASK][edge_weight_by_hour[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
region_train_nonzero = region_edge_weight_by_hour[TRAIN_MASK][region_edge_weight_by_hour[TRAIN_MASK] > 0]
region_weight_scale = region_train_nonzero.std()
region_edge_weight_by_hour = (region_edge_weight_by_hour / region_weight_scale).astype(np.float32)

wdir_sin, wdir_cos = wdir_sin_station, wdir_cos_station
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats

pm25_raw_arr = time_panels["PM25"].to_numpy()
region_pm25 = regional_mean(pm25_raw_arr)

time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                     [wind_speed_arr, wdir_sin, wdir_cos, blh_arr, season_sin, season_cos], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[TRAIN_MASK]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

pm25_col_idx = TIME_FEATS_FULL.index("PM25")

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

def build_windows_interleaved():
    buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
    n_dropped = 0
    for t in range(0, n_time - WINDOW - HORIZON + 1):
        target_t = t + WINDOW + HORIZON - 1
        s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
        if s_start != s_target:
            n_dropped += 1
            continue
        Xl, yreg_l, ycls_l, sl = buckets[s_start]
        Xl.append(time_arr_std[t:t + WINDOW])
        yreg_l.append(time_arr_std[target_t, :, pm25_col_idx])       # station-level continuous target (standardized)
        ycls_l.append(region_episode_label[target_t])                 # region-level classification target
        sl.append(t)
    print(f"windows dropped (straddling a split boundary): {n_dropped}")
    return {k: (np.stack(v[0]), np.stack(v[1]), np.stack(v[2]), np.array(v[3])) for k, v in buckets.items()}

windows_by_split = build_windows_interleaved()
X_train, yreg_train, ycls_train, starts_train = windows_by_split[0]
X_val, yreg_val, ycls_val, starts_val = windows_by_split[1]
X_test, yreg_test, ycls_test, starts_test = windows_by_split[2]
print(f"windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, n_feats={n_feats}, n_regions={n_regions}")

Xtr_t, ytr_reg_t, ytr_cls_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(yreg_train, dtype=torch.float32), torch.tensor(ycls_train, dtype=torch.float32)
Xva_t, yva_reg_t, yva_cls_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(yreg_val, dtype=torch.float32), torch.tensor(ycls_val, dtype=torch.float32)
Xte_t, yte_reg_t, yte_cls_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(yreg_test, dtype=torch.float32), torch.tensor(ycls_test, dtype=torch.float32)
starts_train_t, starts_val_t, starts_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts_train, starts_val, starts_test))

POS_WEIGHT = min(float((ytr_cls_t.numel() - ytr_cls_t.sum()) / ytr_cls_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight for region BCE (capped at 50): {POS_WEIGHT:.2f}")

edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
region_edge_weight_by_hour_t = torch.tensor(region_edge_weight_by_hour)

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def make_gather_fn(ew_by_hour_t):
    def gather_edge_weight_seq(starts_subset):
        idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
        ew = ew_by_hour_t[idx]
        if GRAPH_RECENT_HOURS < WINDOW:
            ew = ew.clone()
            ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
        return ew
    return gather_edge_weight_seq

gather_station_edges = make_gather_fn(edge_weight_by_hour_t)
gather_region_edges = make_gather_fn(region_edge_weight_by_hour_t)

def pool_station_hidden_to_region_max(h_station):   # (B, N, H) -> (B, n_regions, H), MAX pooling
    B, N, H = h_station.shape
    mask = region_membership_t.t().unsqueeze(0).unsqueeze(-1)              # (1, n_regions, N, 1)
    h_expanded = h_station.unsqueeze(1).expand(B, n_regions, N, H)         # (B, n_regions, N, H)
    neg_inf = torch.finfo(h_station.dtype).min
    h_masked = torch.where(mask.bool(), h_expanded, torch.full_like(h_expanded, neg_inf))
    pooled, _ = h_masked.max(dim=2)
    return pooled

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

# ---------------- PHASE 1 models: station-level quantile regression ----------------
class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        raw = self.head(embed)
        return monotonic_quantiles(raw)

class StationQuantileMLP(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        raw = self.head(embed)
        return monotonic_quantiles(raw)

# ---------------- PHASE 2 models: region classification on top of the phase-1 station encoder ----------------
class RegionFromQuantileGCN(nn.Module):
    def __init__(self, station_conv, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.station_conv = station_conv    # copied from trained Phase-1 model, continues to fine-tune
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, station_edge_weight_seq, region_edge_weight_seq):
        B, W, N, Fin = x_window.shape
        station_ei_b = batch_edge_index(edge_index, N, B)
        region_ei_b = batch_edge_index(region_edge_index, n_regions, B)
        num_station_nodes = B * N
        num_region_nodes = B * n_regions
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = pool_station_hidden_to_region_max(h_station).reshape(B * n_regions, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_regions, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_regions, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_regions, -1))
        return self.region_head(embed).squeeze(-1)

class RegionFromQuantileMLP(nn.Module):
    def __init__(self, fc1, fc2, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = fc1    # copied from trained Phase-1 no-graph model
        self.fc2 = fc2
        self.region_fc1 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h_station = torch.relu(self.fc2(h)).reshape(B, N, -1)
            h_region_pooled = pool_station_hidden_to_region_max(h_station).reshape(B * n_regions, -1)
            h_region = torch.relu(self.region_fc1(h_region_pooled))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_regions, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_regions, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_regions, -1))
        return self.region_head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_h36_blh_region_from_quantile"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 8, 1e-4
print(f"device={DEVICE}")
region_membership_t = region_membership_t.to(DEVICE)
edge_index = edge_index.to(DEVICE)
region_edge_index = region_edge_index.to(DEVICE)

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_phase1_epoch(model, use_graph, X, y, starts, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                if use_graph:
                    ew_seq = gather_station_edges(starts[mb_idx]).to(DEVICE)
                    pred = model(xb, edge_index, ew_seq)
                else:
                    pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_phase1(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_phase1_epoch(model, use_graph, Xtr_t, ytr_reg_t, starts_train_t, opt, train=True)
        val_loss = run_phase1_epoch(model, use_graph, Xva_t, yva_reg_t, starts_val_t, opt, train=False)
        print(f"[PHASE1 {name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    print(f"[PHASE1 {name}] BEST epoch={best_epoch}  val_pinball={best_val:.4f}\n", flush=True)
    return model

def run_phase2_epoch(model, use_graph, X, y, starts, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    probs_list = [] if return_probs else None
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                if use_graph:
                    ew_station = gather_station_edges(starts[mb_idx]).to(DEVICE)
                    ew_region = gather_region_edges(starts[mb_idx]).to(DEVICE)
                    logits = model(xb, ew_station, ew_region)
                else:
                    logits = model(xb)
                loss = region_criterion(logits, yb)
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            if return_probs:
                probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    avg_loss = total_loss / total_n
    if return_probs:
        return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

@torch.no_grad()
def predict_proba_phase2(model, use_graph, X, starts, micro_batch=16):
    model.eval()
    n = X.shape[0]
    preds = []
    for start in range(0, n, micro_batch):
        xb = add_static(X[start:start + micro_batch]).to(DEVICE)
        if use_graph:
            ew_station = gather_station_edges(starts[start:start + micro_batch]).to(DEVICE)
            ew_region = gather_region_edges(starts[start:start + micro_batch]).to(DEVICE)
            logits = model(xb, ew_station, ew_region)
        else:
            logits = model(xb)
        preds.append(torch.sigmoid(logits).cpu())
    return torch.cat(preds, dim=0).numpy()

def train_phase2(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val_auc_pr, best_epoch = -1.0, -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    y_val_flat = yva_cls_t.numpy().ravel()
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_phase2_epoch(model, use_graph, Xtr_t, ytr_cls_t, starts_train_t, opt, train=True)
        val_loss, val_probs = run_phase2_epoch(model, use_graph, Xva_t, yva_cls_t, starts_val_t, opt, train=False, return_probs=True)
        val_auc_pr = average_precision_score(y_val_flat, val_probs.ravel())
        print(f"[PHASE2 {name}] epoch {epoch:2d}  train={train_loss:.4f}  val_loss={val_loss:.4f}  val_AUC-PR={val_auc_pr:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_auc_pr > best_val_auc_pr:
            best_val_auc_pr, best_epoch = val_auc_pr, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    proba = predict_proba_phase2(model, use_graph, Xte_t, starts_test_t)
    y_true = yte_cls_t.numpy().ravel()
    y_pred_proba = proba.ravel()
    auc_roc = roc_auc_score(y_true, y_pred_proba)
    auc_pr = average_precision_score(y_true, y_pred_proba)
    pred_pos = y_pred_proba >= 0.5
    actual_pos = y_true >= 0.5
    tp = int(np.sum(pred_pos & actual_pos)); fp = int(np.sum(pred_pos & ~actual_pos))
    fn = int(np.sum(~pred_pos & actual_pos)); tn = int(np.sum(~pred_pos & ~actual_pos))
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    print(f"[PHASE2 {name}] BEST epoch={best_epoch} (val_AUC-PR={best_val_auc_pr:.4f})  "
          f"AUC-ROC={auc_roc:.4f}  AUC-PR={auc_pr:.4f}  P={precision:.4f}  R={recall:.4f}  (tp={tp} fp={fp} fn={fn} tn={tn})\n", flush=True)
    return {"name": name, "best_epoch": best_epoch, "auc_roc": auc_roc, "auc_pr": auc_pr, "precision": precision, "recall": recall}

SEEDS = [0, 1]
all_results = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    phase1_wind = train_phase1(f"phase1_quantile_gcn_seed{seed}", StationQuantileGCN(n_feats), use_graph=True)
    phase2_wind_model = RegionFromQuantileGCN(copy.deepcopy(phase1_wind.station_conv)).to(DEVICE)
    res_wind = train_phase2(f"phase2_region_gcn_seed{seed}", phase2_wind_model, use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_graph"
    all_results.append(res_wind)

    torch.manual_seed(seed); np.random.seed(seed)
    phase1_nograph = train_phase1(f"phase1_quantile_mlp_seed{seed}", StationQuantileMLP(n_feats), use_graph=False)
    phase2_nograph_model = RegionFromQuantileMLP(copy.deepcopy(phase1_nograph.fc1), copy.deepcopy(phase1_nograph.fc2)).to(DEVICE)
    res_nograph = train_phase2(f"phase2_region_mlp_seed{seed}", phase2_nograph_model, use_graph=False)
    res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    all_results.append(res_nograph)

results_df = pd.DataFrame(all_results)
print(results_df[["model", "seed", "best_epoch", "auc_roc", "auc_pr", "precision", "recall"]].to_string(index=False))
print("\n=== quantile-pretrained station encoder -> max-pool -> region wind graph vs no-graph (2 seeds) ===")
print(results_df.groupby("model")[["auc_roc", "auc_pr", "precision", "recall"]].agg(["mean", "std"]))

print("\n=== paired comparison ===")
for metric in ["auc_roc", "auc_pr", "recall", "precision"]:
    pivot = results_df.pivot(index="seed", columns="model", values=metric)
    t, p = stats.ttest_rel(pivot["wind_graph"], pivot["no_graph"])
    wins = int((pivot["wind_graph"] > pivot["no_graph"]).sum())
    print(f"[{metric}] wind={pivot['wind_graph'].mean():.4f}  no_graph={pivot['no_graph'].mean():.4f}  "
          f"paired t={t:.3f} p={p:.4f}  wind_wins={wins}/2")


station graph: 21414 directed edges
region graph: 272 directed edges (fully connected, 17 regions)
windows dropped (straddling a split boundary): 2698
windows: train=30543, val=9251, test=10045, n_feats=23, n_regions=17
pos_weight for region BCE (capped at 50): 50.00
device=mps
[PHASE1 phase1_quantile_gcn_seed0] epoch  1  train=0.1945  val=0.2135  (239s)
[PHASE1 phase1_quantile_gcn_seed0] epoch  2  train=0.1854  val=0.2127  (1022s)
[PHASE1 phase1_quantile_gcn_seed0] epoch  3  train=0.1829  val=0.2142  (444s)
[PHASE1 phase1_quantile_gcn_seed0] epoch  4  train=0.1813  val=0.2152  (211s)


In [1]:
# ================================================================
# Same regional 2h-sustained classifier (advisory_75, matching the
# real system) as before. ONLY CHANGE: DIST_CUTOFF widened from
# 125.6km to 250km, so long-range regional propagation pairs like
# Jeonbuk-Gangwon (233km, confirmed beyond the old cutoff -- and
# exactly the kind of pair showing a real 24-25h staggered onset lag
# in the data) get a direct graph edge. RHO_KM (the distance-decay
# rate, 250.0) is left unchanged -- only the hard cutoff moved, so
# this tests specifically "does connecting the missing long-range
# pairs help" without also changing how sharply weight decays with
# distance.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os, time
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0   # CUTOFF widened from 125.6 -> 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges (was 11446 at 125.6km cutoff)")

# ---- region assignment (nearest centroid to Korea's 17 administrative regions) ----
REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)
print("stations per region:", {region_names[r]: int((station_region_idx == r).sum()) for r in range(n_regions)})

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_counts = region_membership.sum(axis=0)
region_membership_t = torch.tensor(region_membership)
region_counts_t = torch.tensor(region_counts)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months_arr = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

year_offset = years - years.min()
block_id = year_offset * 12 + (months_arr - 1)
n_blocks = int(block_id.max()) + 1
rng = np.random.RandomState(0)
block_order = rng.permutation(n_blocks)
TRAIN_FRAC, VAL_FRAC = 0.6, 0.2
n_train_blocks = int(round(TRAIN_FRAC * n_blocks))
n_val_blocks = int(round(VAL_FRAC * n_blocks))
block_to_split = np.empty(n_blocks, dtype=int)
block_to_split[block_order[:n_train_blocks]] = 0
block_to_split[block_order[n_train_blocks:n_train_blocks + n_val_blocks]] = 1
block_to_split[block_order[n_train_blocks + n_val_blocks:]] = 2
split_id_per_hour = block_to_split[block_id]
print(f"n_blocks={n_blocks}  train_blocks={n_train_blocks}  val_blocks={n_val_blocks}  test_blocks={n_blocks - n_train_blocks - n_val_blocks}")

TRAIN_MASK = split_id_per_hour == 0
train_nonzero = edge_weight_by_hour[TRAIN_MASK][edge_weight_by_hour[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats

pm25_raw_arr = time_panels["PM25"].to_numpy()
region_pm25 = np.zeros((n_time, n_regions), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    region_pm25[:, r] = np.nanmean(pm25_raw_arr[:, cols], axis=1)

time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                     [wind_speed_arr, wdir_sin, wdir_cos, blh_arr, season_sin, season_cos], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[TRAIN_MASK]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)
print(f"regional episode positive rate: train={region_episode_label[TRAIN_MASK].mean():.5f}  "
      f"val={region_episode_label[split_id_per_hour==1].mean():.5f}  test={region_episode_label[split_id_per_hour==2].mean():.5f}")

def build_windows_interleaved():
    buckets = {0: ([], [], []), 1: ([], [], []), 2: ([], [], [])}
    n_dropped = 0
    for t in range(0, n_time - WINDOW - HORIZON + 1):
        target_t = t + WINDOW + HORIZON - 1
        s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
        if s_start != s_target:
            n_dropped += 1
            continue
        Xl, yl, sl = buckets[s_start]
        Xl.append(time_arr[t:t + WINDOW])
        yl.append(region_episode_label[target_t])
        sl.append(t)
    print(f"windows dropped (straddling a split boundary): {n_dropped}")
    out = {}
    for split_val, (Xl, yl, sl) in buckets.items():
        out[split_val] = (np.stack(Xl), np.stack(yl), np.array(sl))
    return out

windows_by_split = build_windows_interleaved()
X_train, y_train, starts_train = windows_by_split[0]
X_val, y_val, starts_val = windows_by_split[1]
X_test, y_test, starts_test = windows_by_split[2]
print(f"windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, n_feats={n_feats}, n_regions={n_regions}")
print(f"positive rate by window: train={y_train.mean():.5f}  val={y_val.mean():.5f}  test={y_test.mean():.5f}")

Xtr_t, ytr_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)
Xva_t, yva_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
Xte_t, yte_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
starts_train_t, starts_val_t, starts_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts_train, starts_val, starts_test))

POS_WEIGHT = min(float((ytr_t.numel() - ytr_t.sum()) / ytr_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight for BCE (capped at 50): {POS_WEIGHT:.2f}")

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def gather_edge_weight_seq(starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = edge_weight_by_hour_t[idx]
    if GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

def pool_station_logits_to_region(station_logits):
    return (station_logits @ region_membership_t) / region_counts_t.unsqueeze(0)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1) if edge_weight_seq is not None else torch.ones(ei_b.shape[1], device=xt.device)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        station_logits = self.head(embed).squeeze(-1)
        return pool_station_logits_to_region(station_logits)

class TemporalOnlyGRU(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index=None, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        station_logits = self.head(embed).squeeze(-1)
        return pool_station_logits_to_region(station_logits)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_h36_blh_regional_clf_wide250"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 8, 1e-4
print(f"device={DEVICE}")
region_membership_t = region_membership_t.to(DEVICE)
region_counts_t = region_counts_t.to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_epoch(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS, return_probs=False):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    probs_list = [] if return_probs else None
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            with torch.set_grad_enabled(train):
                logits = model(xb, ei, ew_seq)
                loss = criterion(logits, yb)
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            if return_probs:
                probs_list.append(torch.sigmoid(logits).detach().cpu())
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    avg_loss = total_loss / total_n
    if return_probs:
        return avg_loss, torch.cat(probs_list, dim=0).numpy()
    return avg_loss

@torch.no_grad()
def predict_proba(model, use_graph, X, starts, micro_batch=16):
    model.eval()
    n = X.shape[0]
    preds = []
    for start in range(0, n, micro_batch):
        xb = add_static(X[start:start + micro_batch]).to(DEVICE)
        ew_seq = gather_edge_weight_seq(starts[start:start + micro_batch]).to(DEVICE) if use_graph else None
        ei = edge_index.to(DEVICE) if use_graph else None
        logits = model(xb, ei, ew_seq)
        preds.append(torch.sigmoid(logits).cpu())
    return torch.cat(preds, dim=0).numpy()

def train_model(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val_auc_pr, best_epoch, best_val_loss = -1.0, -1, None
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    y_val_flat = yva_t.numpy().ravel()
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_epoch(model, Xtr_t, ytr_t, starts_train_t, use_graph, opt, train=True)
        val_loss, val_probs = run_epoch(model, Xva_t, yva_t, starts_val_t, use_graph, opt, train=False, return_probs=True)
        val_auc_pr = average_precision_score(y_val_flat, val_probs.ravel())
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val_loss={val_loss:.4f}  val_AUC-PR={val_auc_pr:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_auc_pr > best_val_auc_pr:
            best_val_auc_pr, best_epoch, best_val_loss = val_auc_pr, epoch, val_loss
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch(model, Xte_t, yte_t, starts_test_t, use_graph, None, train=False)
    proba = predict_proba(model, use_graph, Xte_t, starts_test_t)
    y_true = yte_t.numpy().ravel()
    y_pred_proba = proba.ravel()
    auc_roc = roc_auc_score(y_true, y_pred_proba)
    auc_pr = average_precision_score(y_true, y_pred_proba)
    pred_pos = y_pred_proba >= 0.5
    actual_pos = y_true >= 0.5
    tp = int(np.sum(pred_pos & actual_pos)); fp = int(np.sum(pred_pos & ~actual_pos))
    fn = int(np.sum(~pred_pos & actual_pos)); tn = int(np.sum(~pred_pos & ~actual_pos))
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    print(f"[{name}] BEST epoch={best_epoch} (val_AUC-PR={best_val_auc_pr:.4f}, val_loss={best_val_loss:.4f})  test_loss={test_loss:.4f}  "
          f"AUC-ROC={auc_roc:.4f}  AUC-PR={auc_pr:.4f}  P={precision:.4f}  R={recall:.4f}  "
          f"(tp={tp} fp={fp} fn={fn} tn={tn})\n", flush=True)
    return {"name": name, "best_epoch": best_epoch, "val_auc_pr": best_val_auc_pr, "test_bce": test_loss,
            "auc_roc": auc_roc, "auc_pr": auc_pr, "precision": precision, "recall": recall,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}

SEEDS = [0, 1, 2, 3, 4]
all_results = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    res_wind = train_model(f"wind_gcn_h36_blh_regional_wide250_seed{seed}", WindTemporalGCN(n_feats), use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_graph"
    all_results.append(res_wind)

    torch.manual_seed(seed); np.random.seed(seed)
    res_nograph = train_model(f"no_graph_h36_blh_regional_wide250_seed{seed}", TemporalOnlyGRU(n_feats), use_graph=False)
    res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    all_results.append(res_nograph)

results_df = pd.DataFrame(all_results)
print(results_df[["model", "seed", "best_epoch", "val_auc_pr", "auc_roc", "auc_pr", "precision", "recall"]].to_string(index=False))
print("\n=== summary across seeds (regional, wide cutoff=250km, 2h-sustained, advisory_75) ===")
print(results_df.groupby("model")[["auc_roc", "auc_pr", "precision", "recall"]].agg(["mean", "std"]))

print("\n=== paired significance (wind vs no_graph) ===")
for metric in ["auc_roc", "auc_pr", "recall", "precision"]:
    pivot = results_df.pivot(index="seed", columns="model", values=metric)
    t, p = stats.ttest_rel(pivot["wind_graph"], pivot["no_graph"])
    wins = int((pivot["wind_graph"] > pivot["no_graph"]).sum())
    print(f"[{metric}] wind={pivot['wind_graph'].mean():.4f}  no_graph={pivot['no_graph'].mean():.4f}  "
          f"paired t={t:.3f} p={p:.4f}  wind_wins={wins}/5")


candidate graph: 21414 directed edges (was 11446 at 125.6km cutoff)
stations per region: {'Seoul': 37, 'Busan': 20, 'Daegu': 13, 'Incheon': 20, 'Gwangju': 8, 'Daejeon': 5, 'Ulsan': 9, 'Sejong': 2, 'Gyeonggi': 11, 'Gangwon': 4, 'Chungbuk': 10, 'Chungnam': 2, 'Jeonbuk': 8, 'Jeonnam': 11, 'Gyeongbuk': 4, 'Gyeongnam': 9, 'Jeju': 3}
n_blocks=72  train_blocks=43  val_blocks=14  test_blocks=15
regional episode positive rate: train=0.00451  val=0.01241  test=0.01556
windows dropped (straddling a split boundary): 2698
windows: train=30543, val=9251, test=10045, n_feats=23, n_regions=17
positive rate by window: train=0.00460  val=0.01333  test=0.01585
pos_weight for BCE (capped at 50): 50.00
device=mps
[wind_gcn_h36_blh_regional_wide250_seed0] epoch  1  train=0.4252  val_loss=1.0231  val_AUC-PR=0.0649  (238s)
[wind_gcn_h36_blh_regional_wide250_seed0] epoch  2  train=0.3443  val_loss=0.7688  val_AUC-PR=0.1139  (205s)
[wind_gcn_h36_blh_regional_wide250_seed0] epoch  3  train=0.3172  val_loss=0.668

KeyboardInterrupt: 

In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

pred_train = model_G.predict(X_train_arr)
pred_val = model_G.predict(X_val_arr)
pred_test = model_G.predict(X_test_arr)

vol_col = feature_cols.index("pm25_roll_std24")
vol_train, vol_val, vol_test = X_train_arr[:, vol_col], X_val_arr[:, vol_col], X_test_arr[:, vol_col]

THRESHOLDS = {"bad_36": 36.0, "very_bad_76": 76.0}
calibrators = {}

for name, c in THRESHOLDS.items():
    y_train_label = (y_train_arr > c).astype(int)
    y_val_label = (y_val_arr > c).astype(int)
    y_test_label = (y_test_arr > c).astype(int)

    print(f"\n=== threshold {name} (c={c}) ===")
    print(f"base rate: train={y_train_label.mean():.4f}  val={y_val_label.mean():.4f}  test={y_test_label.mean():.4f}")

    Zval = np.column_stack([pred_val, vol_val])
    S = LogisticRegression(max_iter=1000)
    S.fit(Zval, y_val_label)
    calibrators[name] = S

    Ztest = np.column_stack([pred_test, vol_test])
    p_test = S.predict_proba(Ztest)[:, 1]

    auc = roc_auc_score(y_test_label, p_test)
    pr_auc = average_precision_score(y_test_label, p_test)
    brier = brier_score_loss(y_test_label, p_test)
    print(f"test: AUC={auc:.4f}  PR-AUC={pr_auc:.4f}  Brier={brier:.4f}")

    # reliability check: bin predicted probability, compare to observed frequency
    bins = np.quantile(p_test, np.linspace(0, 1, 11))
    bins[-1] += 1e-6
    bin_idx = np.digitize(p_test, bins) - 1
    print("reliability (predicted vs observed, by decile of predicted prob):")
    for b in range(10):
        mask = bin_idx == b
        if mask.sum() > 0:
            print(f"  decile {b}: n={mask.sum():6d}  mean_pred={p_test[mask].mean():.4f}  observed_rate={y_test_label[mask].mean():.4f}")



=== threshold bad_36 (c=36.0) ===
base rate: train=0.1910  val=0.1553  test=0.0947
test: AUC=0.8427  PR-AUC=0.3869  Brier=0.0707
reliability (predicted vs observed, by decile of predicted prob):
  decile 0: n=308352  mean_pred=0.0172  observed_rate=0.0018
  decile 1: n=308352  mean_pred=0.0271  observed_rate=0.0049
  decile 2: n=308352  mean_pred=0.0371  observed_rate=0.0107
  decile 3: n=308352  mean_pred=0.0483  observed_rate=0.0207
  decile 4: n=308352  mean_pred=0.0618  observed_rate=0.0377
  decile 5: n=308352  mean_pred=0.0793  observed_rate=0.0593
  decile 6: n=308352  mean_pred=0.1032  observed_rate=0.0857
  decile 7: n=308352  mean_pred=0.1403  observed_rate=0.1225
  decile 8: n=308352  mean_pred=0.2127  observed_rate=0.1970
  decile 9: n=308352  mean_pred=0.4591  observed_rate=0.4066

=== threshold very_bad_76 (c=76.0) ===
base rate: train=0.0137  val=0.0218  test=0.0070
test: AUC=0.8376  PR-AUC=0.0508  Brier=0.0075
reliability (predicted vs observed, by decile of predicted 

In [14]:
resid_test6 = y_test6 - pred_test6
test_meta6 = test6[["Station_ID"]].copy()
test_meta6["resid"] = resid_test6
resid_wide6 = test_meta6.pivot_table(index=test_meta6.index, columns="Station_ID", values="resid")

resid_national_mean6 = resid_wide6.mean(axis=1)
resid_local6 = resid_wide6.sub(resid_national_mean6, axis=0).reindex(columns=station_cols)

total_var6 = resid_wide6.var().mean()
national_var6 = resid_national_mean6.var()
print(f"H=6: national common-factor variance: {national_var6:.3f} of {total_var6:.3f} ({national_var6/total_var6:.1%})")

LAG6 = 6  # the boundary of legitimate availability for an H=6 model
resid_local6_lag = resid_local6.shift(LAG6)
combined6 = pd.concat([resid_local6.add_suffix("_now"), resid_local6_lag.add_suffix("_lag")], axis=1)
full_corr6 = combined6.corr()
now_cols6 = [f"{c}_now" for c in station_cols]
lag_cols6 = [f"{c}_lag" for c in station_cols]
cross_corr6_flat = full_corr6.loc[now_cols6, lag_cols6].to_numpy()[mask_offdiag]

print(f"\nH=6 model: LAG={LAG6}h local residual correlation by distance (legitimately available at prediction time):")
for lo, hi in zip(fine_bins[:-1], fine_bins[1:]):
    m = (dist_flat_all >= lo) & (dist_flat_all < hi)
    if m.sum() > 0:
        print(f"{lo:3d}-{hi:3d}km: n_pairs={m.sum():5d}  mean_lagged_local_corr={cross_corr6_flat[m].mean():.4f}")


H=6: national common-factor variance: 23.223 of 78.113 (29.7%)

H=6 model: LAG=6h local residual correlation by distance (legitimately available at prediction time):
  0-  5km: n_pairs=  186  mean_lagged_local_corr=-0.0109
  5- 10km: n_pairs=  526  mean_lagged_local_corr=0.0133
 10- 15km: n_pairs=  638  mean_lagged_local_corr=0.0290
 15- 20km: n_pairs=  644  mean_lagged_local_corr=0.0421
 20- 25km: n_pairs=  652  mean_lagged_local_corr=0.0456
 25- 35km: n_pairs= 1346  mean_lagged_local_corr=0.0483
 35- 50km: n_pairs= 1404  mean_lagged_local_corr=0.0505
 50- 75km: n_pairs= 1504  mean_lagged_local_corr=0.0394
 75-100km: n_pairs= 2196  mean_lagged_local_corr=0.0398


In [16]:
from scipy import stats

LAG = 24
seoul_resid_lag = seoul_resid.shift(LAG)
diffs = []
for s in other_stations2:
    angle_diff = np.abs(((wind_seoul_test - (bearings[s] + 180)) + 180) % 360 - 180)
    downwind_mask = angle_diff <= TOLERANCE
    combined_s = pd.concat([resid_local[s].rename("target"), seoul_resid_lag.rename("seoul_lag"),
                             downwind_mask.rename("downwind")], axis=1).dropna()
    if combined_s["downwind"].sum() > 100 and (~combined_s["downwind"]).sum() > 100:
        c_down = combined_s.loc[combined_s["downwind"], "target"].corr(combined_s.loc[combined_s["downwind"], "seoul_lag"])
        c_other = combined_s.loc[~combined_s["downwind"], "target"].corr(combined_s.loc[~combined_s["downwind"], "seoul_lag"])
        if not (np.isnan(c_down) or np.isnan(c_other)):
            diffs.append(c_down - c_other)

diffs = np.array(diffs)
print(f"n_stations={len(diffs)}  mean(downwind - not_downwind)={diffs.mean():.4f}  std={diffs.std():.4f}")
t_stat, p_value = stats.ttest_1samp(diffs, 0)
print(f"paired t-test vs 0: t={t_stat:.3f}  p={p_value:.5f}")
print(f"fraction of stations where downwind > not_downwind: {(diffs > 0).mean():.1%}")


n_stations=151  mean(downwind - not_downwind)=0.0385  std=0.0665
paired t-test vs 0: t=7.084  p=0.00000
fraction of stations where downwind > not_downwind: 72.2%


In [19]:
val_valid_mask = ~np.isnan(correction_val_flat)
pred_val_corrected2 = pred_val.copy()
pred_val_corrected2[val_valid_mask] = pred_val[val_valid_mask] + alpha + beta * correction_val_flat[val_valid_mask]

vol_val_c = X_val_arr[:, vol_col]
vol_test_c = X_test_arr[:, vol_col]

for name, c in THRESHOLDS.items():
    y_val_label = (val["target"] > c).astype(int).to_numpy()
    y_test_label = (test["target"] > c).astype(int).to_numpy()
    trail_val = np.nan_to_num(val[f"trail_rate_{name}"].to_numpy(), nan=frac_exceeding.mean())
    trail_test = np.nan_to_num(test[f"trail_rate_{name}"].to_numpy(), nan=frac_exceeding.mean())

    Zval_orig = np.column_stack([pred_val, vol_val_c, trail_val])
    S_orig = LogisticRegression(max_iter=1000).fit(Zval_orig, y_val_label)
    p_test_orig = S_orig.predict_proba(np.column_stack([pred_test, vol_test_c, trail_test]))[:, 1]

    Zval_corr = np.column_stack([pred_val_corrected2, vol_val_c, trail_val])
    S_corr = LogisticRegression(max_iter=1000).fit(Zval_corr, y_val_label)
    p_test_corr = S_corr.predict_proba(np.column_stack([pred_test_corrected2, vol_test_c, trail_test]))[:, 1]

    print(f"\n=== {name} (c={c}) ===")
    print(f"AUC:    original={roc_auc_score(y_test_label, p_test_orig):.4f}   wind-corrected={roc_auc_score(y_test_label, p_test_corr):.4f}")
    print(f"PR-AUC: original={average_precision_score(y_test_label, p_test_orig):.4f}   wind-corrected={average_precision_score(y_test_label, p_test_corr):.4f}")
    print(f"Brier:  original={brier_score_loss(y_test_label, p_test_orig):.4f}   wind-corrected={brier_score_loss(y_test_label, p_test_corr):.4f}")



=== bad_36 (c=36.0) ===
AUC:    original=0.8426   wind-corrected=0.8419
PR-AUC: original=0.3849   wind-corrected=0.3845
Brier:  original=0.0704   wind-corrected=0.0705

=== very_bad_76 (c=76.0) ===
AUC:    original=0.8408   wind-corrected=0.8373
PR-AUC: original=0.0502   wind-corrected=0.0523
Brier:  original=0.0073   wind-corrected=0.0075


In [2]:
# ================================================================
# Leave-one-region-out edge ablation. Uses the ALREADY-TRAINED
# wide-cutoff (250km) wind_graph regional classifier checkpoint --
# no retraining, inference only. For each of the 17 regions as a
# candidate "source," zero out every outgoing edge whose origin
# station belongs to that region (across the whole test set), and
# measure how much every OTHER region's prediction quality degrades
# relative to baseline. A region whose downstream AUC-PR drops a lot
# when a specific source is silenced is functionally *depending* on
# that source through the graph's message-passing -- this is the
# direct test of whether edges carry real, usable information,
# independent of the aggregate wind_graph-vs-no_graph comparison.
#
# Uses the seed-0 checkpoint from ckpt_h36_blh_regional_clf_wide250/
# -- adjust CKPT_NAME below if you want a different seed, or wait
# for that run to finish if it hasn't yet (this script needs its
# checkpoint file to exist on disk).
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
CKPT_PATH = f"{BASE}/ckpt_h36_blh_regional_clf_wide250/wind_gcn_h36_blh_regional_wide250_seed0.pt"

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges")

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_counts = region_membership.sum(axis=0)
region_membership_t = torch.tensor(region_membership)
region_counts_t = torch.tensor(region_counts)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months_arr = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

year_offset = years - years.min()
block_id = year_offset * 12 + (months_arr - 1)
n_blocks = int(block_id.max()) + 1
rng = np.random.RandomState(0)
block_order = rng.permutation(n_blocks)
n_train_blocks = int(round(0.6 * n_blocks))
n_val_blocks = int(round(0.2 * n_blocks))
block_to_split = np.empty(n_blocks, dtype=int)
block_to_split[block_order[:n_train_blocks]] = 0
block_to_split[block_order[n_train_blocks:n_train_blocks + n_val_blocks]] = 1
block_to_split[block_order[n_train_blocks + n_val_blocks:]] = 2
split_id_per_hour = block_to_split[block_id]
TRAIN_MASK = split_id_per_hour == 0

train_nonzero = edge_weight_by_hour[TRAIN_MASK][edge_weight_by_hour[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats

pm25_raw_arr = time_panels["PM25"].to_numpy()
region_pm25 = np.zeros((n_time, n_regions), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    region_pm25[:, r] = np.nanmean(pm25_raw_arr[:, cols], axis=1)

time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                     [wind_speed_arr, wdir_sin, wdir_cos, blh_arr, season_sin, season_cos], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[TRAIN_MASK]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

X_list, y_list, starts_list = [], [], []
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    if split_id_per_hour[t] != 2 or split_id_per_hour[target_t] != 2:   # test split only
        continue
    X_list.append(time_arr[t:t + WINDOW])
    y_list.append(region_episode_label[target_t])
    starts_list.append(t)
X_test, y_test, starts_test = np.stack(X_list), np.stack(y_list), np.array(starts_list)
print(f"test windows: {len(X_test)}")
Xte_t = torch.tensor(X_test, dtype=torch.float32)
yte_t = torch.tensor(y_test, dtype=torch.float32)
starts_test_t = torch.tensor(starts_test, dtype=torch.long)

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def make_gather_fn(ew_by_hour_t):
    def gather_edge_weight_seq(starts_subset):
        idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
        ew = ew_by_hour_t[idx]
        if GRAPH_RECENT_HOURS < WINDOW:
            ew = ew.clone()
            ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
        return ew
    return gather_edge_weight_seq

def pool_station_logits_to_region(station_logits):
    return (station_logits @ region_membership_t) / region_counts_t.unsqueeze(0)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        station_logits = self.head(embed).squeeze(-1)
        return pool_station_logits_to_region(station_logits)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
region_membership_t = region_membership_t.to(DEVICE)
region_counts_t = region_counts_t.to(DEVICE)
print(f"device={DEVICE}")

model = WindTemporalGCN(n_feats).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

@torch.no_grad()
def predict_region_proba(ew_by_hour_t, micro_batch=16):
    gather_fn = make_gather_fn(ew_by_hour_t)
    n = Xte_t.shape[0]
    preds = []
    for start in range(0, n, micro_batch):
        xb = add_static(Xte_t[start:start + micro_batch]).to(DEVICE)
        ew_seq = gather_fn(starts_test_t[start:start + micro_batch]).to(DEVICE)
        ei = edge_index.to(DEVICE)
        logits = model(xb, ei, ew_seq)
        preds.append(torch.sigmoid(logits).cpu())
    return torch.cat(preds, dim=0).numpy()   # (n_test, n_regions)

edge_weight_full_t = torch.tensor(edge_weight_by_hour)
y_true = yte_t.numpy()   # (n_test, n_regions)

print("computing baseline predictions...")
baseline_proba = predict_region_proba(edge_weight_full_t)
baseline_auc_pr = {region_names[r]: average_precision_score(y_true[:, r], baseline_proba[:, r])
                    if y_true[:, r].sum() > 0 else np.nan for r in range(n_regions)}

print("\nrunning leave-one-region-out ablations (17 passes)...")
dependency_rows = []
for r_source in range(n_regions):
    source_name = region_names[r_source]
    source_stations = np.where(station_region_idx == r_source)[0]
    ablate_mask = np.isin(src_idx, source_stations)
    ew_ablated = edge_weight_by_hour.copy()
    ew_ablated[:, ablate_mask] = 0.0
    ew_ablated_t = torch.tensor(ew_ablated)

    ablated_proba = predict_region_proba(ew_ablated_t)

    for r_down in range(n_regions):
        if r_down == r_source:
            continue
        if y_true[:, r_down].sum() == 0:
            continue
        ablated_auc_pr = average_precision_score(y_true[:, r_down], ablated_proba[:, r_down])
        base = baseline_auc_pr[region_names[r_down]]
        delta = base - ablated_auc_pr   # positive = downwind region got WORSE without source's edges
        dependency_rows.append({"source": source_name, "downwind": region_names[r_down],
                                 "baseline_auc_pr": base, "ablated_auc_pr": ablated_auc_pr, "delta": delta})
    print(f"  done ablating {source_name} ({r_source+1}/{n_regions})", flush=True)

dep_df = pd.DataFrame(dependency_rows)
print("\n=== top 20 strongest source -> downwind dependencies (largest AUC-PR drop when source silenced) ===")
print(dep_df.sort_values("delta", ascending=False).head(20).to_string(index=False))

print("\n=== Seoul -> Gangwon specifically ===")
print(dep_df[(dep_df.source == "Seoul") & (dep_df.downwind == "Gangwon")].to_string(index=False))

dep_df.to_csv(f"{BASE}/region_dependency_ablation_wide250.csv", index=False)
print(f"\nfull results saved to {BASE}/region_dependency_ablation_wide250.csv")


candidate graph: 21414 directed edges
test windows: 10045
device=mps
computing baseline predictions...

running leave-one-region-out ablations (17 passes)...
  done ablating Seoul (1/17)
  done ablating Busan (2/17)
  done ablating Daegu (3/17)
  done ablating Incheon (4/17)
  done ablating Gwangju (5/17)
  done ablating Daejeon (6/17)
  done ablating Ulsan (7/17)
  done ablating Sejong (8/17)
  done ablating Gyeonggi (9/17)
  done ablating Gangwon (10/17)
  done ablating Chungbuk (11/17)
  done ablating Chungnam (12/17)
  done ablating Jeonbuk (13/17)
  done ablating Jeonnam (14/17)
  done ablating Gyeongbuk (15/17)
  done ablating Gyeongnam (16/17)
  done ablating Jeju (17/17)

=== top 20 strongest source -> downwind dependencies (largest AUC-PR drop when source silenced) ===
   source  downwind  baseline_auc_pr  ablated_auc_pr    delta
    Seoul   Gangwon         0.150296        0.139180 0.011115
    Seoul Gyeongbuk         0.035430        0.025545 0.009885
    Seoul  Gyeonggi      

In [3]:
# ================================================================
# Density-removal degradation curve for Gangwon's upwind corridor.
# Uses the same wide-cutoff (250km) wind_graph checkpoint, inference
# only -- no retraining. Identifies every station with a direct edge
# into any Gangwon station, then progressively removes them (only
# their edges INTO Gangwon -- surgical, since Gangwon's predicted
# probability is a pure function of Gangwon-station hidden states,
# which only depend on edges pointing INTO Gangwon stations) at
# several density levels, averaged over 10 random removal orderings
# per level to get an unbiased "how much does density matter" curve
# rather than one arbitrary removal sequence.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
CKPT_PATH = f"{BASE}/ckpt_h36_blh_regional_clf_wide250/wind_gcn_h36_blh_regional_wide250_seed0.pt"

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges")

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_counts = region_membership.sum(axis=0)
region_membership_t = torch.tensor(region_membership)
region_counts_t = torch.tensor(region_counts)

GANGWON_IDX = region_names.index("Gangwon")
gangwon_stations = np.where(station_region_idx == GANGWON_IDX)[0]
print(f"Gangwon stations: {gangwon_stations}  (n={len(gangwon_stations)})")

# every station with at least one edge INTO a Gangwon station
into_gangwon_mask = np.isin(dst_idx, gangwon_stations)
corridor_stations = np.unique(src_idx[into_gangwon_mask])
corridor_stations = corridor_stations[~np.isin(corridor_stations, gangwon_stations)]  # exclude Gangwon's own stations
N_corridor = len(corridor_stations)
print(f"corridor (upwind-connected) stations: {N_corridor}")
print("corridor stations by region:", {region_names[r]: int((station_region_idx[corridor_stations] == r).sum())
                                        for r in range(n_regions) if (station_region_idx[corridor_stations] == r).sum() > 0})

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months_arr = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

year_offset = years - years.min()
block_id = year_offset * 12 + (months_arr - 1)
n_blocks = int(block_id.max()) + 1
rng = np.random.RandomState(0)
block_order = rng.permutation(n_blocks)
n_train_blocks = int(round(0.6 * n_blocks))
n_val_blocks = int(round(0.2 * n_blocks))
block_to_split = np.empty(n_blocks, dtype=int)
block_to_split[block_order[:n_train_blocks]] = 0
block_to_split[block_order[n_train_blocks:n_train_blocks + n_val_blocks]] = 1
block_to_split[block_order[n_train_blocks + n_val_blocks:]] = 2
split_id_per_hour = block_to_split[block_id]
TRAIN_MASK = split_id_per_hour == 0

train_nonzero = edge_weight_by_hour[TRAIN_MASK][edge_weight_by_hour[TRAIN_MASK] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats

pm25_raw_arr = time_panels["PM25"].to_numpy()
region_pm25 = np.zeros((n_time, n_regions), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    region_pm25[:, r] = np.nanmean(pm25_raw_arr[:, cols], axis=1)

time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                     [wind_speed_arr, wdir_sin, wdir_cos, blh_arr, season_sin, season_cos], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[TRAIN_MASK]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
roll_min_fwd = roll_min_rev[::-1]
region_episode_label = (roll_min_fwd >= EVENT_THRESHOLD).astype(np.float32)

X_list, y_list, starts_list = [], [], []
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    if split_id_per_hour[t] != 2 or split_id_per_hour[target_t] != 2:
        continue
    X_list.append(time_arr[t:t + WINDOW])
    y_list.append(region_episode_label[target_t])
    starts_list.append(t)
X_test, y_test, starts_test = np.stack(X_list), np.stack(y_list), np.array(starts_list)
print(f"test windows: {len(X_test)}")
Xte_t = torch.tensor(X_test, dtype=torch.float32)
yte_t = torch.tensor(y_test, dtype=torch.float32)
starts_test_t = torch.tensor(starts_test, dtype=torch.long)
y_true_gangwon = y_test[:, GANGWON_IDX]
print(f"Gangwon test positive rate: {y_true_gangwon.mean():.5f}")

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def make_gather_fn(ew_by_hour_t):
    def gather_edge_weight_seq(starts_subset):
        idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
        ew = ew_by_hour_t[idx]
        if GRAPH_RECENT_HOURS < WINDOW:
            ew = ew.clone()
            ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
        return ew
    return gather_edge_weight_seq

def pool_station_logits_to_region(station_logits):
    return (station_logits @ region_membership_t) / region_counts_t.unsqueeze(0)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        station_logits = self.head(embed).squeeze(-1)
        return pool_station_logits_to_region(station_logits)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
region_membership_t = region_membership_t.to(DEVICE)
region_counts_t = region_counts_t.to(DEVICE)
print(f"device={DEVICE}")

model = WindTemporalGCN(n_feats).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

@torch.no_grad()
def predict_gangwon_proba(ew_by_hour_t, micro_batch=16):
    gather_fn = make_gather_fn(ew_by_hour_t)
    n = Xte_t.shape[0]
    preds = []
    for start in range(0, n, micro_batch):
        xb = add_static(Xte_t[start:start + micro_batch]).to(DEVICE)
        ew_seq = gather_fn(starts_test_t[start:start + micro_batch]).to(DEVICE)
        ei = edge_index.to(DEVICE)
        logits = model(xb, ei, ew_seq)
        preds.append(torch.sigmoid(logits[:, GANGWON_IDX]).cpu())
    return torch.cat(preds, dim=0).numpy()

DENSITY_FRACS = [1.0, 0.75, 0.5, 0.25, 0.10, 0.0]
N_PERMUTATIONS = 10
into_gangwon_edge_mask = np.isin(dst_idx, gangwon_stations)

results = []
for perm in range(N_PERMUTATIONS):
    perm_rng = np.random.RandomState(perm)
    removal_order = perm_rng.permutation(corridor_stations)
    for frac in DENSITY_FRACS:
        n_remove = N_corridor - int(round(frac * N_corridor))
        stations_to_remove = removal_order[:n_remove]
        ew = edge_weight_by_hour.copy()
        if n_remove > 0:
            remove_mask = into_gangwon_edge_mask & np.isin(src_idx, stations_to_remove)
            ew[:, remove_mask] = 0.0
        ew_t = torch.tensor(ew)
        proba = predict_gangwon_proba(ew_t)
        auc_pr = average_precision_score(y_true_gangwon, proba)
        auc_roc = roc_auc_score(y_true_gangwon, proba) if len(np.unique(y_true_gangwon)) > 1 else np.nan
        results.append({"permutation": perm, "density_frac": frac, "n_remaining": N_corridor - n_remove,
                         "auc_pr": auc_pr, "auc_roc": auc_roc})
    print(f"  permutation {perm+1}/{N_PERMUTATIONS} done", flush=True)

results_df = pd.DataFrame(results)
print("\n=== Gangwon detection vs. corridor density (mean +/- std across 10 random removal orders) ===")
summary = results_df.groupby("density_frac")[["n_remaining", "auc_pr", "auc_roc"]].agg(["mean", "std"])
print(summary)

results_df.to_csv(f"{BASE}/gangwon_corridor_density_curve.csv", index=False)
print(f"\nsaved to {BASE}/gangwon_corridor_density_curve.csv")


candidate graph: 21414 directed edges
Gangwon stations: [ 54 141 142 143]  (n=4)
corridor (upwind-connected) stations: 132
corridor stations by region: {'Seoul': 37, 'Busan': 3, 'Daegu': 13, 'Incheon': 20, 'Gwangju': 2, 'Daejeon': 5, 'Ulsan': 9, 'Sejong': 2, 'Gyeonggi': 11, 'Chungbuk': 10, 'Chungnam': 2, 'Jeonbuk': 8, 'Gyeongbuk': 4, 'Gyeongnam': 6}
test windows: 10045
Gangwon test positive rate: 0.02539
device=mps
  permutation 1/10 done
  permutation 2/10 done
  permutation 3/10 done
  permutation 4/10 done
  permutation 5/10 done
  permutation 6/10 done
  permutation 7/10 done
  permutation 8/10 done
  permutation 9/10 done
  permutation 10/10 done

=== Gangwon detection vs. corridor density (mean +/- std across 10 random removal orders) ===
             n_remaining         auc_pr             auc_roc              
                    mean  std      mean       std      mean           std
density_frac                                                             
0.00                 0.

In [3]:
# ================================================================
# Wind-directed temporal graph model, v2: fixes to the 3 issues
# identified in v1 --
#  1) custom layer doing a TARGET-NORMALIZED weighted MEAN of
#     neighbors (matching the validated closed-form correction),
#     not GCNConv's symmetric spectral normalization (built for
#     undirected graphs, wrong fit for our directed, asymmetric one)
#  2) self-information gets its own dedicated linear pathway,
#     never diluted by however strong the wind happens to be
#  3) one graph-conv layer instead of two (no incidental 2-hop
#     mixing within a single hour's snapshot)
#  5) edge weights scaled by their train-period std so their range
#     is controlled instead of raw, unbounded wind speed
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os, time

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 57.6, 73.6
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges")

WINDOW, HORIZON = 8, 8
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
train_end = int(np.searchsorted(years, 2019))
val_end = int(np.searchsorted(years, 2020))

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy()
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

train_nonzero = edge_weight_by_hour[:train_end][edge_weight_by_hour[:train_end] > 0]
weight_scale = train_nonzero.std()
print(f"edge weight scale (train, nonzero std): {weight_scale:.4f}")
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
print(f"edge_weight_by_hour: {edge_weight_by_hour.shape}, {edge_weight_by_hour.nbytes/1e9:.2f} GB")
edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] + [wind_speed_arr, wdir_sin, wdir_cos], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[:train_end]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

pm25_col_idx = TIME_FEATS_FULL.index("PM25")

def make_windows(start, end):
    X_list, y_list, starts_list = [], [], []
    for t in range(start, end - WINDOW - HORIZON + 1):
        X_list.append(time_arr[t:t + WINDOW])
        y_list.append(time_arr[t + WINDOW + HORIZON - 1, :, pm25_col_idx])
        starts_list.append(t)
    return np.stack(X_list), np.stack(y_list), np.array(starts_list)

X_train, y_train, starts_train = make_windows(0, train_end)
X_val, y_val, starts_val = make_windows(train_end, val_end)
X_test, y_test, starts_test = make_windows(val_end, n_time)
print(f"windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, n_feats={n_feats}")

Xtr_t, ytr_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)
Xva_t, yva_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
Xte_t, yte_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
starts_train_t, starts_val_t, starts_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts_train, starts_val, starts_test))

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def gather_edge_weight_seq(starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    return edge_weight_by_hour_t[idx]

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        return self.lin_self(x) + self.lin_neigh(agg_mean)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1) if edge_weight_seq is not None else torch.ones(ei_b.shape[1], device=xt.device)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

class TemporalOnlyGRU(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index=None, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_wind_temporal_v2"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 8, 1e-4
print(f"device={DEVICE}")

def run_epoch(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            with torch.set_grad_enabled(train):
                pred = model(xb, ei, ew_seq)
                loss = ((pred - yb) ** 2).mean()
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_model(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_epoch(model, Xtr_t, ytr_t, starts_train_t, use_graph, opt, train=True)
        val_loss = run_epoch(model, Xva_t, yva_t, starts_val_t, use_graph, opt, train=False)
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch(model, Xte_t, yte_t, starts_test_t, use_graph, None, train=False)
    print(f"[{name}] BEST epoch={best_epoch}  val={best_val:.4f}  test={test_loss:.4f}\n", flush=True)
    return {"name": name, "best_epoch": best_epoch, "val_mse": best_val, "test_mse": test_loss}

results = []
results.append(train_model("wind_temporal_gcn_v2", WindTemporalGCN(n_feats), use_graph=True))
results.append(train_model("no_graph", TemporalOnlyGRU(n_feats), use_graph=False))

summary = pd.DataFrame(results)
print(summary.to_string(index=False))


candidate graph: 5874 directed edges
edge weight scale (train, nonzero std): 3.8032
edge_weight_by_hour: (52608, 5874), 1.24 GB
windows: train=26289, val=8745, test=17529, n_feats=20
device=mps
[wind_temporal_gcn_v2] epoch  1  train=0.5481  val=0.5320  (18s)
[wind_temporal_gcn_v2] epoch  2  train=0.4881  val=0.5219  (16s)
[wind_temporal_gcn_v2] epoch  3  train=0.4781  val=0.5050  (16s)
[wind_temporal_gcn_v2] epoch  4  train=0.4721  val=0.4945  (16s)
[wind_temporal_gcn_v2] epoch  5  train=0.4681  val=0.4825  (16s)
[wind_temporal_gcn_v2] epoch  6  train=0.4659  val=0.4866  (16s)
[wind_temporal_gcn_v2] epoch  7  train=0.4630  val=0.4905  (16s)
[wind_temporal_gcn_v2] epoch  8  train=0.4614  val=0.4893  (16s)
[wind_temporal_gcn_v2] BEST epoch=5  val=0.4825  test=0.3479

[no_graph] epoch  1  train=0.5625  val=0.5677  (10s)
[no_graph] epoch  2  train=0.5111  val=0.5465  (10s)
[no_graph] epoch  3  train=0.5022  val=0.5520  (10s)
[no_graph] epoch  4  train=0.4974  val=0.5504  (10s)
[no_graph] e

In [23]:
SEEDS = [0, 1, 2]
all_results = []

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    res_wind = train_model(f"wind_temporal_gcn_v2_seed{seed}", WindTemporalGCN(n_feats), use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_temporal_gcn_v2"
    all_results.append(res_wind)

    torch.manual_seed(seed)
    np.random.seed(seed)
    res_nograph = train_model(f"no_graph_seed{seed}", TemporalOnlyGRU(n_feats), use_graph=False)
    res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    all_results.append(res_nograph)

results_df = pd.DataFrame(all_results)
print(results_df[["model", "seed", "best_epoch", "val_mse", "test_mse"]].to_string(index=False))

print("\n=== summary across seeds ===")
print(results_df.groupby("model")[["val_mse", "test_mse"]].agg(["mean", "std"]))

pivot_test = results_df.pivot(index="seed", columns="model", values="test_mse")
pivot_test["wind_wins"] = pivot_test["wind_temporal_gcn_v2"] < pivot_test["no_graph"]
pivot_test["pct_improvement"] = 100 * (pivot_test["no_graph"] - pivot_test["wind_temporal_gcn_v2"]) / pivot_test["no_graph"]
print("\n=== per-seed test MSE comparison ===")
print(pivot_test.to_string())


[wind_temporal_gcn_v2_seed0] epoch  1  train=0.5389  val=0.5446  (17s)
[wind_temporal_gcn_v2_seed0] epoch  2  train=0.4881  val=0.5128  (16s)
[wind_temporal_gcn_v2_seed0] epoch  3  train=0.4782  val=0.5156  (16s)
[wind_temporal_gcn_v2_seed0] epoch  4  train=0.4723  val=0.4989  (16s)
[wind_temporal_gcn_v2_seed0] epoch  5  train=0.4682  val=0.4935  (16s)
[wind_temporal_gcn_v2_seed0] epoch  6  train=0.4657  val=0.4966  (16s)
[wind_temporal_gcn_v2_seed0] epoch  7  train=0.4630  val=0.4919  (16s)
[wind_temporal_gcn_v2_seed0] epoch  8  train=0.4609  val=0.4862  (16s)
[wind_temporal_gcn_v2_seed0] BEST epoch=8  val=0.4862  test=0.3549

[no_graph_seed0] epoch  1  train=0.5557  val=0.5601  (10s)
[no_graph_seed0] epoch  2  train=0.5090  val=0.5470  (10s)
[no_graph_seed0] epoch  3  train=0.5020  val=0.5428  (10s)
[no_graph_seed0] epoch  4  train=0.4965  val=0.5417  (10s)
[no_graph_seed0] epoch  5  train=0.4938  val=0.5419  (10s)
[no_graph_seed0] epoch  6  train=0.4897  val=0.5560  (10s)
[no_graph_

In [24]:
# ================================================================
# Same wind-directed temporal graph model (v2 architecture: target-
# normalized weighted-mean aggregation, protected self-pathway,
# single conv layer, standardized edge weights) vs matched no-graph
# ablation -- now at WINDOW=12h, HORIZON=12h. Run across 3 seeds,
# same as the H=8 robustness check, since that's the rigor level
# this comparison has earned.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os, time

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 57.6, 73.6
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges")

WINDOW, HORIZON = 12, 12
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
train_end = int(np.searchsorted(years, 2019))
val_end = int(np.searchsorted(years, 2020))

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy()
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

train_nonzero = edge_weight_by_hour[:train_end][edge_weight_by_hour[:train_end] > 0]
weight_scale = train_nonzero.std()
print(f"edge weight scale (train, nonzero std): {weight_scale:.4f}")
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
print(f"edge_weight_by_hour: {edge_weight_by_hour.shape}, {edge_weight_by_hour.nbytes/1e9:.2f} GB")
edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] + [wind_speed_arr, wdir_sin, wdir_cos], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[:train_end]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

pm25_col_idx = TIME_FEATS_FULL.index("PM25")

def make_windows(start, end):
    X_list, y_list, starts_list = [], [], []
    for t in range(start, end - WINDOW - HORIZON + 1):
        X_list.append(time_arr[t:t + WINDOW])
        y_list.append(time_arr[t + WINDOW + HORIZON - 1, :, pm25_col_idx])
        starts_list.append(t)
    return np.stack(X_list), np.stack(y_list), np.array(starts_list)

X_train, y_train, starts_train = make_windows(0, train_end)
X_val, y_val, starts_val = make_windows(train_end, val_end)
X_test, y_test, starts_test = make_windows(val_end, n_time)
print(f"windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, n_feats={n_feats}")

Xtr_t, ytr_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)
Xva_t, yva_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
Xte_t, yte_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
starts_train_t, starts_val_t, starts_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts_train, starts_val, starts_test))

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def gather_edge_weight_seq(starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    return edge_weight_by_hour_t[idx]

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        return self.lin_self(x) + self.lin_neigh(agg_mean)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1) if edge_weight_seq is not None else torch.ones(ei_b.shape[1], device=xt.device)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

class TemporalOnlyGRU(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index=None, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_wind_temporal_h12"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 8, 1e-4
print(f"device={DEVICE}")

def run_epoch(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            with torch.set_grad_enabled(train):
                pred = model(xb, ei, ew_seq)
                loss = ((pred - yb) ** 2).mean()
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_model(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_epoch(model, Xtr_t, ytr_t, starts_train_t, use_graph, opt, train=True)
        val_loss = run_epoch(model, Xva_t, yva_t, starts_val_t, use_graph, opt, train=False)
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch(model, Xte_t, yte_t, starts_test_t, use_graph, None, train=False)
    print(f"[{name}] BEST epoch={best_epoch}  val={best_val:.4f}  test={test_loss:.4f}\n", flush=True)
    return {"name": name, "best_epoch": best_epoch, "val_mse": best_val, "test_mse": test_loss}

SEEDS = [0, 1, 2]
all_results = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    res_wind = train_model(f"wind_temporal_gcn_h12_seed{seed}", WindTemporalGCN(n_feats), use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_temporal_gcn"
    all_results.append(res_wind)

    torch.manual_seed(seed); np.random.seed(seed)
    res_nograph = train_model(f"no_graph_h12_seed{seed}", TemporalOnlyGRU(n_feats), use_graph=False)
    res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    all_results.append(res_nograph)

results_df = pd.DataFrame(all_results)
print(results_df[["model", "seed", "best_epoch", "val_mse", "test_mse"]].to_string(index=False))
print("\n=== summary across seeds (H=12) ===")
print(results_df.groupby("model")[["val_mse", "test_mse"]].agg(["mean", "std"]))

pivot_test = results_df.pivot(index="seed", columns="model", values="test_mse")
pivot_test["wind_wins"] = pivot_test["wind_temporal_gcn"] < pivot_test["no_graph"]
pivot_test["pct_improvement"] = 100 * (pivot_test["no_graph"] - pivot_test["wind_temporal_gcn"]) / pivot_test["no_graph"]
print("\n=== per-seed test MSE comparison (H=12) ===")
print(pivot_test.to_string())


candidate graph: 5874 directed edges
edge weight scale (train, nonzero std): 3.8032
edge_weight_by_hour: (52608, 5874), 1.24 GB
windows: train=26281, val=8737, test=17521, n_feats=20
device=mps
[wind_temporal_gcn_h12_seed0] epoch  1  train=0.6216  val=0.6401  (23s)
[wind_temporal_gcn_h12_seed0] epoch  2  train=0.5676  val=0.6193  (22s)
[wind_temporal_gcn_h12_seed0] epoch  3  train=0.5550  val=0.6264  (22s)
[wind_temporal_gcn_h12_seed0] epoch  4  train=0.5481  val=0.5925  (22s)
[wind_temporal_gcn_h12_seed0] epoch  5  train=0.5442  val=0.5907  (22s)
[wind_temporal_gcn_h12_seed0] epoch  6  train=0.5398  val=0.5865  (23s)
[wind_temporal_gcn_h12_seed0] epoch  7  train=0.5363  val=0.5991  (22s)
[wind_temporal_gcn_h12_seed0] epoch  8  train=0.5347  val=0.6012  (22s)
[wind_temporal_gcn_h12_seed0] BEST epoch=6  val=0.5865  test=0.4253

[no_graph_h12_seed0] epoch  1  train=0.6358  val=0.6666  (15s)
[no_graph_h12_seed0] epoch  2  train=0.5917  val=0.6417  (14s)
[no_graph_h12_seed0] epoch  3  trai

In [5]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
bearing_from = bearing_matrix(lats, lons)

pm25_wide = df.pivot(index="Datetime", columns="Station_ID", values="PM25")[station_order]
national_mean = pm25_wide.mean(axis=1)
resid_local_full = pm25_wide.sub(national_mean, axis=0)
resid_arr = resid_local_full.to_numpy().astype(float)

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(resid_local_full.index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(resid_local_full.index).to_numpy().astype(float)
wind_blows_toward = (wind_dir_arr + 180) % 360

T = resid_arr.shape[0]
LAG = 24
ALIGN_COS_THRESH, SPEED_THRESH, MIN_N = 0.5, 1.0, 200
print(f"alignment threshold: cos>{ALIGN_COS_THRESH} (~within {np.degrees(np.arccos(ALIGN_COS_THRESH)):.0f} deg), "
      f"speed>{SPEED_THRESH} m/s, min_n={MIN_N}")

corr_mat = np.full((n_stations, n_stations), np.nan)
n_obs_mat = np.zeros((n_stations, n_stations))

for i in range(n_stations):
    X = np.roll(resid_arr[:, i], LAG); X[:LAG] = np.nan
    wbt_i = np.roll(wind_blows_toward[:, i], LAG); wbt_i[:LAG] = np.nan
    ws_i = np.roll(wind_speed_arr[:, i], LAG); ws_i[:LAG] = np.nan

    align = np.cos(np.radians(wbt_i[:, None] - bearing_from[i, :][None, :]))  # [T, N]
    mask = (align > ALIGN_COS_THRESH) & (ws_i[:, None] > SPEED_THRESH) & (~np.isnan(X))[:, None]
    mask[:, i] = False

    Y = resid_arr
    Xc = np.nan_to_num(X, nan=0.0)
    n = mask.sum(axis=0).astype(float)
    sumX = (mask * Xc[:, None]).sum(axis=0)
    sumX2 = (mask * (Xc[:, None]**2)).sum(axis=0)
    sumY = (mask * Y).sum(axis=0)
    sumY2 = (mask * Y**2).sum(axis=0)
    sumXY = (mask * Xc[:, None] * Y).sum(axis=0)

    with np.errstate(invalid="ignore", divide="ignore"):
        meanX, meanY = sumX/n, sumY/n
        varX, varY = sumX2/n - meanX**2, sumY2/n - meanY**2
        cov = sumXY/n - meanX*meanY
        corr = cov / np.sqrt(varX*varY)

    corr_mat[i, :] = corr
    n_obs_mat[i, :] = n

iu_mask = ~np.eye(n_stations, dtype=bool)
dist_flat, corr_flat, nobs_flat = dist_km[iu_mask], corr_mat[iu_mask], n_obs_mat[iu_mask]
valid = (nobs_flat >= MIN_N) & ~np.isnan(corr_flat)
print(f"\n{valid.sum()} / {len(valid)} directed pairs have >= {MIN_N} wind-aligned observations")
dist_valid, corr_valid = dist_flat[valid], corr_flat[valid]

bins = [0, 25, 50, 75, 100, 150, 200, 250, 300, 400, 500, 600]
bin_mids, bin_means = [], []
print("\nwind-aligned, LAG=24 correlation by distance:")
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (dist_valid >= lo) & (dist_valid < hi)
    if m.sum() > 5:
        mc = corr_valid[m].mean()
        print(f"{lo:3d}-{hi:3d}km: n_pairs={m.sum():4d}  mean_corr={mc:.4f}")
        bin_mids.append((lo + hi) / 2)
        bin_means.append(mc)

bin_mids, bin_means = np.array(bin_mids), np.array(bin_means)

def decay_fn(d, A, rho):
    return A * np.exp(-d / rho)

popt, _ = curve_fit(decay_fn, bin_mids, bin_means, p0=[max(bin_means[0], 0.01), 150], maxfev=5000)
A_fit, rho_fit = popt
print(f"\nfitted decay: A={A_fit:.4f}, RHO_KM={rho_fit:.1f} km")
cutoff_suggestion = -rho_fit * np.log(0.1)
print(f"suggested DIST_CUTOFF (fit reaches 10% of peak): {cutoff_suggestion:.0f} km")


alignment threshold: cos>0.5 (~within 60 deg), speed>1.0 m/s, min_n=200

30800 / 30800 directed pairs have >= 200 wind-aligned observations

wind-aligned, LAG=24 correlation by distance:
  0- 25km: n_pairs=2646  mean_corr=0.1390
 25- 50km: n_pairs=2750  mean_corr=0.1109
 50- 75km: n_pairs=1504  mean_corr=0.0968
 75-100km: n_pairs=2196  mean_corr=0.0659
100-150km: n_pairs=4250  mean_corr=0.0323
150-200km: n_pairs=3668  mean_corr=-0.0097
200-250km: n_pairs=4400  mean_corr=-0.0601
250-300km: n_pairs=4086  mean_corr=-0.0801
300-400km: n_pairs=4822  mean_corr=-0.1096
400-500km: n_pairs= 444  mean_corr=-0.1223
500-600km: n_pairs=  34  mean_corr=-0.0142

fitted decay: A=0.1910, RHO_KM=62.4 km
suggested DIST_CUTOFF (fit reaches 10% of peak): 144 km


In [7]:
import numpy as np

H = 24
ALIGN_COS_THRESH, SPEED_THRESH, MIN_N = 0.5, 1.0, 100
HIGH_THRESH = 12  # at least half the 24h window aligned

T = resid_arr.shape[0]
n_stations = len(station_order)

corr_high = np.full((n_stations, n_stations), np.nan)
corr_low = np.full((n_stations, n_stations), np.nan)
n_high = np.zeros((n_stations, n_stations))
n_low = np.zeros((n_stations, n_stations))

def masked_corr(X, Y, mask):
    Xc = np.nan_to_num(X, nan=0.0)
    n = mask.sum(axis=0).astype(float)
    sumX, sumX2 = (mask*Xc).sum(0), (mask*Xc**2).sum(0)
    sumY, sumY2 = (mask*Y).sum(0), (mask*Y**2).sum(0)
    sumXY = (mask*Xc*Y).sum(0)
    with np.errstate(invalid="ignore", divide="ignore"):
        meanX, meanY = sumX/n, sumY/n
        varX, varY = sumX2/n - meanX**2, sumY2/n - meanY**2
        cov = sumXY/n - meanX*meanY
        corr = cov / np.sqrt(varX*varY)
    return corr, n

for i in range(n_stations):
    aligned_i = ((np.cos(np.radians(wind_blows_toward[:, i][:, None] - bearing_from[i, :][None, :])) > ALIGN_COS_THRESH)
                 & (wind_speed_arr[:, i][:, None] > SPEED_THRESH)).astype(float)
    weighted = aligned_i * resid_arr[:, i][:, None]
    cum_count, cum_weighted = np.cumsum(aligned_i, axis=0), np.cumsum(weighted, axis=0)

    count_win = np.full((T, n_stations), np.nan)
    weighted_win = np.full((T, n_stations), np.nan)
    count_win[H:] = cum_count[H:] - cum_count[:-H]
    weighted_win[H:] = cum_weighted[H:] - cum_weighted[:-H]
    with np.errstate(invalid="ignore", divide="ignore"):
        avg_resid_i = weighted_win / count_win

    valid_t = np.arange(H, T - H)
    X, Y, C = avg_resid_i[valid_t], resid_arr[valid_t + H], count_win[valid_t]

    mask_high = (C >= HIGH_THRESH) & ~np.isnan(X); mask_high[:, i] = False
    mask_low = (C >= 3) & (C < HIGH_THRESH) & ~np.isnan(X); mask_low[:, i] = False

    corr_high[i, :], n_high[i, :] = masked_corr(X, Y, mask_high)
    corr_low[i, :], n_low[i, :] = masked_corr(X, Y, mask_low)

iu_mask = ~np.eye(n_stations, dtype=bool)
dist_flat = dist_km[iu_mask]
bins = [0, 25, 50, 75, 100, 150, 200, 250, 300, 400, 500, 600]

for label, corr_mat, n_mat in [("HIGH sustained (>=12/24h aligned)", corr_high, n_high), ("LOW sustained (3-11/24h aligned)", corr_low, n_low)]:
    corr_f, n_f = corr_mat[iu_mask], n_mat[iu_mask]
    valid = (n_f >= MIN_N) & ~np.isnan(corr_f)
    print(f"\n=== {label}: {valid.sum()}/{len(valid)} pairs with >= {MIN_N} obs ===")
    dv, cv = dist_flat[valid], corr_f[valid]
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (dv >= lo) & (dv < hi)
        if m.sum() > 5:
            print(f"{lo:3d}-{hi:3d}km: n_pairs={m.sum():4d}  mean_corr={cv[m].mean():.4f}")



=== HIGH sustained (>=12/24h aligned): 30800/30800 pairs with >= 100 obs ===
  0- 25km: n_pairs=2646  mean_corr=0.1075
 25- 50km: n_pairs=2750  mean_corr=0.0869
 50- 75km: n_pairs=1504  mean_corr=0.0830
 75-100km: n_pairs=2196  mean_corr=0.0520
100-150km: n_pairs=4250  mean_corr=0.0296
150-200km: n_pairs=3668  mean_corr=-0.0085
200-250km: n_pairs=4400  mean_corr=-0.0479
250-300km: n_pairs=4086  mean_corr=-0.0642
300-400km: n_pairs=4822  mean_corr=-0.0848
400-500km: n_pairs= 444  mean_corr=-0.0985
500-600km: n_pairs=  34  mean_corr=-0.0418

=== LOW sustained (3-11/24h aligned): 30800/30800 pairs with >= 100 obs ===
  0- 25km: n_pairs=2646  mean_corr=0.1065
 25- 50km: n_pairs=2750  mean_corr=0.0911
 50- 75km: n_pairs=1504  mean_corr=0.0805
 75-100km: n_pairs=2196  mean_corr=0.0614
100-150km: n_pairs=4250  mean_corr=0.0296
150-200km: n_pairs=3668  mean_corr=-0.0153
200-250km: n_pairs=4400  mean_corr=-0.0519
250-300km: n_pairs=4086  mean_corr=-0.0719
300-400km: n_pairs=4822  mean_corr=-0.

In [3]:
# ================================================================
# Same wind-directed temporal graph model vs matched no-graph
# ablation -- WINDOW=24h, HORIZON=24h. Adds boundary_layer_height
# as a per-node feature to BOTH models (it was already in G's
# feature set but had never made it into the GNN pipeline) -- keeps
# the comparison fair, same as adding windspeed/direction earlier.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os, time

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 125.6, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges")

WINDOW, HORIZON = 24, 24
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
train_end = int(np.searchsorted(years, 2019))
val_end = int(np.searchsorted(years, 2020))

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

train_nonzero = edge_weight_by_hour[:train_end][edge_weight_by_hour[:train_end] > 0]
weight_scale = train_nonzero.std()
print(f"edge weight scale (train, nonzero std): {weight_scale:.4f}")
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
print(f"edge_weight_by_hour: {edge_weight_by_hour.shape}, {edge_weight_by_hour.nbytes/1e9:.2f} GB")
edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] + [wind_speed_arr, wdir_sin, wdir_cos, blh_arr], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[:train_end]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

pm25_col_idx = TIME_FEATS_FULL.index("PM25")

def make_windows(start, end):
    X_list, y_list, starts_list = [], [], []
    for t in range(start, end - WINDOW - HORIZON + 1):
        X_list.append(time_arr[t:t + WINDOW])
        y_list.append(time_arr[t + WINDOW + HORIZON - 1, :, pm25_col_idx])
        starts_list.append(t)
    return np.stack(X_list), np.stack(y_list), np.array(starts_list)

X_train, y_train, starts_train = make_windows(0, train_end)
X_val, y_val, starts_val = make_windows(train_end, val_end)
X_test, y_test, starts_test = make_windows(val_end, n_time)
print(f"windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, n_feats={n_feats}")

Xtr_t, ytr_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)
Xva_t, yva_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
Xte_t, yte_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
starts_train_t, starts_val_t, starts_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts_train, starts_val, starts_test))

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def gather_edge_weight_seq(starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    return edge_weight_by_hour_t[idx]

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        return self.lin_self(x) + self.lin_neigh(agg_mean)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1) if edge_weight_seq is not None else torch.ones(ei_b.shape[1], device=xt.device)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

class TemporalOnlyGRU(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index=None, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_wind_temporal_h24_blh"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 8, 1e-4
print(f"device={DEVICE}")

def run_epoch(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            with torch.set_grad_enabled(train):
                pred = model(xb, ei, ew_seq)
                loss = ((pred - yb) ** 2).mean()
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_model(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_epoch(model, Xtr_t, ytr_t, starts_train_t, use_graph, opt, train=True)
        val_loss = run_epoch(model, Xva_t, yva_t, starts_val_t, use_graph, opt, train=False)
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch(model, Xte_t, yte_t, starts_test_t, use_graph, None, train=False)
    print(f"[{name}] BEST epoch={best_epoch}  val={best_val:.4f}  test={test_loss:.4f}\n", flush=True)
    return {"name": name, "best_epoch": best_epoch, "val_mse": best_val, "test_mse": test_loss}

SEEDS = [0, 1, 2,3,4]
all_results = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    res_wind = train_model(f"wind_temporal_gcn_h24_blh_seed{seed}", WindTemporalGCN(n_feats), use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_temporal_gcn"
    all_results.append(res_wind)

    torch.manual_seed(seed); np.random.seed(seed)
    res_nograph = train_model(f"no_graph_h24_blh_seed{seed}", TemporalOnlyGRU(n_feats), use_graph=False)
    res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    all_results.append(res_nograph)

results_df = pd.DataFrame(all_results)
print(results_df[["model", "seed", "best_epoch", "val_mse", "test_mse"]].to_string(index=False))
print("\n=== summary across seeds (H=24, +BLH) ===")
print(results_df.groupby("model")[["val_mse", "test_mse"]].agg(["mean", "std"]))

pivot_test = results_df.pivot(index="seed", columns="model", values="test_mse")
pivot_test["wind_wins"] = pivot_test["wind_temporal_gcn"] < pivot_test["no_graph"]
pivot_test["pct_improvement"] = 100 * (pivot_test["no_graph"] - pivot_test["wind_temporal_gcn"]) / pivot_test["no_graph"]
print("\n=== per-seed test MSE comparison (H=24, +BLH) ===")
print(pivot_test.to_string())


candidate graph: 11446 directed edges
edge weight scale (train, nonzero std): 4.2418
edge_weight_by_hour: (52608, 11446), 2.41 GB
windows: train=26257, val=8713, test=17497, n_feats=21
device=mps
[wind_temporal_gcn_h24_blh_seed0] epoch  1  train=0.7536  val=0.8152  (87s)
[wind_temporal_gcn_h24_blh_seed0] epoch  2  train=0.7007  val=0.8549  (66s)
[wind_temporal_gcn_h24_blh_seed0] epoch  3  train=0.6849  val=0.8209  (66s)
[wind_temporal_gcn_h24_blh_seed0] epoch  4  train=0.6764  val=0.8464  (66s)
[wind_temporal_gcn_h24_blh_seed0] epoch  5  train=0.6704  val=0.8207  (66s)
[wind_temporal_gcn_h24_blh_seed0] epoch  6  train=0.6671  val=0.8110  (66s)
[wind_temporal_gcn_h24_blh_seed0] epoch  7  train=0.6606  val=0.8168  (66s)
[wind_temporal_gcn_h24_blh_seed0] epoch  8  train=0.6544  val=0.8107  (66s)
[wind_temporal_gcn_h24_blh_seed0] BEST epoch=8  val=0.8107  test=0.5448

[no_graph_h24_blh_seed0] epoch  1  train=0.7663  val=0.8409  (23s)
[no_graph_h24_blh_seed0] epoch  2  train=0.7290  val=0.8

In [3]:
#36 hours 
# ================================================================
# Same wind-directed temporal graph model vs matched no-graph
# ablation -- now WINDOW=36h, HORIZON=36h, matching Korea's actual
# operational advance-warning target (confirmed: official alerts
# aim for 36h lead time). Everything else unchanged from the H=24
# version -- same features (incl. BLH), same architecture, same
# 5-seed comparison.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os, time

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 125.6, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges")

WINDOW, HORIZON = 36, 36  # CHANGED from 24, 24
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
train_end = int(np.searchsorted(years, 2019))
val_end = int(np.searchsorted(years, 2020))

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

train_nonzero = edge_weight_by_hour[:train_end][edge_weight_by_hour[:train_end] > 0]
weight_scale = train_nonzero.std()
print(f"edge weight scale (train, nonzero std): {weight_scale:.4f}")
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
print(f"edge_weight_by_hour: {edge_weight_by_hour.shape}, {edge_weight_by_hour.nbytes/1e9:.2f} GB")
edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] + [wind_speed_arr, wdir_sin, wdir_cos, blh_arr], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[:train_end]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

pm25_col_idx = TIME_FEATS_FULL.index("PM25")

def make_windows(start, end):
    X_list, y_list, starts_list = [], [], []
    for t in range(start, end - WINDOW - HORIZON + 1):
        X_list.append(time_arr[t:t + WINDOW])
        y_list.append(time_arr[t + WINDOW + HORIZON - 1, :, pm25_col_idx])
        starts_list.append(t)
    return np.stack(X_list), np.stack(y_list), np.array(starts_list)

X_train, y_train, starts_train = make_windows(0, train_end)
X_val, y_val, starts_val = make_windows(train_end, val_end)
X_test, y_test, starts_test = make_windows(val_end, n_time)
print(f"windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, n_feats={n_feats}")

Xtr_t, ytr_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)
Xva_t, yva_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
Xte_t, yte_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
starts_train_t, starts_val_t, starts_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts_train, starts_val, starts_test))

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def gather_edge_weight_seq(starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    return edge_weight_by_hour_t[idx]

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        return self.lin_self(x) + self.lin_neigh(agg_mean)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1) if edge_weight_seq is not None else torch.ones(ei_b.shape[1], device=xt.device)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

class TemporalOnlyGRU(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index=None, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_wind_temporal_h36_blh"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 8, 1e-4
print(f"device={DEVICE}")

def run_epoch(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            with torch.set_grad_enabled(train):
                pred = model(xb, ei, ew_seq)
                loss = ((pred - yb) ** 2).mean()
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_model(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_epoch(model, Xtr_t, ytr_t, starts_train_t, use_graph, opt, train=True)
        val_loss = run_epoch(model, Xva_t, yva_t, starts_val_t, use_graph, opt, train=False)
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch(model, Xte_t, yte_t, starts_test_t, use_graph, None, train=False)
    print(f"[{name}] BEST epoch={best_epoch}  val={best_val:.4f}  test={test_loss:.4f}\n", flush=True)
    return {"name": name, "best_epoch": best_epoch, "val_mse": best_val, "test_mse": test_loss}

SEEDS = [0, 1, 2, 3, 4]
all_results = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    res_wind = train_model(f"wind_temporal_gcn_h36_blh_seed{seed}", WindTemporalGCN(n_feats), use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_temporal_gcn"
    all_results.append(res_wind)

    torch.manual_seed(seed); np.random.seed(seed)
    res_nograph = train_model(f"no_graph_h36_blh_seed{seed}", TemporalOnlyGRU(n_feats), use_graph=False)
    res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    all_results.append(res_nograph)

results_df = pd.DataFrame(all_results)
print(results_df[["model", "seed", "best_epoch", "val_mse", "test_mse"]].to_string(index=False))
print("\n=== summary across seeds (H=36, +BLH) ===")
print(results_df.groupby("model")[["val_mse", "test_mse"]].agg(["mean", "std"]))

pivot_test = results_df.pivot(index="seed", columns="model", values="test_mse")
pivot_test["wind_wins"] = pivot_test["wind_temporal_gcn"] < pivot_test["no_graph"]
pivot_test["pct_improvement"] = 100 * (pivot_test["no_graph"] - pivot_test["wind_temporal_gcn"]) / pivot_test["no_graph"]
print("\n=== per-seed test MSE comparison (H=36, +BLH) ===")
print(pivot_test.to_string())


candidate graph: 11446 directed edges
edge weight scale (train, nonzero std): 4.2418
edge_weight_by_hour: (52608, 11446), 2.41 GB
windows: train=26233, val=8689, test=17473, n_feats=21
device=mps
[wind_temporal_gcn_h36_blh_seed0] epoch  1  train=0.8559  val=0.9624  (122s)
[wind_temporal_gcn_h36_blh_seed0] epoch  2  train=0.7964  val=0.9724  (96s)
[wind_temporal_gcn_h36_blh_seed0] epoch  3  train=0.7786  val=0.9747  (96s)
[wind_temporal_gcn_h36_blh_seed0] epoch  4  train=0.7673  val=0.9721  (96s)
[wind_temporal_gcn_h36_blh_seed0] epoch  5  train=0.7587  val=0.9848  (96s)
[wind_temporal_gcn_h36_blh_seed0] epoch  6  train=0.7512  val=0.9982  (96s)
[wind_temporal_gcn_h36_blh_seed0] epoch  7  train=0.7428  val=0.9774  (96s)
[wind_temporal_gcn_h36_blh_seed0] epoch  8  train=0.7350  val=1.0064  (98s)
[wind_temporal_gcn_h36_blh_seed0] BEST epoch=1  val=0.9624  test=0.6725

[no_graph_h36_blh_seed0] epoch  1  train=0.8660  val=0.9784  (33s)
[no_graph_h36_blh_seed0] epoch  2  train=0.8236  val=0.

In [1]:
# ================================================================
# H=36 / W=36, +BLH -- SAME as your last run, but graph edge weights
# are now truncated to only the most recent GRAPH_RECENT_HOURS of the
# 36h window. Raw node features (AirKorea + wind + BLH) still flow
# through the GRU for the full 36h; only the GRAPH aggregation is
# restricted to recent hours, since wind snapshots ~20-36h before the
# prediction point carry little real transport signal (confirmed via
# partial-correlation decay analysis: 0.254 at 12h lag -> 0.099 at 36h)
# and were likely just injecting noise into the model.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os, time

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 125.6, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges")

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18   # <-- NEW: only the most recent 18h of the window get graph edges
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
train_end = int(np.searchsorted(years, 2019))
val_end = int(np.searchsorted(years, 2020))

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

train_nonzero = edge_weight_by_hour[:train_end][edge_weight_by_hour[:train_end] > 0]
weight_scale = train_nonzero.std()
print(f"edge weight scale (train, nonzero std): {weight_scale:.4f}")
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
print(f"edge_weight_by_hour: {edge_weight_by_hour.shape}, {edge_weight_by_hour.nbytes/1e9:.2f} GB")
edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] + [wind_speed_arr, wdir_sin, wdir_cos, blh_arr], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[:train_end]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

pm25_col_idx = TIME_FEATS_FULL.index("PM25")

def make_windows(start, end):
    X_list, y_list, starts_list = [], [], []
    for t in range(start, end - WINDOW - HORIZON + 1):
        X_list.append(time_arr[t:t + WINDOW])
        y_list.append(time_arr[t + WINDOW + HORIZON - 1, :, pm25_col_idx])
        starts_list.append(t)
    return np.stack(X_list), np.stack(y_list), np.array(starts_list)

X_train, y_train, starts_train = make_windows(0, train_end)
X_val, y_val, starts_val = make_windows(train_end, val_end)
X_test, y_test, starts_test = make_windows(val_end, n_time)
print(f"windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, n_feats={n_feats}")

Xtr_t, ytr_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)
Xva_t, yva_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
Xte_t, yte_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
starts_train_t, starts_val_t, starts_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts_train, starts_val, starts_test))

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def gather_edge_weight_seq(starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = edge_weight_by_hour_t[idx]
    if GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0   # zero out edges older than the recent window
    return ew

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        return self.lin_self(x) + self.lin_neigh(agg_mean)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1) if edge_weight_seq is not None else torch.ones(ei_b.shape[1], device=xt.device)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

class TemporalOnlyGRU(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index=None, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_wind_temporal_h36_blh_recent18"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 8, 1e-4
print(f"device={DEVICE}")

def run_epoch(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            with torch.set_grad_enabled(train):
                pred = model(xb, ei, ew_seq)
                loss = ((pred - yb) ** 2).mean()
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_model(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_epoch(model, Xtr_t, ytr_t, starts_train_t, use_graph, opt, train=True)
        val_loss = run_epoch(model, Xva_t, yva_t, starts_val_t, use_graph, opt, train=False)
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch(model, Xte_t, yte_t, starts_test_t, use_graph, None, train=False)
    print(f"[{name}] BEST epoch={best_epoch}  val={best_val:.4f}  test={test_loss:.4f}\n", flush=True)
    return {"name": name, "best_epoch": best_epoch, "val_mse": best_val, "test_mse": test_loss}

# only the wind model needs rerunning -- no_graph is unaffected by GRAPH_RECENT_HOURS
# (it never used edge weights in the first place), so we reuse the H36 no_graph results
# already validated. If you want a fresh no_graph run anyway for perfect apples-to-apples
# checkpoint provenance, uncomment its lines below.
SEEDS = [0, 1, 2, 3, 4]
all_results = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    res_wind = train_model(f"wind_temporal_gcn_h36_blh_recent18_seed{seed}", WindTemporalGCN(n_feats), use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_temporal_gcn_recent18"
    all_results.append(res_wind)

    # torch.manual_seed(seed); np.random.seed(seed)
    # res_nograph = train_model(f"no_graph_h36_blh_seed{seed}", TemporalOnlyGRU(n_feats), use_graph=False)
    # res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    # all_results.append(res_nograph)

results_df = pd.DataFrame(all_results)
print(results_df[["model", "seed", "best_epoch", "val_mse", "test_mse"]].to_string(index=False))
print("\n=== summary across seeds (H=36, +BLH, recent-18h graph) ===")
print(results_df.groupby("model")[["val_mse", "test_mse"]].agg(["mean", "std"]))


candidate graph: 11446 directed edges
edge weight scale (train, nonzero std): 4.2418
edge_weight_by_hour: (52608, 11446), 2.41 GB
windows: train=26233, val=8689, test=17473, n_feats=21
device=mps
[wind_temporal_gcn_h36_blh_recent18_seed0] epoch  1  train=0.8569  val=0.9611  (125s)
[wind_temporal_gcn_h36_blh_recent18_seed0] epoch  2  train=0.8037  val=0.9710  (96s)
[wind_temporal_gcn_h36_blh_recent18_seed0] epoch  3  train=0.7883  val=0.9760  (96s)
[wind_temporal_gcn_h36_blh_recent18_seed0] epoch  4  train=0.7787  val=0.9663  (97s)
[wind_temporal_gcn_h36_blh_recent18_seed0] epoch  5  train=0.7713  val=0.9685  (97s)
[wind_temporal_gcn_h36_blh_recent18_seed0] epoch  6  train=0.7650  val=0.9777  (98s)
[wind_temporal_gcn_h36_blh_recent18_seed0] epoch  7  train=0.7580  val=0.9575  (97s)
[wind_temporal_gcn_h36_blh_recent18_seed0] epoch  8  train=0.7518  val=0.9797  (96s)
[wind_temporal_gcn_h36_blh_recent18_seed0] BEST epoch=7  val=0.9575  test=0.6499

[wind_temporal_gcn_h36_blh_recent18_seed1

In [2]:
# ================================================================
# Sweep GRAPH_RECENT_HOURS over {12, 24} at H=36/W=36, +BLH.
# (18h and full-36h/no-truncation were already run and validated:
#  18h -> wind mean=0.6337, no_graph mean=0.6577, t=5.40, p=0.0057, 5/5 wins
#  36h(full) -> wind mean=0.6428, t=1.49, p=0.21, 3/5 wins)
# no_graph is unaffected by this parameter, so only the wind model
# is retrained, for each of the two new GRAPH_RECENT_HOURS values.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os, time, gc
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 125.6, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges")

WINDOW, HORIZON = 36, 36
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
train_end = int(np.searchsorted(years, 2019))
val_end = int(np.searchsorted(years, 2020))

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)
del wbt_src, cos_align, speed_src, wind_blows_toward

train_nonzero = edge_weight_by_hour[:train_end][edge_weight_by_hour[:train_end] > 0]
weight_scale = train_nonzero.std()
print(f"edge weight scale (train, nonzero std): {weight_scale:.4f}")
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
print(f"edge_weight_by_hour: {edge_weight_by_hour.shape}, {edge_weight_by_hour.nbytes/1e9:.2f} GB")
edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)
del edge_weight_by_hour, train_nonzero
gc.collect()

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] + [wind_speed_arr, wdir_sin, wdir_cos, blh_arr], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del time_panels, wind_dir_arr, wind_speed_arr, blh_arr, wdir_sin, wdir_cos
gc.collect()

time_train = time_arr[:train_end]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

pm25_col_idx = TIME_FEATS_FULL.index("PM25")

def make_windows(start, end):
    X_list, y_list, starts_list = [], [], []
    for t in range(start, end - WINDOW - HORIZON + 1):
        X_list.append(time_arr[t:t + WINDOW])
        y_list.append(time_arr[t + WINDOW + HORIZON - 1, :, pm25_col_idx])
        starts_list.append(t)
    return np.stack(X_list), np.stack(y_list), np.array(starts_list)

X_train, y_train, starts_train = make_windows(0, train_end)
X_val, y_val, starts_val = make_windows(train_end, val_end)
X_test, y_test, starts_test = make_windows(val_end, n_time)
print(f"windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, n_feats={n_feats}")

Xtr_t, ytr_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)
Xva_t, yva_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
Xte_t, yte_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
starts_train_t, starts_val_t, starts_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts_train, starts_val, starts_test))
del X_train, y_train, X_val, y_val, X_test, y_test, time_arr
gc.collect()

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

GRAPH_RECENT_HOURS = WINDOW  # overwritten inside the sweep loop below

def gather_edge_weight_seq(starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = edge_weight_by_hour_t[idx]
    if GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        return self.lin_self(x) + self.lin_neigh(agg_mean)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1) if edge_weight_seq is not None else torch.ones(ei_b.shape[1], device=xt.device)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 8, 1e-4
print(f"device={DEVICE}")

def run_epoch(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            with torch.set_grad_enabled(train):
                pred = model(xb, ei, ew_seq)
                loss = ((pred - yb) ** 2).mean()
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_model(name, model, use_graph, ckpt_dir, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{ckpt_dir}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_epoch(model, Xtr_t, ytr_t, starts_train_t, use_graph, opt, train=True)
        val_loss = run_epoch(model, Xva_t, yva_t, starts_val_t, use_graph, opt, train=False)
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch(model, Xte_t, yte_t, starts_test_t, use_graph, None, train=False)
    print(f"[{name}] BEST epoch={best_epoch}  val={best_val:.4f}  test={test_loss:.4f}\n", flush=True)
    return {"name": name, "best_epoch": best_epoch, "val_mse": best_val, "test_mse": test_loss}

SEEDS = [0, 1, 2, 3, 4]
NO_GRAPH_H36 = np.array([0.660700, 0.656495, 0.661686, 0.660987, 0.648388])  # already validated, reused as baseline

sweep_results = {}
for RECENT_HOURS in [12, 24]:
    GRAPH_RECENT_HOURS = RECENT_HOURS
    ckpt_dir = f"{BASE}/ckpt_wind_temporal_h36_blh_recent{RECENT_HOURS}"
    os.makedirs(ckpt_dir, exist_ok=True)
    seed_results = []
    for seed in SEEDS:
        torch.manual_seed(seed); np.random.seed(seed)
        res = train_model(f"wind_temporal_gcn_h36_blh_recent{RECENT_HOURS}_seed{seed}",
                           WindTemporalGCN(n_feats), use_graph=True, ckpt_dir=ckpt_dir)
        seed_results.append(res["test_mse"])
    seed_results = np.array(seed_results)
    t, p = stats.ttest_rel(NO_GRAPH_H36, seed_results)
    d = (NO_GRAPH_H36 - seed_results).mean() / (NO_GRAPH_H36 - seed_results).std(ddof=1)
    wins = int((seed_results < NO_GRAPH_H36).sum())
    sweep_results[RECENT_HOURS] = {
        "test_mse_mean": seed_results.mean(), "test_mse_std": seed_results.std(ddof=1),
        "t": t, "p": p, "cohens_d": d, "wins": wins, "per_seed": seed_results.tolist()
    }
    print(f"\n=== GRAPH_RECENT_HOURS={RECENT_HOURS} done: mean={seed_results.mean():.4f} std={seed_results.std(ddof=1):.4f} "
          f"t={t:.3f} p={p:.4f} wins={wins}/5 ===\n", flush=True)

print("\n\n========== FULL SWEEP SUMMARY (H=36, +BLH) ==========")
print(f"{'recent_hours':>13} {'mean':>8} {'std':>8} {'t':>7} {'p':>8} {'wins':>6}")
known = {
    18: {"test_mse_mean": 0.6337, "test_mse_std": 0.0121, "t": 5.40, "p": 0.0057, "wins": 5},
    36: {"test_mse_mean": 0.6428, "test_mse_std": 0.0205, "t": 1.49, "p": 0.2114, "wins": 3},
}
all_points = {**sweep_results, **known}
for rh in sorted(all_points):
    r = all_points[rh]
    print(f"{rh:>13} {r['test_mse_mean']:>8.4f} {r['test_mse_std']:>8.4f} {r['t']:>7.2f} {r['p']:>8.4f} {r['wins']:>6}/5")


candidate graph: 11446 directed edges
edge weight scale (train, nonzero std): 4.2418
edge_weight_by_hour: (52608, 11446), 2.41 GB
windows: train=26233, val=8689, test=17473, n_feats=21
device=mps
[wind_temporal_gcn_h36_blh_recent12_seed0] epoch  1  train=0.8580  val=0.9608  (111s)
[wind_temporal_gcn_h36_blh_recent12_seed0] epoch  2  train=0.8071  val=0.9716  (96s)
[wind_temporal_gcn_h36_blh_recent12_seed0] epoch  3  train=0.7924  val=0.9771  (96s)
[wind_temporal_gcn_h36_blh_recent12_seed0] epoch  4  train=0.7831  val=0.9646  (96s)
[wind_temporal_gcn_h36_blh_recent12_seed0] epoch  5  train=0.7763  val=0.9695  (96s)
[wind_temporal_gcn_h36_blh_recent12_seed0] epoch  6  train=0.7708  val=0.9764  (95s)
[wind_temporal_gcn_h36_blh_recent12_seed0] epoch  7  train=0.7648  val=0.9576  (95s)
[wind_temporal_gcn_h36_blh_recent12_seed0] epoch  8  train=0.7596  val=0.9791  (96s)
[wind_temporal_gcn_h36_blh_recent12_seed0] BEST epoch=7  val=0.9576  test=0.6473

[wind_temporal_gcn_h36_blh_recent12_seed1

In [5]:
#retrain models for extreme event classification. 
# ================================================================
# Quantile regression version, H=36/W=36, +BLH. Same architecture
# and same wind-graph config (GRAPH_RECENT_HOURS=18, the sweep
# winner) vs no-graph, but the output head now predicts multiple
# conditional quantiles [0.50, 0.75, 0.90, 0.95, 0.99] via pinball
# loss instead of a single MSE point estimate, with a monotonicity
# constraint so predicted quantiles can't cross. This directly
# targets the tail instead of retrofitting a threshold onto a
# mean-regression model that never learned to care about it.
#
# After training: (1) paired significance test on test pinball loss,
# wind vs no_graph, to confirm the graph still wins at 36h with the
# 18h-recent edge construction; (2) classification eval at the real
# alert thresholds (75, 150), flagging via the 95th/99th percentile
# predictions respectively.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 125.6, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges")

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
train_end = int(np.searchsorted(years, 2019))
val_end = int(np.searchsorted(years, 2020))

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

train_nonzero = edge_weight_by_hour[:train_end][edge_weight_by_hour[:train_end] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] + [wind_speed_arr, wdir_sin, wdir_cos, blh_arr], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[:train_end]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

pm25_col_idx = TIME_FEATS_FULL.index("PM25")
pm25_mean_train = float(t_mean[0, 0, pm25_col_idx])
pm25_std_train = float(t_std[0, 0, pm25_col_idx])
print(f"pm25_mean_train={pm25_mean_train:.4f}  pm25_std_train={pm25_std_train:.4f}")

def make_windows(start, end):
    X_list, y_list, starts_list = [], [], []
    for t in range(start, end - WINDOW - HORIZON + 1):
        X_list.append(time_arr[t:t + WINDOW])
        y_list.append(time_arr[t + WINDOW + HORIZON - 1, :, pm25_col_idx])
        starts_list.append(t)
    return np.stack(X_list), np.stack(y_list), np.array(starts_list)

X_train, y_train, starts_train = make_windows(0, train_end)
X_val, y_val, starts_val = make_windows(train_end, val_end)
X_test, y_test, starts_test = make_windows(val_end, n_time)
print(f"windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, n_feats={n_feats}")

Xtr_t, ytr_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)
Xva_t, yva_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
Xte_t, yte_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
starts_train_t, starts_val_t, starts_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts_train, starts_val, starts_test))

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def gather_edge_weight_seq(starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = edge_weight_by_hour_t[idx]
    if GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

def pinball_loss(preds, target, quantiles):
    # preds: (..., Q)  target: (...)  -> broadcast target over Q
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    loss = torch.max(q_tensor * diff, (q_tensor - 1) * diff)
    return loss.mean()

def monotonic_quantiles(raw):
    # raw: (..., Q) unconstrained -> non-decreasing quantile predictions
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        return self.lin_self(x) + self.lin_neigh(agg_mean)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3, n_quantiles=N_QUANTILES):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, n_quantiles)

    def forward(self, x_window, edge_index, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1) if edge_weight_seq is not None else torch.ones(ei_b.shape[1], device=xt.device)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        raw = self.head(embed)
        return monotonic_quantiles(raw)

class TemporalOnlyGRU(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3, n_quantiles=N_QUANTILES):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, n_quantiles)

    def forward(self, x_window, edge_index=None, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        raw = self.head(embed)
        return monotonic_quantiles(raw)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_h36_blh_quantile"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 8, 1e-4
print(f"device={DEVICE}")

def run_epoch(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            with torch.set_grad_enabled(train):
                pred = model(xb, ei, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_model(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_epoch(model, Xtr_t, ytr_t, starts_train_t, use_graph, opt, train=True)
        val_loss = run_epoch(model, Xva_t, yva_t, starts_val_t, use_graph, opt, train=False)
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch(model, Xte_t, yte_t, starts_test_t, use_graph, None, train=False)
    print(f"[{name}] BEST epoch={best_epoch}  val={best_val:.4f}  test={test_loss:.4f}\n", flush=True)
    return model, {"name": name, "best_epoch": best_epoch, "val_pinball": best_val, "test_pinball": test_loss}

@torch.no_grad()
def predict_full(model, use_graph, micro_batch=16):
    model.eval()
    n = Xte_t.shape[0]
    preds = []
    for start in range(0, n, micro_batch):
        xb = add_static(Xte_t[start:start + micro_batch]).to(DEVICE)
        ew_seq = gather_edge_weight_seq(starts_test_t[start:start + micro_batch]).to(DEVICE) if use_graph else None
        ei = edge_index.to(DEVICE) if use_graph else None
        pred = model(xb, ei, ew_seq)
        preds.append(pred.cpu())
    return torch.cat(preds, dim=0).numpy()  # (n_test, N, n_quantiles)

def unstd(x):
    return x * pm25_std_train + pm25_mean_train

actual_raw = unstd(y_test)  # (n_test, N)
THRESHOLDS = {"advisory_75": (75.0, 0.95), "warning_150": (150.0, 0.99)}  # (level, quantile used to flag)

def classify_metrics(pred_quantile_raw, actual_raw, threshold):
    pred_pos = pred_quantile_raw >= threshold
    actual_pos = actual_raw >= threshold
    tp = int(np.sum(pred_pos & actual_pos))
    fp = int(np.sum(pred_pos & ~actual_pos))
    fn = int(np.sum(~pred_pos & actual_pos))
    tn = int(np.sum(~pred_pos & ~actual_pos))
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else float("nan")
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=precision, recall=recall, f1=f1)

SEEDS = [0, 1, 2, 3, 4]
reg_results = []
clf_rows = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    wind_model, res_wind = train_model(f"wind_gcn_h36_blh_recent18_quantile_seed{seed}", WindTemporalGCN(n_feats), use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_graph"
    reg_results.append(res_wind)

    torch.manual_seed(seed); np.random.seed(seed)
    nograph_model, res_nograph = train_model(f"no_graph_h36_blh_quantile_seed{seed}", TemporalOnlyGRU(n_feats), use_graph=False)
    res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    reg_results.append(res_nograph)

    pred_wind = unstd(predict_full(wind_model, use_graph=True))       # (n_test, N, Q)
    pred_nograph = unstd(predict_full(nograph_model, use_graph=False))

    for thresh_name, (thresh_val, tau) in THRESHOLDS.items():
        q_idx = QUANTILES.index(tau)
        m_wind = classify_metrics(pred_wind[:, :, q_idx], actual_raw, thresh_val)
        m_nograph = classify_metrics(pred_nograph[:, :, q_idx], actual_raw, thresh_val)
        clf_rows.append({"seed": seed, "threshold": thresh_name, "model": "wind_graph", **m_wind})
        clf_rows.append({"seed": seed, "threshold": thresh_name, "model": "no_graph", **m_nograph})

reg_df = pd.DataFrame(reg_results)
print(reg_df[["model", "seed", "best_epoch", "val_pinball", "test_pinball"]].to_string(index=False))
print("\n=== quantile-regression summary (H=36, +BLH, wind=recent18) ===")
print(reg_df.groupby("model")[["val_pinball", "test_pinball"]].agg(["mean", "std"]))

pivot = reg_df.pivot(index="seed", columns="model", values="test_pinball")
t, p = stats.ttest_rel(pivot["no_graph"], pivot["wind_graph"])
d = (pivot["no_graph"] - pivot["wind_graph"]).mean() / (pivot["no_graph"] - pivot["wind_graph"]).std(ddof=1)
wins = int((pivot["wind_graph"] < pivot["no_graph"]).sum())
print(f"\n=== does the graph still win at 36h (recent-18h edges), now under quantile loss? ===")
print(f"wind mean={pivot['wind_graph'].mean():.4f}  no_graph mean={pivot['no_graph'].mean():.4f}")
print(f"paired t-test: t={t:.4f} p={p:.4f} cohens_d={d:.4f} wins={wins}/5")

clf_df = pd.DataFrame(clf_rows)
print("\n=== per-seed classification metrics (flagged via upper-quantile predictions) ===")
print(clf_df[["threshold", "model", "seed", "tp", "fp", "fn", "tn", "precision", "recall", "f1"]].to_string(index=False))
print("\n=== classification summary across seeds ===")
print(clf_df.groupby(["threshold", "model"])[["precision", "recall", "f1"]].agg(["mean", "std"]))

print("\n=== paired significance (wind vs no_graph) on classification ===")
for thresh_name in THRESHOLDS:
    sub = clf_df[clf_df.threshold == thresh_name]
    wr = sub[sub.model == "wind_graph"].sort_values("seed")["recall"].to_numpy()
    nr = sub[sub.model == "no_graph"].sort_values("seed")["recall"].to_numpy()
    tr, pr = stats.ttest_rel(wr, nr)
    print(f"[{thresh_name}] recall: wind={np.nanmean(wr):.4f} no_graph={np.nanmean(nr):.4f}  paired t={tr:.3f} p={pr:.4f}")


candidate graph: 11446 directed edges
pm25_mean_train=24.5797  pm25_std_train=16.9176
windows: train=26233, val=8689, test=17473, n_feats=21
device=mps
[wind_gcn_h36_blh_recent18_quantile_seed0] epoch  1  train=0.1993  val=0.2024  (132s)
[wind_gcn_h36_blh_recent18_quantile_seed0] epoch  2  train=0.1918  val=0.1984  (102s)
[wind_gcn_h36_blh_recent18_quantile_seed0] epoch  3  train=0.1900  val=0.1996  (102s)
[wind_gcn_h36_blh_recent18_quantile_seed0] epoch  4  train=0.1885  val=0.2016  (102s)
[wind_gcn_h36_blh_recent18_quantile_seed0] epoch  5  train=0.1875  val=0.2011  (102s)
[wind_gcn_h36_blh_recent18_quantile_seed0] epoch  6  train=0.1867  val=0.1989  (102s)
[wind_gcn_h36_blh_recent18_quantile_seed0] epoch  7  train=0.1861  val=0.2004  (102s)
[wind_gcn_h36_blh_recent18_quantile_seed0] epoch  8  train=0.1855  val=0.2004  (102s)
[wind_gcn_h36_blh_recent18_quantile_seed0] BEST epoch=2  val=0.1984  test=0.1654

[no_graph_h36_blh_quantile_seed0] epoch  1  train=0.2030  val=0.2024  (37s)
[n

In [3]:
# ================================================================
# Step 2: per-sensor extreme-event classification at H=36, using
# the VALIDATED models -- wind-graph (GRAPH_RECENT_HOURS=18, the
# sweep winner) vs no-graph, both already trained. No retraining:
# just load the 5 checkpoints per model and run inference on the
# held-out test set. Thresholds are Korea's real regional alert
# levels (75=advisory, 150=warning), applied per-sensor per-hour.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 125.6, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
train_end = int(np.searchsorted(years, 2019))
val_end = int(np.searchsorted(years, 2020))

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

train_nonzero = edge_weight_by_hour[:train_end][edge_weight_by_hour[:train_end] > 0]
weight_scale = train_nonzero.std()
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] + [wind_speed_arr, wdir_sin, wdir_cos, blh_arr], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[:train_end]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

pm25_col_idx = TIME_FEATS_FULL.index("PM25")
pm25_mean_train = t_mean[0, 0, pm25_col_idx]
pm25_std_train = t_std[0, 0, pm25_col_idx]
print(f"pm25_mean_train={pm25_mean_train:.4f}  pm25_std_train={pm25_std_train:.4f}")

def make_windows(start, end, arr):
    X_list, y_list, starts_list = [], [], []
    for t in range(start, end - WINDOW - HORIZON + 1):
        X_list.append(arr[t:t + WINDOW])
        y_list.append(arr[t + WINDOW + HORIZON - 1, :, pm25_col_idx])
        starts_list.append(t)
    return np.stack(X_list), np.stack(y_list), np.array(starts_list)

X_test, y_test, starts_test = make_windows(val_end, n_time, time_arr_std)
print(f"test windows: {len(X_test)}")
Xte_t, yte_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
starts_test_t = torch.tensor(starts_test, dtype=torch.long)

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def gather_edge_weight_seq(starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = edge_weight_by_hour_t[idx]
    if GRAPH_RECENT_HOURS < WINDOW:
        ew = ew.clone()
        ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        return self.lin_self(x) + self.lin_neigh(agg_mean)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1) if edge_weight_seq is not None else torch.ones(ei_b.shape[1], device=xt.device)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

class TemporalOnlyGRU(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index=None, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
WIND_CKPT_DIR = f"{BASE}/ckpt_wind_temporal_h36_blh_recent18"
NOGRAPH_CKPT_DIR = f"{BASE}/ckpt_wind_temporal_h36_blh"
print(f"device={DEVICE}")

@torch.no_grad()
def predict_full(model, use_graph, micro_batch=16):
    model.eval()
    n = Xte_t.shape[0]
    preds = []
    for start in range(0, n, micro_batch):
        xb = add_static(Xte_t[start:start + micro_batch]).to(DEVICE)
        ew_seq = gather_edge_weight_seq(starts_test_t[start:start + micro_batch]).to(DEVICE) if use_graph else None
        ei = edge_index.to(DEVICE) if use_graph else None
        pred = model(xb, ei, ew_seq)
        preds.append(pred.cpu())
    return torch.cat(preds, dim=0).numpy()

def unstd(x):
    return x * pm25_std_train + pm25_mean_train

actual_raw = unstd(y_test)  # (n_test, n_stations)
THRESHOLDS = {"advisory_75": 75.0, "warning_150": 150.0}
SEEDS = [0, 1, 2, 3, 4]

def classify_metrics(pred_raw, actual_raw, threshold):
    pred_pos = pred_raw >= threshold
    actual_pos = actual_raw >= threshold
    tp = int(np.sum(pred_pos & actual_pos))
    fp = int(np.sum(pred_pos & ~actual_pos))
    fn = int(np.sum(~pred_pos & actual_pos))
    tn = int(np.sum(~pred_pos & ~actual_pos))
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else float("nan")
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=precision, recall=recall, f1=f1,
                n_actual_positive=int(actual_pos.sum()))

all_rows = []
for seed in SEEDS:
    torch.manual_seed(seed)
    wind_model = WindTemporalGCN(n_feats).to(DEVICE)
    wind_model.load_state_dict(torch.load(f"{WIND_CKPT_DIR}/wind_temporal_gcn_h36_blh_recent18_seed{seed}.pt", map_location=DEVICE))
    pred_wind_raw = unstd(predict_full(wind_model, use_graph=True))

    nograph_model = TemporalOnlyGRU(n_feats).to(DEVICE)
    nograph_model.load_state_dict(torch.load(f"{NOGRAPH_CKPT_DIR}/no_graph_h36_blh_seed{seed}.pt", map_location=DEVICE))
    pred_nograph_raw = unstd(predict_full(nograph_model, use_graph=False))

    for thresh_name, thresh_val in THRESHOLDS.items():
        m_wind = classify_metrics(pred_wind_raw, actual_raw, thresh_val)
        m_nograph = classify_metrics(pred_nograph_raw, actual_raw, thresh_val)
        all_rows.append({"seed": seed, "threshold": thresh_name, "model": "wind_graph", **m_wind})
        all_rows.append({"seed": seed, "threshold": thresh_name, "model": "no_graph", **m_nograph})
    print(f"seed {seed} done", flush=True)

results_df = pd.DataFrame(all_rows)
print("\n=== per-seed classification metrics (H=36, wind=recent18) ===")
print(results_df[["threshold", "model", "seed", "tp", "fp", "fn", "tn", "precision", "recall", "f1"]].to_string(index=False))

print("\n=== summary across seeds (mean +/- std) ===")
summary = results_df.groupby(["threshold", "model"])[["precision", "recall", "f1"]].agg(["mean", "std"])
print(summary)

print("\n=== paired significance (wind vs no_graph), per threshold ===")
for thresh_name in THRESHOLDS:
    sub = results_df[results_df.threshold == thresh_name]
    wind_recall = sub[sub.model == "wind_graph"].sort_values("seed")["recall"].to_numpy()
    nograph_recall = sub[sub.model == "no_graph"].sort_values("seed")["recall"].to_numpy()
    t, p = stats.ttest_rel(wind_recall, nograph_recall)
    print(f"[{thresh_name}] recall: wind={wind_recall.mean():.4f} no_graph={nograph_recall.mean():.4f}  paired t={t:.3f} p={p:.4f}")

    wind_prec = sub[sub.model == "wind_graph"].sort_values("seed")["precision"].to_numpy()
    nograph_prec = sub[sub.model == "no_graph"].sort_values("seed")["precision"].to_numpy()
    t2, p2 = stats.ttest_rel(wind_prec, nograph_prec)
    print(f"[{thresh_name}] precision: wind={wind_prec.mean():.4f} no_graph={nograph_prec.mean():.4f}  paired t={t2:.3f} p={p2:.4f}")

    n_pos = sub[sub.model == "wind_graph"]["n_actual_positive"].mean()
    print(f"[{thresh_name}] avg actual-positive test samples per seed: {n_pos:.0f}\n")


pm25_mean_train=24.5797  pm25_std_train=16.9176
test windows: 17473
device=mps
seed 0 done
seed 1 done
seed 2 done
seed 3 done
seed 4 done

=== per-seed classification metrics (H=36, wind=recent18) ===
  threshold      model  seed  tp  fp    fn      tn  precision  recall  f1
advisory_75 wind_graph     0   0   0 24163 3051085        NaN     0.0 NaN
advisory_75   no_graph     0   0   0 24163 3051085        NaN     0.0 NaN
warning_150 wind_graph     0   0   0   931 3074317        NaN     0.0 NaN
warning_150   no_graph     0   0   0   931 3074317        NaN     0.0 NaN
advisory_75 wind_graph     1   0   0 24163 3051085        NaN     0.0 NaN
advisory_75   no_graph     1   0   0 24163 3051085        NaN     0.0 NaN
warning_150 wind_graph     1   0   0   931 3074317        NaN     0.0 NaN
warning_150   no_graph     1   0   0   931 3074317        NaN     0.0 NaN
advisory_75 wind_graph     2   0   0 24163 3051085        NaN     0.0 NaN
advisory_75   no_graph     2   0   0 24163 3051085        

In [4]:
print("pred_wind_raw (seed 4):    ", np.percentile(pred_wind_raw, [50, 90, 95, 99, 99.9, 99.99]), "max=", pred_wind_raw.max())
print("pred_nograph_raw (seed 4): ", np.percentile(pred_nograph_raw, [50, 90, 95, 99, 99.9, 99.99]), "max=", pred_nograph_raw.max())
print("actual_raw (test set):    ", np.percentile(actual_raw, [50, 90, 95, 99, 99.9, 99.99]), "max=", actual_raw.max())
print("\nfraction of actual test samples >= 75:", (actual_raw >= 75).mean())
print("fraction of actual test samples >= 150:", (actual_raw >= 150).mean())


pred_wind_raw (seed 4):     [18.6701353  29.62461116 33.34623655 40.36969506 48.2771549  52.9781158 ] max= 58.21236610751623
pred_nograph_raw (seed 4):  [19.56007975 29.72412586 33.26324961 39.61613091 45.3091545  48.89523159] max= 51.95965982659584
actual_raw (test set):     [ 15.  36.  45.  70. 117. 180.] max= 589.0000000000001

fraction of actual test samples >= 75: 0.007857252488254605
fraction of actual test samples >= 150: 0.00030273981155340966


In [3]:
# ================================================================
# Same wind-directed temporal graph model vs matched no-graph
# ablation -- WINDOW=24h, HORIZON=24h. SAME as before, except the
# data split: train=2016-2019, val=2020, test=2021 (was
# train=2016-2018, val=2019, test=2020-2021). Nothing else changed.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os, time

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 125.6, 250.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges")

WINDOW, HORIZON = 24, 24
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
train_end = int(np.searchsorted(years, 2020))  # CHANGED: train now includes 2019 too
val_end = int(np.searchsorted(years, 2021))    # CHANGED: val is 2020 only, test is 2021 only

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

train_nonzero = edge_weight_by_hour[:train_end][edge_weight_by_hour[:train_end] > 0]
weight_scale = train_nonzero.std()
print(f"edge weight scale (train, nonzero std): {weight_scale:.4f}")
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
print(f"edge_weight_by_hour: {edge_weight_by_hour.shape}, {edge_weight_by_hour.nbytes/1e9:.2f} GB")
edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] + [wind_speed_arr, wdir_sin, wdir_cos, blh_arr], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[:train_end]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

pm25_col_idx = TIME_FEATS_FULL.index("PM25")

def make_windows(start, end):
    X_list, y_list, starts_list = [], [], []
    for t in range(start, end - WINDOW - HORIZON + 1):
        X_list.append(time_arr[t:t + WINDOW])
        y_list.append(time_arr[t + WINDOW + HORIZON - 1, :, pm25_col_idx])
        starts_list.append(t)
    return np.stack(X_list), np.stack(y_list), np.array(starts_list)

X_train, y_train, starts_train = make_windows(0, train_end)
X_val, y_val, starts_val = make_windows(train_end, val_end)
X_test, y_test, starts_test = make_windows(val_end, n_time)
print(f"windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, n_feats={n_feats}")

Xtr_t, ytr_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)
Xva_t, yva_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
Xte_t, yte_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
starts_train_t, starts_val_t, starts_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts_train, starts_val, starts_test))

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def gather_edge_weight_seq(starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    return edge_weight_by_hour_t[idx]

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        return self.lin_self(x) + self.lin_neigh(agg_mean)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1) if edge_weight_seq is not None else torch.ones(ei_b.shape[1], device=xt.device)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

class TemporalOnlyGRU(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index=None, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_wind_temporal_h24_blh_altsplit"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 8, 1e-4
print(f"device={DEVICE}")

def run_epoch(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            with torch.set_grad_enabled(train):
                pred = model(xb, ei, ew_seq)
                loss = ((pred - yb) ** 2).mean()
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_model(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_epoch(model, Xtr_t, ytr_t, starts_train_t, use_graph, opt, train=True)
        val_loss = run_epoch(model, Xva_t, yva_t, starts_val_t, use_graph, opt, train=False)
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch(model, Xte_t, yte_t, starts_test_t, use_graph, None, train=False)
    print(f"[{name}] BEST epoch={best_epoch}  val={best_val:.4f}  test={test_loss:.4f}\n", flush=True)
    return {"name": name, "best_epoch": best_epoch, "val_mse": best_val, "test_mse": test_loss}

SEEDS = [0, 1, 2, 3, 4]
all_results = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    res_wind = train_model(f"wind_temporal_gcn_h24_blh_altsplit_seed{seed}", WindTemporalGCN(n_feats), use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_temporal_gcn"
    all_results.append(res_wind)

    torch.manual_seed(seed); np.random.seed(seed)
    res_nograph = train_model(f"no_graph_h24_blh_altsplit_seed{seed}", TemporalOnlyGRU(n_feats), use_graph=False)
    res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    all_results.append(res_nograph)

results_df = pd.DataFrame(all_results)
print(results_df[["model", "seed", "best_epoch", "val_mse", "test_mse"]].to_string(index=False))
print("\n=== summary across seeds (H=24, +BLH, alt split: train 2016-19, val 2020, test 2021) ===")
print(results_df.groupby("model")[["val_mse", "test_mse"]].agg(["mean", "std"]))

pivot_test = results_df.pivot(index="seed", columns="model", values="test_mse")
pivot_test["wind_wins"] = pivot_test["wind_temporal_gcn"] < pivot_test["no_graph"]
pivot_test["pct_improvement"] = 100 * (pivot_test["no_graph"] - pivot_test["wind_temporal_gcn"]) / pivot_test["no_graph"]
print("\n=== per-seed test MSE comparison (alt split) ===")
print(pivot_test.to_string())


candidate graph: 11446 directed edges
edge weight scale (train, nonzero std): 4.2190
edge_weight_by_hour: (52608, 11446), 2.41 GB
windows: train=35017, val=8737, test=8713, n_feats=21
device=mps
[wind_temporal_gcn_h24_blh_altsplit_seed0] epoch  1  train=0.7298  val=0.4401  (103s)
[wind_temporal_gcn_h24_blh_altsplit_seed0] epoch  2  train=0.6842  val=0.4186  (84s)
[wind_temporal_gcn_h24_blh_altsplit_seed0] epoch  3  train=0.6703  val=0.4238  (84s)
[wind_temporal_gcn_h24_blh_altsplit_seed0] epoch  4  train=0.6631  val=0.4108  (84s)
[wind_temporal_gcn_h24_blh_altsplit_seed0] epoch  5  train=0.6567  val=0.4112  (83s)
[wind_temporal_gcn_h24_blh_altsplit_seed0] epoch  6  train=0.6517  val=0.4243  (83s)
[wind_temporal_gcn_h24_blh_altsplit_seed0] epoch  7  train=0.6476  val=0.4188  (84s)
[wind_temporal_gcn_h24_blh_altsplit_seed0] epoch  8  train=0.6431  val=0.4076  (84s)
[wind_temporal_gcn_h24_blh_altsplit_seed0] BEST epoch=8  val=0.4076  test=0.5932

[no_graph_h24_blh_altsplit_seed0] epoch  1

In [4]:
import numpy as np
import torch
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, accuracy_score, confusion_matrix, f1_score, precision_recall_curve

def eval_predict(model, X, starts, use_graph, micro_batch=MICRO_BATCH):
    model.eval()
    n = X.shape[0]
    preds = []
    with torch.no_grad():
        for start in range(0, n, micro_batch):
            mb_idx = torch.arange(start, min(start + micro_batch, n))
            xb = add_static(X[mb_idx]).to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            preds.append(model(xb, ei, ew_seq).cpu().numpy())
    return np.concatenate(preds, axis=0)

pm25_mean_train = t_mean[0, 0, pm25_col_idx]
pm25_std_train = t_std[0, 0, pm25_col_idx]

SEEDS = [0, 1, 2, 3, 4]
val_preds_wind, test_preds_wind = [], []
val_preds_ng, test_preds_ng = [], []

for seed in SEEDS:
    wind_model = WindTemporalGCN(n_feats).to(DEVICE)
    wind_model.load_state_dict(torch.load(f"{CKPT_DIR}/wind_temporal_gcn_h24_blh_altsplit_seed{seed}.pt"))
    val_preds_wind.append(eval_predict(wind_model, Xva_t, starts_val_t, True))
    test_preds_wind.append(eval_predict(wind_model, Xte_t, starts_test_t, True))

    ng_model = TemporalOnlyGRU(n_feats).to(DEVICE)
    ng_model.load_state_dict(torch.load(f"{CKPT_DIR}/no_graph_h24_blh_altsplit_seed{seed}.pt"))
    val_preds_ng.append(eval_predict(ng_model, Xva_t, starts_val_t, False))
    test_preds_ng.append(eval_predict(ng_model, Xte_t, starts_test_t, False))

val_pred_wind_ens = np.mean(val_preds_wind, axis=0) * pm25_std_train + pm25_mean_train
test_pred_wind_ens = np.mean(test_preds_wind, axis=0) * pm25_std_train + pm25_mean_train
val_pred_ng_ens = np.mean(val_preds_ng, axis=0) * pm25_std_train + pm25_mean_train
test_pred_ng_ens = np.mean(test_preds_ng, axis=0) * pm25_std_train + pm25_mean_train

y_val_raw = y_val * pm25_std_train + pm25_mean_train
y_test_raw = y_test * pm25_std_train + pm25_mean_train

THRESHOLDS = {"bad_36": 36.0, "very_bad_76": 76.0}

print("=== How often are we RIGHT when predicting extreme events? (5-seed ensemble, alt split test=2021) ===\n")
for model_name, val_pred, test_pred in [("wind_graph", val_pred_wind_ens, test_pred_wind_ens),
                                          ("no_graph", val_pred_ng_ens, test_pred_ng_ens)]:
    print(f"--- {model_name} ---")
    for thresh_name, c in THRESHOLDS.items():
        yv_label = (y_val_raw.flatten() > c).astype(int)
        yt_label = (y_test_raw.flatten() > c).astype(int)

        S = LogisticRegression(max_iter=1000)
        S.fit(val_pred.flatten().reshape(-1, 1), yv_label)
        p_val = S.predict_proba(val_pred.flatten().reshape(-1, 1))[:, 1]
        p_test = S.predict_proba(test_pred.flatten().reshape(-1, 1))[:, 1]

        # pick the decision threshold that maximizes F1 on VAL (never touches test)
        prec, rec, thresh_grid = precision_recall_curve(yv_label, p_val)
        f1s = 2 * prec * rec / (prec + rec + 1e-12)
        best_idx = np.nanargmax(f1s[:-1])
        best_thresh = thresh_grid[best_idx]

        pred_label_test = (p_test >= best_thresh).astype(int)
        prec_t = precision_score(yt_label, pred_label_test, zero_division=0)
        rec_t = recall_score(yt_label, pred_label_test, zero_division=0)
        acc_t = accuracy_score(yt_label, pred_label_test)
        f1_t = f1_score(yt_label, pred_label_test, zero_division=0)
        cm = confusion_matrix(yt_label, pred_label_test)

        print(f"  {thresh_name} (c={c}, decision threshold chosen via F1 on val: P>={best_thresh:.3f}):")
        print(f"    base rate (test): {yt_label.mean():.4f}")
        print(f"    precision={prec_t:.3f}  recall={rec_t:.3f}  accuracy={acc_t:.3f}  F1={f1_t:.3f}")
        print(f"    confusion matrix [[TN,FP],[FN,TP]]:\n{cm}")
    print()


=== How often are we RIGHT when predicting extreme events? (5-seed ensemble, alt split test=2021) ===

--- wind_graph ---
  bad_36 (c=36.0, decision threshold chosen via F1 on val: P>=0.181):
    base rate (test): 0.0903
    precision=0.379  recall=0.418  accuracy=0.886  F1=0.397
    confusion matrix [[TN,FP],[FN,TP]]:
[[1300197   94871]
 [  80600   57820]]
  very_bad_76 (c=76.0, decision threshold chosen via F1 on val: P>=0.049):
    base rate (test): 0.0109
    precision=0.239  recall=0.180  accuracy=0.985  F1=0.205
    confusion matrix [[TN,FP],[FN,TP]]:
[[1507113    9609]
 [  13753    3013]]

--- no_graph ---
  bad_36 (c=36.0, decision threshold chosen via F1 on val: P>=0.170):
    base rate (test): 0.0903
    precision=0.348  recall=0.443  accuracy=0.875  F1=0.389
    confusion matrix [[TN,FP],[FN,TP]]:
[[1280015  115053]
 [  77142   61278]]
  very_bad_76 (c=76.0, decision threshold chosen via F1 on val: P>=0.044):
    base rate (test): 0.0109
    precision=0.220  recall=0.230  ac

In [5]:
import numpy as np
import torch

def gather_edge_weight_seq_custom(starts_subset, weight_tensor):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    return weight_tensor[idx]

def eval_predict_custom(model, X, starts, weight_tensor, micro_batch=MICRO_BATCH):
    model.eval()
    n = X.shape[0]
    preds = []
    with torch.no_grad():
        for start in range(0, n, micro_batch):
            mb_idx = torch.arange(start, min(start + micro_batch, n))
            xb = add_static(X[mb_idx]).to(DEVICE)
            ew_seq = gather_edge_weight_seq_custom(starts[mb_idx], weight_tensor).to(DEVICE)
            ei = edge_index.to(DEVICE)
            preds.append(model(xb, ei, ew_seq).cpu().numpy())
    return np.concatenate(preds, axis=0)

# the "helps most" inland stations identified earlier (63-79km from coast) --
# testing whether their presence as an upwind SOURCE currently helps their
# downwind neighbors, as a proxy for "would a new station in a similar
# position help the stations that currently lack one"
ABLATE_STATIONS = [221211, 633123, 632121]
SEEDS = [0, 1, 2, 3, 4]

for ablate_id in ABLATE_STATIONS:
    k_idx = station_order.index(ablate_id)
    ablate_edge_mask = (src_idx == k_idx)
    n_outgoing = ablate_edge_mask.sum()
    dependent_targets = np.unique(dst_idx[ablate_edge_mask])
    print(f"\n=== ablating station {ablate_id} as a source ===")
    print(f"{n_outgoing} outgoing edges to {len(dependent_targets)} dependent stations")

    edge_weight_ablated = edge_weight_by_hour.copy()
    edge_weight_ablated[:, ablate_edge_mask] = 0.0
    edge_weight_ablated_t = torch.tensor(edge_weight_ablated)

    normal_preds, ablated_preds = [], []
    for seed in SEEDS:
        wind_model = WindTemporalGCN(n_feats).to(DEVICE)
        wind_model.load_state_dict(torch.load(f"{CKPT_DIR}/wind_temporal_gcn_h24_blh_altsplit_seed{seed}.pt"))
        normal_preds.append(eval_predict_custom(wind_model, Xte_t, starts_test_t, edge_weight_by_hour_t))
        ablated_preds.append(eval_predict_custom(wind_model, Xte_t, starts_test_t, edge_weight_ablated_t))

    normal_ens = np.mean(normal_preds, axis=0)
    ablated_ens = np.mean(ablated_preds, axis=0)

    mse_normal = ((normal_ens - y_test) ** 2).mean(axis=0)   # [N]
    mse_ablated = ((ablated_ens - y_test) ** 2).mean(axis=0)  # [N]

    degradation = mse_ablated[dependent_targets] - mse_normal[dependent_targets]
    pct_worse = 100 * degradation.mean() / mse_normal[dependent_targets].mean()
    print(f"mean MSE degradation for dependent stations: {degradation.mean():.5f} ({pct_worse:.2f}% worse)")
    print(f"dependent stations that got WORSE: {(degradation > 0).sum()}/{len(dependent_targets)}")

    overall_degradation = (mse_ablated - mse_normal).mean()
    print(f"overall network-wide MSE change (for comparison): {overall_degradation:.5f}")



=== ablating station 221211 as a source ===
54 outgoing edges to 54 dependent stations
mean MSE degradation for dependent stations: -0.00046 (-0.13% worse)
dependent stations that got WORSE: 1/54
overall network-wide MSE change (for comparison): -0.00014

=== ablating station 633123 as a source ===
87 outgoing edges to 87 dependent stations


KeyboardInterrupt: 

In [6]:
import numpy as np
import pandas as pd

pm25_wide = df.pivot(index="Datetime", columns="Station_ID", values="PM25")[station_order].reindex(dt_index).to_numpy()
frac_exceeding_36 = (pm25_wide > 36).mean(axis=1)  # [T], fraction of stations over "Bad" each hour
frac_exceeding_76 = (pm25_wide > 76).mean(axis=1)  # [T], fraction over "Very Bad"

graph_density = edge_weight_by_hour.mean(axis=1)       # mean edge weight across all candidate edges, per hour
graph_total = edge_weight_by_hour.sum(axis=1)          # total connectivity
graph_active_frac = (edge_weight_by_hour > 0).mean(axis=1)  # fraction of edges with any wind alignment at all

H = 24
valid_t = np.arange(len(graph_density) - H)

print("does current graph structure predict the fraction of the network in an extreme state, H hours later?\n")
for label, X in [("mean edge weight", graph_density), ("total connectivity", graph_total), ("fraction active edges", graph_active_frac)]:
    for thresh_label, Y_full in [("Bad (36+)", frac_exceeding_36), ("Very Bad (76+)", frac_exceeding_76)]:
        c = np.corrcoef(X[valid_t], Y_full[valid_t + H])[0, 1]
        print(f"  {label:<22} vs future %{thresh_label:<15}: corr = {c:+.4f}")


does current graph structure predict the fraction of the network in an extreme state, H hours later?

  mean edge weight       vs future %Bad (36+)      : corr = -0.2601
  mean edge weight       vs future %Very Bad (76+) : corr = -0.1047
  total connectivity     vs future %Bad (36+)      : corr = -0.2601
  total connectivity     vs future %Very Bad (76+) : corr = -0.1047
  fraction active edges  vs future %Bad (36+)      : corr = -0.1995
  fraction active edges  vs future %Very Bad (76+) : corr = -0.0844


In [7]:
national_windspeed = wind_speed_arr.mean(axis=1)  # simple national mean wind speed, no graph at all
national_blh = blh_arr.mean(axis=1)                # simple national mean boundary layer height

H = 24
valid_t = np.arange(len(graph_density) - H)

candidates = {
    "graph: mean edge weight":         graph_density,
    "graph: total connectivity":       graph_total,
    "graph: fraction active edges":    graph_active_frac,
    "weather: national mean windspeed": national_windspeed,
    "weather: national mean BLH":       national_blh,
}

print("does current signal predict the fraction of the network in an extreme state, H hours later?\n")
for label, X in candidates.items():
    for thresh_label, Y_full in [("Bad (36+)", frac_exceeding_36), ("Very Bad (76+)", frac_exceeding_76)]:
        c = np.corrcoef(X[valid_t], Y_full[valid_t + H])[0, 1]
        print(f"  {label:<34} vs future %{thresh_label:<15}: corr = {c:+.4f}")


does current signal predict the fraction of the network in an extreme state, H hours later?

  graph: mean edge weight            vs future %Bad (36+)      : corr = -0.2601
  graph: mean edge weight            vs future %Very Bad (76+) : corr = -0.1047
  graph: total connectivity          vs future %Bad (36+)      : corr = -0.2601
  graph: total connectivity          vs future %Very Bad (76+) : corr = -0.1047
  graph: fraction active edges       vs future %Bad (36+)      : corr = -0.1995
  graph: fraction active edges       vs future %Very Bad (76+) : corr = -0.0844
  weather: national mean windspeed   vs future %Bad (36+)      : corr = -0.2437
  weather: national mean windspeed   vs future %Very Bad (76+) : corr = -0.0977
  weather: national mean BLH         vs future %Bad (36+)      : corr = -0.2088
  weather: national mean BLH         vs future %Very Bad (76+) : corr = -0.0994


In [4]:
import numpy as np
import torch
import pandas as pd

def eval_mae(model, X, y, starts, use_graph, micro_batch=MICRO_BATCH):
    model.eval()
    n = X.shape[0]
    total_abs_err, total_n = 0.0, 0
    with torch.no_grad():
        for start in range(0, n, micro_batch):
            mb_idx = torch.arange(start, min(start + micro_batch, n))
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            pred = model(xb, ei, ew_seq)
            total_abs_err += (pred - yb).abs().sum().item()
            total_n += yb.numel()
    return total_abs_err / total_n

pm25_std_train = t_std[0, 0, pm25_col_idx]
print(f"train-period PM2.5 std used for conversion: {pm25_std_train:.4f} ug/m3")

mae_results = []
for seed in SEEDS:
    wind_model = WindTemporalGCN(n_feats).to(DEVICE)
    wind_model.load_state_dict(torch.load(f"{CKPT_DIR}/wind_temporal_gcn_h24_blh_seed{seed}.pt"))
    mae_val_w = eval_mae(wind_model, Xva_t, yva_t, starts_val_t, use_graph=True)
    mae_test_w = eval_mae(wind_model, Xte_t, yte_t, starts_test_t, use_graph=True)

    nograph_model = TemporalOnlyGRU(n_feats).to(DEVICE)
    nograph_model.load_state_dict(torch.load(f"{CKPT_DIR}/no_graph_h24_blh_seed{seed}.pt"))
    mae_val_n = eval_mae(nograph_model, Xva_t, yva_t, starts_val_t, use_graph=False)
    mae_test_n = eval_mae(nograph_model, Xte_t, yte_t, starts_test_t, use_graph=False)

    mae_results.append({"seed": seed, "model": "wind_temporal_gcn",
                         "val_mae_ugm3": mae_val_w * pm25_std_train, "test_mae_ugm3": mae_test_w * pm25_std_train})
    mae_results.append({"seed": seed, "model": "no_graph",
                         "val_mae_ugm3": mae_val_n * pm25_std_train, "test_mae_ugm3": mae_test_n * pm25_std_train})

mae_df = pd.DataFrame(mae_results)
print(mae_df.to_string(index=False))
print("\n=== summary across seeds (MAE, ug/m3) ===")
print(mae_df.groupby("model")[["val_mae_ugm3", "test_mae_ugm3"]].agg(["mean", "std"]))


train-period PM2.5 std used for conversion: 16.9176 ug/m3
 seed             model  val_mae_ugm3  test_mae_ugm3
    0 wind_temporal_gcn     10.272544       8.870502
    0          no_graph     10.419262       9.035810
    1 wind_temporal_gcn     10.446626       9.028843
    1          no_graph     10.504655       9.195256
    2 wind_temporal_gcn     10.178477       8.612471
    2          no_graph     10.371374       9.034159
    3 wind_temporal_gcn     10.225021       8.744991
    3          no_graph     10.492899       9.149829
    4 wind_temporal_gcn     10.252401       8.773269
    4          no_graph     10.518947       9.123435

=== summary across seeds (MAE, ug/m3) ===
                  val_mae_ugm3           test_mae_ugm3          
                          mean       std          mean       std
model                                                           
no_graph             10.461427  0.063354      9.107698  0.071178
wind_temporal_gcn    10.275014  0.102209      8.806015  

In [5]:
#paired t test per station 
import numpy as np
import torch
from scipy import stats

def eval_predict(model, X, starts, use_graph, micro_batch=MICRO_BATCH):
    model.eval()
    n = X.shape[0]
    preds = []
    with torch.no_grad():
        for start in range(0, n, micro_batch):
            mb_idx = torch.arange(start, min(start + micro_batch, n))
            xb = add_static(X[mb_idx]).to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            preds.append(model(xb, ei, ew_seq).cpu().numpy())
    return np.concatenate(preds, axis=0)

SEEDS = [0, 1, 2, 3, 4]
per_station_mse_wind = np.zeros((len(SEEDS), n_stations))
per_station_mse_nograph = np.zeros((len(SEEDS), n_stations))

for i, seed in enumerate(SEEDS):
    wind_model = WindTemporalGCN(n_feats).to(DEVICE)
    wind_model.load_state_dict(torch.load(f"{CKPT_DIR}/wind_temporal_gcn_h24_blh_seed{seed}.pt"))
    pred_wind = eval_predict(wind_model, Xte_t, starts_test_t, use_graph=True)
    per_station_mse_wind[i] = ((pred_wind - y_test) ** 2).mean(axis=0)

    nograph_model = TemporalOnlyGRU(n_feats).to(DEVICE)
    nograph_model.load_state_dict(torch.load(f"{CKPT_DIR}/no_graph_h24_blh_seed{seed}.pt"))
    pred_ng = eval_predict(nograph_model, Xte_t, starts_test_t, use_graph=False)
    per_station_mse_nograph[i] = ((pred_ng - y_test) ** 2).mean(axis=0)

# average across the 5 seeds first, so each station gets one robust MSE per model
station_mse_wind = per_station_mse_wind.mean(axis=0)      # [176]
station_mse_nograph = per_station_mse_nograph.mean(axis=0)  # [176]

diff = station_mse_nograph - station_mse_wind
n_wins = (diff > 0).sum()
print(f"wind beats no_graph at {n_wins} / {n_stations} stations")
print(f"mean per-station improvement: {diff.mean():.5f} ({100*diff.mean()/station_mse_nograph.mean():.2f}% relative)")
print(f"std of per-station differences: {diff.std(ddof=1):.5f}")

t_stat, p_value = stats.ttest_rel(station_mse_nograph, station_mse_wind)
print(f"\npaired t-test across {n_stations} stations: t={t_stat:.3f}, p={p_value:.3e}")

d = diff.mean() / diff.std(ddof=1)
print(f"effect size (Cohen's d): {d:.3f}")

w_stat, w_p = stats.wilcoxon(station_mse_nograph, station_mse_wind)
print(f"Wilcoxon signed-rank test (non-parametric check): W={w_stat:.1f}, p={w_p:.3e}")

worst_idx = np.argsort(diff)[:5]
best_idx = np.argsort(diff)[-5:][::-1]
print("\nstations where wind helps LEAST (or hurts):")
for idx in worst_idx:
    print(f"  station {station_order[idx]}: no_graph_mse={station_mse_nograph[idx]:.4f}  wind_mse={station_mse_wind[idx]:.4f}  diff={diff[idx]:.4f}")
print("\nstations where wind helps MOST:")
for idx in best_idx:
    print(f"  station {station_order[idx]}: no_graph_mse={station_mse_nograph[idx]:.4f}  wind_mse={station_mse_wind[idx]:.4f}  diff={diff[idx]:.4f}")


wind beats no_graph at 171 / 176 stations
mean per-station improvement: 0.02387 (4.21% relative)
std of per-station differences: 0.01275

paired t-test across 176 stations: t=24.838, p=2.917e-59
effect size (Cohen's d): 1.872
Wilcoxon signed-rank test (non-parametric check): W=117.0, p=9.054e-30

stations where wind helps LEAST (or hurts):
  station 221231: no_graph_mse=0.3624  wind_mse=0.3779  diff=-0.0155
  station 632161: no_graph_mse=0.3056  wind_mse=0.3200  diff=-0.0144
  station 632132: no_graph_mse=0.3264  wind_mse=0.3405  diff=-0.0142
  station 221251: no_graph_mse=0.3196  wind_mse=0.3235  diff=-0.0039
  station 339111: no_graph_mse=0.4915  wind_mse=0.4950  diff=-0.0035

stations where wind helps MOST:
  station 221211: no_graph_mse=0.4938  wind_mse=0.4247  diff=0.0690
  station 633123: no_graph_mse=0.5941  wind_mse=0.5377  diff=0.0564
  station 632121: no_graph_mse=0.6175  wind_mse=0.5658  diff=0.0517
  station 633211: no_graph_mse=0.6495  wind_mse=0.5981  diff=0.0514
  statio

In [10]:
import numpy as np
import torch
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

def eval_predict(model, X, starts, use_graph, micro_batch=MICRO_BATCH):
    model.eval()
    n = X.shape[0]
    preds = []
    with torch.no_grad():
        for start in range(0, n, micro_batch):
            mb_idx = torch.arange(start, min(start + micro_batch, n))
            xb = add_static(X[mb_idx]).to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            pred = model(xb, ei, ew_seq)
            preds.append(pred.cpu().numpy())
    return np.concatenate(preds, axis=0)

pm25_mean_train = t_mean[0, 0, pm25_col_idx]
pm25_std_train = t_std[0, 0, pm25_col_idx]

y_val_raw = y_val * pm25_std_train + pm25_mean_train
y_test_raw = y_test * pm25_std_train + pm25_mean_train

THRESHOLDS = {"bad_36": 36.0, "very_bad_76": 76.0}
classification_results = []

for seed in SEEDS:
    wind_model = WindTemporalGCN(n_feats).to(DEVICE)
    wind_model.load_state_dict(torch.load(f"{CKPT_DIR}/wind_temporal_gcn_h24_blh_seed{seed}.pt"))
    pred_val_wind = eval_predict(wind_model, Xva_t, starts_val_t, use_graph=True) * pm25_std_train + pm25_mean_train
    pred_test_wind = eval_predict(wind_model, Xte_t, starts_test_t, use_graph=True) * pm25_std_train + pm25_mean_train

    nograph_model = TemporalOnlyGRU(n_feats).to(DEVICE)
    nograph_model.load_state_dict(torch.load(f"{CKPT_DIR}/no_graph_h24_blh_seed{seed}.pt"))
    pred_val_ng = eval_predict(nograph_model, Xva_t, starts_val_t, use_graph=False) * pm25_std_train + pm25_mean_train
    pred_test_ng = eval_predict(nograph_model, Xte_t, starts_test_t, use_graph=False) * pm25_std_train + pm25_mean_train

    for model_name, pv, pt in [("wind_temporal_gcn", pred_val_wind, pred_test_wind),
                                 ("no_graph", pred_val_ng, pred_test_ng)]:
        for thresh_name, c in THRESHOLDS.items():
            yv_label = (y_val_raw.flatten() > c).astype(int)
            yt_label = (y_test_raw.flatten() > c).astype(int)
            pv_flat = pv.flatten().reshape(-1, 1)
            pt_flat = pt.flatten().reshape(-1, 1)

            S = LogisticRegression(max_iter=1000)
            S.fit(pv_flat, yv_label)
            p_test = S.predict_proba(pt_flat)[:, 1]

            classification_results.append({
                "seed": seed, "model": model_name, "threshold": thresh_name,
                "AUC": roc_auc_score(yt_label, p_test),
                "PR_AUC": average_precision_score(yt_label, p_test),
                "Brier": brier_score_loss(yt_label, p_test),
            })

cls_df = pd.DataFrame(classification_results)
print(cls_df.to_string(index=False))
print("\n=== summary across seeds ===")
print(cls_df.groupby(["threshold", "model"])[["AUC", "PR_AUC", "Brier"]].agg(["mean", "std"]))


 seed             model   threshold      AUC   PR_AUC    Brier
    0 wind_temporal_gcn      bad_36 0.839982 0.387369 0.070331
    0 wind_temporal_gcn very_bad_76 0.812330 0.055149 0.007511
    0          no_graph      bad_36 0.829934 0.352843 0.072380
    0          no_graph very_bad_76 0.825495 0.081488 0.007197
    1 wind_temporal_gcn      bad_36 0.839870 0.379442 0.070401
    1 wind_temporal_gcn very_bad_76 0.830313 0.072455 0.007241
    1          no_graph      bad_36 0.828389 0.353054 0.072368
    1          no_graph very_bad_76 0.819362 0.071094 0.007222
    2 wind_temporal_gcn      bad_36 0.847302 0.392982 0.069703
    2 wind_temporal_gcn very_bad_76 0.839009 0.063023 0.007380
    2          no_graph      bad_36 0.828806 0.355633 0.072123
    2          no_graph very_bad_76 0.812012 0.076422 0.007166

=== summary across seeds ===
                                    AUC              PR_AUC            \
                                   mean       std      mean       std   
thres

In [11]:
import numpy as np
import torch
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ALPHA, W_THRESH, W_SCALE, W_CAP = 3.0, 36.0, 40.0, 3.0  # weight ramps up above "Bad" threshold, capped at 1+ALPHA*W_CAP

def sample_weight(yb_raw):
    return 1.0 + ALPHA * torch.clamp((yb_raw - W_THRESH) / W_SCALE, min=0.0, max=W_CAP)

def run_epoch_weighted(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            with torch.set_grad_enabled(train):
                pred = model(xb, ei, ew_seq)
                yb_raw = yb * pm25_std_train + pm25_mean_train
                w = sample_weight(yb_raw)
                loss = (w * (pred - yb) ** 2).mean()
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_model_weighted(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        train_loss = run_epoch_weighted(model, Xtr_t, ytr_t, starts_train_t, use_graph, opt, train=True)
        val_loss = run_epoch_weighted(model, Xva_t, yva_t, starts_val_t, use_graph, opt, train=False)
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch_weighted(model, Xte_t, yte_t, starts_test_t, use_graph, None, train=False)
    print(f"[{name}] BEST epoch={best_epoch}  val={best_val:.4f}  test={test_loss:.4f}\n", flush=True)
    return best_epoch, best_val, test_loss

SEEDS = [0, 1, 2]
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    train_model_weighted(f"wind_temporal_gcn_h24_blh_weighted_seed{seed}", WindTemporalGCN(n_feats), use_graph=True)
    torch.manual_seed(seed); np.random.seed(seed)
    train_model_weighted(f"no_graph_h24_blh_weighted_seed{seed}", TemporalOnlyGRU(n_feats), use_graph=False)

# --- classification check, same as before, now on the weighted-loss checkpoints ---
def eval_predict(model, X, starts, use_graph, micro_batch=MICRO_BATCH):
    model.eval()
    n = X.shape[0]
    preds = []
    with torch.no_grad():
        for start in range(0, n, micro_batch):
            mb_idx = torch.arange(start, min(start + micro_batch, n))
            xb = add_static(X[mb_idx]).to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            preds.append(model(xb, ei, ew_seq).cpu().numpy())
    return np.concatenate(preds, axis=0)

y_val_raw = y_val * pm25_std_train + pm25_mean_train
y_test_raw = y_test * pm25_std_train + pm25_mean_train
THRESHOLDS = {"bad_36": 36.0, "very_bad_76": 76.0}
classification_results = []

for seed in SEEDS:
    wind_model = WindTemporalGCN(n_feats).to(DEVICE)
    wind_model.load_state_dict(torch.load(f"{CKPT_DIR}/wind_temporal_gcn_h24_blh_weighted_seed{seed}.pt"))
    pred_val_wind = eval_predict(wind_model, Xva_t, starts_val_t, True) * pm25_std_train + pm25_mean_train
    pred_test_wind = eval_predict(wind_model, Xte_t, starts_test_t, True) * pm25_std_train + pm25_mean_train

    nograph_model = TemporalOnlyGRU(n_feats).to(DEVICE)
    nograph_model.load_state_dict(torch.load(f"{CKPT_DIR}/no_graph_h24_blh_weighted_seed{seed}.pt"))
    pred_val_ng = eval_predict(nograph_model, Xva_t, starts_val_t, False) * pm25_std_train + pm25_mean_train
    pred_test_ng = eval_predict(nograph_model, Xte_t, starts_test_t, False) * pm25_std_train + pm25_mean_train

    for model_name, pv, pt in [("wind_temporal_gcn", pred_val_wind, pred_test_wind), ("no_graph", pred_val_ng, pred_test_ng)]:
        for thresh_name, c in THRESHOLDS.items():
            yv_label = (y_val_raw.flatten() > c).astype(int)
            yt_label = (y_test_raw.flatten() > c).astype(int)
            S = LogisticRegression(max_iter=1000)
            S.fit(pv.flatten().reshape(-1, 1), yv_label)
            p_test = S.predict_proba(pt.flatten().reshape(-1, 1))[:, 1]
            classification_results.append({
                "seed": seed, "model": model_name, "threshold": thresh_name,
                "AUC": roc_auc_score(yt_label, p_test),
                "PR_AUC": average_precision_score(yt_label, p_test),
                "Brier": brier_score_loss(yt_label, p_test),
            })

cls_df = pd.DataFrame(classification_results)
print(cls_df.to_string(index=False))
print("\n=== summary across seeds (weighted loss) ===")
print(cls_df.groupby(["threshold", "model"])[["AUC", "PR_AUC", "Brier"]].agg(["mean", "std"]))


[wind_temporal_gcn_h24_blh_weighted_seed0] epoch  1  train=1.6996  val=2.2328
[wind_temporal_gcn_h24_blh_weighted_seed0] epoch  2  train=1.5252  val=2.6209
[wind_temporal_gcn_h24_blh_weighted_seed0] epoch  3  train=1.4801  val=2.5243
[wind_temporal_gcn_h24_blh_weighted_seed0] epoch  4  train=1.4569  val=2.4744
[wind_temporal_gcn_h24_blh_weighted_seed0] epoch  5  train=1.4366  val=2.4084
[wind_temporal_gcn_h24_blh_weighted_seed0] epoch  6  train=1.4255  val=2.3922
[wind_temporal_gcn_h24_blh_weighted_seed0] epoch  7  train=1.4107  val=2.4078
[wind_temporal_gcn_h24_blh_weighted_seed0] epoch  8  train=1.3984  val=2.4600
[wind_temporal_gcn_h24_blh_weighted_seed0] BEST epoch=1  val=2.2328  test=1.4566

[no_graph_h24_blh_weighted_seed0] epoch  1  train=1.7408  val=2.3938
[no_graph_h24_blh_weighted_seed0] epoch  2  train=1.6128  val=2.3304
[no_graph_h24_blh_weighted_seed0] epoch  3  train=1.5792  val=2.3792
[no_graph_h24_blh_weighted_seed0] epoch  4  train=1.5591  val=2.3901
[no_graph_h24_blh_

In [10]:
import time
import torch
import torch.nn as nn
import pandas as pd
from torch_geometric.nn import GATConv

class VanillaGAT(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, heads=1, dropout=0.3):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden, heads=heads, concat=False, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_fixed=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.conv1(xt, ei_b))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

DEVICE_CPU = torch.device("cpu")
ei_cpu = edge_index_std.to(DEVICE_CPU)

def run_epoch_gat_cpu(model, X, y, optimizer, train, desc="", micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS, print_every_frac=0.1):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    starts = list(range(0, n, eff_batch))
    print_every = max(1, int(len(starts) * print_every_frac))
    for i, start in enumerate(starts):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx])
            yb = y[mb_idx]
            with torch.set_grad_enabled(train):
                pred = model(xb, ei_cpu, None)
                loss = ((pred - yb) ** 2).mean()
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
        if (i + 1) % print_every == 0 or (i + 1) == len(starts):
            pct = 100 * (i + 1) / len(starts)
            print(f"  {desc}: {pct:5.1f}% ({i+1}/{len(starts)} steps)  running_loss={total_loss/max(total_n,1):.4f}", flush=True)
    return total_loss / total_n

def train_gat_cpu(name, model, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE_CPU)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_epoch_gat_cpu(model, Xtr_t, ytr_t, opt, train=True, desc=f"[{name}] epoch {epoch} train")
        val_loss = run_epoch_gat_cpu(model, Xva_t, yva_t, opt, train=False, desc=f"[{name}] epoch {epoch} val")
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch_gat_cpu(model, Xte_t, yte_t, opt, train=False, desc=f"[{name}] test")
    print(f"[{name}] BEST epoch={best_epoch}  val={best_val:.4f}  test={test_loss:.4f}\n", flush=True)
    return {"name": name, "best_epoch": best_epoch, "val_mse": best_val, "test_mse": test_loss}

SEEDS_GAT = [0, 1, 2]
gat_results = []
for seed in SEEDS_GAT:
    torch.manual_seed(seed); np.random.seed(seed)
    r = train_gat_cpu(f"vanilla_gat_h24_seed{seed}", VanillaGAT(n_feats))
    r["seed"], r["model"] = seed, "vanilla_gat"
    gat_results.append(r)

gat_df = pd.DataFrame(gat_results)
print(gat_df.to_string(index=False))


  [vanilla_gat_h24_seed0] epoch 1 train:  10.0% (41/411 steps)  running_loss=0.8781
  [vanilla_gat_h24_seed0] epoch 1 train:  20.0% (82/411 steps)  running_loss=0.8375
  [vanilla_gat_h24_seed0] epoch 1 train:  29.9% (123/411 steps)  running_loss=0.8267
  [vanilla_gat_h24_seed0] epoch 1 train:  39.9% (164/411 steps)  running_loss=0.8123
  [vanilla_gat_h24_seed0] epoch 1 train:  49.9% (205/411 steps)  running_loss=0.8028
  [vanilla_gat_h24_seed0] epoch 1 train:  59.9% (246/411 steps)  running_loss=0.7936
  [vanilla_gat_h24_seed0] epoch 1 train:  69.8% (287/411 steps)  running_loss=0.7922
  [vanilla_gat_h24_seed0] epoch 1 train:  79.8% (328/411 steps)  running_loss=0.7899
  [vanilla_gat_h24_seed0] epoch 1 train:  89.8% (369/411 steps)  running_loss=0.7894
  [vanilla_gat_h24_seed0] epoch 1 train:  99.8% (410/411 steps)  running_loss=0.7899
  [vanilla_gat_h24_seed0] epoch 1 train: 100.0% (411/411 steps)  running_loss=0.7901
  [vanilla_gat_h24_seed0] epoch 1 val:   9.5% (13/137 steps)  runni

In [1]:
import numpy as np
import torch
import pandas as pd
from scipy import stats

SPLITS = {
    "A_original":  {"train": [2016,2017,2018], "val": [2019], "test": [2020,2021]},
    "B_expanding": {"train": [2016,2017,2018,2019], "val": [2020], "test": [2021]},
    "C_sliding":   {"train": [2017,2018,2019], "val": [2020], "test": [2021]},
}

years_arr = dt_index.year.to_numpy()
SEEDS_SPLIT = [0, 1, 2]
split_results = []

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] + [wind_speed_arr, wdir_sin, wdir_cos, blh_arr], axis=-1)
print(f"time_arr_raw rebuilt: {time_arr_raw.shape}")

for split_name, yrs in SPLITS.items():
    print(f"\n{'='*60}\nSPLIT {split_name}: train={yrs['train']} val={yrs['val']} test={yrs['test']}\n{'='*60}", flush=True)

    train_idx = np.where(np.isin(years_arr, yrs["train"]))[0]
    val_idx = np.where(np.isin(years_arr, yrs["val"]))[0]
    test_idx = np.where(np.isin(years_arr, yrs["test"]))[0]
    tr_start, tr_end = train_idx.min(), train_idx.max() + 1
    va_start, va_end = val_idx.min(), val_idx.max() + 1
    te_start, te_end = test_idx.min(), test_idx.max() + 1

    # re-standardize using THIS split's own training period
    time_train_split = time_arr_raw[tr_start:tr_end]
    t_mean_s = np.nanmean(time_train_split, axis=(0, 1), keepdims=True)
    t_std_s = np.nanstd(time_train_split, axis=(0, 1), keepdims=True) + 1e-6
    time_arr_s = np.nan_to_num((time_arr_raw - t_mean_s) / t_std_s, nan=0.0)

    def make_windows_split(start, end):
        X_list, y_list, starts_list = [], [], []
        for t in range(start, end - WINDOW - HORIZON + 1):
            X_list.append(time_arr_s[t:t + WINDOW])
            y_list.append(time_arr_s[t + WINDOW + HORIZON - 1, :, pm25_col_idx])
            starts_list.append(t)
        return np.stack(X_list), np.stack(y_list), np.array(starts_list)

    Xtr_s, ytr_s, str_s = make_windows_split(tr_start, tr_end)
    Xva_s, yva_s, sva_s = make_windows_split(va_start, va_end)
    Xte_s, yte_s, ste_s = make_windows_split(te_start, te_end)
    print(f"windows: train={len(Xtr_s)}, val={len(Xva_s)}, test={len(Xte_s)}", flush=True)

    Xtr_st, ytr_st = torch.tensor(Xtr_s, dtype=torch.float32), torch.tensor(ytr_s, dtype=torch.float32)
    Xva_st, yva_st = torch.tensor(Xva_s, dtype=torch.float32), torch.tensor(yva_s, dtype=torch.float32)
    Xte_st, yte_st = torch.tensor(Xte_s, dtype=torch.float32), torch.tensor(yte_s, dtype=torch.float32)
    str_st, sva_st, ste_st = map(lambda a: torch.tensor(a, dtype=torch.long), (str_s, sva_s, ste_s))

    def run_epoch_split(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
        n = X.shape[0]
        idx = torch.randperm(n) if train else torch.arange(n)
        model.train(train)
        total_loss, total_n = 0.0, 0
        eff_batch = micro_batch * accum_steps
        for start in range(0, n, eff_batch):
            if train:
                optimizer.zero_grad()
            batch_idx = idx[start:start + eff_batch]
            for micro_start in range(0, len(batch_idx), micro_batch):
                mb_idx = batch_idx[micro_start:micro_start + micro_batch]
                if len(mb_idx) == 0:
                    continue
                xb = add_static(X[mb_idx]).to(DEVICE)
                yb = y[mb_idx].to(DEVICE)
                ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
                ei = edge_index.to(DEVICE) if use_graph else None
                with torch.set_grad_enabled(train):
                    pred = model(xb, ei, ew_seq)
                    loss = ((pred - yb) ** 2).mean()
                if train:
                    (loss * len(mb_idx) / len(batch_idx)).backward()
                total_loss += loss.item() * len(mb_idx)
                total_n += len(mb_idx)
            if train:
                optimizer.step()
        return total_loss / total_n

    def train_split(name, model, use_graph, max_epochs=MAX_EPOCHS):
        model = model.to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
        best_val, best_epoch = float("inf"), -1
        ckpt_path = f"{CKPT_DIR}/{name}.pt"
        for epoch in range(1, max_epochs + 1):
            run_epoch_split(model, Xtr_st, ytr_st, str_st, use_graph, opt, train=True)
            val_loss = run_epoch_split(model, Xva_st, yva_st, sva_st, use_graph, opt, train=False)
            if val_loss < best_val:
                best_val, best_epoch = val_loss, epoch
                torch.save(model.state_dict(), ckpt_path)
        model.load_state_dict(torch.load(ckpt_path))
        test_loss = run_epoch_split(model, Xte_st, yte_st, ste_st, use_graph, None, train=False)
        print(f"  [{name}] best_epoch={best_epoch} val={best_val:.4f} test={test_loss:.4f}", flush=True)
        return best_epoch, best_val, test_loss

    for seed in SEEDS_SPLIT:
        torch.manual_seed(seed); np.random.seed(seed)
        be_w, bv_w, te_w = train_split(f"wind_{split_name}_seed{seed}", WindTemporalGCN(n_feats), use_graph=True)
        torch.manual_seed(seed); np.random.seed(seed)
        be_n, bv_n, te_n = train_split(f"nograph_{split_name}_seed{seed}", TemporalOnlyGRU(n_feats), use_graph=False)
        split_results.append({"split": split_name, "seed": seed, "model": "wind", "test_mse": te_w})
        split_results.append({"split": split_name, "seed": seed, "model": "no_graph", "test_mse": te_n})

split_df = pd.DataFrame(split_results)
print("\n" + "="*60)
print(split_df.pivot_table(index=["split","seed"], columns="model", values="test_mse"))
print("\n=== per-split summary ===")
summary = split_df.groupby(["split","model"])["test_mse"].agg(["mean","std"])
print(summary)

print("\n=== per-split paired significance ===")
for split_name in SPLITS:
    sub = split_df[split_df["split"] == split_name].pivot(index="seed", columns="model", values="test_mse")
    t, p = stats.ttest_rel(sub["no_graph"], sub["wind"])
    diff = (sub["no_graph"] - sub["wind"]).mean()
    print(f"{split_name}: mean improvement={diff:.5f} ({100*diff/sub['no_graph'].mean():.2f}%)  t={t:.3f} p={p:.4f}")


NameError: name 'dt_index' is not defined